# DIMER Workshop: Comparing Learned Image-Matching Models

**Profile:** `TASK-INFERENCE` · **Mode:** `WORKSHOP` · **Notebook Specification:** 2.1 · **Standalone:** yes

This workshop compares two live DIMER image-matching systems on the **same photographs, same homographies, and same geometric metrics**:

- **LightGlue + ALIKED** — sparse learned matching over ALIKED keypoints
- **XoFTR** — detector-free / semi-dense learned correspondence

No model is fine-tuned here. The goal is to study where the frozen matchers succeed or fail under controlled geometric and photometric changes.

Three transformation tiers are evaluated:

1. **easy** — up to ±10° rotation;
2. **standard-hard** — up to ±35° rotation, matching the XoFTR carrier's hard tier; and
3. **extreme-rotation** — up to ±150° rotation, matching the LightGlue carrier's stress tier.

The two repository packages are embedded at authoring time from immutable commits and materialized locally. The notebook never clones or downloads DIMER repository source at runtime.


## Learning goals

You will learn to:

1. distinguish sparse keypoint matching from detector-free/semi-dense correspondence;
2. evaluate matches against a known homography rather than judging only a visualization;
3. interpret precision at 1/3/5 px, match/inlier counts, median reprojection error, and homography recovery accuracy;
4. compare model behavior as rotation and appearance changes become more severe;
5. use identity and local patch-neighbour baselines as sanity floors; and
6. separate *failure-mode evidence on this controlled sample* from claims about general matcher superiority.


In [ ]:
# @title 0. Workshop controls and runtime
USE_BYOD = False  # @param {type:"boolean"}
BYOD_ZIP_PATH = ""  # @param {type:"string"}
MAX_EVAL_PAIRS = 96  # @param {type:"integer"}
PLOT_METRIC = "precision_3px"  # @param ["precision_1px","precision_3px","precision_5px","homography_acc_3px","homography_acc_5px"]

import importlib.metadata
import json
import os
import platform
import stat
import subprocess
import sys
import time
import venv
import zipfile
from pathlib import Path, PurePosixPath

from packaging.version import Version

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"This reviewed workshop supports Python 3.12; found {platform.python_version()}.")

def require_version(name, minimum, maximum):
    value=Version(importlib.metadata.version(name))
    if not (Version(minimum) <= value < Version(maximum)):
        raise RuntimeError(f"{name} {value} is outside [{minimum}, {maximum}).")
    return str(value)

CONTROL_VERSIONS={
    "numpy":require_version("numpy","1.26","3.0"),
    "pandas":require_version("pandas","2.2","4.0"),
    "matplotlib":require_version("matplotlib","3.8","4.0"),
    "packaging":require_version("packaging","24.0","27.0"),
}
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if MAX_EVAL_PAIRS < 12 or MAX_EVAL_PAIRS > 96:
    raise ValueError("MAX_EVAL_PAIRS must be in 12..96.")

ROOT=Path.cwd()
RUNTIME_ROOT=ROOT/"workshop_runtime"
EMBEDDED_ROOT=RUNTIME_ROOT/"embedded_src"
OUTPUT_ROOT=ROOT/"outputs"
for path in (RUNTIME_ROOT,EMBEDDED_ROOT,OUTPUT_ROOT):
    path.mkdir(parents=True,exist_ok=True)

def gpu_info():
    try:
        return subprocess.check_output(
            ["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader,nounits"],
            text=True,stderr=subprocess.STDOUT
        ).strip()
    except Exception:
        return ""
GPU_INFO=gpu_info()
if not GPU_INFO:
    raise RuntimeError("Select a CUDA GPU runtime before Run all; the two live matchers are GPU-oriented.")
print({"python":platform.python_version(),"control_versions":CONTROL_VERSIONS,"gpu":GPU_INFO})


## 1. Embedded implementation provenance

The notebook carries the exact package source used to construct this comparative carrier:

- LightGlue package source: `kurtvalcorza/lightglue-matching-pipeline@86996ba0671c8cdcf1c922d95d9f53ca156f3130`
- XoFTR package source: `kurtvalcorza/xoftr-image-matching-pipeline@bb795eb064bf12742beb2e842e2a1aa7dc4093d8`

Model identities remain the live DIMER identities:

- LightGlue matcher: `cvg/LightGlue` release `v0.1_arxiv`, paired with ALIKED-N16 at `Shiaoming/ALIKED@683d7c65197395c0b3f01ebe76e1084a27e73a65`; source pickles are statically audited and deterministically converted to SafeTensors before model loading.
- XoFTR: `vismatch/xoftr@d8ee7d89be3c9e5c157db3886db1c0f0e038b321`, `xoftr_640.safetensors`.

The manifests below are also embedded. Runtime network access is limited to the model weight origins and the pinned public sample photographs.


In [ ]:
# @title 1.1 Materialize embedded packages and weight manifests
EMBEDDED_SOURCES = {"lightglue_pipeline/__init__.py":"\"\"\"DIMER-oriented LightGlue + ALIKED image-matching pipeline: two pinned GitHub-hosted checkpoints\naudited and converted once to safetensors, vendored networks, an inference contract with an\nexact-reference evaluation, a bounded matcher adaptation and a safetensors adapter.\"\"\"\n\nfrom .config import (\n    ALLOWED_CHECKPOINT_FILES,\n    CKPT_ALLOWED_GLOBALS,\n    CONVERTED_FILENAMES,\n    DEFAULT_MODEL_KEY,\n    DEPTH_CONFIDENCE,\n    DETECTION_THRESHOLD,\n    DIVISIBLE_BY,\n    EXTRACTOR_COMMIT,\n    EXTRACTOR_FILENAME,\n    EXTRACTOR_LICENSE,\n    EXTRACTOR_PARAMETER_COUNT,\n    EXTRACTOR_PICKLE_AUDIT_SHA256,\n    EXTRACTOR_REPOSITORY,\n    EXTRACTOR_SHA256,\n    EXTRACTOR_SIZE_BYTES,\n    EXTRACTOR_SOURCE_FILENAME,\n    EXTRACTOR_SOURCE_SHA256,\n    EXTRACTOR_SOURCE_SIZE_BYTES,\n    EXTRACTOR_SOURCE_URL,\n    EXTRACTOR_STATE_TENSORS,\n    FILTER_THRESHOLD,\n    MATCHER_FILENAME,\n    MATCHER_PARAMETER_COUNT,\n    MATCHER_PICKLE_AUDIT_SHA256,\n    MATCHER_SHA256,\n    MATCHER_SIZE_BYTES,\n    MATCHER_SOURCE_FILENAME,\n    MATCHER_SOURCE_SHA256,\n    MATCHER_SOURCE_SIZE_BYTES,\n    MATCHER_SOURCE_URL,\n    MATCHER_STATE_TENSORS,\n    MAX_KEYPOINTS,\n    MAX_SIDE,\n    MIN_SIDE,\n    MODEL_FILENAME,\n    MODEL_ID,\n    MODEL_LICENSE,\n    MODEL_REVISION,\n    MODEL_REVISION_KIND,\n    MODEL_SHA256,\n    MODEL_SIZE_BYTES,\n    NMS_RADIUS,\n    SOURCE_FILENAMES,\n    UNSAFE_WEIGHT_EXTENSIONS,\n    WIDTH_CONFIDENCE,\n)\nfrom .metrics import (\n    METRIC_DEFINITIONS,\n    corner_error,\n    dlt_homography,\n    identity_baseline,\n    matching_metrics,\n    mutual_nn_matches,\n    pair_metrics,\n    patch_neighbour_baseline,\n    ransac_homography,\n    reprojection_errors,\n    warp_points,\n)\nfrom .model import (\n    DEFAULT_WEIGHTS_DIR,\n    MANIFEST_NAME,\n    audit_pickle,\n    build_models,\n    convert_sources,\n    load_components,\n    stage_missing_files,\n    verify_checkpoint,\n    verify_snapshot,\n)\nfrom .modeling import ALIKED, ALIKED_REPOSITORY, UPSTREAM_COMMIT, UPSTREAM_REPOSITORY, LightGlue\nfrom .pipeline import (\n    ARTIFACT_FORMAT,\n    DEFAULT_TRAINABLE_LAYERS,\n    INPUT_SCHEMA,\n    MATCHER_LAYERS,\n    MAX_EVAL_RECORDS,\n    MIN_SCORED_RECORDS,\n    NEGATIVE_PX,\n    PARAMETER_COUNT,\n    POSITIVE_PX,\n    LightGluePipeline,\n    evaluation_report,\n    load_pipeline,\n    validate_inputs,\n)\nfrom .provenance import build_provenance, write_provenance\nfrom .samples import (\n    CORPUS_BASE_URL,\n    CORPUS_BYTES,\n    CORPUS_LICENSE,\n    CORPUS_NAME,\n    CORPUS_RELEASE,\n    MAX_IMAGE_SIDE,\n    MAX_RECORDS,\n    MIN_RECORDS,\n    SAMPLE_RECORDS,\n    SAMPLE_SEED,\n    SAMPLE_SPLIT,\n    SPECIES,\n    TIER_PARAMS,\n    TIERS,\n    WORKING_LONG_SIDE,\n    build_sample_dataset,\n    check_homography,\n    check_split_disjoint,\n    dataset_digest,\n    fetch_corpus,\n    fetch_sample_dataset,\n    image_digest,\n    load_byod_dataset,\n    make_pair,\n    make_pairs,\n    observer_overlap,\n    prepare_image,\n    read_corpus,\n    sample_homography,\n    split_dataset,\n    validate_dataset,\n    warp_image,\n    working_size,\n    write_dataset_csv,\n)\n\n__all__ = [\n    \"ALIKED\",\n    \"ALIKED_REPOSITORY\",\n    \"ALLOWED_CHECKPOINT_FILES\",\n    \"ARTIFACT_FORMAT\",\n    \"CKPT_ALLOWED_GLOBALS\",\n    \"CONVERTED_FILENAMES\",\n    \"CORPUS_BASE_URL\",\n    \"CORPUS_BYTES\",\n    \"CORPUS_LICENSE\",\n    \"CORPUS_NAME\",\n    \"CORPUS_RELEASE\",\n    \"DEFAULT_MODEL_KEY\",\n    \"DEFAULT_TRAINABLE_LAYERS\",\n    \"DEFAULT_WEIGHTS_DIR\",\n    \"DEPTH_CONFIDENCE\",\n    \"DETECTION_THRESHOLD\",\n    \"DIVISIBLE_BY\",\n    \"EXTRACTOR_COMMIT\",\n    \"EXTRACTOR_FILENAME\",\n    \"EXTRACTOR_LICENSE\",\n    \"EXTRACTOR_PARAMETER_COUNT\",\n    \"EXTRACTOR_PICKLE_AUDIT_SHA256\",\n    \"EXTRACTOR_REPOSITORY\",\n    \"EXTRACTOR_SHA256\",\n    \"EXTRACTOR_SIZE_BYTES\",\n    \"EXTRACTOR_SOURCE_FILENAME\",\n    \"EXTRACTOR_SOURCE_SHA256\",\n    \"EXTRACTOR_SOURCE_SIZE_BYTES\",\n    \"EXTRACTOR_SOURCE_URL\",\n    \"EXTRACTOR_STATE_TENSORS\",\n    \"FILTER_THRESHOLD\",\n    \"INPUT_SCHEMA\",\n    \"LightGlue\",\n    \"LightGluePipeline\",\n    \"MANIFEST_NAME\",\n    \"MATCHER_FILENAME\",\n    \"MATCHER_LAYERS\",\n    \"MATCHER_PARAMETER_COUNT\",\n    \"MATCHER_PICKLE_AUDIT_SHA256\",\n    \"MATCHER_SHA256\",\n    \"MATCHER_SIZE_BYTES\",\n    \"MATCHER_SOURCE_FILENAME\",\n    \"MATCHER_SOURCE_SHA256\",\n    \"MATCHER_SOURCE_SIZE_BYTES\",\n    \"MATCHER_SOURCE_URL\",\n    \"MATCHER_STATE_TENSORS\",\n    \"MAX_EVAL_RECORDS\",\n    \"MAX_IMAGE_SIDE\",\n    \"MAX_KEYPOINTS\",\n    \"MAX_RECORDS\",\n    \"MAX_SIDE\",\n    \"METRIC_DEFINITIONS\",\n    \"MIN_RECORDS\",\n    \"MIN_SCORED_RECORDS\",\n    \"MIN_SIDE\",\n    \"MODEL_FILENAME\",\n    \"MODEL_ID\",\n    \"MODEL_LICENSE\",\n    \"MODEL_REVISION\",\n    \"MODEL_REVISION_KIND\",\n    \"MODEL_SHA256\",\n    \"MODEL_SIZE_BYTES\",\n    \"NEGATIVE_PX\",\n    \"NMS_RADIUS\",\n    \"PARAMETER_COUNT\",\n    \"POSITIVE_PX\",\n    \"SAMPLE_RECORDS\",\n    \"SAMPLE_SEED\",\n    \"SAMPLE_SPLIT\",\n    \"SOURCE_FILENAMES\",\n    \"SPECIES\",\n    \"TIERS\",\n    \"TIER_PARAMS\",\n    \"UNSAFE_WEIGHT_EXTENSIONS\",\n    \"UPSTREAM_COMMIT\",\n    \"UPSTREAM_REPOSITORY\",\n    \"WIDTH_CONFIDENCE\",\n    \"WORKING_LONG_SIDE\",\n    \"audit_pickle\",\n    \"build_models\",\n    \"build_provenance\",\n    \"build_sample_dataset\",\n    \"check_homography\",\n    \"check_split_disjoint\",\n    \"convert_sources\",\n    \"corner_error\",\n    \"dataset_digest\",\n    \"dlt_homography\",\n    \"evaluation_report\",\n    \"fetch_corpus\",\n    \"fetch_sample_dataset\",\n    \"identity_baseline\",\n    \"image_digest\",\n    \"load_byod_dataset\",\n    \"load_components\",\n    \"load_pipeline\",\n    \"make_pair\",\n    \"make_pairs\",\n    \"matching_metrics\",\n    \"mutual_nn_matches\",\n    \"observer_overlap\",\n    \"pair_metrics\",\n    \"patch_neighbour_baseline\",\n    \"prepare_image\",\n    \"ransac_homography\",\n    \"read_corpus\",\n    \"reprojection_errors\",\n    \"sample_homography\",\n    \"split_dataset\",\n    \"stage_missing_files\",\n    \"validate_dataset\",\n    \"validate_inputs\",\n    \"verify_checkpoint\",\n    \"verify_snapshot\",\n    \"warp_image\",\n    \"warp_points\",\n    \"working_size\",\n    \"write_dataset_csv\",\n    \"write_provenance\",\n]\n","lightglue_pipeline/config.py":"from __future__ import annotations\n\n# LightGlue (Lindenberger, Sarlin and Pollefeys, ICCV 2023) with ALIKED local features. LightGlue\n# publishes no Hugging Face repository: the matcher checkpoint is an asset of the immutable GitHub\n# release tag below, and the ALIKED extractor checkpoint is a file of the ALIKED repository at a\n# pinned commit. Both are plain PyTorch state-dict pickles, audited statically and converted once to\n# safetensors (the only files the network is ever loaded from).\nMODEL_ID = \"cvg/LightGlue\"\n# GitHub release tag (2023-06-26); its assets are immutable.\nMODEL_REVISION = \"v0.1_arxiv\"\nMODEL_REVISION_KIND = \"github-release-tag\"\n# LightGlue code and weights.\nMODEL_LICENSE = \"Apache-2.0\"\nEXTRACTOR_LICENSE = \"BSD-3-Clause\"  # ALIKED code (the port carried in modeling.py) and weights\n\n# --- the matcher: LightGlue trained on ALIKED features -------------------------------------------\nMATCHER_SOURCE_FILENAME = \"aliked_lightglue.pth\"\nMATCHER_SOURCE_URL = (\n    f\"https://github.com/{MODEL_ID}/releases/download/{MODEL_REVISION}/{MATCHER_SOURCE_FILENAME}\"\n)\nMATCHER_SOURCE_SHA256 = \"d975e965b105311a6143194852297dff4f02aea5cc2e10cecfed966ca0e22503\"\nMATCHER_SOURCE_SIZE_BYTES = 47_632_827\nMATCHER_PICKLE_AUDIT_SHA256 = \"e7b998d087a5dcadd37713daf30b63cc571160c3180ebc138500ab662197e932\"\nMATCHER_FILENAME = \"aliked_lightglue.safetensors\"  # the deterministic conversion of the source\nMATCHER_SHA256 = \"9c630a386c74c534428370ce46253e1d0968655db180f97074cb6ad797bd2bc6\"\nMATCHER_SIZE_BYTES = 47_564_948\nMATCHER_STATE_TENSORS = 253  # all float32 parameters; the confidence-threshold buffer is computed\nMATCHER_PARAMETER_COUNT = 11_884_625\n\n# --- the extractor: ALIKED-N(16) ----------------------------------------------------------------\nEXTRACTOR_REPOSITORY = \"Shiaoming/ALIKED\"\nEXTRACTOR_COMMIT = \"683d7c65197395c0b3f01ebe76e1084a27e73a65\"\nEXTRACTOR_SOURCE_FILENAME = \"aliked-n16.pth\"\nEXTRACTOR_SOURCE_URL = (\n    f\"https://raw.githubusercontent.com/{EXTRACTOR_REPOSITORY}/{EXTRACTOR_COMMIT}\"\n    f\"/models/{EXTRACTOR_SOURCE_FILENAME}\"\n)\nEXTRACTOR_SOURCE_SHA256 = \"5be8704840ed662d9d8c561bf7279c222092674e7eb05fd0feab94899e9d82f2\"\nEXTRACTOR_SOURCE_SIZE_BYTES = 2_738_091\nEXTRACTOR_PICKLE_AUDIT_SHA256 = \"5b9f0ba08490293d6c17b9cef219991e1a6edda31609429679f8dca1af5a7b10\"\nEXTRACTOR_FILENAME = \"aliked-n16.safetensors\"\nEXTRACTOR_SHA256 = \"3c8ca40c0c985cd4d641e96e4b408b14d067b5b3521ac17b36590447d49d115a\"\nEXTRACTOR_SIZE_BYTES = 2_719_928\nEXTRACTOR_STATE_TENSORS = 76  # 68 float32 parameter / running-statistic tensors + 8 int64 counters\nEXTRACTOR_PARAMETER_COUNT = 677_356\n\n# The globals a checkpoint pickle may import (the fleet's four); anything else fails the audit.\nCKPT_ALLOWED_GLOBALS = frozenset(\n    {\n        \"collections.OrderedDict\",\n        \"torch.FloatStorage\",\n        \"torch.LongStorage\",\n        \"torch._utils._rebuild_tensor_v2\",\n    }\n)\n\n# The served (primary) weight file for the fleet's single-file conventions is the matcher.\nMODEL_FILENAME = MATCHER_FILENAME\nMODEL_SHA256 = MATCHER_SHA256\nMODEL_SIZE_BYTES = MATCHER_SIZE_BYTES\nSOURCE_FILENAMES = (MATCHER_SOURCE_FILENAME, EXTRACTOR_SOURCE_FILENAME)\nCONVERTED_FILENAMES = (MATCHER_FILENAME, EXTRACTOR_FILENAME)\n\nDEFAULT_MODEL_KEY = \"lightglue-aliked\"\nUNSAFE_WEIGHT_EXTENSIONS = (\n    \".bin\",\n    \".pt\",\n    \".pth\",\n    \".ckpt\",\n    \".pkl\",\n    \".pickle\",\n    \".h5\",\n    \".msgpack\",\n)\nALLOWED_CHECKPOINT_FILES = CONVERTED_FILENAMES\n\n# Inference contract.\nMAX_KEYPOINTS = 2048  # ALIKED keypoints per image (upstream demo default)\nDETECTION_THRESHOLD = 0.2  # ALIKED keypoint score threshold (upstream default)\nNMS_RADIUS = 2  # ALIKED non-maximum suppression radius, px (upstream default)\nFILTER_THRESHOLD = 0.1  # LightGlue match threshold on the assignment score (upstream default)\nDEPTH_CONFIDENCE = -1.0  # upstream's adaptive early exit (0.95) is off: all nine layers always run\nWIDTH_CONFIDENCE = -1.0  # upstream's adaptive point pruning (0.99) is off: no keypoint is dropped\nDIVISIBLE_BY = 8  # sample pairs are built with sides that are multiples of this (ALIKED pads to 32)\nMIN_SIDE = 64\nMAX_SIDE = 1024\n","lightglue_pipeline/metrics.py":"\"\"\"Homography-supervised matching metrics and two non-neural baselines, in numpy.\n\nA record pairs an image with a warped copy of itself under a known 3 × 3 homography `H` (image0 → image1),\nso every match has an exact reference: the reprojection error of `(x0, y0)` mapped by `H` against `(x1, y1)`.\nFor a set of records the pipeline reports:\n\n- **precision at 3 px** (the fraction of returned matches with reprojection error under 3 px — the\n  matching-precision reading), also at 1 px and 5 px;\n- **matches per pair** and **inliers per pair** at 3 px (how much a downstream solver has to work with);\n- **median reprojection error** of the inliers (sub-pixel accuracy);\n- **homography accuracy at 3 px / 5 px**: the fraction of pairs whose homography, estimated from the\n  matches by a normalised DLT inside a plain RANSAC loop, moves the four image corners by less than the\n  threshold on average against the reference `H` (the usual HPatches-style reading; pairs with fewer than\n  four inliers count as failures).\n\nThree baselines a matcher must beat: the **identity guess** (every grid point maps to itself — correct only\nwhere the warp is small), a **patch nearest neighbour** (for each grid point of image0 the best\nnormalised-cross-correlation 15 × 15 patch of image1 within a search window — a matcher that knows the\nimages through raw intensities) and, in the pipeline, the **descriptor mutual nearest neighbour** (the same\nALIKED keypoints and descriptors the model matches, paired by mutual nearest neighbour in descriptor space\nwith `mutual_nn_matches` below — what LightGlue is replacing). All return the same match structure the model\ndoes and are scored by the same code.\n\"\"\"\n# ruff: noqa: E501  -- fleet metrics module written at the 110-column fleet width; this repo lints at 100\n\nfrom __future__ import annotations\n\nimport math\nfrom collections.abc import Mapping, Sequence\nfrom typing import Any\n\nimport numpy as np\nfrom PIL import Image\n\nMETRIC_DEFINITIONS = {\n    \"precision_3px\": \"fraction of returned matches whose reprojection error under the reference homography is below 3 px, averaged over pairs (a pair with no matches scores 0); in 0..1\",\n    \"precision_1px\": \"the same at 1 px\",\n    \"precision_5px\": \"the same at 5 px\",\n    \"matches_per_pair\": \"mean number of returned matches per pair\",\n    \"inliers_per_pair\": \"mean number of returned matches under 3 px per pair\",\n    \"median_error_px\": \"median reprojection error of the inliers under 3 px, pooled over pairs; px\",\n    \"homography_acc_3px\": \"fraction of pairs whose RANSAC-DLT homography from the matches moves the four corners by less than 3 px on average against the reference; in 0..1\",\n    \"homography_acc_5px\": \"the same at 5 px\",\n}\nTHRESHOLDS = (1.0, 3.0, 5.0)\nINLIER_PX = 3.0\n\n\ndef warp_points(points: np.ndarray, homography: np.ndarray) -> np.ndarray:\n    \"\"\"Apply a 3 × 3 homography to (N, 2) pixel coordinates.\"\"\"\n    pts = np.asarray(points, dtype=np.float64)\n    if pts.ndim != 2 or pts.shape[1] != 2:\n        raise ValueError(\"points must be an (N, 2) array\")\n    hom = np.concatenate([pts, np.ones((len(pts), 1))], axis=1) @ np.asarray(homography, dtype=np.float64).T\n    with np.errstate(divide=\"ignore\", invalid=\"ignore\"):  # points at infinity under a degenerate candidate\n        return hom[:, :2] / hom[:, 2:3]\n\n\ndef reprojection_errors(kpts0: np.ndarray, kpts1: np.ndarray, homography: np.ndarray) -> np.ndarray:\n    \"\"\"Per-match distance between `H · kpts0` and `kpts1`, in px.\"\"\"\n    kpts0 = np.asarray(kpts0, dtype=np.float64).reshape(-1, 2)\n    kpts1 = np.asarray(kpts1, dtype=np.float64).reshape(-1, 2)\n    if len(kpts0) != len(kpts1):\n        raise ValueError(\"kpts0 and kpts1 must have the same length\")\n    if len(kpts0) == 0:\n        return np.zeros((0,), dtype=np.float64)\n    errors = np.linalg.norm(warp_points(kpts0, homography) - kpts1, axis=1)\n    return np.where(np.isfinite(errors), errors, np.inf)\n\n\ndef _normalise(points: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n    mean = points.mean(axis=0)\n    scale = math.sqrt(2.0) / max(float(np.sqrt(((points - mean) ** 2).sum(axis=1)).mean()), 1e-9)\n    transform = np.array([[scale, 0.0, -scale * mean[0]], [0.0, scale, -scale * mean[1]], [0.0, 0.0, 1.0]])\n    hom = np.concatenate([points, np.ones((len(points), 1))], axis=1) @ transform.T\n    return hom[:, :2], transform\n\n\ndef dlt_homography(kpts0: np.ndarray, kpts1: np.ndarray) -> np.ndarray | None:\n    \"\"\"Normalised direct linear transform from at least four correspondences; None when degenerate.\"\"\"\n    kpts0 = np.asarray(kpts0, dtype=np.float64)\n    kpts1 = np.asarray(kpts1, dtype=np.float64)\n    if len(kpts0) < 4:\n        return None\n    p0, t0 = _normalise(kpts0)\n    p1, t1 = _normalise(kpts1)\n    rows = []\n    for (x, y), (u, v) in zip(p0, p1, strict=True):\n        rows.append([-x, -y, -1.0, 0.0, 0.0, 0.0, u * x, u * y, u])\n        rows.append([0.0, 0.0, 0.0, -x, -y, -1.0, v * x, v * y, v])\n    a = np.asarray(rows)\n    try:\n        _u, sigma, vt = np.linalg.svd(a)\n    except np.linalg.LinAlgError:\n        return None\n    if sigma[-2] < 1e-12:  # rank-deficient: collinear points\n        return None\n    h_norm = vt[-1].reshape(3, 3)\n    homography = np.linalg.inv(t1) @ h_norm @ t0\n    if abs(homography[2, 2]) < 1e-12:\n        return None\n    return homography / homography[2, 2]\n\n\ndef ransac_homography(\n    kpts0: np.ndarray,\n    kpts1: np.ndarray,\n    *,\n    threshold: float = INLIER_PX,\n    iterations: int = 500,\n    seed: int = 0,\n) -> tuple[np.ndarray | None, np.ndarray]:\n    \"\"\"A plain RANSAC over four-point DLT samples, refit on the consensus set. Returns (H or None, inlier mask).\"\"\"\n    kpts0 = np.asarray(kpts0, dtype=np.float64).reshape(-1, 2)\n    kpts1 = np.asarray(kpts1, dtype=np.float64).reshape(-1, 2)\n    n = len(kpts0)\n    if n < 4:\n        return None, np.zeros((n,), dtype=bool)\n    rng = np.random.default_rng(seed)\n    best_mask = np.zeros((n,), dtype=bool)\n    for _ in range(iterations):\n        sample = rng.choice(n, size=4, replace=False)\n        candidate = dlt_homography(kpts0[sample], kpts1[sample])\n        if candidate is None:\n            continue\n        mask = reprojection_errors(kpts0, kpts1, candidate) < threshold\n        if mask.sum() > best_mask.sum():\n            best_mask = mask\n            if best_mask.sum() == n:\n                break\n    if best_mask.sum() < 4:\n        return None, best_mask\n    refit = dlt_homography(kpts0[best_mask], kpts1[best_mask])\n    if refit is None:\n        return None, best_mask\n    return refit, reprojection_errors(kpts0, kpts1, refit) < threshold\n\n\ndef corner_error(estimated: np.ndarray, reference: np.ndarray, size: tuple[int, int]) -> float:\n    \"\"\"Mean displacement of the four image corners between two homographies, in px.\"\"\"\n    width, height = size\n    corners = np.array([[0.0, 0.0], [width - 1.0, 0.0], [width - 1.0, height - 1.0], [0.0, height - 1.0]])\n    return float(np.linalg.norm(warp_points(corners, estimated) - warp_points(corners, reference), axis=1).mean())\n\n\ndef pair_metrics(match: Mapping[str, Any], homography: np.ndarray, size: tuple[int, int]) -> dict[str, Any]:\n    \"\"\"Per-pair scores for one match result `{kpts0, kpts1}` against the reference homography.\"\"\"\n    kpts0 = np.asarray(match[\"kpts0\"], dtype=np.float64).reshape(-1, 2)\n    kpts1 = np.asarray(match[\"kpts1\"], dtype=np.float64).reshape(-1, 2)\n    errors = reprojection_errors(kpts0, kpts1, homography)\n    out: dict[str, Any] = {\"n_matches\": int(len(errors))}\n    for t in THRESHOLDS:\n        out[f\"precision_{int(t)}px\"] = float((errors < t).mean()) if len(errors) else 0.0\n    inliers = errors[errors < INLIER_PX]\n    out[\"n_inliers\"] = int(len(inliers))\n    out[\"inlier_errors\"] = inliers.tolist()\n    estimated, _mask = ransac_homography(kpts0, kpts1)\n    out[\"corner_error_px\"] = corner_error(estimated, homography, size) if estimated is not None else math.inf\n    return out\n\n\ndef matching_metrics(per_pair: Sequence[Mapping[str, Any]]) -> dict[str, Any]:\n    \"\"\"Aggregate `pair_metrics` rows over a set of pairs.\"\"\"\n    if not per_pair:\n        raise ValueError(\"at least one pair is required\")\n    pooled = [e for row in per_pair for e in row[\"inlier_errors\"]]\n    out: dict[str, Any] = {\n        \"n\": len(per_pair),\n        \"matches_per_pair\": float(np.mean([row[\"n_matches\"] for row in per_pair])),\n        \"inliers_per_pair\": float(np.mean([row[\"n_inliers\"] for row in per_pair])),\n        \"median_error_px\": float(np.median(pooled)) if pooled else math.inf,\n        \"homography_acc_3px\": float(np.mean([row[\"corner_error_px\"] < 3.0 for row in per_pair])),\n        \"homography_acc_5px\": float(np.mean([row[\"corner_error_px\"] < 5.0 for row in per_pair])),\n        \"definitions\": METRIC_DEFINITIONS,\n    }\n    for t in THRESHOLDS:\n        key = f\"precision_{int(t)}px\"\n        out[key] = float(np.mean([row[key] for row in per_pair]))\n    return out\n\n\n# --------------------------------------------------------------------------------------------------\n# non-neural baselines\n# --------------------------------------------------------------------------------------------------\n\n\ndef grid_points(size: tuple[int, int], step: int = 32, margin: int = 16) -> np.ndarray:\n    width, height = size\n    xs = np.arange(margin, width - margin, step, dtype=np.float64)\n    ys = np.arange(margin, height - margin, step, dtype=np.float64)\n    gx, gy = np.meshgrid(xs, ys)\n    return np.stack([gx.ravel(), gy.ravel()], axis=1)\n\n\ndef identity_baseline(image0: Image.Image, image1: Image.Image, *, step: int = 32) -> dict[str, Any]:\n    \"\"\"Every grid point of image0 is matched to the same coordinates in image1 (no motion assumed).\"\"\"\n    pts = grid_points(image0.size, step=step)\n    return {\"kpts0\": pts, \"kpts1\": pts.copy(), \"confidence\": np.ones(len(pts)), \"baseline\": \"identity guess\"}\n\n\ndef _gray(image: Image.Image) -> np.ndarray:\n    return np.asarray(image.convert(\"L\"), dtype=np.float64)\n\n\ndef _ncc(patch: np.ndarray, window: np.ndarray) -> np.ndarray:\n    \"\"\"Normalised cross-correlation of a (p, p) patch over every (p, p) position of a (h, w) window.\"\"\"\n    p = patch.shape[0]\n    h, w = window.shape\n    if h < p or w < p:\n        return np.zeros((0, 0))\n    strides = np.lib.stride_tricks.sliding_window_view(window, (p, p))  # (h-p+1, w-p+1, p, p)\n    tiles = strides.reshape(strides.shape[0], strides.shape[1], -1)\n    tiles = tiles - tiles.mean(axis=2, keepdims=True)\n    flat = (patch - patch.mean()).ravel()\n    denom = np.sqrt((tiles**2).sum(axis=2) * (flat**2).sum()) + 1e-9\n    return (tiles @ flat) / denom\n\n\ndef patch_neighbour_baseline(\n    image0: Image.Image,\n    image1: Image.Image,\n    *,\n    step: int = 32,\n    patch: int = 15,\n    search: int = 48,\n) -> dict[str, Any]:\n    \"\"\"For each grid point of image0, the position in image1 (within ±`search` px) whose `patch` × `patch`\n    neighbourhood has the highest normalised cross-correlation with the point's own patch.\"\"\"\n    g0, g1 = _gray(image0), _gray(image1)\n    half = patch // 2\n    pts0 = grid_points(image0.size, step=step, margin=max(16, half + 1))\n    kpts0, kpts1, conf = [], [], []\n    for x, y in pts0:\n        xi, yi = int(round(x)), int(round(y))\n        tile = g0[yi - half : yi + half + 1, xi - half : xi + half + 1]\n        y0, y1 = max(0, yi - search - half), min(g1.shape[0], yi + search + half + 1)\n        x0, x1 = max(0, xi - search - half), min(g1.shape[1], xi + search + half + 1)\n        scores = _ncc(tile, g1[y0:y1, x0:x1])\n        if scores.size == 0:\n            continue\n        best = np.unravel_index(int(np.argmax(scores)), scores.shape)\n        kpts0.append([x, y])\n        kpts1.append([x0 + best[1] + half, y0 + best[0] + half])\n        conf.append(float(scores[best]))\n    return {\n        \"kpts0\": np.asarray(kpts0, dtype=np.float64).reshape(-1, 2),\n        \"kpts1\": np.asarray(kpts1, dtype=np.float64).reshape(-1, 2),\n        \"confidence\": np.asarray(conf, dtype=np.float64),\n        \"baseline\": f\"patch nearest neighbour ({patch}x{patch} NCC, ±{search} px search)\",\n    }\n\n\ndef mutual_nn_matches(\n    desc0: np.ndarray, desc1: np.ndarray, *, min_similarity: float = 0.0\n) -> tuple[np.ndarray, np.ndarray]:\n    \"\"\"Mutual nearest neighbours of two L2-normalised descriptor sets by cosine similarity: index pairs\n    (M, 2) and their similarities (M,). The classical detector-and-describe matcher without any learned\n    context — the reference for what a learned matcher adds on the same keypoints.\"\"\"\n    d0 = np.asarray(desc0, dtype=np.float64)\n    d1 = np.asarray(desc1, dtype=np.float64)\n    if d0.ndim != 2 or d1.ndim != 2 or d0.shape[1] != d1.shape[1]:\n        raise ValueError(\"descriptors must be (N, D) arrays with a common D\")\n    if len(d0) == 0 or len(d1) == 0:\n        return np.zeros((0, 2), dtype=np.int64), np.zeros((0,), dtype=np.float64)\n    sim = d0 @ d1.T\n    nn01 = sim.argmax(axis=1)\n    nn10 = sim.argmax(axis=0)\n    i = np.arange(len(d0))\n    mutual = nn10[nn01] == i\n    scores = sim[i, nn01]\n    keep = mutual & (scores >= min_similarity)\n    pairs = np.stack([i[keep], nn01[keep]], axis=1)\n    return pairs.astype(np.int64), scores[keep]\n","lightglue_pipeline/model.py":"from __future__ import annotations\n\nimport hashlib\nimport io\nimport json\nimport os\nimport pickletools\nimport time\nimport urllib.request\nimport zipfile\nfrom collections.abc import Callable\nfrom pathlib import Path\nfrom typing import Any\n\nimport torch\n\nfrom .config import (\n    CKPT_ALLOWED_GLOBALS,\n    CONVERTED_FILENAMES,\n    DEFAULT_MODEL_KEY,\n    DEPTH_CONFIDENCE,\n    DETECTION_THRESHOLD,\n    EXTRACTOR_FILENAME,\n    EXTRACTOR_PARAMETER_COUNT,\n    EXTRACTOR_PICKLE_AUDIT_SHA256,\n    EXTRACTOR_SHA256,\n    EXTRACTOR_SIZE_BYTES,\n    EXTRACTOR_SOURCE_FILENAME,\n    EXTRACTOR_SOURCE_SHA256,\n    EXTRACTOR_SOURCE_SIZE_BYTES,\n    EXTRACTOR_SOURCE_URL,\n    EXTRACTOR_STATE_TENSORS,\n    FILTER_THRESHOLD,\n    MATCHER_FILENAME,\n    MATCHER_PARAMETER_COUNT,\n    MATCHER_PICKLE_AUDIT_SHA256,\n    MATCHER_SHA256,\n    MATCHER_SIZE_BYTES,\n    MATCHER_SOURCE_FILENAME,\n    MATCHER_SOURCE_SHA256,\n    MATCHER_SOURCE_SIZE_BYTES,\n    MATCHER_SOURCE_URL,\n    MATCHER_STATE_TENSORS,\n    MAX_KEYPOINTS,\n    MODEL_ID,\n    MODEL_REVISION,\n    NMS_RADIUS,\n    SOURCE_FILENAMES,\n    UNSAFE_WEIGHT_EXTENSIONS,\n    WIDTH_CONFIDENCE,\n)\n\nMANIFEST_NAME = \"dimer-base-manifest.json\"\n#: Fleet snapshot scheme (DIMER NOTEBOOK_SPEC 1.1 MOD13): the pinned files live in a repository-\n#: local snapshot directory named by the model key and described by the committed manifest; a\n#: standalone notebook carries that manifest inline and stages/verifies a working-directory copy.\nDEFAULT_WEIGHTS_DIR = Path(__file__).resolve().parents[2] / \"weights\" / DEFAULT_MODEL_KEY\n\n#: The two pinned sources (name -> (url, sha256, bytes, pickle-audit digest)) and the two converted\n#: files they become (name -> (sha256, bytes, state tensors)).\nSOURCES: dict[str, tuple[str, str, int, str]] = {\n    MATCHER_SOURCE_FILENAME: (\n        MATCHER_SOURCE_URL,\n        MATCHER_SOURCE_SHA256,\n        MATCHER_SOURCE_SIZE_BYTES,\n        MATCHER_PICKLE_AUDIT_SHA256,\n    ),\n    EXTRACTOR_SOURCE_FILENAME: (\n        EXTRACTOR_SOURCE_URL,\n        EXTRACTOR_SOURCE_SHA256,\n        EXTRACTOR_SOURCE_SIZE_BYTES,\n        EXTRACTOR_PICKLE_AUDIT_SHA256,\n    ),\n}\nCONVERTED: dict[str, tuple[str, int, int]] = {\n    MATCHER_FILENAME: (MATCHER_SHA256, MATCHER_SIZE_BYTES, MATCHER_STATE_TENSORS),\n    EXTRACTOR_FILENAME: (EXTRACTOR_SHA256, EXTRACTOR_SIZE_BYTES, EXTRACTOR_STATE_TENSORS),\n}\nSOURCE_OF: dict[str, str] = {\n    MATCHER_FILENAME: MATCHER_SOURCE_FILENAME,\n    EXTRACTOR_FILENAME: EXTRACTOR_SOURCE_FILENAME,\n}\n\n\ndef _sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open(\"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef verify_checkpoint(\n    snapshot_path: str | Path,\n    *,\n    require_configs: bool = False,\n    return_manifest_verified: bool = False,\n) -> Path | tuple[Path, bool]:\n    \"\"\"Assert the two converted safetensors files against their pinned byte counts and digests,\n    refuse any weight file in an unsafe format other than the two pinned (digest-checked, never\n    loaded) sources, and size/digest-check every manifest entry that is present.\"\"\"\n    root = Path(snapshot_path)\n    if not root.is_dir():\n        raise RuntimeError(f\"Checkpoint directory does not exist: {root}\")\n\n    for name in CONVERTED_FILENAMES:\n        if not (root / name).is_file():\n            raise RuntimeError(\n                f\"Pinned checkpoint is missing {name}; run convert_sources() on the audited \"\n                f\"{SOURCE_OF[name]} first\"\n            )\n\n    unsafe = sorted(\n        p.name\n        for p in root.iterdir()\n        if p.is_file()\n        and p.suffix.lower() in UNSAFE_WEIGHT_EXTENSIONS\n        and p.name not in SOURCE_FILENAMES\n    )\n    if unsafe:\n        raise RuntimeError(f\"Refusing unsafe weight files: {unsafe}\")\n\n    manifest_path = root / MANIFEST_NAME\n    manifest_verified = False\n\n    if manifest_path.is_file():\n        try:\n            manifest = json.loads(manifest_path.read_text(encoding=\"utf-8\"))\n        except Exception as exc:\n            raise RuntimeError(f\"Corrupt manifest {MANIFEST_NAME}: {exc}\") from exc\n\n        files = manifest.get(\"files\") or []\n        if not files:\n            raise RuntimeError(f\"Manifest {MANIFEST_NAME} contains no files\")\n\n        for entry in files:\n            rel_path = entry.get(\"path\")\n            if not rel_path:\n                continue\n            target = root / rel_path\n            if not target.is_file():\n                if rel_path in SOURCE_FILENAMES:\n                    continue  # converted-only (DIMER-hosted) shape: the sources are not required\n                raise RuntimeError(f\"Manifest file missing: {rel_path}\")\n            exp_bytes = entry.get(\"bytes\")\n            if exp_bytes is not None and target.stat().st_size != exp_bytes:\n                raise RuntimeError(\n                    f\"Size mismatch for {rel_path}: {target.stat().st_size} != {exp_bytes}\"\n                )\n            exp_sha = entry.get(\"sha256\")\n            if exp_sha is not None and _sha256(target) != exp_sha:\n                raise RuntimeError(f\"SHA-256 mismatch for {rel_path}\")\n\n        manifest_verified = True\n\n    for name, (sha, size_bytes, _tensors) in CONVERTED.items():\n        weight_path = root / name\n        size = weight_path.stat().st_size\n        if size != size_bytes:\n            raise RuntimeError(f\"Unexpected {name} size: {size}; expected {size_bytes}\")\n        digest = _sha256(weight_path)\n        if digest != sha:\n            raise RuntimeError(f\"Unexpected {name} SHA-256: {digest}; expected {sha}\")\n\n    # The networks' configurations are carried in code (modeling.ALIKED.cfgs and\n    # modeling.LightGlue.features); with require_configs there is nothing else to require.\n\n    if return_manifest_verified:\n        return root, manifest_verified\n    return root\n\n\ndef _read_manifest(root: Path) -> dict[str, Any]:\n    \"\"\"Load and identity-check ``<root>/dimer-base-manifest.json``.\"\"\"\n    manifest_path = root / MANIFEST_NAME\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f\"snapshot manifest not found: {manifest_path}\")\n    try:\n        manifest = json.loads(manifest_path.read_text(encoding=\"utf-8\"))\n    except ValueError as exc:\n        raise RuntimeError(f\"Corrupt manifest {MANIFEST_NAME}: {exc}\") from exc\n    if manifest.get(\"modelId\") != MODEL_ID or manifest.get(\"revision\") != MODEL_REVISION:\n        raise ValueError(\n            f\"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, \"\n            f\"package pins {MODEL_ID}@{MODEL_REVISION}; refusing\"\n        )\n    if not manifest.get(\"files\"):\n        raise RuntimeError(f\"Manifest {MANIFEST_NAME} contains no files\")\n    return manifest\n\n\ndef _check_manifest_files(root: Path, manifest: dict[str, Any]) -> None:\n    for entry in manifest[\"files\"]:\n        target = root / entry[\"path\"]\n        if not target.is_file():\n            raise RuntimeError(f\"Manifest file missing: {entry['path']}\")\n        if target.stat().st_size != entry.get(\"bytes\"):\n            raise RuntimeError(f\"Size mismatch for {entry['path']}\")\n        if _sha256(target) != entry.get(\"sha256\"):\n            raise RuntimeError(f\"SHA-256 mismatch for {entry['path']}\")\n\n\ndef verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:\n    \"\"\"Manifest-driven verification of a fleet snapshot directory; raise on the first mismatch.\n\n    The identity in the manifest must be the pinned one. Before conversion the two source pickles\n    are size- and SHA-256-checked (never unpickled here); once both converted files exist,\n    :func:`verify_checkpoint` asserts their pinned digests and byte counts and checks every manifest\n    entry still present. Returns ``{\"path\": ..., **manifest, \"converted\": bool}``.\n    \"\"\"\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    manifest = _read_manifest(root)\n    converted = all((root / name).is_file() for name in CONVERTED_FILENAMES)\n    if not converted:\n        _check_manifest_files(root, manifest)\n        return {\"path\": str(root), **manifest, \"converted\": False}\n    _, manifest_verified = verify_checkpoint(\n        root, require_configs=True, return_manifest_verified=True\n    )\n    if not manifest_verified:\n        raise RuntimeError(f\"manifest at {root} was not verified\")  # pragma: no cover\n    return {\"path\": str(root), **manifest, \"converted\": True}\n\n\ndef _release_download(relative_path: str, root: Path) -> None:\n    \"\"\"Fetch one pinned source from its immutable URL: the LightGlue release asset or the ALIKED\n    file at the pinned commit (never a branch). The digest is re-checked by ``verify_snapshot`` and\n    again by the static audit before anything is unpickled: a substituted file is caught first.\"\"\"\n    if relative_path not in SOURCES:\n        raise ValueError(f\"{relative_path} is not a downloadable manifest entry\")\n    url = SOURCES[relative_path][0]\n    root.mkdir(parents=True, exist_ok=True)\n    target = root / relative_path\n    tmp = target.with_suffix(target.suffix + \".part\")\n    with urllib.request.urlopen(url, timeout=120) as response, open(tmp, \"wb\") as fh:  # noqa: S310\n        while chunk := response.read(1 << 20):\n            fh.write(chunk)\n    tmp.replace(target)\n\n\ndef stage_missing_files(\n    path: str | Path | None = None,\n    *,\n    allow_download: bool = False,\n    downloader: Callable[[str, Path], None] | None = None,\n) -> list[str]:\n    \"\"\"Fetch manifest-listed sources that are absent locally (a fresh clone commits the manifest but\n    git-ignores the weights). A source is not needed when its converted file is already present.\n    Returns the relative paths fetched; :func:`verify_snapshot` still runs after.\"\"\"\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    manifest = _read_manifest(root)\n    missing = [entry[\"path\"] for entry in manifest[\"files\"] if not (root / entry[\"path\"]).is_file()]\n    for converted_name, source_name in SOURCE_OF.items():\n        if (root / converted_name).is_file() and source_name in missing:\n            missing.remove(source_name)  # converted-only shape: the source pickle is not needed\n    if not missing:\n        return []\n    if not allow_download:\n        raise FileNotFoundError(\n            f\"snapshot at {root} is missing {missing}; \"\n            f\"pass allow_download=True to fetch them from the {MODEL_REVISION} release assets\"\n        )\n    fetch = downloader or _release_download\n    for relative_path in missing:\n        fetch(relative_path, root)\n    return missing\n\n\ndef _pickle_globals(data: bytes) -> dict[str, int]:\n    \"\"\"Every global a pickle stream would import, collected with `pickletools.genops` (no\n    execution).\"\"\"\n    found: dict[str, int] = {}\n    stack: list[Any] = []\n    for op, arg, _pos in pickletools.genops(io.BytesIO(data)):\n        if op.name == \"GLOBAL\":  # pickletools renders the (module, name) pair space-separated\n            key = arg.replace(\"\\n\", \" \").replace(\" \", \".\", 1)\n            found[key] = found.get(key, 0) + 1\n        elif op.name == \"STACK_GLOBAL\":\n            key = f\"{stack[-2]}.{stack[-1]}\"\n            found[key] = found.get(key, 0) + 1\n        if op.name in (\"SHORT_BINUNICODE\", \"BINUNICODE\", \"UNICODE\", \"SHORT_BINSTRING\", \"BINSTRING\"):\n            stack.append(arg)\n        elif op.name in (\"MEMOIZE\", \"BINPUT\", \"LONG_BINPUT\", \"PUT\"):\n            pass\n        else:\n            stack.append(None)\n    return found\n\n\ndef audit_pickle(\n    path: str | Path, *, allowed: frozenset[str] = CKPT_ALLOWED_GLOBALS\n) -> dict[str, Any]:\n    \"\"\"Statically list the globals a pickle (plain, or inside a torch zip archive) would import and\n    refuse any outside `allowed`. Executes nothing. Returns the sorted globals and their digest.\"\"\"\n    file_path = Path(path)\n    if not file_path.is_file():\n        raise FileNotFoundError(f\"file not found: {file_path}\")\n    data = file_path.read_bytes()\n    found: dict[str, int] = {}\n    nested = 0\n    if data[:4] == b\"PK\\x03\\x04\":\n        archive = zipfile.ZipFile(io.BytesIO(data))\n        for name in archive.namelist():\n            if name.endswith(\".pkl\"):\n                nested += 1\n                for key, count in _pickle_globals(archive.read(name)).items():\n                    found[key] = found.get(key, 0) + count\n    else:\n        found = _pickle_globals(data)\n    violations = sorted(name for name in found if name not in allowed)\n    summary = {\n        \"file\": file_path.name,\n        \"torch_archive\": data[:4] == b\"PK\\x03\\x04\",\n        \"pickles\": nested if nested else 1,\n        \"globals\": sorted(found),\n        \"violations\": violations,\n        \"audit_sha256\": hashlib.sha256(\"\\n\".join(sorted(found)).encode(\"utf-8\")).hexdigest(),\n    }\n    if violations:\n        raise ValueError(\n            f\"{file_path.name}: pickle audit failed, globals outside the allow-list: {violations}\"\n        )\n    return summary\n\n\ndef _check_pinned_source(root: Path, name: str) -> dict[str, Any]:\n    _url, sha, size_bytes, audit_sha = SOURCES[name]\n    source = root / name\n    if not source.is_file():\n        raise FileNotFoundError(f\"source file not found: {source}\")\n    size = source.stat().st_size\n    if size != size_bytes:\n        raise ValueError(f\"{name}: size {size} != pinned {size_bytes}\")\n    digest = _sha256(source)\n    if digest != sha:\n        raise ValueError(f\"{name}: sha256 {digest} != pinned {sha}\")\n    audit = audit_pickle(source)\n    if audit[\"audit_sha256\"] != audit_sha:\n        raise ValueError(\n            f\"{name}: pickle audit digest {audit['audit_sha256']} != pinned {audit_sha}\"\n        )\n    return {\"path\": name, \"bytes\": size, \"sha256\": digest, \"audit\": audit}\n\n\ndef build_models(\n    *,\n    max_keypoints: int = MAX_KEYPOINTS,\n    detection_threshold: float = DETECTION_THRESHOLD,\n    filter_threshold: float = FILTER_THRESHOLD,\n) -> tuple[Any, Any]:\n    \"\"\"The vendored ALIKED-N(16) extractor and the ALIKED-feature LightGlue matcher at random\n    initialisation, in the inference configuration of this contract (nine layers, no pruning).\"\"\"\n    from .modeling import ALIKED, LightGlue\n\n    extractor = ALIKED(\n        max_num_keypoints=max_keypoints,\n        detection_threshold=detection_threshold,\n        nms_radius=NMS_RADIUS,\n    )\n    matcher = LightGlue(\n        features=\"aliked\",\n        filter_threshold=filter_threshold,\n        depth_confidence=DEPTH_CONFIDENCE,\n        width_confidence=WIDTH_CONFIDENCE,\n    )\n    return extractor, matcher\n\n\ndef convert_sources(path: str | Path | None = None) -> dict[str, Any]:\n    \"\"\"Convert the two pinned pickles into their safetensors files, deterministically, after size,\n    digest and static-audit checks: torch's weights-only unpickler, a strict load into the vendored\n    network, and the network's own state dict saved (sorted keys, contiguous tensors). Each pickle\n    is unpickled exactly once, here. Converted files that already exist are left as they are.\"\"\"\n    from safetensors.torch import save_file\n\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    started = time.perf_counter()\n    extractor, matcher = build_models()\n    report: dict[str, Any] = {\"converted\": [], \"skipped\": []}\n    for converted_name, module in ((MATCHER_FILENAME, matcher), (EXTRACTOR_FILENAME, extractor)):\n        target = root / converted_name\n        sha, size_bytes, n_tensors = CONVERTED[converted_name]\n        if target.is_file():\n            if target.stat().st_size != size_bytes or _sha256(target) != sha:\n                raise ValueError(\n                    f\"{converted_name}: existing file does not match its pinned digest\"\n                )\n            report[\"skipped\"].append(converted_name)\n            continue\n        source_name = SOURCE_OF[converted_name]\n        source = _check_pinned_source(root, source_name)\n        state = torch.load(root / source_name, map_location=\"cpu\", weights_only=True)\n        if not isinstance(state, dict) or any(\n            not isinstance(v, torch.Tensor) for v in state.values()\n        ):\n            raise ValueError(f\"{source_name} did not unpickle to a state dict of tensors\")\n        if len(state) != n_tensors:\n            raise ValueError(\n                f\"{source_name}: state dict has {len(state)} tensors, expected {n_tensors}\"\n            )\n        module.load_state_dict(state, strict=True)\n        canonical = {k: v.contiguous() for k, v in module.state_dict().items()}\n        if len(canonical) != n_tensors:\n            raise ValueError(\n                f\"converted state dict has {len(canonical)} tensors; expected {n_tensors}\"\n            )\n        save_file(canonical, str(target), metadata={\"format\": \"pt\"})\n        size = target.stat().st_size\n        digest = _sha256(target)\n        if size != size_bytes or digest != sha:\n            target.unlink()\n            raise ValueError(\n                f\"{converted_name}: converted file {size} B / {digest} != pinned \"\n                f\"{size_bytes} B / {sha}\"\n            )\n        report[\"converted\"].append(\n            {\n                \"source\": {k: v for k, v in source.items() if k != \"audit\"},\n                \"audit\": source[\"audit\"],\n                \"state_dict_tensors\": len(state),\n                \"parameters\": sum(p.numel() for p in module.parameters()),\n                \"converted\": {\"path\": converted_name, \"bytes\": size, \"sha256\": digest},\n            }\n        )\n    report[\"seconds\"] = round(time.perf_counter() - started, 2)\n    return report\n\n\ndef resolve_weights_path(\n    weights_path: str | Path | None = None,\n    cache_dir: str | Path | None = None,\n) -> tuple[Path, str]:\n    \"\"\"Resolve weights path with precedence:\n\n    1. Explicit argument `weights_path` -> 'explicit_path'\n    2. Environment variable `LIGHTGLUE_WEIGHTS_DIR` -> 'env_var'\n    3. Source checkout convention `weights/lightglue-aliked` -> 'repo_offline'\n       (only if pyproject.toml exists at repo root and the directory holds the manifest)\n\n    There is no Hub fallback: the checkpoints are GitHub-hosted files staged by\n    :func:`stage_missing_files` into a manifest-described snapshot directory.\n    \"\"\"\n    if weights_path is not None:\n        return Path(weights_path), \"explicit_path\"\n\n    env_dir = os.environ.get(\"LIGHTGLUE_WEIGHTS_DIR\")\n    if env_dir:\n        return Path(env_dir), \"env_var\"\n\n    repo_root = Path(__file__).resolve().parents[2]\n    if (repo_root / \"pyproject.toml\").is_file():\n        repo_weights = repo_root / \"weights\" / DEFAULT_MODEL_KEY\n        if (repo_weights / MANIFEST_NAME).is_file():\n            return repo_weights, \"repo_offline\"\n\n    raise FileNotFoundError(\n        \"no weights directory: pass weights_path / weights_dir, set LIGHTGLUE_WEIGHTS_DIR, or run \"\n        f\"from a checkout holding weights/{DEFAULT_MODEL_KEY}/{MANIFEST_NAME}\"\n    )\n\n\n_resolve_weights_path = resolve_weights_path\n\n\ndef _resolve_device(device: str | torch.device | None) -> torch.device:\n    if device is not None:\n        return torch.device(device)\n    return torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n\n\ndef load_components(\n    *,\n    device: str | torch.device | None = None,\n    cache_dir: str | Path | None = None,\n    weights_path: str | Path | None = None,\n    return_metadata: bool = False,\n    max_keypoints: int = MAX_KEYPOINTS,\n    detection_threshold: float = DETECTION_THRESHOLD,\n    filter_threshold: float = FILTER_THRESHOLD,\n) -> tuple[Any, Any, torch.device, Path] | tuple[Any, Any, torch.device, Path, dict[str, Any]]:\n    \"\"\"Verify and load the two converted checkpoints into the vendored networks (strict state-dict\n    loads from safetensors; no pickle is opened here).\"\"\"\n    from safetensors.torch import load_file\n\n    candidate_path, source = resolve_weights_path(\n        weights_path=weights_path,\n        cache_dir=cache_dir,\n    )\n\n    verified, manifest_verified = verify_checkpoint(\n        candidate_path,\n        require_configs=True,\n        return_manifest_verified=True,\n    )\n    target_device = _resolve_device(device)\n\n    extractor, matcher = build_models(\n        max_keypoints=max_keypoints,\n        detection_threshold=detection_threshold,\n        filter_threshold=filter_threshold,\n    )\n    loaded: dict[str, Any] = {}\n    for name, module, n_tensors, n_params in (\n        (EXTRACTOR_FILENAME, extractor, EXTRACTOR_STATE_TENSORS, EXTRACTOR_PARAMETER_COUNT),\n        (MATCHER_FILENAME, matcher, MATCHER_STATE_TENSORS, MATCHER_PARAMETER_COUNT),\n    ):\n        state = load_file(str(verified / name))\n        if len(state) != n_tensors:\n            raise RuntimeError(f\"{name}: {len(state)} tensors, expected {n_tensors}\")\n        module.load_state_dict(state, strict=True)\n        count = sum(p.numel() for p in module.parameters())\n        if count != n_params:\n            raise RuntimeError(f\"{name}: network has {count} parameters, expected {n_params}\")\n        module.eval().to(target_device)\n        loaded[name] = {\"state_tensors\": len(state), \"parameters\": count}\n\n    weight_file = verified / MATCHER_FILENAME\n    metadata: dict[str, Any] = {\n        \"checkpoint_path\": verified,\n        \"checkpoint_source\": source,\n        \"manifest_verified\": manifest_verified,\n        \"weight_sha256\": _sha256(weight_file),\n        \"weight_size_bytes\": weight_file.stat().st_size,\n        \"extractor_sha256\": _sha256(verified / EXTRACTOR_FILENAME),\n        \"extractor_size_bytes\": (verified / EXTRACTOR_FILENAME).stat().st_size,\n        \"device\": str(target_device),\n        \"state_tensors\": loaded[MATCHER_FILENAME][\"state_tensors\"],\n        \"parameters\": loaded[MATCHER_FILENAME][\"parameters\"],\n        \"extractor_state_tensors\": loaded[EXTRACTOR_FILENAME][\"state_tensors\"],\n        \"extractor_parameters\": loaded[EXTRACTOR_FILENAME][\"parameters\"],\n    }\n\n    if return_metadata:\n        return extractor, matcher, target_device, verified, metadata\n    return extractor, matcher, target_device, verified\n","lightglue_pipeline/modeling.py":"\"\"\"ALIKED (Zhao et al., IEEE TIM 2023) local-feature extractor and LightGlue (Lindenberger, Sarlin and Pollefeys,\nICCV 2023) matcher, vendored from https://github.com/cvg/LightGlue at commit\neb42fee2d71449efb0aa5c10549752b5d75384d8 (Apache-2.0; ``lightglue/aliked.py`` is the authors' BSD-3-Clause port of\nhttps://github.com/Shiaoming/ALIKED and keeps its licence header below): ``lightglue/utils.py`` (the ``Extractor``\nbase only), ``lightglue/aliked.py`` and ``lightglue/lightglue.py`` concatenated in dependency order with the\npackage-relative imports removed. Nothing is fetched or unpickled at construction: ``model.load_components``\nloads the audited, converted safetensors state dicts strictly.\n\nEdits against upstream, each marked ``# vendored:`` in place and listed in ``docs/WEIGHTS.md``: the kornia\n``grayscale_to_rgb`` call is a channel ``expand``; ``torchvision.models.resnet.conv1x1`` / ``conv3x3`` are the two\none-line helpers below (``torchvision.ops.deform_conv2d`` is the one torchvision call kept); the optional\n``flash_attn`` import, the download-at-construction code of both classes, ``ALIKED.describe`` and the cv2 /\nkornia image utilities are not carried; and LightGlue's ``confidence_thresholds`` buffer is registered\n``persistent=False`` (it is computed from the configuration and absent from the checkpoint) so the checkpoint\nloads with ``strict=True`` instead of upstream's ``strict=False``.\n\"\"\"\n# ruff: noqa: E501, N801, N802, N803, N806, E741, B905, B006, B008, F841, E712, UP004, UP006, UP008, UP031, UP032, UP035, UP045  -- vendored code kept as upstream wrote it, for auditability\n\nfrom __future__ import annotations\n\nimport warnings\nfrom types import SimpleNamespace\nfrom typing import Callable, List, Optional, Tuple\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom torch.nn.modules.utils import _pair\nfrom torchvision.ops import deform_conv2d\n\nUPSTREAM_REPOSITORY = \"https://github.com/cvg/LightGlue\"\nUPSTREAM_COMMIT = \"eb42fee2d71449efb0aa5c10549752b5d75384d8\"\nALIKED_REPOSITORY = \"https://github.com/Shiaoming/ALIKED\"\n\n\ndef conv3x3(in_planes: int, out_planes: int, stride: int = 1, groups: int = 1, dilation: int = 1) -> nn.Conv2d:\n    \"\"\"``torchvision.models.resnet.conv3x3``, carried so the vendored code does not import torchvision.models.\"\"\"\n    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=dilation, groups=groups, bias=False, dilation=dilation)\n\n\ndef conv1x1(in_planes: int, out_planes: int, stride: int = 1) -> nn.Conv2d:\n    \"\"\"``torchvision.models.resnet.conv1x1``.\"\"\"\n    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)\n\n\n# --------------------------------------------------------------------------------------------------\n# lightglue/utils.py (Extractor base only)\n# --------------------------------------------------------------------------------------------------\n\n\nclass Extractor(torch.nn.Module):\n    # upstream `lightglue/utils.py`: only the configuration base is carried; `extract` (kornia resize) is not\n    def __init__(self, **conf):\n        super().__init__()\n        self.conf = SimpleNamespace(**{**self.default_conf, **conf})\n\n\n# --------------------------------------------------------------------------------------------------\n# lightglue/aliked.py\n# --------------------------------------------------------------------------------------------------\n\n# BSD 3-Clause License\n\n# Copyright (c) 2022, Zhao Xiaoming\n# All rights reserved.\n\n# Redistribution and use in source and binary forms, with or without\n# modification, are permitted provided that the following conditions are met:\n\n# 1. Redistributions of source code must retain the above copyright notice, this\n#    list of conditions and the following disclaimer.\n\n# 2. Redistributions in binary form must reproduce the above copyright notice,\n#    this list of conditions and the following disclaimer in the documentation\n#    and/or other materials provided with the distribution.\n\n# 3. Neither the name of the copyright holder nor the names of its\n#    contributors may be used to endorse or promote products derived from\n#    this software without specific prior written permission.\n\n# THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS \"AS IS\"\n# AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE\n# IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\n# DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE\n# FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\n# DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\n# SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\n# CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,\n# OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE\n# OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n\n# Authors:\n# Xiaoming Zhao, Xingming Wu, Weihai Chen, Peter C.Y. Chen, Qingsong Xu, and Zhengguo Li\n# Code from https://github.com/Shiaoming/ALIKED\n\n\n\ndef get_patches(\n    tensor: torch.Tensor, required_corners: torch.Tensor, ps: int\n) -> torch.Tensor:\n    c, h, w = tensor.shape\n    corner = (required_corners - ps / 2 + 1).long()\n    corner[:, 0] = corner[:, 0].clamp(min=0, max=w - 1 - ps)\n    corner[:, 1] = corner[:, 1].clamp(min=0, max=h - 1 - ps)\n    offset = torch.arange(0, ps)\n\n    kw = {\"indexing\": \"ij\"} if torch.__version__ >= \"1.10\" else {}\n    x, y = torch.meshgrid(offset, offset, **kw)\n    patches = torch.stack((x, y)).permute(2, 1, 0).unsqueeze(2)\n    patches = patches.to(corner) + corner[None, None]\n    pts = patches.reshape(-1, 2)\n    sampled = tensor.permute(1, 2, 0)[tuple(pts.T)[::-1]]\n    sampled = sampled.reshape(ps, ps, -1, c)\n    assert sampled.shape[:3] == patches.shape[:3]\n    return sampled.permute(2, 3, 0, 1)\n\n\ndef simple_nms(scores: torch.Tensor, nms_radius: int):\n    \"\"\"Fast Non-maximum suppression to remove nearby points\"\"\"\n\n    zeros = torch.zeros_like(scores)\n    max_mask = scores == torch.nn.functional.max_pool2d(\n        scores, kernel_size=nms_radius * 2 + 1, stride=1, padding=nms_radius\n    )\n\n    for _ in range(2):\n        supp_mask = (\n            torch.nn.functional.max_pool2d(\n                max_mask.float(),\n                kernel_size=nms_radius * 2 + 1,\n                stride=1,\n                padding=nms_radius,\n            )\n            > 0\n        )\n        supp_scores = torch.where(supp_mask, zeros, scores)\n        new_max_mask = supp_scores == torch.nn.functional.max_pool2d(\n            supp_scores, kernel_size=nms_radius * 2 + 1, stride=1, padding=nms_radius\n        )\n        max_mask = max_mask | (new_max_mask & (~supp_mask))\n    return torch.where(max_mask, scores, zeros)\n\n\nclass DKD(nn.Module):\n    def __init__(\n        self,\n        radius: int = 2,\n        top_k: int = 0,\n        scores_th: float = 0.2,\n        n_limit: int = 20000,\n    ):\n        \"\"\"\n        Args:\n            radius: soft detection radius, kernel size is (2 * radius + 1)\n            top_k: top_k > 0: return top k keypoints\n            scores_th: top_k <= 0 threshold mode:\n                scores_th > 0: return keypoints with scores>scores_th\n                else: return keypoints with scores > scores.mean()\n            n_limit: max number of keypoint in threshold mode\n        \"\"\"\n        super().__init__()\n        self.radius = radius\n        self.top_k = top_k\n        self.scores_th = scores_th\n        self.n_limit = n_limit\n        self.kernel_size = 2 * self.radius + 1\n        self.temperature = 0.1  # tuned temperature\n        self.unfold = nn.Unfold(kernel_size=self.kernel_size, padding=self.radius)\n        # local xy grid\n        x = torch.linspace(-self.radius, self.radius, self.kernel_size)\n        # (kernel_size*kernel_size) x 2 : (w,h)\n        kw = {\"indexing\": \"ij\"} if torch.__version__ >= \"1.10\" else {}\n        self.hw_grid = (\n            torch.stack(torch.meshgrid([x, x], **kw)).view(2, -1).t()[:, [1, 0]]\n        )\n\n    def forward(\n        self,\n        scores_map: torch.Tensor,\n        sub_pixel: bool = True,\n        image_size: Optional[torch.Tensor] = None,\n    ):\n        \"\"\"\n        :param scores_map: Bx1xHxW\n        :param descriptor_map: BxCxHxW\n        :param sub_pixel: whether to use sub-pixel keypoint detection\n        :return: kpts: list[Nx2,...]; kptscores: list[N,....] normalised position: -1~1\n        \"\"\"\n        b, c, h, w = scores_map.shape\n        scores_nograd = scores_map.detach()\n        nms_scores = simple_nms(scores_nograd, self.radius)\n\n        # remove border\n        nms_scores[:, :, : self.radius, :] = 0\n        nms_scores[:, :, :, : self.radius] = 0\n        if image_size is not None:\n            for i in range(scores_map.shape[0]):\n                w, h = image_size[i].long()\n                nms_scores[i, :, h.item() - self.radius :, :] = 0\n                nms_scores[i, :, :, w.item() - self.radius :] = 0\n        else:\n            nms_scores[:, :, -self.radius :, :] = 0\n            nms_scores[:, :, :, -self.radius :] = 0\n\n        # detect keypoints without grad\n        if self.top_k > 0:\n            topk = torch.topk(nms_scores.view(b, -1), self.top_k)\n            indices_keypoints = [topk.indices[i] for i in range(b)]  # B x top_k\n        else:\n            if self.scores_th > 0:\n                masks = nms_scores > self.scores_th\n                if masks.sum() == 0:\n                    th = scores_nograd.reshape(b, -1).mean(dim=1)  # th = self.scores_th\n                    masks = nms_scores > th.reshape(b, 1, 1, 1)\n            else:\n                th = scores_nograd.reshape(b, -1).mean(dim=1)  # th = self.scores_th\n                masks = nms_scores > th.reshape(b, 1, 1, 1)\n            masks = masks.reshape(b, -1)\n\n            indices_keypoints = []  # list, B x (any size)\n            scores_view = scores_nograd.reshape(b, -1)\n            for mask, scores in zip(masks, scores_view):\n                indices = mask.nonzero()[:, 0]\n                if len(indices) > self.n_limit:\n                    kpts_sc = scores[indices]\n                    sort_idx = kpts_sc.sort(descending=True)[1]\n                    sel_idx = sort_idx[: self.n_limit]\n                    indices = indices[sel_idx]\n                indices_keypoints.append(indices)\n\n        wh = torch.tensor([w - 1, h - 1], device=scores_nograd.device)\n\n        keypoints = []\n        scoredispersitys = []\n        kptscores = []\n        if sub_pixel:\n            # detect soft keypoints with grad backpropagation\n            patches = self.unfold(scores_map)  # B x (kernel**2) x (H*W)\n            self.hw_grid = self.hw_grid.to(scores_map)  # to device\n            for b_idx in range(b):\n                patch = patches[b_idx].t()  # (H*W) x (kernel**2)\n                indices_kpt = indices_keypoints[\n                    b_idx\n                ]  # one dimension vector, say its size is M\n                patch_scores = patch[indices_kpt]  # M x (kernel**2)\n                keypoints_xy_nms = torch.stack(\n                    [indices_kpt % w, torch.div(indices_kpt, w, rounding_mode=\"trunc\")],\n                    dim=1,\n                )  # Mx2\n\n                # max is detached to prevent undesired backprop loops in the graph\n                max_v = patch_scores.max(dim=1).values.detach()[:, None]\n                x_exp = (\n                    (patch_scores - max_v) / self.temperature\n                ).exp()  # M * (kernel**2), in [0, 1]\n\n                # \\frac{ \\sum{(i,j) \\times \\exp(x/T)} }{ \\sum{\\exp(x/T)} }\n                xy_residual = (\n                    x_exp @ self.hw_grid / x_exp.sum(dim=1)[:, None]\n                )  # Soft-argmax, Mx2\n\n                hw_grid_dist2 = (\n                    torch.norm(\n                        (self.hw_grid[None, :, :] - xy_residual[:, None, :])\n                        / self.radius,\n                        dim=-1,\n                    )\n                    ** 2\n                )\n                scoredispersity = (x_exp * hw_grid_dist2).sum(dim=1) / x_exp.sum(dim=1)\n\n                # compute result keypoints\n                keypoints_xy = keypoints_xy_nms + xy_residual\n                keypoints_xy = keypoints_xy / wh * 2 - 1  # (w,h) -> (-1~1,-1~1)\n\n                kptscore = torch.nn.functional.grid_sample(\n                    scores_map[b_idx].unsqueeze(0),\n                    keypoints_xy.view(1, 1, -1, 2),\n                    mode=\"bilinear\",\n                    align_corners=True,\n                )[\n                    0, 0, 0, :\n                ]  # CxN\n\n                keypoints.append(keypoints_xy)\n                scoredispersitys.append(scoredispersity)\n                kptscores.append(kptscore)\n        else:\n            for b_idx in range(b):\n                indices_kpt = indices_keypoints[\n                    b_idx\n                ]  # one dimension vector, say its size is M\n                # To avoid warning: UserWarning: __floordiv__ is deprecated\n                keypoints_xy_nms = torch.stack(\n                    [indices_kpt % w, torch.div(indices_kpt, w, rounding_mode=\"trunc\")],\n                    dim=1,\n                )  # Mx2\n                keypoints_xy = keypoints_xy_nms / wh * 2 - 1  # (w,h) -> (-1~1,-1~1)\n                kptscore = torch.nn.functional.grid_sample(\n                    scores_map[b_idx].unsqueeze(0),\n                    keypoints_xy.view(1, 1, -1, 2),\n                    mode=\"bilinear\",\n                    align_corners=True,\n                )[\n                    0, 0, 0, :\n                ]  # CxN\n                keypoints.append(keypoints_xy)\n                scoredispersitys.append(kptscore)  # for jit.script compatability\n                kptscores.append(kptscore)\n\n        return keypoints, kptscores, scoredispersitys\n\n\nclass InputPadder(object):\n    \"\"\"Pads images such that dimensions are divisible by 8\"\"\"\n\n    def __init__(self, h: int, w: int, divis_by: int = 8):\n        self.ht = h\n        self.wd = w\n        pad_ht = (((self.ht // divis_by) + 1) * divis_by - self.ht) % divis_by\n        pad_wd = (((self.wd // divis_by) + 1) * divis_by - self.wd) % divis_by\n        self._pad = [\n            pad_wd // 2,\n            pad_wd - pad_wd // 2,\n            pad_ht // 2,\n            pad_ht - pad_ht // 2,\n        ]\n\n    def pad(self, x: torch.Tensor):\n        assert x.ndim == 4\n        return F.pad(x, self._pad, mode=\"replicate\")\n\n    def unpad(self, x: torch.Tensor):\n        assert x.ndim == 4\n        ht = x.shape[-2]\n        wd = x.shape[-1]\n        c = [self._pad[2], ht - self._pad[3], self._pad[0], wd - self._pad[1]]\n        return x[..., c[0] : c[1], c[2] : c[3]]\n\n\nclass DeformableConv2d(nn.Module):\n    def __init__(\n        self,\n        in_channels,\n        out_channels,\n        kernel_size=3,\n        stride=1,\n        padding=1,\n        bias=False,\n        mask=False,\n    ):\n        super(DeformableConv2d, self).__init__()\n\n        self.padding = padding\n        self.mask = mask\n\n        self.channel_num = (\n            3 * kernel_size * kernel_size if mask else 2 * kernel_size * kernel_size\n        )\n        self.offset_conv = nn.Conv2d(\n            in_channels,\n            self.channel_num,\n            kernel_size=kernel_size,\n            stride=stride,\n            padding=self.padding,\n            bias=True,\n        )\n\n        self.regular_conv = nn.Conv2d(\n            in_channels=in_channels,\n            out_channels=out_channels,\n            kernel_size=kernel_size,\n            stride=stride,\n            padding=self.padding,\n            bias=bias,\n        )\n\n    def forward(self, x):\n        h, w = x.shape[2:]\n        max_offset = max(h, w) / 4.0\n\n        out = self.offset_conv(x)\n        if self.mask:\n            o1, o2, mask = torch.chunk(out, 3, dim=1)\n            offset = torch.cat((o1, o2), dim=1)\n            mask = torch.sigmoid(mask)\n        else:\n            offset = out\n            mask = None\n        offset = offset.clamp(-max_offset, max_offset)\n        x = deform_conv2d(\n            input=x,\n            offset=offset,\n            weight=self.regular_conv.weight,\n            bias=self.regular_conv.bias,\n            padding=self.padding,\n            mask=mask,\n        )\n        return x\n\n\ndef get_conv(\n    inplanes,\n    planes,\n    kernel_size=3,\n    stride=1,\n    padding=1,\n    bias=False,\n    conv_type=\"conv\",\n    mask=False,\n):\n    if conv_type == \"conv\":\n        conv = nn.Conv2d(\n            inplanes,\n            planes,\n            kernel_size=kernel_size,\n            stride=stride,\n            padding=padding,\n            bias=bias,\n        )\n    elif conv_type == \"dcn\":\n        conv = DeformableConv2d(\n            inplanes,\n            planes,\n            kernel_size=kernel_size,\n            stride=stride,\n            padding=_pair(padding),\n            bias=bias,\n            mask=mask,\n        )\n    else:\n        raise TypeError\n    return conv\n\n\nclass ConvBlock(nn.Module):\n    def __init__(\n        self,\n        in_channels,\n        out_channels,\n        gate: Optional[Callable[..., nn.Module]] = None,\n        norm_layer: Optional[Callable[..., nn.Module]] = None,\n        conv_type: str = \"conv\",\n        mask: bool = False,\n    ):\n        super().__init__()\n        if gate is None:\n            self.gate = nn.ReLU(inplace=True)\n        else:\n            self.gate = gate\n        if norm_layer is None:\n            norm_layer = nn.BatchNorm2d\n        self.conv1 = get_conv(\n            in_channels, out_channels, kernel_size=3, conv_type=conv_type, mask=mask\n        )\n        self.bn1 = norm_layer(out_channels)\n        self.conv2 = get_conv(\n            out_channels, out_channels, kernel_size=3, conv_type=conv_type, mask=mask\n        )\n        self.bn2 = norm_layer(out_channels)\n\n    def forward(self, x):\n        x = self.gate(self.bn1(self.conv1(x)))  # B x in_channels x H x W\n        x = self.gate(self.bn2(self.conv2(x)))  # B x out_channels x H x W\n        return x\n\n\n# modified based on torchvision\\models\\resnet.py#27->BasicBlock\nclass ResBlock(nn.Module):\n    expansion: int = 1\n\n    def __init__(\n        self,\n        inplanes: int,\n        planes: int,\n        stride: int = 1,\n        downsample: Optional[nn.Module] = None,\n        groups: int = 1,\n        base_width: int = 64,\n        dilation: int = 1,\n        gate: Optional[Callable[..., nn.Module]] = None,\n        norm_layer: Optional[Callable[..., nn.Module]] = None,\n        conv_type: str = \"conv\",\n        mask: bool = False,\n    ) -> None:\n        super(ResBlock, self).__init__()\n        if gate is None:\n            self.gate = nn.ReLU(inplace=True)\n        else:\n            self.gate = gate\n        if norm_layer is None:\n            norm_layer = nn.BatchNorm2d\n        if groups != 1 or base_width != 64:\n            raise ValueError(\"ResBlock only supports groups=1 and base_width=64\")\n        if dilation > 1:\n            raise NotImplementedError(\"Dilation > 1 not supported in ResBlock\")\n        # Both self.conv1 and self.downsample layers\n        # downsample the input when stride != 1\n        self.conv1 = get_conv(\n            inplanes, planes, kernel_size=3, conv_type=conv_type, mask=mask\n        )\n        self.bn1 = norm_layer(planes)\n        self.conv2 = get_conv(\n            planes, planes, kernel_size=3, conv_type=conv_type, mask=mask\n        )\n        self.bn2 = norm_layer(planes)\n        self.downsample = downsample\n        self.stride = stride\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        identity = x\n\n        out = self.conv1(x)\n        out = self.bn1(out)\n        out = self.gate(out)\n\n        out = self.conv2(out)\n        out = self.bn2(out)\n\n        if self.downsample is not None:\n            identity = self.downsample(x)\n\n        out += identity\n        out = self.gate(out)\n\n        return out\n\n\nclass SDDH(nn.Module):\n    def __init__(\n        self,\n        dims: int,\n        kernel_size: int = 3,\n        n_pos: int = 8,\n        gate=nn.ReLU(),\n        conv2D=False,\n        mask=False,\n    ):\n        super(SDDH, self).__init__()\n        self.kernel_size = kernel_size\n        self.n_pos = n_pos\n        self.conv2D = conv2D\n        self.mask = mask\n\n        self.get_patches_func = get_patches\n\n        # estimate offsets\n        self.channel_num = 3 * n_pos if mask else 2 * n_pos\n        self.offset_conv = nn.Sequential(\n            nn.Conv2d(\n                dims,\n                self.channel_num,\n                kernel_size=kernel_size,\n                stride=1,\n                padding=0,\n                bias=True,\n            ),\n            gate,\n            nn.Conv2d(\n                self.channel_num,\n                self.channel_num,\n                kernel_size=1,\n                stride=1,\n                padding=0,\n                bias=True,\n            ),\n        )\n\n        # sampled feature conv\n        self.sf_conv = nn.Conv2d(\n            dims, dims, kernel_size=1, stride=1, padding=0, bias=False\n        )\n\n        # convM\n        if not conv2D:\n            # deformable desc weights\n            agg_weights = torch.nn.Parameter(torch.rand(n_pos, dims, dims))\n            self.register_parameter(\"agg_weights\", agg_weights)\n        else:\n            self.convM = nn.Conv2d(\n                dims * n_pos, dims, kernel_size=1, stride=1, padding=0, bias=False\n            )\n\n    def forward(self, x, keypoints):\n        # x: [B,C,H,W]\n        # keypoints: list, [[N_kpts,2], ...] (w,h)\n        b, c, h, w = x.shape\n        wh = torch.tensor([[w - 1, h - 1]], device=x.device)\n        max_offset = max(h, w) / 4.0\n\n        offsets = []\n        descriptors = []\n        # get offsets for each keypoint\n        for ib in range(b):\n            xi, kptsi = x[ib], keypoints[ib]\n            kptsi_wh = (kptsi / 2 + 0.5) * wh\n            N_kpts = len(kptsi)\n\n            if self.kernel_size > 1:\n                patch = self.get_patches_func(\n                    xi, kptsi_wh.long(), self.kernel_size\n                )  # [N_kpts, C, K, K]\n            else:\n                kptsi_wh_long = kptsi_wh.long()\n                patch = (\n                    xi[:, kptsi_wh_long[:, 1], kptsi_wh_long[:, 0]]\n                    .permute(1, 0)\n                    .reshape(N_kpts, c, 1, 1)\n                )\n\n            offset = self.offset_conv(patch).clamp(\n                -max_offset, max_offset\n            )  # [N_kpts, 2*n_pos, 1, 1]\n            if self.mask:\n                offset = (\n                    offset[:, :, 0, 0].view(N_kpts, 3, self.n_pos).permute(0, 2, 1)\n                )  # [N_kpts, n_pos, 3]\n                offset = offset[:, :, :-1]  # [N_kpts, n_pos, 2]\n                mask_weight = torch.sigmoid(offset[:, :, -1])  # [N_kpts, n_pos]\n            else:\n                offset = (\n                    offset[:, :, 0, 0].view(N_kpts, 2, self.n_pos).permute(0, 2, 1)\n                )  # [N_kpts, n_pos, 2]\n            offsets.append(offset)  # for visualization\n\n            # get sample positions\n            pos = kptsi_wh.unsqueeze(1) + offset  # [N_kpts, n_pos, 2]\n            pos = 2.0 * pos / wh[None] - 1\n            pos = pos.reshape(1, N_kpts * self.n_pos, 1, 2)\n\n            # sample features\n            features = F.grid_sample(\n                xi.unsqueeze(0), pos, mode=\"bilinear\", align_corners=True\n            )  # [1,C,(N_kpts*n_pos),1]\n            features = features.reshape(c, N_kpts, self.n_pos, 1).permute(\n                1, 0, 2, 3\n            )  # [N_kpts, C, n_pos, 1]\n            if self.mask:\n                features = torch.einsum(\"ncpo,np->ncpo\", features, mask_weight)\n\n            features = torch.selu_(self.sf_conv(features)).squeeze(\n                -1\n            )  # [N_kpts, C, n_pos]\n            # convM\n            if not self.conv2D:\n                descs = torch.einsum(\n                    \"ncp,pcd->nd\", features, self.agg_weights\n                )  # [N_kpts, C]\n            else:\n                features = features.reshape(N_kpts, -1)[\n                    :, :, None, None\n                ]  # [N_kpts, C*n_pos, 1, 1]\n                descs = self.convM(features).squeeze()  # [N_kpts, C]\n\n            # normalize\n            descs = F.normalize(descs, p=2.0, dim=1)\n            descriptors.append(descs)\n\n        return descriptors, offsets\n\n\nclass ALIKED(Extractor):\n    default_conf = {\n        \"model_name\": \"aliked-n16\",\n        \"max_num_keypoints\": -1,\n        \"detection_threshold\": 0.2,\n        \"nms_radius\": 2,\n    }\n\n    checkpoint_url = \"https://github.com/Shiaoming/ALIKED/raw/main/models/{}.pth\"\n\n    n_limit_max = 20000\n\n    # c1, c2, c3, c4, dim, K, M\n    cfgs = {\n        \"aliked-t16\": [8, 16, 32, 64, 64, 3, 16],\n        \"aliked-n16\": [16, 32, 64, 128, 128, 3, 16],\n        \"aliked-n16rot\": [16, 32, 64, 128, 128, 3, 16],\n        \"aliked-n32\": [16, 32, 64, 128, 128, 3, 32],\n    }\n    preprocess_conf = {\n        \"resize\": 1024,\n    }\n\n    required_data_keys = [\"image\"]\n\n    def __init__(self, **conf):\n        super().__init__(**conf)  # Update with default configuration.\n        conf = self.conf\n        c1, c2, c3, c4, dim, K, M = self.cfgs[conf.model_name]\n        conv_types = [\"conv\", \"conv\", \"dcn\", \"dcn\"]\n        conv2D = False\n        mask = False\n\n        # build model\n        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)\n        self.pool4 = nn.AvgPool2d(kernel_size=4, stride=4)\n        self.norm = nn.BatchNorm2d\n        self.gate = nn.SELU(inplace=True)\n        self.block1 = ConvBlock(3, c1, self.gate, self.norm, conv_type=conv_types[0])\n        self.block2 = self.get_resblock(c1, c2, conv_types[1], mask)\n        self.block3 = self.get_resblock(c2, c3, conv_types[2], mask)\n        self.block4 = self.get_resblock(c3, c4, conv_types[3], mask)\n\n        self.conv1 = conv1x1(c1, dim // 4)\n        self.conv2 = conv1x1(c2, dim // 4)\n        self.conv3 = conv1x1(c3, dim // 4)\n        self.conv4 = conv1x1(dim, dim // 4)\n        self.upsample2 = nn.Upsample(\n            scale_factor=2, mode=\"bilinear\", align_corners=True\n        )\n        self.upsample4 = nn.Upsample(\n            scale_factor=4, mode=\"bilinear\", align_corners=True\n        )\n        self.upsample8 = nn.Upsample(\n            scale_factor=8, mode=\"bilinear\", align_corners=True\n        )\n        self.upsample32 = nn.Upsample(\n            scale_factor=32, mode=\"bilinear\", align_corners=True\n        )\n        self.score_head = nn.Sequential(\n            conv1x1(dim, 8),\n            self.gate,\n            conv3x3(8, 4),\n            self.gate,\n            conv3x3(4, 4),\n            self.gate,\n            conv3x3(4, 1),\n        )\n        self.desc_head = SDDH(dim, K, M, gate=self.gate, conv2D=conv2D, mask=mask)\n        self.dkd = DKD(\n            radius=conf.nms_radius,\n            top_k=-1 if conf.detection_threshold > 0 else conf.max_num_keypoints,\n            scores_th=conf.detection_threshold,\n            n_limit=(\n                conf.max_num_keypoints\n                if conf.max_num_keypoints > 0\n                else self.n_limit_max\n            ),\n        )\n\n        # vendored: no download at construction; `model.load_components` loads the audited, converted\n        # `aliked-n16.safetensors` strictly\n\n    def get_resblock(self, c_in, c_out, conv_type, mask):\n        return ResBlock(\n            c_in,\n            c_out,\n            1,\n            nn.Conv2d(c_in, c_out, 1),\n            gate=self.gate,\n            norm_layer=self.norm,\n            conv_type=conv_type,\n            mask=mask,\n        )\n\n    def extract_dense_map(self, image):\n        # Pads images such that dimensions are divisible by\n        div_by = 2**5\n        padder = InputPadder(image.shape[-2], image.shape[-1], div_by)\n        image = padder.pad(image)\n\n        # ================================== feature encoder\n        x1 = self.block1(image)  # B x c1 x H x W\n        x2 = self.pool2(x1)\n        x2 = self.block2(x2)  # B x c2 x H/2 x W/2\n        x3 = self.pool4(x2)\n        x3 = self.block3(x3)  # B x c3 x H/8 x W/8\n        x4 = self.pool4(x3)\n        x4 = self.block4(x4)  # B x dim x H/32 x W/32\n        # ================================== feature aggregation\n        x1 = self.gate(self.conv1(x1))  # B x dim//4 x H x W\n        x2 = self.gate(self.conv2(x2))  # B x dim//4 x H//2 x W//2\n        x3 = self.gate(self.conv3(x3))  # B x dim//4 x H//8 x W//8\n        x4 = self.gate(self.conv4(x4))  # B x dim//4 x H//32 x W//32\n        x2_up = self.upsample2(x2)  # B x dim//4 x H x W\n        x3_up = self.upsample8(x3)  # B x dim//4 x H x W\n        x4_up = self.upsample32(x4)  # B x dim//4 x H x W\n        x1234 = torch.cat([x1, x2_up, x3_up, x4_up], dim=1)\n        # ================================== score head\n        score_map = torch.sigmoid(self.score_head(x1234))\n        feature_map = torch.nn.functional.normalize(x1234, p=2, dim=1)\n\n        # Unpads images\n        feature_map = padder.unpad(feature_map)\n        score_map = padder.unpad(score_map)\n\n        return feature_map, score_map\n\n    # vendored: `describe` (kornia-resized re-description of given keypoints) is not carried\n\n    def forward(self, data: dict) -> dict:\n        image = data[\"image\"]\n        if image.shape[1] == 1:\n            image = image.expand(-1, 3, -1, -1)  # vendored: kornia.color.grayscale_to_rgb replaced\n        feature_map, score_map = self.extract_dense_map(image)\n        keypoints, kptscores, scoredispersitys = self.dkd(\n            score_map, image_size=data.get(\"image_size\")\n        )\n        descriptors, offsets = self.desc_head(feature_map, keypoints)\n\n        _, _, h, w = image.shape\n        wh = torch.tensor([w - 1, h - 1], device=image.device)\n        # no padding required\n        # we can set detection_threshold=-1 and conf.max_num_keypoints > 0\n        return {\n            \"keypoints\": wh * (torch.stack(keypoints) + 1) / 2.0,  # B x N x 2\n            \"descriptors\": torch.stack(descriptors),  # B x N x D\n            \"keypoint_scores\": torch.stack(kptscores),  # B x N\n        }\n\n\n# --------------------------------------------------------------------------------------------------\n# lightglue/lightglue.py\n# --------------------------------------------------------------------------------------------------\n\nFlashCrossAttention = None  # vendored: the optional flash-attn import is not carried; torch SDPA is used\n\nif FlashCrossAttention or hasattr(F, \"scaled_dot_product_attention\"):\n    FLASH_AVAILABLE = True\nelse:\n    FLASH_AVAILABLE = False\n\ntorch.backends.cudnn.deterministic = True\n\n\nAMP_CUSTOM_FWD_F32 = (\n    torch.amp.custom_fwd(cast_inputs=torch.float32, device_type=\"cuda\")\n    if hasattr(torch, \"amp\") and hasattr(torch.amp, \"custom_fwd\")\n    else torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)\n)\n\n\n@AMP_CUSTOM_FWD_F32\ndef normalize_keypoints(\n    kpts: torch.Tensor, size: Optional[torch.Tensor] = None\n) -> torch.Tensor:\n    if size is None:\n        size = 1 + kpts.max(-2).values - kpts.min(-2).values\n    elif not isinstance(size, torch.Tensor):\n        size = torch.tensor(size, device=kpts.device, dtype=kpts.dtype)\n    size = size.to(kpts)\n    shift = size / 2\n    scale = size.max(-1).values / 2\n    kpts = (kpts - shift[..., None, :]) / scale[..., None, None]\n    return kpts\n\n\ndef pad_to_length(x: torch.Tensor, length: int) -> Tuple[torch.Tensor]:\n    if length <= x.shape[-2]:\n        return x, torch.ones_like(x[..., :1], dtype=torch.bool)\n    pad = torch.ones(\n        *x.shape[:-2], length - x.shape[-2], x.shape[-1], device=x.device, dtype=x.dtype\n    )\n    y = torch.cat([x, pad], dim=-2)\n    mask = torch.zeros(*y.shape[:-1], 1, dtype=torch.bool, device=x.device)\n    mask[..., : x.shape[-2], :] = True\n    return y, mask\n\n\ndef rotate_half(x: torch.Tensor) -> torch.Tensor:\n    x = x.unflatten(-1, (-1, 2))\n    x1, x2 = x.unbind(dim=-1)\n    return torch.stack((-x2, x1), dim=-1).flatten(start_dim=-2)\n\n\ndef apply_cached_rotary_emb(freqs: torch.Tensor, t: torch.Tensor) -> torch.Tensor:\n    return (t * freqs[0]) + (rotate_half(t) * freqs[1])\n\n\nclass LearnableFourierPositionalEncoding(nn.Module):\n    def __init__(self, M: int, dim: int, F_dim: int = None, gamma: float = 1.0) -> None:\n        super().__init__()\n        F_dim = F_dim if F_dim is not None else dim\n        self.gamma = gamma\n        self.Wr = nn.Linear(M, F_dim // 2, bias=False)\n        nn.init.normal_(self.Wr.weight.data, mean=0, std=self.gamma**-2)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        \"\"\"encode position vector\"\"\"\n        projected = self.Wr(x)\n        cosines, sines = torch.cos(projected), torch.sin(projected)\n        emb = torch.stack([cosines, sines], 0).unsqueeze(-3)\n        return emb.repeat_interleave(2, dim=-1)\n\n\nclass TokenConfidence(nn.Module):\n    def __init__(self, dim: int) -> None:\n        super().__init__()\n        self.token = nn.Sequential(nn.Linear(dim, 1), nn.Sigmoid())\n\n    def forward(self, desc0: torch.Tensor, desc1: torch.Tensor):\n        \"\"\"get confidence tokens\"\"\"\n        return (\n            self.token(desc0.detach()).squeeze(-1),\n            self.token(desc1.detach()).squeeze(-1),\n        )\n\n\nclass Attention(nn.Module):\n    def __init__(self, allow_flash: bool) -> None:\n        super().__init__()\n        if allow_flash and not FLASH_AVAILABLE:\n            warnings.warn(\n                \"FlashAttention is not available. For optimal speed, \"\n                \"consider installing torch >= 2.0 or flash-attn.\",\n                stacklevel=2,\n            )\n        self.enable_flash = allow_flash and FLASH_AVAILABLE\n        self.has_sdp = hasattr(F, \"scaled_dot_product_attention\")\n        if allow_flash and FlashCrossAttention:\n            self.flash_ = FlashCrossAttention()\n        if self.has_sdp:\n            torch.backends.cuda.enable_flash_sdp(allow_flash)\n\n    def forward(self, q, k, v, mask: Optional[torch.Tensor] = None) -> torch.Tensor:\n        if q.shape[-2] == 0 or k.shape[-2] == 0:\n            return q.new_zeros((*q.shape[:-1], v.shape[-1]))\n        if self.enable_flash and q.device.type == \"cuda\":\n            # use torch 2.0 scaled_dot_product_attention with flash\n            if self.has_sdp:\n                args = [x.half().contiguous() for x in [q, k, v]]\n                v = F.scaled_dot_product_attention(*args, attn_mask=mask).to(q.dtype)\n                return v if mask is None else v.nan_to_num()\n            else:\n                assert mask is None\n                q, k, v = [x.transpose(-2, -3).contiguous() for x in [q, k, v]]\n                m = self.flash_(q.half(), torch.stack([k, v], 2).half())\n                return m.transpose(-2, -3).to(q.dtype).clone()\n        elif self.has_sdp:\n            args = [x.contiguous() for x in [q, k, v]]\n            v = F.scaled_dot_product_attention(*args, attn_mask=mask)\n            return v if mask is None else v.nan_to_num()\n        else:\n            s = q.shape[-1] ** -0.5\n            sim = torch.einsum(\"...id,...jd->...ij\", q, k) * s\n            if mask is not None:\n                sim.masked_fill(~mask, -float(\"inf\"))\n            attn = F.softmax(sim, -1)\n            return torch.einsum(\"...ij,...jd->...id\", attn, v)\n\n\nclass SelfBlock(nn.Module):\n    def __init__(\n        self, embed_dim: int, num_heads: int, flash: bool = False, bias: bool = True\n    ) -> None:\n        super().__init__()\n        self.embed_dim = embed_dim\n        self.num_heads = num_heads\n        assert self.embed_dim % num_heads == 0\n        self.head_dim = self.embed_dim // num_heads\n        self.Wqkv = nn.Linear(embed_dim, 3 * embed_dim, bias=bias)\n        self.inner_attn = Attention(flash)\n        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=bias)\n        self.ffn = nn.Sequential(\n            nn.Linear(2 * embed_dim, 2 * embed_dim),\n            nn.LayerNorm(2 * embed_dim, elementwise_affine=True),\n            nn.GELU(),\n            nn.Linear(2 * embed_dim, embed_dim),\n        )\n\n    def forward(\n        self,\n        x: torch.Tensor,\n        encoding: torch.Tensor,\n        mask: Optional[torch.Tensor] = None,\n    ) -> torch.Tensor:\n        qkv = self.Wqkv(x)\n        qkv = qkv.unflatten(-1, (self.num_heads, -1, 3)).transpose(1, 2)\n        q, k, v = qkv[..., 0], qkv[..., 1], qkv[..., 2]\n        q = apply_cached_rotary_emb(encoding, q)\n        k = apply_cached_rotary_emb(encoding, k)\n        context = self.inner_attn(q, k, v, mask=mask)\n        message = self.out_proj(context.transpose(1, 2).flatten(start_dim=-2))\n        return x + self.ffn(torch.cat([x, message], -1))\n\n\nclass CrossBlock(nn.Module):\n    def __init__(\n        self, embed_dim: int, num_heads: int, flash: bool = False, bias: bool = True\n    ) -> None:\n        super().__init__()\n        self.heads = num_heads\n        dim_head = embed_dim // num_heads\n        self.scale = dim_head**-0.5\n        inner_dim = dim_head * num_heads\n        self.to_qk = nn.Linear(embed_dim, inner_dim, bias=bias)\n        self.to_v = nn.Linear(embed_dim, inner_dim, bias=bias)\n        self.to_out = nn.Linear(inner_dim, embed_dim, bias=bias)\n        self.ffn = nn.Sequential(\n            nn.Linear(2 * embed_dim, 2 * embed_dim),\n            nn.LayerNorm(2 * embed_dim, elementwise_affine=True),\n            nn.GELU(),\n            nn.Linear(2 * embed_dim, embed_dim),\n        )\n        if flash and FLASH_AVAILABLE:\n            self.flash = Attention(True)\n        else:\n            self.flash = None\n\n    def map_(self, func: Callable, x0: torch.Tensor, x1: torch.Tensor):\n        return func(x0), func(x1)\n\n    def forward(\n        self, x0: torch.Tensor, x1: torch.Tensor, mask: Optional[torch.Tensor] = None\n    ) -> List[torch.Tensor]:\n        qk0, qk1 = self.map_(self.to_qk, x0, x1)\n        v0, v1 = self.map_(self.to_v, x0, x1)\n        qk0, qk1, v0, v1 = map(\n            lambda t: t.unflatten(-1, (self.heads, -1)).transpose(1, 2),\n            (qk0, qk1, v0, v1),\n        )\n        if self.flash is not None and qk0.device.type == \"cuda\":\n            m0 = self.flash(qk0, qk1, v1, mask)\n            m1 = self.flash(\n                qk1, qk0, v0, mask.transpose(-1, -2) if mask is not None else None\n            )\n        else:\n            qk0, qk1 = qk0 * self.scale**0.5, qk1 * self.scale**0.5\n            sim = torch.einsum(\"bhid, bhjd -> bhij\", qk0, qk1)\n            if mask is not None:\n                sim = sim.masked_fill(~mask, -float(\"inf\"))\n            attn01 = F.softmax(sim, dim=-1)\n            attn10 = F.softmax(sim.transpose(-2, -1).contiguous(), dim=-1)\n            m0 = torch.einsum(\"bhij, bhjd -> bhid\", attn01, v1)\n            m1 = torch.einsum(\"bhji, bhjd -> bhid\", attn10.transpose(-2, -1), v0)\n            if mask is not None:\n                m0, m1 = m0.nan_to_num(), m1.nan_to_num()\n        m0, m1 = self.map_(lambda t: t.transpose(1, 2).flatten(start_dim=-2), m0, m1)\n        m0, m1 = self.map_(self.to_out, m0, m1)\n        x0 = x0 + self.ffn(torch.cat([x0, m0], -1))\n        x1 = x1 + self.ffn(torch.cat([x1, m1], -1))\n        return x0, x1\n\n\nclass TransformerLayer(nn.Module):\n    def __init__(self, *args, **kwargs):\n        super().__init__()\n        self.self_attn = SelfBlock(*args, **kwargs)\n        self.cross_attn = CrossBlock(*args, **kwargs)\n\n    def forward(\n        self,\n        desc0,\n        desc1,\n        encoding0,\n        encoding1,\n        mask0: Optional[torch.Tensor] = None,\n        mask1: Optional[torch.Tensor] = None,\n    ):\n        if mask0 is not None and mask1 is not None:\n            return self.masked_forward(desc0, desc1, encoding0, encoding1, mask0, mask1)\n        else:\n            desc0 = self.self_attn(desc0, encoding0)\n            desc1 = self.self_attn(desc1, encoding1)\n            return self.cross_attn(desc0, desc1)\n\n    # This part is compiled and allows padding inputs\n    def masked_forward(self, desc0, desc1, encoding0, encoding1, mask0, mask1):\n        mask = mask0 & mask1.transpose(-1, -2)\n        mask0 = mask0 & mask0.transpose(-1, -2)\n        mask1 = mask1 & mask1.transpose(-1, -2)\n        desc0 = self.self_attn(desc0, encoding0, mask0)\n        desc1 = self.self_attn(desc1, encoding1, mask1)\n        return self.cross_attn(desc0, desc1, mask)\n\n\ndef sigmoid_log_double_softmax(\n    sim: torch.Tensor, z0: torch.Tensor, z1: torch.Tensor\n) -> torch.Tensor:\n    \"\"\"create the log assignment matrix from logits and similarity\"\"\"\n    b, m, n = sim.shape\n    certainties = F.logsigmoid(z0) + F.logsigmoid(z1).transpose(1, 2)\n    scores0 = F.log_softmax(sim, 2)\n    scores1 = F.log_softmax(sim.transpose(-1, -2).contiguous(), 2).transpose(-1, -2)\n    scores = sim.new_full((b, m + 1, n + 1), 0)\n    scores[:, :m, :n] = scores0 + scores1 + certainties\n    scores[:, :-1, -1] = F.logsigmoid(-z0.squeeze(-1))\n    scores[:, -1, :-1] = F.logsigmoid(-z1.squeeze(-1))\n    return scores\n\n\nclass MatchAssignment(nn.Module):\n    def __init__(self, dim: int) -> None:\n        super().__init__()\n        self.dim = dim\n        self.matchability = nn.Linear(dim, 1, bias=True)\n        self.final_proj = nn.Linear(dim, dim, bias=True)\n\n    def forward(self, desc0: torch.Tensor, desc1: torch.Tensor):\n        \"\"\"build assignment matrix from descriptors\"\"\"\n        mdesc0, mdesc1 = self.final_proj(desc0), self.final_proj(desc1)\n        _, _, d = mdesc0.shape\n        mdesc0, mdesc1 = mdesc0 / d**0.25, mdesc1 / d**0.25\n        sim = torch.einsum(\"bmd,bnd->bmn\", mdesc0, mdesc1)\n        z0 = self.matchability(desc0)\n        z1 = self.matchability(desc1)\n        scores = sigmoid_log_double_softmax(sim, z0, z1)\n        return scores, sim\n\n    def get_matchability(self, desc: torch.Tensor):\n        return torch.sigmoid(self.matchability(desc)).squeeze(-1)\n\n\ndef filter_matches(scores: torch.Tensor, th: float):\n    \"\"\"obtain matches from a log assignment matrix [Bx M+1 x N+1]\"\"\"\n    max0, max1 = scores[:, :-1, :-1].max(2), scores[:, :-1, :-1].max(1)\n    m0, m1 = max0.indices, max1.indices\n    indices0 = torch.arange(m0.shape[1], device=m0.device)[None]\n    indices1 = torch.arange(m1.shape[1], device=m1.device)[None]\n    mutual0 = indices0 == m1.gather(1, m0)\n    mutual1 = indices1 == m0.gather(1, m1)\n    max0_exp = max0.values.exp()\n    zero = max0_exp.new_tensor(0)\n    mscores0 = torch.where(mutual0, max0_exp, zero)\n    mscores1 = torch.where(mutual1, mscores0.gather(1, m1), zero)\n    valid0 = mutual0 & (mscores0 > th)\n    valid1 = mutual1 & valid0.gather(1, m1)\n    m0 = torch.where(valid0, m0, -1)\n    m1 = torch.where(valid1, m1, -1)\n    return m0, m1, mscores0, mscores1\n\n\nclass LightGlue(nn.Module):\n    default_conf = {\n        \"name\": \"lightglue\",  # just for interfacing\n        \"input_dim\": 256,  # input descriptor dimension (autoselected from weights)\n        \"descriptor_dim\": 256,\n        \"add_scale_ori\": False,\n        \"n_layers\": 9,\n        \"num_heads\": 4,\n        \"flash\": True,  # enable FlashAttention if available.\n        \"mp\": False,  # enable mixed precision\n        \"depth_confidence\": 0.95,  # early stopping, disable with -1\n        \"width_confidence\": 0.99,  # point pruning, disable with -1\n        \"filter_threshold\": 0.1,  # match threshold\n        \"weights\": None,\n    }\n\n    # Point pruning involves an overhead (gather).\n    # Therefore, we only activate it if there are enough keypoints.\n    pruning_keypoint_thresholds = {\n        \"cpu\": -1,\n        \"mps\": -1,\n        \"cuda\": 1024,\n        \"flash\": 1536,\n    }\n\n    required_data_keys = [\"image0\", \"image1\"]\n\n    version = \"v0.1_arxiv\"  # the release whose `aliked_lightglue.pth` this repository pins\n\n    features = {\n        \"superpoint\": {\n            \"weights\": \"superpoint_lightglue\",\n            \"input_dim\": 256,\n        },\n        \"disk\": {\n            \"weights\": \"disk_lightglue\",\n            \"input_dim\": 128,\n        },\n        \"aliked\": {\n            \"weights\": \"aliked_lightglue\",\n            \"input_dim\": 128,\n        },\n        \"raco-aliked\": {\n            \"weights\": \"raco_aliked_lightglue\",\n            \"input_dim\": 128,\n        },\n        \"sift\": {\n            \"weights\": \"sift_lightglue\",\n            \"input_dim\": 128,\n            \"add_scale_ori\": True,\n        },\n        \"doghardnet\": {\n            \"weights\": \"doghardnet_lightglue\",\n            \"input_dim\": 128,\n            \"add_scale_ori\": True,\n        },\n    }\n\n    def __init__(self, features=\"superpoint\", **conf) -> None:\n        super().__init__()\n        self.conf = conf = SimpleNamespace(**{**self.default_conf, **conf})\n        if features is not None:\n            if features not in self.features:\n                raise ValueError(\n                    f\"Unsupported features: {features} not in \"\n                    f\"{{{','.join(self.features)}}}\"\n                )\n            for k, v in self.features[features].items():\n                setattr(conf, k, v)\n\n        if conf.input_dim != conf.descriptor_dim:\n            self.input_proj = nn.Linear(conf.input_dim, conf.descriptor_dim, bias=True)\n        else:\n            self.input_proj = nn.Identity()\n\n        head_dim = conf.descriptor_dim // conf.num_heads\n        self.posenc = LearnableFourierPositionalEncoding(\n            2 + 2 * self.conf.add_scale_ori, head_dim, head_dim\n        )\n\n        h, n, d = conf.num_heads, conf.n_layers, conf.descriptor_dim\n\n        self.transformers = nn.ModuleList(\n            [TransformerLayer(d, h, conf.flash) for _ in range(n)]\n        )\n\n        self.log_assignment = nn.ModuleList([MatchAssignment(d) for _ in range(n)])\n        self.token_confidence = nn.ModuleList(\n            [TokenConfidence(d) for _ in range(n - 1)]\n        )\n        self.register_buffer(\n            \"confidence_thresholds\",\n            torch.Tensor(\n                [self.confidence_threshold(i) for i in range(self.conf.n_layers)]\n            ),\n            persistent=False,  # vendored: computed from the configuration, absent from the checkpoint -> strict load\n        )\n\n        # vendored: no download and no torch.load at construction; `model.load_components` loads the\n        # audited, converted `aliked_lightglue.safetensors` strictly (the release checkpoint already uses the\n        # `transformers.{i}.self_attn` key layout, so upstream's rename pass is not needed)\n\n        # static lengths LightGlue is compiled for (only used with torch.compile)\n        self.static_lengths = None\n\n    def compile(\n        self, mode=\"reduce-overhead\", static_lengths=[256, 512, 768, 1024, 1280, 1536]\n    ):\n        if self.conf.width_confidence != -1:\n            warnings.warn(\n                \"Point pruning is partially disabled for compiled forward.\",\n                stacklevel=2,\n            )\n\n        torch._inductor.cudagraph_mark_step_begin()\n        for i in range(self.conf.n_layers):\n            self.transformers[i].masked_forward = torch.compile(\n                self.transformers[i].masked_forward, mode=mode, fullgraph=True\n            )\n\n        self.static_lengths = static_lengths\n\n    def forward(self, data: dict) -> dict:\n        \"\"\"\n        Match keypoints and descriptors between two images\n\n        Input (dict):\n            image0: dict\n                keypoints: [B x M x 2]\n                descriptors: [B x M x D]\n                image: [B x C x H x W] or image_size: [B x 2]\n            image1: dict\n                keypoints: [B x N x 2]\n                descriptors: [B x N x D]\n                image: [B x C x H x W] or image_size: [B x 2]\n        Output (dict):\n            matches0: [B x M]\n            matching_scores0: [B x M]\n            matches1: [B x N]\n            matching_scores1: [B x N]\n            matches: List[[Si x 2]]\n            scores: List[[Si]]\n            stop: int\n            prune0: [B x M]\n            prune1: [B x N]\n        \"\"\"\n        with torch.autocast(enabled=self.conf.mp, device_type=\"cuda\"):\n            return self._forward(data)\n\n    def _forward(self, data: dict) -> dict:\n        for key in self.required_data_keys:\n            assert key in data, f\"Missing key {key} in data\"\n        data0, data1 = data[\"image0\"], data[\"image1\"]\n        kpts0, kpts1 = data0[\"keypoints\"], data1[\"keypoints\"]\n        b, m, _ = kpts0.shape\n        b, n, _ = kpts1.shape\n        device = kpts0.device\n        size0, size1 = data0.get(\"image_size\"), data1.get(\"image_size\")\n        kpts0 = normalize_keypoints(kpts0, size0).clone()\n        kpts1 = normalize_keypoints(kpts1, size1).clone()\n\n        if self.conf.add_scale_ori:\n            kpts0 = torch.cat(\n                [kpts0] + [data0[k].unsqueeze(-1) for k in (\"scales\", \"oris\")], -1\n            )\n            kpts1 = torch.cat(\n                [kpts1] + [data1[k].unsqueeze(-1) for k in (\"scales\", \"oris\")], -1\n            )\n        desc0 = data0[\"descriptors\"].detach().contiguous()\n        desc1 = data1[\"descriptors\"].detach().contiguous()\n\n        assert desc0.shape[-1] == self.conf.input_dim\n        assert desc1.shape[-1] == self.conf.input_dim\n\n        if torch.is_autocast_enabled():\n            desc0 = desc0.half()\n            desc1 = desc1.half()\n\n        mask0, mask1 = None, None\n        c = max(m, n)\n        do_compile = self.static_lengths and c <= max(self.static_lengths)\n        if do_compile:\n            kn = min([k for k in self.static_lengths if k >= c])\n            desc0, mask0 = pad_to_length(desc0, kn)\n            desc1, mask1 = pad_to_length(desc1, kn)\n            kpts0, _ = pad_to_length(kpts0, kn)\n            kpts1, _ = pad_to_length(kpts1, kn)\n        desc0 = self.input_proj(desc0)\n        desc1 = self.input_proj(desc1)\n        # cache positional embeddings\n        encoding0 = self.posenc(kpts0)\n        encoding1 = self.posenc(kpts1)\n\n        # GNN + final_proj + assignment\n        do_early_stop = self.conf.depth_confidence > 0\n        do_point_pruning = self.conf.width_confidence > 0 and not do_compile\n        pruning_th = self.pruning_min_kpts(device)\n        if do_point_pruning:\n            ind0 = torch.arange(0, m, device=device)[None]\n            ind1 = torch.arange(0, n, device=device)[None]\n            # We store the index of the layer at which pruning is detected.\n            prune0 = torch.ones_like(ind0)\n            prune1 = torch.ones_like(ind1)\n        token0, token1 = None, None\n        for i in range(self.conf.n_layers):\n            if desc0.shape[1] == 0 or desc1.shape[1] == 0:  # no keypoints\n                break\n            desc0, desc1 = self.transformers[i](\n                desc0, desc1, encoding0, encoding1, mask0=mask0, mask1=mask1\n            )\n            if i == self.conf.n_layers - 1:\n                continue  # no early stopping or adaptive width at last layer\n\n            if do_early_stop:\n                token0, token1 = self.token_confidence[i](desc0, desc1)\n                if self.check_if_stop(token0[..., :m], token1[..., :n], i, m + n):\n                    break\n            if do_point_pruning and desc0.shape[-2] > pruning_th:\n                scores0 = self.log_assignment[i].get_matchability(desc0)\n                prunemask0 = self.get_pruning_mask(token0, scores0, i)\n                keep0 = torch.where(prunemask0)[1]\n                ind0 = ind0.index_select(1, keep0)\n                desc0 = desc0.index_select(1, keep0)\n                encoding0 = encoding0.index_select(-2, keep0)\n                prune0[:, ind0] += 1\n            if do_point_pruning and desc1.shape[-2] > pruning_th:\n                scores1 = self.log_assignment[i].get_matchability(desc1)\n                prunemask1 = self.get_pruning_mask(token1, scores1, i)\n                keep1 = torch.where(prunemask1)[1]\n                ind1 = ind1.index_select(1, keep1)\n                desc1 = desc1.index_select(1, keep1)\n                encoding1 = encoding1.index_select(-2, keep1)\n                prune1[:, ind1] += 1\n\n        if desc0.shape[1] == 0 or desc1.shape[1] == 0:  # no keypoints\n            m0 = desc0.new_full((b, m), -1, dtype=torch.long)\n            m1 = desc1.new_full((b, n), -1, dtype=torch.long)\n            mscores0 = desc0.new_zeros((b, m))\n            mscores1 = desc1.new_zeros((b, n))\n            matches = desc0.new_empty((b, 0, 2), dtype=torch.long)\n            mscores = desc0.new_empty((b, 0))\n            if not do_point_pruning:\n                prune0 = torch.ones_like(mscores0) * self.conf.n_layers\n                prune1 = torch.ones_like(mscores1) * self.conf.n_layers\n            return {\n                \"matches0\": m0,\n                \"matches1\": m1,\n                \"matching_scores0\": mscores0,\n                \"matching_scores1\": mscores1,\n                \"stop\": i + 1,\n                \"matches\": matches,\n                \"scores\": mscores,\n                \"prune0\": prune0,\n                \"prune1\": prune1,\n            }\n\n        desc0, desc1 = desc0[..., :m, :], desc1[..., :n, :]  # remove padding\n        scores, _ = self.log_assignment[i](desc0, desc1)\n        m0, m1, mscores0, mscores1 = filter_matches(scores, self.conf.filter_threshold)\n        matches, mscores = [], []\n        for k in range(b):\n            valid = m0[k] > -1\n            m_indices_0 = torch.where(valid)[0]\n            m_indices_1 = m0[k][valid]\n            if do_point_pruning:\n                m_indices_0 = ind0[k, m_indices_0]\n                m_indices_1 = ind1[k, m_indices_1]\n            matches.append(torch.stack([m_indices_0, m_indices_1], -1))\n            mscores.append(mscores0[k][valid])\n\n        # TODO: Remove when hloc switches to the compact format.\n        if do_point_pruning:\n            m0_ = torch.full((b, m), -1, device=m0.device, dtype=m0.dtype)\n            m1_ = torch.full((b, n), -1, device=m1.device, dtype=m1.dtype)\n            m0_[:, ind0] = torch.where(m0 == -1, -1, ind1.gather(1, m0.clamp(min=0)))\n            m1_[:, ind1] = torch.where(m1 == -1, -1, ind0.gather(1, m1.clamp(min=0)))\n            mscores0_ = torch.zeros((b, m), device=mscores0.device)\n            mscores1_ = torch.zeros((b, n), device=mscores1.device)\n            mscores0_[:, ind0] = mscores0\n            mscores1_[:, ind1] = mscores1\n            m0, m1, mscores0, mscores1 = m0_, m1_, mscores0_, mscores1_\n        else:\n            prune0 = torch.ones_like(mscores0) * self.conf.n_layers\n            prune1 = torch.ones_like(mscores1) * self.conf.n_layers\n\n        return {\n            \"matches0\": m0,\n            \"matches1\": m1,\n            \"matching_scores0\": mscores0,\n            \"matching_scores1\": mscores1,\n            \"stop\": i + 1,\n            \"matches\": matches,\n            \"scores\": mscores,\n            \"prune0\": prune0,\n            \"prune1\": prune1,\n        }\n\n    def confidence_threshold(self, layer_index: int) -> float:\n        \"\"\"scaled confidence threshold\"\"\"\n        threshold = 0.8 + 0.1 * np.exp(-4.0 * layer_index / self.conf.n_layers)\n        return np.clip(threshold, 0, 1)\n\n    def get_pruning_mask(\n        self, confidences: torch.Tensor, scores: torch.Tensor, layer_index: int\n    ) -> torch.Tensor:\n        \"\"\"mask points which should be removed\"\"\"\n        keep = scores > (1 - self.conf.width_confidence)\n        if confidences is not None:  # Low-confidence points are never pruned.\n            keep |= confidences <= self.confidence_thresholds[layer_index]\n        return keep\n\n    def check_if_stop(\n        self,\n        confidences0: torch.Tensor,\n        confidences1: torch.Tensor,\n        layer_index: int,\n        num_points: int,\n    ) -> torch.Tensor:\n        \"\"\"evaluate stopping condition\"\"\"\n        confidences = torch.cat([confidences0, confidences1], -1)\n        threshold = self.confidence_thresholds[layer_index]\n        ratio_confident = 1.0 - (confidences < threshold).float().sum() / num_points\n        return ratio_confident > self.conf.depth_confidence\n\n    def pruning_min_kpts(self, device: torch.device):\n        if self.conf.flash and FLASH_AVAILABLE and device.type == \"cuda\":\n            return self.pruning_keypoint_thresholds[\"flash\"]\n        else:\n            return self.pruning_keypoint_thresholds[device.type]\n","lightglue_pipeline/pipeline.py":"\"\"\"LightGlue + ALIKED matching pipeline: the inference contract (`extract`, `match`), homography-supervised\nevaluation with three baselines, a bounded adaptation of the matcher's last layers with the LightGlue\nassignment loss, and a digest-manifested safetensors adapter.\"\"\"\n# ruff: noqa: E501  -- docstrings and record literals kept on single lines at the fleet width\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport time\nfrom collections.abc import Callable, Mapping, Sequence\nfrom io import BytesIO\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport torch\nfrom PIL import Image\n\nfrom .config import (\n    DEFAULT_MODEL_KEY,\n    DETECTION_THRESHOLD,\n    EXTRACTOR_FILENAME,\n    EXTRACTOR_SHA256,\n    FILTER_THRESHOLD,\n    MATCHER_FILENAME,\n    MATCHER_PARAMETER_COUNT,\n    MATCHER_SHA256,\n    MAX_KEYPOINTS,\n    MAX_SIDE,\n    MIN_SIDE,\n    MODEL_ID,\n    MODEL_REVISION,\n)\nfrom .metrics import (\n    identity_baseline,\n    matching_metrics,\n    mutual_nn_matches,\n    pair_metrics,\n    patch_neighbour_baseline,\n    warp_points,\n)\nfrom .model import convert_sources, load_components, stage_missing_files, verify_snapshot\n\nImageInput = str | Path | bytes | Image.Image\n\n# --------------------------------------------------------------------------\n# Adaptation contract (E2E): bounded fine-tuning of the matcher's last transformer layers and its final\n# assignment head with the LightGlue assignment loss, supervised by the pairs' exact homographies.\n# --------------------------------------------------------------------------\nMATCHER_LAYERS = 9  # LightGlue transformer layers (self + cross attention each)\nDEFAULT_TRAINABLE_LAYERS = 2  # the last two layers + the final assignment head (2,567,169 params)\nPOSITIVE_PX = 3.0  # a keypoint pair is a ground-truth match under this symmetric reprojection error\nNEGATIVE_PX = 5.0  # a keypoint with no partner under this error is ground-truth unmatched\nMAX_EVAL_RECORDS = 5_000\nMIN_SCORED_RECORDS = 30  # below this a scored set is labelled a small sample\nARTIFACT_FORMAT = \"org.valcorza.lightglue.adapter.v1\"\nARTIFACT_FORMAT_VERSION = \"1.0\"\nARTIFACT_WEIGHTS_NAME = \"adapter.safetensors\"\nARTIFACT_MANIFEST_NAME = \"manifest.json\"\nPARAMETER_COUNT = MATCHER_PARAMETER_COUNT\n\nINPUT_SCHEMA: dict[str, Any] = {\n    \"images\": (\n        \"PIL.Image.Image, raw bytes, or a local path decodable by Pillow; any mode, converted to RGB; \"\n        \"remote URLs are refused\"\n    ),\n    \"image_size\": (\n        f\"sides in [{MIN_SIDE}, {MAX_SIDE}] px; no resizing or cropping (ALIKED pads to a multiple of 32 \"\n        \"internally and reports keypoints in the input frame)\"\n    ),\n    \"keypoints\": {\"max_per_image\": MAX_KEYPOINTS, \"detection_threshold\": DETECTION_THRESHOLD},\n    \"thresholds\": {\"match\": FILTER_THRESHOLD},\n    \"output\": (\n        \"kpts0 / kpts1 (M, 2) float32 pixel coordinates (x, y) of the matched ALIKED keypoints and \"\n        \"confidence (M,) in (0, 1] — the assignment score of each mutual match\"\n    ),\n    \"validation\": (\n        \"size and decodability only. Nothing checks that the two images show the same scene: any two \"\n        \"images are matched, and a pair with no overlap still returns whatever passes the threshold\"\n    ),\n}\n\n\ndef _sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with open(path, \"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1 << 20), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _coerce_image(value: ImageInput) -> Image.Image:\n    if isinstance(value, Image.Image):\n        return value.convert(\"RGB\")\n    if isinstance(value, bytes):\n        image = Image.open(BytesIO(value))\n        image.load()\n        return image.convert(\"RGB\")\n    if isinstance(value, str | Path):\n        text = str(value)\n        if text.lower().startswith((\"http://\", \"https://\")):\n            raise ValueError(\"remote image URLs are not accepted; pass a local path, bytes or a PIL image\")\n        path = Path(text)\n        if not path.is_file():\n            raise ValueError(f\"image file not found: {path}\")\n        image = Image.open(path)\n        image.load()\n        return image.convert(\"RGB\")\n    raise ValueError(\"image must be a local path, bytes or a PIL.Image.Image\")\n\n\ndef _check_size(image: Image.Image, what: str) -> None:\n    if min(image.size) < MIN_SIDE or max(image.size) > MAX_SIDE:\n        raise ValueError(f\"{what}: sides must lie in [{MIN_SIDE}, {MAX_SIDE}] px; got {image.size}\")\n\n\ndef _to_tensor(image: Image.Image) -> torch.Tensor:\n    \"\"\"RGB float tensor (1, 3, H, W) in [0, 1] — the extractor's input, at the image's own size.\"\"\"\n    array = np.asarray(image.convert(\"RGB\"), dtype=np.float32) / 255.0\n    return torch.from_numpy(array).permute(2, 0, 1)[None].contiguous()\n\n\ndef validate_inputs(\n    image0: ImageInput, image1: ImageInput, *, names: Sequence[str] | None = None\n) -> dict[str, Any]:\n    \"\"\"Validation stage: exactly the checks `match` applies, reported as an input manifest before the model runs.\"\"\"\n    findings: list[dict[str, Any]] = []\n    observations = []\n    labels = list(names) if names else [\"image0\", \"image1\"]\n    for label, value in zip(labels, (image0, image1), strict=True):\n        image = _coerce_image(value)\n        _check_size(image, label)\n        width, height = image.size\n        observations.append({\"name\": label, \"size\": [width, height], \"mode\": \"RGB after conversion\"})\n    return {\"schema\": INPUT_SCHEMA, \"images\": observations, \"findings\": findings, \"verdict\": \"accepted\"}\n\n\nclass LightGluePipeline:\n    def __init__(\n        self,\n        extractor: Any,\n        matcher: Any,\n        *,\n        device: str | torch.device = \"cpu\",\n        checkpoint_path: Path | str | None = None,\n        checkpoint_source: str | None = None,\n        manifest_verified: bool = False,\n        weight_sha256: str | None = None,\n        weight_size_bytes: int | None = None,\n        extractor_sha256: str | None = None,\n    ) -> None:\n        self.extractor = extractor\n        self.matcher = matcher\n        self.model = matcher  # the adaptable network, under the fleet's attribute name\n        self.device = torch.device(device)\n        self.checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else None\n        self.checkpoint_source = checkpoint_source\n        self.manifest_verified = manifest_verified\n        self.weight_sha256 = weight_sha256\n        self.weight_size_bytes = weight_size_bytes\n        self.extractor_sha256 = extractor_sha256\n        self.adapter: dict[str, Any] | None = None\n        self.conversion: dict[str, Any] | None = None\n        for module in (self.extractor, self.matcher):\n            if hasattr(module, \"parameters\"):\n                module.eval()  # ALIKED's BatchNorm statistics must never drift: a train-mode extractor updates them on every forward\n                for param in module.parameters():\n                    param.requires_grad_(False)\n\n    @classmethod\n    def from_pretrained(\n        cls,\n        *,\n        device: str | torch.device | None = None,\n        cache_dir: str | Path | None = None,\n        weights_path: str | Path | None = None,\n        weights_dir: str | Path | None = None,\n        allow_download: bool = False,\n        max_keypoints: int = MAX_KEYPOINTS,\n        detection_threshold: float = DETECTION_THRESHOLD,\n        filter_threshold: float = FILTER_THRESHOLD,\n        progress: Callable[[Any], None] | None = None,\n    ) -> LightGluePipeline:\n        \"\"\"Load the two pinned checkpoints into the vendored networks.\n\n        ``weights_dir`` names a fleet snapshot directory holding ``dimer-base-manifest.json``: absent sources\n        are staged with :func:`stage_missing_files` (only when ``allow_download=True``), the directory is\n        verified against the manifest by :func:`verify_snapshot`, the two safetensors files are produced by\n        :func:`convert_sources` when absent (static pickle audit, one weights-only unpickle each, strict load,\n        deterministic save, pinned-digest check), and :func:`load_components` loads them as an explicit path.\n        \"\"\"\n        conversion = None\n        if weights_dir is not None:\n            if weights_path is not None:\n                raise ValueError(\"pass either weights_dir or weights_path, not both\")\n            stage_missing_files(weights_dir, allow_download=allow_download)\n            snapshot = verify_snapshot(weights_dir)\n            if not snapshot.get(\"converted\"):\n                conversion = convert_sources(weights_dir)  # audit + one-time weights-only unpickle\n                if progress is not None:\n                    progress({\"conversion\": conversion})\n                verify_snapshot(weights_dir)\n            weights_path = weights_dir\n        extractor, matcher, target_device, _, metadata = load_components(\n            device=device,\n            cache_dir=cache_dir,\n            weights_path=weights_path,\n            return_metadata=True,\n            max_keypoints=max_keypoints,\n            detection_threshold=detection_threshold,\n            filter_threshold=filter_threshold,\n        )\n        pipe = cls(\n            extractor,\n            matcher,\n            device=target_device,\n            checkpoint_path=metadata.get(\"checkpoint_path\"),\n            checkpoint_source=metadata.get(\"checkpoint_source\"),\n            manifest_verified=metadata.get(\"manifest_verified\", False),\n            weight_sha256=metadata.get(\"weight_sha256\"),\n            weight_size_bytes=metadata.get(\"weight_size_bytes\"),\n            extractor_sha256=metadata.get(\"extractor_sha256\"),\n        )\n        pipe.conversion = conversion\n        return pipe\n\n    # ------------------------------------------------------------------ inference contract\n\n    def _features(self, image: Image.Image) -> dict[str, torch.Tensor]:\n        \"\"\"ALIKED keypoints, descriptors and scores of one image, as batched tensors on the device.\"\"\"\n        tensor = _to_tensor(image).to(self.device)\n        with torch.inference_mode():\n            feats = self.extractor({\"image\": tensor})\n        feats = {k: v.clone() for k, v in feats.items()}  # leave inference mode: tensors may feed autograd\n        feats[\"image_size\"] = torch.tensor([[tensor.shape[-1], tensor.shape[-2]]], dtype=torch.float32, device=self.device)\n        return feats\n\n    def extract(self, image: ImageInput) -> dict[str, Any]:\n        \"\"\"The extractor half of the contract: up to `MAX_KEYPOINTS` ALIKED keypoints of one image with their\n        128-d L2-normalised descriptors and detection scores, in the image's own pixel frame.\"\"\"\n        img = _coerce_image(image)\n        _check_size(img, \"image\")\n        feats = self._features(img)\n        return {\n            \"keypoints\": feats[\"keypoints\"][0].cpu().numpy().astype(np.float32),\n            \"descriptors\": feats[\"descriptors\"][0].cpu().numpy().astype(np.float32),\n            \"scores\": feats[\"keypoint_scores\"][0].cpu().numpy().astype(np.float32),\n            \"size\": [int(img.width), int(img.height)],\n        }\n\n    def match(self, image0: ImageInput, image1: ImageInput) -> dict[str, Any]:\n        \"\"\"Sparse matches between two images: `{kpts0, kpts1, confidence, size0, size1, n_keypoints0,\n        n_keypoints1}` with keypoints as (M, 2) float32 (x, y) pixel coordinates in each image's own frame.\"\"\"\n        img0, img1 = _coerce_image(image0), _coerce_image(image1)\n        _check_size(img0, \"image0\")\n        _check_size(img1, \"image1\")\n        feats0, feats1 = self._features(img0), self._features(img1)\n        with torch.inference_mode():\n            out = self.matcher({\"image0\": feats0, \"image1\": feats1})\n        pairs = out[\"matches\"][0]\n        kpts0 = feats0[\"keypoints\"][0][pairs[:, 0]].cpu().numpy().astype(np.float32)\n        kpts1 = feats1[\"keypoints\"][0][pairs[:, 1]].cpu().numpy().astype(np.float32)\n        conf = out[\"scores\"][0].cpu().numpy().astype(np.float32)\n        return {\n            \"kpts0\": kpts0.reshape(-1, 2),\n            \"kpts1\": kpts1.reshape(-1, 2),\n            \"confidence\": conf.reshape(-1),\n            \"size0\": [int(img0.width), int(img0.height)],\n            \"size1\": [int(img1.width), int(img1.height)],\n            \"n_keypoints0\": int(feats0[\"keypoints\"].shape[1]),\n            \"n_keypoints1\": int(feats1[\"keypoints\"].shape[1]),\n            \"layers\": int(out[\"stop\"]),\n        }\n\n    def descriptor_nn_match(self, image0: ImageInput, image1: ImageInput) -> dict[str, Any]:\n        \"\"\"The third baseline: the same ALIKED keypoints and descriptors, paired by mutual nearest neighbour in\n        descriptor space — no LightGlue.\"\"\"\n        img0, img1 = _coerce_image(image0), _coerce_image(image1)\n        _check_size(img0, \"image0\")\n        _check_size(img1, \"image1\")\n        feats0, feats1 = self._features(img0), self._features(img1)\n        pairs, sims = mutual_nn_matches(feats0[\"descriptors\"][0].cpu().numpy(), feats1[\"descriptors\"][0].cpu().numpy())\n        kpts0 = feats0[\"keypoints\"][0].cpu().numpy()[pairs[:, 0]]\n        kpts1 = feats1[\"keypoints\"][0].cpu().numpy()[pairs[:, 1]]\n        return {\n            \"kpts0\": np.asarray(kpts0, dtype=np.float64).reshape(-1, 2),\n            \"kpts1\": np.asarray(kpts1, dtype=np.float64).reshape(-1, 2),\n            \"confidence\": np.asarray(sims, dtype=np.float64),\n            \"size0\": [int(img0.width), int(img0.height)],\n            \"baseline\": \"ALIKED descriptors, mutual nearest neighbour (no learned matcher)\",\n        }\n\n    # ------------------------------------------------------------------ evaluation\n\n    def evaluate(\n        self,\n        records: Sequence[Mapping[str, Any]],\n        *,\n        matcher: Callable[[Image.Image, Image.Image], Mapping[str, Any]] | None = None,\n    ) -> dict[str, Any]:\n        \"\"\"Homography-supervised scoring of a validated pair dataset with `metrics.matching_metrics`; `matcher`\n        substitutes a baseline for the model (same record structure, same scoring).\"\"\"\n        from .samples import validate_dataset\n\n        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)[\"records\"]\n        started = time.perf_counter()\n        rows = []\n        for record in checked:\n            result = (\n                matcher(record[\"image0\"], record[\"image1\"])\n                if matcher is not None\n                else self.match(record[\"image0\"], record[\"image1\"])\n            )\n            size = tuple(result.get(\"size0\", record[\"image0\"].size))\n            row = pair_metrics(result, np.asarray(record[\"homography\"]), (int(size[0]), int(size[1])))\n            row.update({\"id\": record[\"id\"], \"tier\": record[\"tier\"]})\n            rows.append(row)\n        out = matching_metrics(rows)\n        tiers = sorted({r[\"tier\"] for r in rows})\n        out[\"by_tier\"] = {\n            tier: {\n                k: v\n                for k, v in matching_metrics([r for r in rows if r[\"tier\"] == tier]).items()\n                if k != \"definitions\"\n            }\n            for tier in tiers\n        }\n        out.update(\n            {\n                \"per_pair\": [{k: v for k, v in r.items() if k != \"inlier_errors\"} for r in rows],\n                \"verdict\": \"measured\" if len(checked) >= MIN_SCORED_RECORDS else \"measured-small-sample\",\n                \"adapted\": self.adapter is not None,\n                \"matcher\": \"model\" if matcher is None else \"baseline\",\n                \"seconds\": round(time.perf_counter() - started, 3),\n                \"model_id\": MODEL_ID,\n                \"model_revision\": MODEL_REVISION,\n            }\n        )\n        return out\n\n    def evaluate_baselines(self, records: Sequence[Mapping[str, Any]]) -> dict[str, dict[str, Any]]:\n        \"\"\"The three references scored exactly as the model is: two non-neural (identity guess, patch nearest\n        neighbour) and the same keypoints without the learned matcher (descriptor mutual nearest neighbour).\"\"\"\n        out = {}\n        for name, fn in (\n            (\"identity\", identity_baseline),\n            (\"patch_neighbour\", patch_neighbour_baseline),\n            (\"descriptor_nn\", self.descriptor_nn_match),\n        ):\n            result = self.evaluate(records, matcher=fn)\n            result[\"baseline\"] = name\n            out[name] = result\n        return out\n\n    # ------------------------------------------------------------------ adaptation\n\n    def _assignment_forward(self, feats0: Mapping[str, torch.Tensor], feats1: Mapping[str, torch.Tensor]) -> torch.Tensor:\n        \"\"\"The matcher's forward to the final log-assignment matrix with gradients: keypoint normalisation,\n        input projection, positional encoding, all nine transformer layers (no early exit, no pruning) and the\n        last assignment head. Returns (1, M + 1, N + 1) log scores with the dustbin row and column.\"\"\"\n        from .modeling import normalize_keypoints\n\n        m = self.matcher\n        kpts0 = normalize_keypoints(feats0[\"keypoints\"], feats0[\"image_size\"]).clone()\n        kpts1 = normalize_keypoints(feats1[\"keypoints\"], feats1[\"image_size\"]).clone()\n        desc0 = m.input_proj(feats0[\"descriptors\"].detach().contiguous())\n        desc1 = m.input_proj(feats1[\"descriptors\"].detach().contiguous())\n        encoding0, encoding1 = m.posenc(kpts0), m.posenc(kpts1)\n        for layer in m.transformers:\n            desc0, desc1 = layer(desc0, desc1, encoding0, encoding1)\n        scores, _sim = m.log_assignment[-1](desc0, desc1)\n        return scores\n\n    @staticmethod\n    def match_ground_truth(\n        kpts0: np.ndarray,\n        kpts1: np.ndarray,\n        homography: np.ndarray,\n        *,\n        positive_px: float = POSITIVE_PX,\n        negative_px: float = NEGATIVE_PX,\n    ) -> dict[str, np.ndarray]:\n        \"\"\"Ground-truth assignment of two keypoint sets under a homography, the LightGlue training rule: the\n        symmetric reprojection distance (`H · k0` against `k1` and `H⁻¹ · k1` against `k0`, the larger of the\n        two), mutual nearest neighbours under `positive_px` are the positive pairs, keypoints with no partner\n        under `negative_px` are unmatched, and the rest are ignored by the loss.\"\"\"\n        k0 = np.asarray(kpts0, dtype=np.float64).reshape(-1, 2)\n        k1 = np.asarray(kpts1, dtype=np.float64).reshape(-1, 2)\n        m, n = len(k0), len(k1)\n        if m == 0 or n == 0:\n            return {\"positives\": np.zeros((0, 2), dtype=np.int64), \"unmatched0\": np.ones(m, dtype=bool), \"unmatched1\": np.ones(n, dtype=bool)}\n        p01 = warp_points(k0, homography)\n        p10 = warp_points(k1, np.linalg.inv(np.asarray(homography, dtype=np.float64)))\n        d01 = np.linalg.norm(p01[:, None, :] - k1[None, :, :], axis=2)\n        d10 = np.linalg.norm(k0[:, None, :] - p10[None, :, :], axis=2)\n        dist = np.maximum(np.nan_to_num(d01, nan=np.inf), np.nan_to_num(d10, nan=np.inf))\n        nn0 = dist.argmin(axis=1)\n        nn1 = dist.argmin(axis=0)\n        i = np.arange(m)\n        best0 = dist[i, nn0]\n        positive = (nn1[nn0] == i) & (best0 < positive_px)\n        positives = np.stack([i[positive], nn0[positive]], axis=1).astype(np.int64)\n        unmatched0 = dist.min(axis=1) > negative_px\n        unmatched1 = dist.min(axis=0) > negative_px\n        return {\"positives\": positives, \"unmatched0\": unmatched0, \"unmatched1\": unmatched1}\n\n    @staticmethod\n    def assignment_loss(scores: torch.Tensor, ground_truth: Mapping[str, np.ndarray]) -> torch.Tensor:\n        \"\"\"The LightGlue assignment loss on one pair: the negative log-likelihood of the positive pairs in the\n        log-assignment matrix, plus half the mean negative log-likelihood of the dustbin entries of the\n        unmatched keypoints of each image (`log(1 − σ)`), each term averaged over its own set.\"\"\"\n        s = scores[0]\n        zero = s.sum() * 0.0\n        pos = torch.as_tensor(ground_truth[\"positives\"], dtype=torch.long, device=s.device)\n        un0 = torch.as_tensor(ground_truth[\"unmatched0\"], dtype=torch.bool, device=s.device)\n        un1 = torch.as_tensor(ground_truth[\"unmatched1\"], dtype=torch.bool, device=s.device)\n        nll_pos = -s[pos[:, 0], pos[:, 1]].mean() if len(pos) else zero\n        nll_neg0 = -s[:-1, -1][un0].mean() if bool(un0.any()) else zero\n        nll_neg1 = -s[-1, :-1][un1].mean() if bool(un1.any()) else zero\n        return nll_pos + 0.5 * (nll_neg0 + nll_neg1)\n\n    def _trainable_names(self, trainable_layers: int) -> list[str]:\n        if (\n            isinstance(trainable_layers, bool)\n            or not isinstance(trainable_layers, int)\n            or not 1 <= trainable_layers <= MATCHER_LAYERS\n        ):\n            raise ValueError(f\"trainable_layers must be an int in 1..{MATCHER_LAYERS}\")\n        first = MATCHER_LAYERS - trainable_layers\n        prefixes = tuple(f\"transformers.{k}.\" for k in range(first, MATCHER_LAYERS)) + (\n            f\"log_assignment.{MATCHER_LAYERS - 1}.\",\n        )\n        return [name for name, _p in self.matcher.named_parameters() if name.startswith(prefixes)]\n\n    def adapt(\n        self,\n        train: Sequence[Mapping[str, Any]],\n        val: Sequence[Mapping[str, Any]] | None = None,\n        *,\n        epochs: int = 3,\n        lr: float = 1e-4,\n        batch_size: int = 4,\n        trainable_layers: int = DEFAULT_TRAINABLE_LAYERS,\n        seed: int = 0,\n        progress: Callable[[dict[str, Any]], None] | None = None,\n    ) -> dict[str, Any]:\n        \"\"\"Bounded fine-tuning of the matcher on validated pairs.\n\n        Only the last `trainable_layers` transformer layers of LightGlue and its final assignment head\n        (`log_assignment.8`: matchability and final projection) train — 2,567,169 of 11,884,625 parameters by\n        default; ALIKED, the input projection, the positional encoding, the earlier layers and the token\n        confidences stay frozen. ALIKED features of the training pairs are extracted once and cached (the\n        extractor is frozen). Each pair runs the matcher to the final log-assignment matrix and is scored with\n        the LightGlue assignment loss against the homography's ground-truth assignment; `batch_size` pairs are\n        accumulated per AdamW step (pairs have different keypoint counts, so they are not stacked), gradients\n        are clipped at 1.0, the order is seeded, no scheduler. Epoch 0 records the frozen model's validation\n        metrics; the epoch with the highest validation precision at 3 px is kept (ties broken by homography\n        accuracy at 3 px). Transactional: any failure restores the base tensors.\"\"\"\n        from .samples import validate_dataset\n\n        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:\n            raise ValueError(\"epochs must be an int in 1..20\")\n        if not (0.0 < lr <= 1e-3):\n            raise ValueError(\"lr must be in (0, 1e-3]\")\n        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 32:\n            raise ValueError(\"batch_size must be an int in 1..32\")\n        names = self._trainable_names(trainable_layers)\n        train_checked = validate_dataset(train)[\"records\"]\n        val_checked = validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)[\"records\"] if val else []\n        model = self.matcher\n        torch.manual_seed(seed)\n        started = time.perf_counter()\n        wanted = set(names)\n        for name, param in model.named_parameters():\n            param.requires_grad_(name in wanted)\n        params = [p for p in model.parameters() if p.requires_grad]\n        n_trainable = sum(p.numel() for p in params)\n        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)\n        cached = []\n        for record in train_checked:\n            feats0, feats1 = self._features(record[\"image0\"]), self._features(record[\"image1\"])\n            gt = self.match_ground_truth(feats0[\"keypoints\"][0].cpu().numpy(), feats1[\"keypoints\"][0].cpu().numpy(), np.asarray(record[\"homography\"]))\n            cached.append((feats0, feats1, gt))\n        n_positive = int(sum(len(gt[\"positives\"]) for _f0, _f1, gt in cached))\n        feature_seconds = round(time.perf_counter() - started, 2)\n\n        def score_val() -> dict[str, Any] | None:\n            if not val_checked:\n                return None\n            model.eval()\n            result = self.evaluate(val_checked)\n            return {k: result[k] for k in (\"precision_3px\", \"homography_acc_3px\", \"inliers_per_pair\", \"matches_per_pair\", \"n\")}\n\n        def key(entry: dict[str, Any]) -> tuple[float, float]:\n            return (entry[\"val\"][\"precision_3px\"], entry[\"val\"][\"homography_acc_3px\"]) if entry[\"val\"] else (-math.inf, -math.inf)\n\n        history: list[dict[str, Any]] = []\n        entry: dict[str, Any] = {\"epoch\": 0, \"train_loss\": None, \"val\": score_val(), \"note\": \"frozen model\"}\n        history.append(entry)\n        if progress:\n            progress(entry)\n        best_key = key(entry)\n        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}\n        initial_state = {k: v.clone() for k, v in best_state.items()}\n        best_epoch = 0\n        generator = torch.Generator().manual_seed(seed)\n        try:\n            for epoch in range(1, epochs + 1):\n                model.train()\n                self.extractor.eval()  # the features are cached, but any later _features call must not touch BatchNorm statistics\n                order = torch.randperm(len(cached), generator=generator).tolist()\n                losses = []\n                for start in range(0, len(order), batch_size):\n                    optimiser.zero_grad(set_to_none=True)\n                    chunk = order[start : start + batch_size]\n                    total = 0.0\n                    for i in chunk:\n                        feats0, feats1, gt = cached[i]\n                        scores = self._assignment_forward(feats0, feats1)\n                        loss = self.assignment_loss(scores, gt) / len(chunk)\n                        loss.backward()\n                        total += float(loss.detach())\n                    torch.nn.utils.clip_grad_norm_(params, 1.0)\n                    optimiser.step()\n                    losses.append(total)\n                model.eval()\n                entry = {\"epoch\": epoch, \"train_loss\": sum(losses) / len(losses), \"val\": score_val()}\n                history.append(entry)\n                if progress:\n                    progress(entry)\n                if not entry[\"val\"] or key(entry) > best_key:\n                    best_key = key(entry)\n                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}\n                    best_epoch = epoch\n        except BaseException:\n            restore = dict(model.state_dict())\n            restore.update(initial_state)\n            model.load_state_dict(restore, strict=True)\n            model.eval()\n            for param in model.parameters():\n                param.requires_grad_(False)\n            self.adapter = None\n            raise\n        merged = dict(model.state_dict())\n        merged.update(best_state)\n        model.load_state_dict(merged, strict=True)\n        model.eval()\n        for param in model.parameters():\n            param.requires_grad_(False)\n        self.adapter = {\n            \"trainable_layers\": trainable_layers,\n            \"trainable_names\": names,\n            \"n_trainable\": n_trainable,\n            \"n_total\": sum(p.numel() for p in model.parameters()),\n            \"epochs\": epochs,\n            \"best_epoch\": best_epoch,\n            \"selection\": \"highest validation precision at 3 px (ties: homography accuracy at 3 px)\"\n            if val_checked\n            else \"final epoch (no validation split)\",\n            \"lr\": lr,\n            \"batch_size\": batch_size,\n            \"n_train\": len(train_checked),\n            \"n_val\": len(val_checked),\n            \"ground_truth\": {\"positive_px\": POSITIVE_PX, \"negative_px\": NEGATIVE_PX, \"positive_pairs\": n_positive},\n            \"feature_seconds\": feature_seconds,\n            \"seed\": seed,\n            \"history\": history,\n            \"seconds\": round(time.perf_counter() - started, 2),\n        }\n        return dict(self.adapter)\n\n    # ------------------------------------------------------------------ artifacts\n\n    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:\n        \"\"\"Write the adapted matcher tensors as safetensors plus a base manifest.\"\"\"\n        if self.adapter is None:\n            raise ValueError(\"nothing to save: call adapt() first\")\n        from safetensors.torch import save_file\n\n        out = Path(output_dir)\n        out.mkdir(parents=True, exist_ok=True)\n        names = set(self.adapter[\"trainable_names\"])\n        tensors = {k: v.detach().cpu().contiguous() for k, v in self.matcher.state_dict().items() if k in names}\n        weights_path = out / ARTIFACT_WEIGHTS_NAME\n        save_file(tensors, str(weights_path), metadata={\"format\": \"pt\"})\n        manifest = {\n            \"format\": ARTIFACT_FORMAT,\n            \"format_version\": ARTIFACT_FORMAT_VERSION,\n            \"base_model\": {\n                \"id\": MODEL_ID,\n                \"revision\": MODEL_REVISION,\n                \"key\": DEFAULT_MODEL_KEY,\n                \"weight_file\": MATCHER_FILENAME,\n                \"weight_sha256\": MATCHER_SHA256,\n                \"extractor_file\": EXTRACTOR_FILENAME,\n                \"extractor_sha256\": EXTRACTOR_SHA256,\n            },\n            \"adapter\": {k: v for k, v in self.adapter.items() if k not in (\"history\", \"trainable_names\")},\n            \"history\": self.adapter[\"history\"],\n            \"tensors\": sorted(tensors),\n            \"files\": [\n                {\n                    \"path\": ARTIFACT_WEIGHTS_NAME,\n                    \"bytes\": weights_path.stat().st_size,\n                    \"sha256\": _sha256_file(weights_path),\n                }\n            ],\n            \"metadata\": dict(metadata or {}),\n        }\n        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding=\"utf-8\")\n        return out\n\n    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:\n        \"\"\"Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite\n        exactly the tensors it carries.\"\"\"\n        root = Path(artifact_dir)\n        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding=\"utf-8\"))\n        if manifest.get(\"format\") != ARTIFACT_FORMAT:\n            raise ValueError(f\"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}\")\n        if manifest.get(\"format_version\") != ARTIFACT_FORMAT_VERSION:\n            raise ValueError(f\"artifact format_version {manifest.get('format_version')!r} != {ARTIFACT_FORMAT_VERSION!r}\")\n        base = manifest.get(\"base_model\") or {}\n        if (base.get(\"id\"), base.get(\"revision\"), base.get(\"weight_sha256\")) != (MODEL_ID, MODEL_REVISION, MATCHER_SHA256):\n            raise ValueError(\"artifact was adapted from a different base model, revision or weight file\")\n        if base.get(\"weight_file\", MATCHER_FILENAME) != MATCHER_FILENAME or base.get(\"extractor_sha256\", EXTRACTOR_SHA256) != EXTRACTOR_SHA256:\n            raise ValueError(\"artifact was adapted from a different base weight file or extractor\")\n        files = manifest.get(\"files\")\n        if not isinstance(files, list) or len(files) != 1 or files[0].get(\"path\") != ARTIFACT_WEIGHTS_NAME:\n            raise ValueError(f\"artifact manifest must list exactly {ARTIFACT_WEIGHTS_NAME!r}\")\n        weights_path = (root / ARTIFACT_WEIGHTS_NAME).resolve()\n        if weights_path.parent != root.resolve():\n            raise ValueError(\"artifact weight path must resolve inside the artifact directory\")\n        layers = (manifest.get(\"adapter\") or {}).get(\"trainable_layers\")\n        expected = sorted(self._trainable_names(layers))\n        if sorted(manifest.get(\"tensors\") or []) != expected:\n            raise ValueError(\"artifact tensor list does not match its recorded configuration\")\n        entry = files[0]\n        if not weights_path.is_file():\n            raise FileNotFoundError(f\"artifact weights missing: {weights_path}\")\n        if _sha256_file(weights_path) != entry[\"sha256\"] or weights_path.stat().st_size != entry[\"bytes\"]:\n            raise ValueError(f\"{entry['path']}: digest or size mismatch; refusing to load\")\n        from safetensors.torch import load_file\n\n        tensors = load_file(str(weights_path))\n        if sorted(tensors) != expected:\n            raise ValueError(\"artifact tensor names differ from its manifest\")\n        state = self.matcher.state_dict()\n        for key, value in tensors.items():\n            if key not in state or not key.startswith((\"transformers.\", \"log_assignment.\")):\n                raise ValueError(f\"artifact tensor {key} is not an adaptable matcher tensor of the base\")\n            if tuple(value.shape) != tuple(state[key].shape):\n                raise ValueError(f\"artifact tensor {key} has shape {tuple(value.shape)}, base has {tuple(state[key].shape)}\")\n        merged = dict(state)\n        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})\n        self.matcher.load_state_dict(merged, strict=True)\n        self.matcher.eval()\n        self.adapter = {**manifest[\"adapter\"], \"trainable_names\": manifest[\"tensors\"], \"history\": manifest.get(\"history\", [])}\n        return manifest\n\n    @classmethod\n    def from_artifact(\n        cls,\n        artifact_dir: str | Path,\n        *,\n        device: str | torch.device | None = None,\n        weights_dir: str | Path | None = None,\n        allow_download: bool = False,\n    ) -> LightGluePipeline:\n        \"\"\"A fresh pipeline from the pinned base with an adapter overlaid.\"\"\"\n        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)\n        pipe.load_artifact(artifact_dir)\n        return pipe\n\n\ndef load_pipeline(**kwargs: Any) -> LightGluePipeline:\n    return LightGluePipeline.from_pretrained(**kwargs)\n\n\ndef evaluation_report(result: Mapping[str, Any], reference: Mapping[str, Any] | None = None, *, sample_kind: str = \"synthetic\") -> dict[str, Any]:\n    \"\"\"Evaluation stage for the drawn-shape sanity pair: a machine-readable report even when nothing is\n    measurable. With a `reference` homography and image size the report carries the pair metrics with the\n    verdict `sample-sanity`; without it the verdict is `not-measurable`.\"\"\"\n    base: dict[str, Any] = {\n        \"task\": \"sparse image matching (ALIKED keypoints, LightGlue assignment)\",\n        \"score_semantics\": (\n            \"match confidences are LightGlue assignment scores in (0, 1] (mutual matches over the threshold), \"\n            \"not calibrated probabilities that a match is correct; the decision rule is the upstream match \"\n            \"threshold; no geometric verification ships\"\n        ),\n        \"sample_kind\": sample_kind,\n        \"n_matches\": int(len(np.asarray(result.get(\"kpts0\", [])).reshape(-1, 2))),\n    }\n    if not reference:\n        return {**base, \"metrics\": [], \"verdict\": \"not-measurable\"}\n    row = pair_metrics(result, np.asarray(reference[\"homography\"]), tuple(reference[\"size\"]))\n    metrics = [\n        {\"id\": \"precision_3px\", \"value\": row[\"precision_3px\"], \"definition\": \"fraction of matches under 3 px reprojection error\"},\n        {\"id\": \"n_inliers\", \"value\": row[\"n_inliers\"], \"definition\": \"matches under 3 px\"},\n        {\"id\": \"corner_error_px\", \"value\": row[\"corner_error_px\"], \"definition\": \"mean corner displacement of the RANSAC-DLT homography vs the reference\"},\n    ]\n    return {**base, \"metrics\": metrics, \"verdict\": \"sample-sanity\"}\n\n\n__all__ = [\n    \"ARTIFACT_FORMAT\",\n    \"DEFAULT_TRAINABLE_LAYERS\",\n    \"INPUT_SCHEMA\",\n    \"LightGluePipeline\",\n    \"MATCHER_LAYERS\",\n    \"MAX_EVAL_RECORDS\",\n    \"MIN_SCORED_RECORDS\",\n    \"NEGATIVE_PX\",\n    \"PARAMETER_COUNT\",\n    \"POSITIVE_PX\",\n    \"evaluation_report\",\n    \"load_pipeline\",\n    \"validate_inputs\",\n]\n","lightglue_pipeline/provenance.py":"from __future__ import annotations\n\nimport json\nimport platform\nimport sys\nfrom importlib.metadata import PackageNotFoundError, version\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .config import (\n    DEPTH_CONFIDENCE,\n    DETECTION_THRESHOLD,\n    EXTRACTOR_COMMIT,\n    EXTRACTOR_FILENAME,\n    EXTRACTOR_REPOSITORY,\n    EXTRACTOR_SHA256,\n    EXTRACTOR_SIZE_BYTES,\n    FILTER_THRESHOLD,\n    MATCHER_FILENAME,\n    MATCHER_SHA256,\n    MATCHER_SIZE_BYTES,\n    MAX_KEYPOINTS,\n    MAX_SIDE,\n    MIN_SIDE,\n    MODEL_ID,\n    MODEL_REVISION,\n    MODEL_REVISION_KIND,\n    NMS_RADIUS,\n    WIDTH_CONFIDENCE,\n)\nfrom .modeling import UPSTREAM_COMMIT, UPSTREAM_REPOSITORY\n\n_RUNTIME_PACKAGES = (\n    \"numpy\",\n    \"pillow\",\n    \"safetensors\",\n    \"torch\",\n    \"torchvision\",\n)\n\n\ndef _package_version(name: str) -> str | None:\n    try:\n        return version(name)\n    except PackageNotFoundError:\n        return None\n\n\ndef build_provenance(\n    *,\n    pipeline: Any | None = None,\n    checkpoint_path: str | Path | None = None,\n    include_runtime: bool = True,\n) -> dict[str, Any]:\n    checkpoint_source = None\n    manifest_verified = False\n    weight_sha256 = MATCHER_SHA256\n    weight_size = MATCHER_SIZE_BYTES\n    extractor_sha256 = EXTRACTOR_SHA256\n    device_str = None\n    resolved_checkpoint_path = None\n    adapter = None\n\n    if pipeline is not None:\n        if getattr(pipeline, \"checkpoint_path\", None) is not None:\n            resolved_checkpoint_path = str(pipeline.checkpoint_path)\n        checkpoint_source = getattr(pipeline, \"checkpoint_source\", None)\n        manifest_verified = bool(getattr(pipeline, \"manifest_verified\", False))\n        if getattr(pipeline, \"weight_sha256\", None):\n            weight_sha256 = pipeline.weight_sha256\n        if getattr(pipeline, \"weight_size_bytes\", None):\n            weight_size = pipeline.weight_size_bytes\n        if getattr(pipeline, \"extractor_sha256\", None):\n            extractor_sha256 = pipeline.extractor_sha256\n        if getattr(pipeline, \"device\", None) is not None:\n            device_str = str(pipeline.device)\n        if getattr(pipeline, \"adapter\", None):\n            skip = (\"history\", \"trainable_names\")\n            adapter = {k: v for k, v in pipeline.adapter.items() if k not in skip}\n    if checkpoint_path is not None:\n        resolved_checkpoint_path = str(checkpoint_path)\n\n    provenance: dict[str, Any] = {\n        \"schema_version\": 1,\n        \"model\": {\n            \"id\": MODEL_ID,\n            \"revision\": MODEL_REVISION,\n            \"revision_kind\": MODEL_REVISION_KIND,\n            \"weight_file\": MATCHER_FILENAME,\n            \"weight_sha256\": weight_sha256,\n            \"weight_size_bytes\": weight_size,\n            \"weight_format\": \"safetensors converted once from the audited release pickle\",\n            \"extractor\": {\n                \"repository\": EXTRACTOR_REPOSITORY,\n                \"commit\": EXTRACTOR_COMMIT,\n                \"weight_file\": EXTRACTOR_FILENAME,\n                \"weight_sha256\": extractor_sha256,\n                \"weight_size_bytes\": EXTRACTOR_SIZE_BYTES,\n            },\n            \"checkpoint_source\": checkpoint_source,\n            \"checkpoint_path\": resolved_checkpoint_path,\n            \"manifest_verified\": manifest_verified,\n            \"vendored_code\": {\"repository\": UPSTREAM_REPOSITORY, \"commit\": UPSTREAM_COMMIT},\n        },\n        \"preprocessing\": {\n            \"rgb\": True,\n            \"resize\": \"none (the sample pairs are built at 640 px on the long side)\",\n            \"side_range_px\": [MIN_SIDE, MAX_SIDE],\n            \"padding\": \"inside ALIKED to a multiple of 32; keypoints reported in the input frame\",\n        },\n        \"inference\": {\n            \"max_keypoints\": MAX_KEYPOINTS,\n            \"detection_threshold\": DETECTION_THRESHOLD,\n            \"nms_radius\": NMS_RADIUS,\n            \"match_threshold\": FILTER_THRESHOLD,\n            \"depth_confidence\": DEPTH_CONFIDENCE,\n            \"width_confidence\": WIDTH_CONFIDENCE,\n            \"match_confidence_semantics\": \"assignment_scores_not_calibrated_probability\",\n            \"geometric_verification\": (\n                \"none in the pipeline; the evaluation's RANSAC-DLT is a metric, not a filter\"\n            ),\n            \"device\": device_str,\n        },\n        \"adapter\": adapter,\n    }\n    if include_runtime:\n        provenance[\"runtime\"] = {\n            \"python\": platform.python_version(),\n            \"implementation\": platform.python_implementation(),\n            \"platform\": sys.platform,\n            \"packages\": {name: _package_version(name) for name in _RUNTIME_PACKAGES},\n        }\n    return provenance\n\n\ndef write_provenance(\n    path: str | Path,\n    *,\n    pipeline: Any | None = None,\n    checkpoint_path: str | Path | None = None,\n    include_runtime: bool = True,\n) -> Path:\n    record = build_provenance(\n        pipeline=pipeline, checkpoint_path=checkpoint_path, include_runtime=include_runtime\n    )\n    out = Path(path)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    out.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding=\"utf-8\")\n    return out\n","lightglue_pipeline/samples.py":"\"\"\"Image-pair dataset contract for adapting the matcher: the pinned iNaturalist photograph corpus, seeded\nhomography pairs with exact references, validation, splitting, BYOD loaders and CSV export.\n\nThe images are **real**: 360 CC0-licensed, research-grade iNaturalist photographs of six North American bird\nspecies (the fleet's SigLIP sample; 60 per species, one per observer), pinned here by photo id, byte size and\nSHA-256 of the served `medium` JPEG and fetched from the iNaturalist open-data bucket at run time, refused on any\nbyte-size or SHA-256 mismatch; the repository redistributes none of them, and every record keeps its\nobservation page and observer login. Each photograph becomes one **pair**: the photograph (long side scaled to\n640 px, sides cropped to multiples of 8 — the same photographs and split the fleet's XoFTR row uses) and a copy warped by a seeded homography with\nseeded photometric changes, so the reference `H` (image0 → image1) is exact and every returned match has a\nreprojection error. Two difficulty tiers alternate per photograph: `easy` (corner jitter up to 6 % of the side,\nrotation ±10°, scale 0.9–1.1, mild brightness / contrast / noise) and `hard` (jitter up to 18 %, **rotation ±150°**,\nscale 0.6–1.4, strong brightness / contrast / gamma / blur / noise). The hard tier is this row's own: ALIKED\ndescriptors and LightGlue's positional encoding are not rotation-invariant, and the XoFTR row's ±35° left the\nfrozen matcher with nothing for an adaptation to move.\n\nA record is ``{id, image0, image1, homography}`` (PIL images or paths, and a 3 × 3 list); `SPECIES` maps the\nphotograph's species key to its names for provenance.\n\"\"\"\n# ruff: noqa: E501  -- fleet dataset module written at the 110-column fleet width; this repo lints at 100\n\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport random\nimport re\nimport urllib.request\nimport zipfile\nfrom collections.abc import Mapping, Sequence\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nfrom PIL import Image, ImageEnhance, ImageFilter\n\nfrom .config import DIVISIBLE_BY, MAX_SIDE, MIN_SIDE, MODEL_ID\n\nMAX_IMAGE_SIDE = 4096  # pixels; larger uploads are rejected before any decode-to-tensor work\nWORKING_LONG_SIDE = 640  # the pairs are built at the checkpoint's training resolution\n\nCORPUS_NAME = \"iNaturalist CC0 bird photographs (six species)\"\nCORPUS_RELEASE = (\n    \"iNaturalist open-data bucket, research-grade CC0 photos selected 2026-09-19 (60 per species)\"\n)\nCORPUS_BASE_URL = \"https://inaturalist-open-data.s3.amazonaws.com/photos/\"\nCORPUS_LICENSE = (\n    \"CC0 1.0 (each photo's own license_code on iNaturalist; observers credited in the records)\"\n)\nCORPUS_BYTES = 39_223_447\nDEFAULT_CACHE_DIR = Path(\"weights\") / \"inat-birds\"\nSPECIES: dict[str, tuple[str, str]] = {\n    \"song_sparrow\": (\"Melospiza melodia\", \"Song Sparrow\"),\n    \"chipping_sparrow\": (\"Spizella passerina\", \"Chipping Sparrow\"),\n    \"white_throated_sparrow\": (\"Zonotrichia albicollis\", \"White-throated Sparrow\"),\n    \"dark_eyed_junco\": (\"Junco hyemalis\", \"Dark-eyed Junco\"),\n    \"house_finch\": (\"Haemorhous mexicanus\", \"House Finch\"),\n    \"american_goldfinch\": (\"Spinus tristis\", \"American Goldfinch\"),\n}\n# (id, species, iNat photo id, iNat observation id, observer login, bytes, sha256 of the served\n#  <photo id>/medium.<ext>, ext) — the bucket serves each photo under its original extension\n#  (jpg or jpeg); the digest pins the served bytes\nSAMPLE_RECORDS: tuple[tuple[str, str, int, int, str, int, str, str], ...] = (\n    (\n        \"song_sparrow-00\",\n        \"song_sparrow\",\n        129376982,\n        79016324,\n        \"andywilson\",\n        43427,\n        \"7a9d9304a82f202e992655ec5f65477cd3d7c1dce03aa89a214c2daa38f9d61d\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-01\",\n        \"song_sparrow\",\n        480991086,\n        267636534,\n        \"lyneisfilm\",\n        162073,\n        \"11f77ff277dd2703c1000f2c787136ff0c3ca7ffad7017056892fe789d65efec\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-02\",\n        \"song_sparrow\",\n        546060381,\n        302980489,\n        \"swpollinators\",\n        27899,\n        \"4e70b9519c6e5f7384a4495b91b45465f2b1599f86491d8f9f9b12635a4046f6\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-03\",\n        \"song_sparrow\",\n        308625896,\n        177450028,\n        \"radrat\",\n        70961,\n        \"1211da4fdb24ae85ef0c6c3e2d03542c430457856aec661fe8f5f2de0027eee5\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-04\",\n        \"song_sparrow\",\n        494793016,\n        275349085,\n        \"k-simpkins\",\n        58410,\n        \"255538cf450197257e86ed3d41dc69fb78e594434e9cb338c6314288c6cff26e\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-05\",\n        \"song_sparrow\",\n        674054489,\n        369029444,\n        \"ben142\",\n        220573,\n        \"1ae24622888d9d449ffd6b5c65cac1b9b14870fd8e12dbed8aa0acc2f5030123\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-06\",\n        \"song_sparrow\",\n        339623726,\n        193339933,\n        \"rawcomposition\",\n        25012,\n        \"d2cde085277a71886a2bf211a1eec26941a73752375708726a8af29aa4995a8b\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-07\",\n        \"song_sparrow\",\n        181658744,\n        107953669,\n        \"gcart043\",\n        98482,\n        \"a662a6abb24f42b256fb6e2d6f02c3e128a1053ef534f454461aa5cadcc03fe4\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-08\",\n        \"song_sparrow\",\n        222768957,\n        130949329,\n        \"davidfbird\",\n        110773,\n        \"0feee62753f409d0aa365e9aa017ddef436ad70a2673dc847feb3b5386af27ff\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-09\",\n        \"song_sparrow\",\n        148994242,\n        90171417,\n        \"glennberry\",\n        102702,\n        \"64333977d24957d723a1d26886004f222d810557617685579f72a6b553e508fa\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-10\",\n        \"song_sparrow\",\n        637315932,\n        349374463,\n        \"sooji\",\n        136572,\n        \"a5cebbc0cc2325d3805c4ac854e103f7ba1c22e3a34fb204f30d58681e865033\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-11\",\n        \"song_sparrow\",\n        471146686,\n        262252507,\n        \"jeanpaulboerekamps\",\n        98601,\n        \"4301f06b52b8dcf1e137567c32412e384edb71cb90e2e35459a0405b2e56529b\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-12\",\n        \"song_sparrow\",\n        640811426,\n        351179648,\n        \"erikschiff\",\n        107695,\n        \"27aecce184a485ee888c0e8101cb4a34899b8a185b62dc064f3dc2ec9902182b\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-13\",\n        \"song_sparrow\",\n        123859010,\n        75689904,\n        \"w_mark_c\",\n        190819,\n        \"6b6057a1c50b83ffeb4d9e34367b3e6a9b355236dbcae9820b509e1484f8fe33\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-14\",\n        \"song_sparrow\",\n        108520869,\n        67204020,\n        \"dugald\",\n        52496,\n        \"e483364889fb95c540db84b5edf5a2400febf7d2eb62a1e49303b13ad329d1b8\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-15\",\n        \"song_sparrow\",\n        63537800,\n        40010230,\n        \"nathanael15\",\n        51441,\n        \"5ae55f868e779a4e8ee34f6ad077b40c41f20aa699d967f373ebd659a89137a8\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-16\",\n        \"song_sparrow\",\n        435251292,\n        244042351,\n        \"carterdorscht\",\n        166662,\n        \"d0cdd9ddcf7202a91ac2c47910a0c230639bce50283e8511fb8311151763233a\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-17\",\n        \"song_sparrow\",\n        120710033,\n        73898230,\n        \"tys_rbg\",\n        126820,\n        \"515b3b32b64e88d401990bfd1d8e2c1281443b5d1e763a09e86d3d3664e1df41\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-18\",\n        \"song_sparrow\",\n        349361283,\n        198243065,\n        \"sean579\",\n        42820,\n        \"5eb0031a8d6066650c66b265fb1724413273e095e4e531054d2817eb75910074\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-19\",\n        \"song_sparrow\",\n        393408927,\n        222067370,\n        \"irenemacaulay_\",\n        101681,\n        \"22e326807de963352b4307790bd40c5506adb0fab1f9844968edaf4bc5665e1c\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-20\",\n        \"song_sparrow\",\n        614258628,\n        337847585,\n        \"joy4birds\",\n        70552,\n        \"aa767aa5a74a76cfd985aba5282e589f670a9ce0a8fcdc6a096c0357992dec67\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-21\",\n        \"song_sparrow\",\n        608802621,\n        335154467,\n        \"jamesadney\",\n        112564,\n        \"6e94bce1b5135f48b0b19b64e21a0c76cff4f9b9a28fcbc2b7f5c8ddda6ef513\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-22\",\n        \"song_sparrow\",\n        634100352,\n        347744524,\n        \"zorthesosen\",\n        214044,\n        \"6a70dba26dfb3e98a244a9ec9badb812ddaa8b3612e7a36f66aba057bb757ecd\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-23\",\n        \"song_sparrow\",\n        50065474,\n        31954532,\n        \"truthseqr\",\n        82521,\n        \"4186fbf344e92038358d4338102aa440098bff5f558ab1d197f72fefbcec0b15\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-24\",\n        \"song_sparrow\",\n        8656044,\n        6803564,\n        \"glmory\",\n        297964,\n        \"e8cab773436ccfaad4699e5112ef4edb516236d413d6dbf034eea7bf9c88c8f5\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-25\",\n        \"song_sparrow\",\n        131051149,\n        79988590,\n        \"funvill\",\n        72572,\n        \"0a694b5bb03aee6eb5a3a6132e855d47172b21b84cc4cc7c64367d1a31de7894\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-26\",\n        \"song_sparrow\",\n        12923156,\n        9491600,\n        \"gambolingquail\",\n        69559,\n        \"6a92bff4fc76820c6f21e15c5f254385dd976a32264911bfc08417a1c2bf053d\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-27\",\n        \"song_sparrow\",\n        12077500,\n        8959545,\n        \"reuvenm\",\n        74010,\n        \"3e3c5e94c839f45610ef3ef7ffaf3575f4bfe365fc94454c30bfdb718ca3c593\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-28\",\n        \"song_sparrow\",\n        132730299,\n        80955309,\n        \"steph123456\",\n        154309,\n        \"5f8854dd231a302c643b22521b49ed3007203d74b8b8413a23abc79ffe0b0270\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-29\",\n        \"song_sparrow\",\n        124840110,\n        76316566,\n        \"terrimewbornagain\",\n        81046,\n        \"2020092b0b67397a78cc2b0df267fb671b3ba18e03fb0e35e664e0784061e3c6\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-30\",\n        \"song_sparrow\",\n        130188079,\n        79482300,\n        \"fake_id\",\n        180462,\n        \"1d9ad7b45536b85d46fa0fd9a29d4a8ae828bc223836796a3eb9df1fe8fe4fc4\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-31\",\n        \"song_sparrow\",\n        136405967,\n        83075831,\n        \"tom_lazar\",\n        178554,\n        \"f86c3c5899d20ac8fb73afb76d45666c610245636d7285a6d517b0081b9d6cc9\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-32\",\n        \"song_sparrow\",\n        118885385,\n        72867352,\n        \"ellyne\",\n        105156,\n        \"aab53fb1e7c12adc697711215778c81983f51af83e20bfffe841b80c1d144673\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-33\",\n        \"song_sparrow\",\n        676467457,\n        370293984,\n        \"sholsenbeck\",\n        59104,\n        \"b7c230ca942cedc1c53f42902e1f70bc2d09497e57d2e9f31dfe72a5f6c84d16\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-34\",\n        \"song_sparrow\",\n        688316087,\n        376461614,\n        \"doublecritch\",\n        109416,\n        \"7f11c69df733f99d6e8663e91d807327f8cc57f404049fdbe95e2fd1ce1066e1\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-35\",\n        \"song_sparrow\",\n        691785499,\n        378254663,\n        \"karuquebec\",\n        144522,\n        \"9862b079bcf496aa8d9ce9e716e65b03a3d9a70a5c2077a223630cf9dcc6d5cc\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-36\",\n        \"song_sparrow\",\n        691561091,\n        378140427,\n        \"vaughnshirey\",\n        87827,\n        \"a2c197f91c0543ec7bd3a9aac4867a34417b2eb2444bf3062060f9f1e3580045\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-37\",\n        \"song_sparrow\",\n        582182748,\n        321880490,\n        \"dougbrown\",\n        44795,\n        \"465ce1214c40efab6afdddfbfd34fc5076e5640d130b0442e8e8ac2731ec8a1f\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-38\",\n        \"song_sparrow\",\n        704199014,\n        384692464,\n        \"enspring\",\n        85269,\n        \"1aa25fd1ed376b9245127432925832cf9253e5d156f4f7d35298eff9a8002043\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-39\",\n        \"song_sparrow\",\n        583754170,\n        322666295,\n        \"rociherrera\",\n        97743,\n        \"5e093fc6d59b13bbd7ee0889613cef115ef7bb14432634d0af753a8124951737\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-40\",\n        \"song_sparrow\",\n        590270305,\n        325961498,\n        \"damienxw\",\n        133375,\n        \"a12a152c94ef9c27e5f6dc557e74bf57f6c9ca3015a5e28ee9c30a27ed0e7e19\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-41\",\n        \"song_sparrow\",\n        423534602,\n        237940479,\n        \"don54\",\n        25229,\n        \"69b0f0dfb05e8219700e3878c9b7e4a2d43d0a2df3905414630913397a565dea\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-42\",\n        \"song_sparrow\",\n        418537114,\n        235351206,\n        \"memoosborne\",\n        95540,\n        \"3432cf5a95939b898989c5a962c66b2d71b92b17195d5e695020a776361af3fa\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-43\",\n        \"song_sparrow\",\n        280724703,\n        162358184,\n        \"shirleymorrison\",\n        43015,\n        \"f64af9b3fc4559f251ba0f1e95bf8889bfa629123586532adeb2b81e4c2871b2\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-44\",\n        \"song_sparrow\",\n        641009256,\n        351278414,\n        \"robinlanark\",\n        49640,\n        \"e4af33fd4a504a876b6e7a111a5024cf4197fc888c90f2039df63e9be13d5f06\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-45\",\n        \"song_sparrow\",\n        531314531,\n        295151228,\n        \"steveplumb\",\n        128710,\n        \"8806861909c0833b29f43b10e9428e104deaccb99d41b42039b76f7da0ae4600\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-46\",\n        \"song_sparrow\",\n        114487462,\n        70425556,\n        \"leahmfulton\",\n        50091,\n        \"d0b3c9408b72514f8485b19f61116b94be002be99368db96e126d625a17bd343\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-47\",\n        \"song_sparrow\",\n        118804811,\n        72823547,\n        \"heibudas\",\n        191643,\n        \"8dbe5a56d5fea83b6c25257aa0f9e552ea971f1e0311f583691d24fc9668556f\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-48\",\n        \"song_sparrow\",\n        118498513,\n        72671334,\n        \"seanwashington1\",\n        80800,\n        \"940762ff238d94815922d06051ca00fe4a025279b1da08921a44733084adcdb6\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-49\",\n        \"song_sparrow\",\n        119696805,\n        73327300,\n        \"rambryum\",\n        68983,\n        \"65f7007d6b90654b77b7f4ec0557ff2d0f310c572368cb3492dd69f9d082e90d\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-50\",\n        \"song_sparrow\",\n        4251416,\n        3670620,\n        \"swells\",\n        49271,\n        \"8c53fa5c02e603227117293720d269c59a13555abd537a83640210805fbd844d\",\n        \"JPG\",\n    ),\n    (\n        \"song_sparrow-51\",\n        \"song_sparrow\",\n        303954368,\n        174929167,\n        \"dianeclark6280\",\n        157478,\n        \"14446920b7c55472de3b172f417fbbeac312090add7356a3c947cfb16911dc95\",\n        \"png\",\n    ),\n    (\n        \"song_sparrow-52\",\n        \"song_sparrow\",\n        392544542,\n        221653008,\n        \"emckenziewhat\",\n        138683,\n        \"7fd2f3ad3c835e541a7c720372d662a8f953f88f4d8223864a03e4b0f70a42d3\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-53\",\n        \"song_sparrow\",\n        270614122,\n        156495271,\n        \"radkins21\",\n        92783,\n        \"7bf8ceb5944de93cbf01aebe4d464b9acb848569c07fbe6e732b701d309f0b25\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-54\",\n        \"song_sparrow\",\n        139634990,\n        84920252,\n        \"raffib128\",\n        56293,\n        \"865cf92a61d89cb3420c9d3cbf585a0724df2e314918b05d705804aee356f1a5\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-55\",\n        \"song_sparrow\",\n        25251350,\n        16733911,\n        \"kemper\",\n        54034,\n        \"99f19fa1a37d23712538e43f6a82339f1fd37defe283de3baacc17719f05857e\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-56\",\n        \"song_sparrow\",\n        139129879,\n        84639025,\n        \"wewantashrubbery\",\n        104736,\n        \"aabbffe279d90ffdf737175cc720bea92d634483b830c90acecf920901419643\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-57\",\n        \"song_sparrow\",\n        128242284,\n        78367372,\n        \"stevestevens\",\n        112226,\n        \"918dfc8094d0a697beb1dcd04499cfd7b0c4b12f9dac4d915d1566ab5690b29b\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-58\",\n        \"song_sparrow\",\n        478733235,\n        266446909,\n        \"saintaardvark\",\n        298253,\n        \"d3c8d3a58b44d28c6af4748e4e4195a9a2af7dc6fe91541429dc44aa3a659d21\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-59\",\n        \"song_sparrow\",\n        623006598,\n        342147975,\n        \"ryman56\",\n        60213,\n        \"73e18df350a69ce00158f03c7ec4fda8672539e0a7292b75f1dfa22dbec7ee30\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-00\",\n        \"chipping_sparrow\",\n        198992636,\n        117809422,\n        \"k-simpkins\",\n        62563,\n        \"cf9f3b0c1863808e21af596b2e609b047ddbc28cb2ed076625e2546425ff0adf\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-01\",\n        \"chipping_sparrow\",\n        248210057,\n        144599194,\n        \"w_mark_c\",\n        193389,\n        \"497d0a0fef81c326bcc87b5d1eb97fe987d559b8f2b71bb60fe422ae34dce150\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-02\",\n        \"chipping_sparrow\",\n        156350853,\n        94266719,\n        \"ellyne\",\n        142332,\n        \"6a60ebac34476a372b4790a87d823432cdda8f72590930b5d18ae166cf4c7ba2\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-03\",\n        \"chipping_sparrow\",\n        16128796,\n        11327134,\n        \"reuvenm\",\n        70532,\n        \"5ad36c9cdd6c92e225a1b8ab3c04d2f65bc4e971d0243958f8ac090b0996115c\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-04\",\n        \"chipping_sparrow\",\n        391300648,\n        220982684,\n        \"carterdorscht\",\n        208567,\n        \"c974a676c3c227c2844ff822d429c784bc87bb41f40d5074a0ebadd4c6785b21\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-05\",\n        \"chipping_sparrow\",\n        339456083,\n        193252042,\n        \"rawcomposition\",\n        46329,\n        \"41c9260ca9107430e3a8090ba01cebf3e3c25f2dad15bea4f77c8e824d9fc496\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-06\",\n        \"chipping_sparrow\",\n        40457652,\n        26076708,\n        \"andywilson\",\n        156894,\n        \"b70edd35e00fe672a39d5f0441ea4e9bb9fac19b0f2415ae13e22160bc6091a6\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-07\",\n        \"chipping_sparrow\",\n        84151914,\n        52921135,\n        \"davidfbird\",\n        145714,\n        \"ea65ce5ed881ded9a8157b5756fa907948f41e0eefb6735b48a1c43993e977f9\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-08\",\n        \"chipping_sparrow\",\n        523674610,\n        291074747,\n        \"rwp84\",\n        47983,\n        \"41eac0a5fb578b089f7524c4d2e6cb38c5508aa81815d6cfc46948c081daa800\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-09\",\n        \"chipping_sparrow\",\n        292695018,\n        168861389,\n        \"tim_kirsten\",\n        64812,\n        \"cbb2a03f5dbd2209d56f1cca8b49f342dfbaf44e51fe1014ead75b2018c6678d\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-10\",\n        \"chipping_sparrow\",\n        92501489,\n        57964052,\n        \"tniernberger\",\n        78380,\n        \"44ddb9923026a97ee77ed362bc944c3d2cbbe4fbd4636000880e81613634e6c5\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-11\",\n        \"chipping_sparrow\",\n        220257388,\n        129611350,\n        \"gcart043\",\n        129377,\n        \"1bdce32e7ac58321dd6beacc084afaf45973469215d66b61a0f7c612da3db361\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-12\",\n        \"chipping_sparrow\",\n        478480946,\n        266314208,\n        \"russnamitz\",\n        61359,\n        \"8200cc2ec779d47be1b5afa261d934910a251a39f5620d3dc705106fb2bf5e0b\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-13\",\n        \"chipping_sparrow\",\n        538480223,\n        298914072,\n        \"hiltonward\",\n        201271,\n        \"2a6479556f14a20a8c9a69ff0c1026fe4deb376944234d1d4429c558042c4346\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-14\",\n        \"chipping_sparrow\",\n        58192408,\n        36778771,\n        \"bradenjudson\",\n        22352,\n        \"7c630d7a5b24d94e677e563a8ecb36f57fb29babe6ac1568f11537f10a5edf75\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-15\",\n        \"chipping_sparrow\",\n        80410971,\n        50636049,\n        \"radrat\",\n        55646,\n        \"0c71ca735502e8c94db81302ecd006428206eb16d75472eb0c706e62e2656667\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-16\",\n        \"chipping_sparrow\",\n        300376697,\n        172991803,\n        \"matthias55\",\n        81986,\n        \"7a9a5cbfb6a7d0581c98c75b5fdeb53a049d60f352eda43a39b1f9c2a3347f3b\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-17\",\n        \"chipping_sparrow\",\n        370917657,\n        209626382,\n        \"craigmartin\",\n        101736,\n        \"feb94d319fe5b12c01e63a80dc8e44c45e15a0bfb6534ac5d2c0641205dfe1d5\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-18\",\n        \"chipping_sparrow\",\n        264119989,\n        152960401,\n        \"laurelthrone\",\n        194107,\n        \"13e7336628f9ad784757dca82557c79eda22fdce561e0b52a10a503e72979b8e\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-19\",\n        \"chipping_sparrow\",\n        220323755,\n        129631264,\n        \"enspring\",\n        85687,\n        \"886b7e6faa39943f0c9754e4aef637e2a2f5ad57fe8e8c7ec08360ea95af45b1\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-20\",\n        \"chipping_sparrow\",\n        210319996,\n        124105091,\n        \"bunnymom20\",\n        188328,\n        \"2f6b6f7ac5ed9d91a361c8fed102484f4fca599ec39c9a2bfe9af1df53b28899\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-21\",\n        \"chipping_sparrow\",\n        117013164,\n        71832078,\n        \"terrimewbornagain\",\n        49527,\n        \"4456adb52cdafdefdaeb6dd8cec5955f93c1b8f40abdbb1182b2f605a0e1d486\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-22\",\n        \"chipping_sparrow\",\n        660951395,\n        362143224,\n        \"mike_cove\",\n        148079,\n        \"41d85eaa47b3a41a97ada54728a84c0bfbfed3bcff92de3b6ba88d4c9aface48\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-23\",\n        \"chipping_sparrow\",\n        691160217,\n        377927837,\n        \"ben142\",\n        235922,\n        \"f8137e51e90f0a1964c64fac1a775beaa9ae96c90a1d9aad4f1c870f143e34fd\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-24\",\n        \"chipping_sparrow\",\n        707633397,\n        386482138,\n        \"leannestacy\",\n        238800,\n        \"3fef93babe26eed119ab3b2363902007c46a58a02f91beb0d3bc650821f0dc28\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-25\",\n        \"chipping_sparrow\",\n        696064427,\n        380468188,\n        \"rdnwoods\",\n        309547,\n        \"b56f8f043112021367cbb02861021ac2ef6d2831359315f49d1445bf92477a07\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-26\",\n        \"chipping_sparrow\",\n        702557361,\n        383835541,\n        \"seanwashington1\",\n        159095,\n        \"2940c235b38f84101504b6f50e59c359d0f13e3dc384cadbd43976d1e5e9de91\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-27\",\n        \"chipping_sparrow\",\n        660337945,\n        361814902,\n        \"kristinpiston\",\n        158651,\n        \"a2272e4168edf260001ace410774fbe874ea6b1f6632b8502efc1bc02ae0f866\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-28\",\n        \"chipping_sparrow\",\n        546647394,\n        303298450,\n        \"kzoebel\",\n        86457,\n        \"c37a4aff4263de570ca82f94de92f82d9f9711922e5880547af836104f1883df\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-29\",\n        \"chipping_sparrow\",\n        713495256,\n        389515472,\n        \"lina47741\",\n        287986,\n        \"52e8a37dca8d985a57fdd0d90205e1d813388d961dbf4280d5cbccca9c6af5d3\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-30\",\n        \"chipping_sparrow\",\n        269951645,\n        156136769,\n        \"conhawn\",\n        106540,\n        \"11b48d8bf97084d9ff57aa69517b61539f03b72525022930192330fcb05a160c\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-31\",\n        \"chipping_sparrow\",\n        535714979,\n        297462154,\n        \"dissectedfrog\",\n        200498,\n        \"cd2f2e1be8c76240a1eeff9535e9a60b8f4fb9d771c7e487d65cde48b7e24cf6\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-32\",\n        \"chipping_sparrow\",\n        557301829,\n        308874987,\n        \"mar_y_sierra_silvestre\",\n        115174,\n        \"ce58e87584df623a86643705b08750228903705ca9b02d93b0206401a24eee9e\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-33\",\n        \"chipping_sparrow\",\n        652401709,\n        357680572,\n        \"dianeclark6280\",\n        46782,\n        \"b77c0eea6899d49f8ab97fd1ad6fc772b6055e5195628fa9f6f989e2d51cedd1\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-34\",\n        \"chipping_sparrow\",\n        657945260,\n        360569392,\n        \"jasonleduc\",\n        36363,\n        \"e3471104742fa68b9eeb149255b0136c53306df23d14f2b2633cea80c9e46eea\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-35\",\n        \"chipping_sparrow\",\n        434553038,\n        243681621,\n        \"stariplativky\",\n        57489,\n        \"e253bd9d70dea9d0438b795f6bebcbc2b26c48b16a90729f0d3155695fa812fd\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-36\",\n        \"chipping_sparrow\",\n        207498409,\n        122551474,\n        \"askalotl\",\n        98446,\n        \"d84e28017be8142c22758e59f14b8c52990e79584a68e6f929123aab4997ac68\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-37\",\n        \"chipping_sparrow\",\n        179570776,\n        106840941,\n        \"artemis224\",\n        138087,\n        \"74cad843089d2aada7a124f9e704ae1c1419b7cfcade7e2ae29e2c4eda0a6d36\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-38\",\n        \"chipping_sparrow\",\n        81287617,\n        51172999,\n        \"thyg\",\n        55659,\n        \"e1f075addb4bd1feea3ab256e37945a464d2e8c77768b7be131d0655382268d7\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-39\",\n        \"chipping_sparrow\",\n        343116928,\n        195090930,\n        \"umamimomma\",\n        52983,\n        \"a8948a0189d19b3d7b8df65271f4844b14bd5118f510d0b9ef458c942ac41ce2\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-40\",\n        \"chipping_sparrow\",\n        480169680,\n        267207764,\n        \"cvharris\",\n        144339,\n        \"d1ab8643c48f2ea823f9b61843016795def21b290c35ce3d6045e4933b5dfa0a\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-41\",\n        \"chipping_sparrow\",\n        352953833,\n        200088501,\n        \"aster-asti\",\n        105786,\n        \"82c35e04e4e22fe35c9b364bce7737fa5226cb5c469c761adfc61ef0956da9db\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-42\",\n        \"chipping_sparrow\",\n        622149696,\n        341721858,\n        \"wilderbombyx\",\n        293652,\n        \"227c39a25941a81e9551b5ccef39f95d320e7daa02c96de247123ba62f7b9824\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-43\",\n        \"chipping_sparrow\",\n        503555630,\n        280377982,\n        \"jtdavis05\",\n        42352,\n        \"24ccb474659505cd785502295f385ec369a36c6248296aff85261ed3475a4372\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-44\",\n        \"chipping_sparrow\",\n        509135437,\n        283351340,\n        \"martyndrabik\",\n        125948,\n        \"1604bf4f5512207c59637a84c78842285c40f76a4cf5335bf6d39afe896ed03c\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-45\",\n        \"chipping_sparrow\",\n        8076820,\n        6402608,\n        \"msieges\",\n        166588,\n        \"6c379f94141a0bcdeb8c9743ea140ded50f8c7864baecd6cef0ee1c724b9fd80\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-46\",\n        \"chipping_sparrow\",\n        131917390,\n        80486493,\n        \"tom_lazar\",\n        137355,\n        \"10ab0a9c5af2ca2203a18e4e3df799dcf129494d7b637a3c46ffdcf0f3b3856c\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-47\",\n        \"chipping_sparrow\",\n        141348323,\n        85865218,\n        \"j-dehoog\",\n        71473,\n        \"d3fae9ca7f515c43c216d749f085231d19ddfd101bd37cc4ffae25fc946a34fb\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-48\",\n        \"chipping_sparrow\",\n        28879424,\n        18850970,\n        \"andy71\",\n        194983,\n        \"ea5af97ddef4f743596c176653c86e457f6bddde65d86f5ab16ddff939574683\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-49\",\n        \"chipping_sparrow\",\n        148831027,\n        90084486,\n        \"ian-wolfe\",\n        181942,\n        \"cfab51f0c0598120f5312b354ae249bafb15124dd74723a8c29cdd7319206e49\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-50\",\n        \"chipping_sparrow\",\n        641370647,\n        351455126,\n        \"hrachski\",\n        130262,\n        \"9bc25019bf6a71d73daac63b810d18039d360887504014525598cffce2035908\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-51\",\n        \"chipping_sparrow\",\n        648268506,\n        355462611,\n        \"potatocthulhu\",\n        193138,\n        \"f526135f7b71e0f9f8d80c17d8e6c1a3cd89ba068333b9104e66d02ced7a7eaa\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-52\",\n        \"chipping_sparrow\",\n        645412478,\n        353758550,\n        \"ctlqh\",\n        154844,\n        \"e34228baa058ddb18c6aacf193c20574dab67d8c1f4d6940eec6e1acb03d993b\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-53\",\n        \"chipping_sparrow\",\n        121777317,\n        74504842,\n        \"hickl\",\n        48576,\n        \"a7f0980bf777cacb9f61855d157b94d895f6dd629485b2a038ce20519c53856c\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-54\",\n        \"chipping_sparrow\",\n        147992761,\n        89606238,\n        \"benkeen\",\n        461732,\n        \"8b6e8f03650c270624ebf82f85dd0b37db3e693157d4d3d7a1787c855846da9d\",\n        \"png\",\n    ),\n    (\n        \"chipping_sparrow-55\",\n        \"chipping_sparrow\",\n        27976676,\n        18310312,\n        \"schoenitz\",\n        66424,\n        \"64a0186d15abaf1537a34bd329b67625f7580b7aa544dded6bc626dd389b72fe\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-56\",\n        \"chipping_sparrow\",\n        190706448,\n        112884695,\n        \"jbeusmans\",\n        109296,\n        \"1777945eeb6f916ec6f9c0dc10c81014e8dc0ba22d87268e7c7dc1173193b1e6\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-57\",\n        \"chipping_sparrow\",\n        184174664,\n        109291439,\n        \"chrismcv\",\n        91566,\n        \"c5d9a1dbaf1fad839680c417c7be7d4a680396e38e7c2aba0ee78fd213f7d32c\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-58\",\n        \"chipping_sparrow\",\n        72765715,\n        45879059,\n        \"henryfrye\",\n        146855,\n        \"f0b957698f5a600e3a6c45471848013f9c74d1602895e40d513e2116a9b7320a\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-59\",\n        \"chipping_sparrow\",\n        74100878,\n        46721618,\n        \"aredmer\",\n        132045,\n        \"ce6f4d4324d684cfc0f40d6fe9c4a2b03a7024080e7448083ae7e3fc0c06ac6e\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-00\",\n        \"white_throated_sparrow\",\n        339621218,\n        193338380,\n        \"rawcomposition\",\n        31152,\n        \"d1c08bfaca721bf0873437455b4cc010c6860d08b4777136c775007b0b9d07b6\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-01\",\n        \"white_throated_sparrow\",\n        166821399,\n        99992799,\n        \"dziakj1\",\n        125954,\n        \"c595a41fbc8948b0d918b59117340dc320e2ba80d29e92bb9dacaaed5b404852\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-02\",\n        \"white_throated_sparrow\",\n        469820434,\n        261505977,\n        \"joy4birds\",\n        111767,\n        \"7ce091492c73c68395667bb45578dfff11457511b0958d1d0d320d1df3e55ceb\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-03\",\n        \"white_throated_sparrow\",\n        260413022,\n        150969515,\n        \"andywilson\",\n        45958,\n        \"fb7c533c92239775da6a5b353ff398f6bdd8f65c13445fc1dd72986af5a46466\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-04\",\n        \"white_throated_sparrow\",\n        99351488,\n        62040646,\n        \"bradenjudson\",\n        23399,\n        \"e103968e2a6c9efb6f0bcb548a6457aef950a4a720838a624b6796b90411538e\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-05\",\n        \"white_throated_sparrow\",\n        628148203,\n        344731686,\n        \"lavenderdame\",\n        106872,\n        \"36379abf3af51d865ba6e6804ba3dd48fc9efd12ecc0b7d97e03fc04a16178a8\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-06\",\n        \"white_throated_sparrow\",\n        104660609,\n        65043951,\n        \"allan7\",\n        42443,\n        \"10791891945e83a0c908c14fea07a257a0f25f51c0e708bee35184213833a635\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-07\",\n        \"white_throated_sparrow\",\n        250718938,\n        145903421,\n        \"stevestevens\",\n        138735,\n        \"bd52e6d2f48c247f72db0fa393ec4fa13af8510c95f0ca9134cbef71caddb6d4\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-08\",\n        \"white_throated_sparrow\",\n        15105971,\n        10793852,\n        \"schylerbrown\",\n        31467,\n        \"6496e7e6d3abf13b1a538f769e3cc402280cf1e9a2a1ebdf81c68dcfd7ed01e2\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-09\",\n        \"white_throated_sparrow\",\n        171460784,\n        102554447,\n        \"w_mark_c\",\n        251287,\n        \"3828c41af9209b408fe0d8ec6541edf35d13fc730b74fd27a82980d4f3771137\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-10\",\n        \"white_throated_sparrow\",\n        244260317,\n        142471526,\n        \"deejay\",\n        75694,\n        \"d67efb1ec61f6700b8a6c6552e2da9cd981e68ae81af16d6f1e0ef17fcaff84f\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-11\",\n        \"white_throated_sparrow\",\n        194732675,\n        115373159,\n        \"wildreturn\",\n        128989,\n        \"7ed4bde480b35734576bb4c5f9d1453e77c1f095380432b43b1b682050255831\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-12\",\n        \"white_throated_sparrow\",\n        341990462,\n        194541980,\n        \"ethologist\",\n        149260,\n        \"3e2710fac082cc347e6cd114d42d39f25ec47b82c25936922232eba916808cdb\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-13\",\n        \"white_throated_sparrow\",\n        267625208,\n        154862375,\n        \"laurelthrone\",\n        148616,\n        \"0cac1bc7053c891ce5ae1c33e3f94b69f1b19b958bd08c686db2ccaa7ba0fbd1\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-14\",\n        \"white_throated_sparrow\",\n        330539586,\n        188793537,\n        \"efalquet\",\n        119214,\n        \"c3db4583124472b28dbfb4c6fd9b3829e451d508a6b0a095eccc28f17e7982b4\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-15\",\n        \"white_throated_sparrow\",\n        193091403,\n        114346779,\n        \"ian-wolfe\",\n        73751,\n        \"e28974c8782fccb00f5ea5420140e01248ead859f151563a95a26bf67acba479\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-16\",\n        \"white_throated_sparrow\",\n        691050773,\n        377873006,\n        \"dinomariobob\",\n        167744,\n        \"cc5e25afa5041ce2d6ea4e0d726793843f3a867f30b8d9ccf55892d6617da887\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-17\",\n        \"white_throated_sparrow\",\n        440986574,\n        246671481,\n        \"suzannehale\",\n        141857,\n        \"4a354c189225de2e7b4e94a6df9cbd3a671dac0c8a7d2d4c75e933f93d8b83bf\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-18\",\n        \"white_throated_sparrow\",\n        113217820,\n        69733707,\n        \"kemper\",\n        97802,\n        \"4de7da3ac53a086f2c343555be5a2b62a683715db9f80636a76673cc380fa7f6\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-19\",\n        \"white_throated_sparrow\",\n        575307217,\n        318327478,\n        \"k-simpkins\",\n        90416,\n        \"af2b4e72a098bd90b920c8a49632e1bbff18b73954d8ac541ca81ed4baf0530b\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-20\",\n        \"white_throated_sparrow\",\n        340420076,\n        193765555,\n        \"don54\",\n        62714,\n        \"7f6eea434fc700166af1d343951d15f3a9c83eff06cfb0518c3c4291019a5556\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-21\",\n        \"white_throated_sparrow\",\n        562344160,\n        311510652,\n        \"rrfc\",\n        45842,\n        \"f7c6394649765db6e3313a03ae013291db1ae20d89d3b8f0bdf6641757f31ed0\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-22\",\n        \"white_throated_sparrow\",\n        74598266,\n        47039878,\n        \"ianrwhyte\",\n        141502,\n        \"b0633999572a4499a80310955ab928a86f4fe08774e669b4c4949cd26254428b\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-23\",\n        \"white_throated_sparrow\",\n        588001234,\n        324825124,\n        \"portablecity\",\n        370187,\n        \"11e3a97f43de5d61266d15028fe9023eef3900cfea2d027bd94ad847ecba9607\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-24\",\n        \"white_throated_sparrow\",\n        694103481,\n        379467387,\n        \"memoosborne\",\n        120656,\n        \"82684f82f62f6e036e206eec315fff8e1ba36d308b667eba06750b75d1fada3a\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-25\",\n        \"white_throated_sparrow\",\n        655749108,\n        359408087,\n        \"russnamitz\",\n        58449,\n        \"2a7d2b4226f1941d0950adc04bfd8fe62345f1138c521d0f5f0adf8bc5838da1\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-26\",\n        \"white_throated_sparrow\",\n        599110899,\n        330395670,\n        \"jd_flores\",\n        121343,\n        \"5eeef4f5533a4b0a0222dcf7be000a2b34f3331f3be2b939703155ba8c8ba3ae\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-27\",\n        \"white_throated_sparrow\",\n        653888173,\n        358443775,\n        \"toknowtheland\",\n        144197,\n        \"20a4933b00967fe4488296b2b6e89c12ecbcca0e40da591bd95927cca46ae2f7\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-28\",\n        \"white_throated_sparrow\",\n        295037357,\n        170132018,\n        \"sturuss\",\n        117077,\n        \"d309f75ad5f4008c906ba7b3d6138aa017f919a0a17a1d5b431fa7ab47d5f2d3\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-29\",\n        \"white_throated_sparrow\",\n        405042885,\n        228270930,\n        \"carterdorscht\",\n        122157,\n        \"f1fb32bcfb78f66f50ccd20f2418832afe72c7a5847bdeb8e4dc19546455cfaf\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-30\",\n        \"white_throated_sparrow\",\n        84971412,\n        53425343,\n        \"davidfbird\",\n        168442,\n        \"95c5363704d17d6076f3c7a4c9f5e41d235edf7d5dc9d6022116774f2fd544e6\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-31\",\n        \"white_throated_sparrow\",\n        476335868,\n        265210970,\n        \"aster-asti\",\n        91852,\n        \"cd300cbc823e8ebc0902b7bd80c99eb323e5900bbabf63303b0d9399624c3e54\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-32\",\n        \"white_throated_sparrow\",\n        481170792,\n        267732071,\n        \"kcthetc1\",\n        46021,\n        \"2194a78339aae99549df56d2d3d7fc91e0bf98a371e7d0da7958b4312fa3b487\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-33\",\n        \"white_throated_sparrow\",\n        623602892,\n        342440773,\n        \"imperialwoodpecker26\",\n        112678,\n        \"66653f47ce6a54d38cc1f54b0a85e965f5c5ee7cce65b02c78c1c185cb586c85\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-34\",\n        \"white_throated_sparrow\",\n        622732944,\n        342010560,\n        \"tom_lazar\",\n        34257,\n        \"584070c4b5231db49087d1507aabec8f51adc4e09dcdda8b93b3a78159d47146\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-35\",\n        \"white_throated_sparrow\",\n        505347090,\n        281327144,\n        \"erikschiff\",\n        85652,\n        \"91d50bda6afc3cd5596bf4d3176999bf473c31fb6680b64445ec1cbeaabea847\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-36\",\n        \"white_throated_sparrow\",\n        7072170,\n        5706496,\n        \"jmutter\",\n        171582,\n        \"de90ea2739e8c65ca89f0e511abaf44219501e18377eb49d9e5c7f10d3a6d5b2\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-37\",\n        \"white_throated_sparrow\",\n        16303076,\n        11431253,\n        \"robw\",\n        61348,\n        \"e7d81cfe027c7b0914c79a0431f0ce1431db1ebe78368a2fe5f31e2aab59c3c7\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-38\",\n        \"white_throated_sparrow\",\n        7149657,\n        5761350,\n        \"bradleysaul\",\n        108304,\n        \"adf755b8393fb6b4db99e788d4112c2c72197c33ecffeec2ca7b3ba5dd9d650e\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-39\",\n        \"white_throated_sparrow\",\n        27513220,\n        17998912,\n        \"mefisher\",\n        49587,\n        \"f1716d49782f70455ec97ecdda634505b6b2afc375105b1d779353e704dda07b\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-40\",\n        \"white_throated_sparrow\",\n        642121459,\n        351843522,\n        \"jeffcherry\",\n        53348,\n        \"c9e82d7e47ca2e51c856f9bd4d8f2965d8f69948f081c08c9fa6ebf44fcde711\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-41\",\n        \"white_throated_sparrow\",\n        642355950,\n        351962427,\n        \"robinlanark\",\n        54504,\n        \"c0da16d5146afac102fa274fffea2e0b203282078ed692f3b7ad4a1340b0d4cd\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-42\",\n        \"white_throated_sparrow\",\n        638199267,\n        349826931,\n        \"wildaboutwildlife\",\n        32981,\n        \"845f7d38d6f35d2e6e9dca61234c9566778fec1e274500c97e47977fcecd232f\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-43\",\n        \"white_throated_sparrow\",\n        119839965,\n        73407944,\n        \"terrimewbornagain\",\n        61951,\n        \"dbfbaaa2f72caa812c026122f41512caef00f8f8fdd21c2307fcdf9d79921757\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-44\",\n        \"white_throated_sparrow\",\n        6236370,\n        5078910,\n        \"ruggedbynature\",\n        126403,\n        \"3d6030593bf57bc5734fd6c0706c1e350bb117a32370a16b194ef4488668f123\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-45\",\n        \"white_throated_sparrow\",\n        32221828,\n        20882923,\n        \"christinan\",\n        132485,\n        \"b92ce41128bb1c5b53a0c487b46ba80eea83b9a443ee9c72f100005e5d80e411\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-46\",\n        \"white_throated_sparrow\",\n        36598789,\n        23643589,\n        \"jessm-c\",\n        45861,\n        \"eda75171590418596e372c6cba058364acb64f1279d250dff6c5df61fa17868b\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-47\",\n        \"white_throated_sparrow\",\n        163198257,\n        98065830,\n        \"ellyne\",\n        109261,\n        \"864fb42a82561684c1ac22ae71bd023ddeb15c6052b7732ebd2570a98b552a1a\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-48\",\n        \"white_throated_sparrow\",\n        162150240,\n        97493574,\n        \"c_burns802\",\n        188464,\n        \"0fb2fb2e04bca82c436315b77a974adc665d2d1fb4e419dd61afa068ab412254\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-49\",\n        \"white_throated_sparrow\",\n        161756778,\n        97282492,\n        \"raffib128\",\n        67237,\n        \"77384a65eb120bdb345c3015cb8c22fcec9c89df49b0e0c00954e3476834da9f\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-50\",\n        \"white_throated_sparrow\",\n        165146429,\n        99107203,\n        \"philippthompson\",\n        84518,\n        \"8582544ca8b3cb1c73174fc42a9d66971fdae0c90cd4739083eb4ef4983eb6e7\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-51\",\n        \"white_throated_sparrow\",\n        177451999,\n        105722646,\n        \"natepow\",\n        44983,\n        \"c81a68c46264d1a5fcae3042e7ab24653c3615665cef6293c31622d5f342d840\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-52\",\n        \"white_throated_sparrow\",\n        67630852,\n        42591255,\n        \"schoenitz\",\n        138616,\n        \"34594c49a4f601b17e49dfc4d4098db7b73ff5a69d3830b773378e4d695ff181\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-53\",\n        \"white_throated_sparrow\",\n        99106006,\n        61898962,\n        \"glennberry\",\n        122576,\n        \"fd8cd1c0b0c1fdd7636b8588c86c6d4f3cddc233e9bd4bccade5e2ed275d0e28\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-54\",\n        \"white_throated_sparrow\",\n        102162606,\n        63634844,\n        \"eug302\",\n        73329,\n        \"692114df14fb8a75fdf38428a196467a4a394e5ded95605fbb48cf64903ccb1b\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-55\",\n        \"white_throated_sparrow\",\n        370403033,\n        209307218,\n        \"kuykenwil\",\n        124312,\n        \"d0700068b06ad0f79eeac331a92e5d633bd76be264c3d8801a9a6eb301193969\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-56\",\n        \"white_throated_sparrow\",\n        500790892,\n        278893388,\n        \"quillipede\",\n        186273,\n        \"0db918123c7f529ca76c04129886eb9aeb947bd3e90cdfd968c4605931c16f85\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-57\",\n        \"white_throated_sparrow\",\n        364738073,\n        205369541,\n        \"sandra1142\",\n        144424,\n        \"4cd383751871426904f4d8ff7526a312ba33a6c894faf7121c6ec0ff6ce0d7b1\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-58\",\n        \"white_throated_sparrow\",\n        374502828,\n        211884837,\n        \"halliefromcali\",\n        62476,\n        \"0073652fddd7378dc936f280c18d7e44c4a7160aadc2f051e7b5e2896b1a476c\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-59\",\n        \"white_throated_sparrow\",\n        109241547,\n        67591707,\n        \"blkillin\",\n        130154,\n        \"e5b8c1caa0f7cc7926d46eac42df582ccb0882b4a0733fc0740b8b4597a4e014\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-00\",\n        \"dark_eyed_junco\",\n        172110799,\n        102901486,\n        \"schylerbrown\",\n        182973,\n        \"185209c7a1111fc626a068e136ec3cfcdff3174d15f1c60af209d7b32fe7bef9\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-01\",\n        \"dark_eyed_junco\",\n        46691943,\n        29901256,\n        \"haida_gwaii\",\n        46823,\n        \"a92dca21e6e58c375fc313f0d1da2c86c11408beb0df8fd96fc60ae035ae80d5\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-02\",\n        \"dark_eyed_junco\",\n        707222551,\n        386266764,\n        \"ben142\",\n        289608,\n        \"7bf320edf4d34a4848f3e9d175cfafa66cc5bab6a1a8f97c41c670263de3ff89\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-03\",\n        \"dark_eyed_junco\",\n        346777340,\n        196961623,\n        \"zacharyfoster\",\n        46712,\n        \"ea8f9f0eebb4d2f86193c705344a8fab09bf634bc2217a0874ee57c4f0f5b4ab\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-04\",\n        \"dark_eyed_junco\",\n        8793471,\n        6892999,\n        \"truthseqr\",\n        45302,\n        \"f8241eab39797c4e097a1432b13658466287d61ed515448457907c17e60cf5e2\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-05\",\n        \"dark_eyed_junco\",\n        192557376,\n        114006980,\n        \"k-simpkins\",\n        172398,\n        \"206f012add8a5d0fa434e07c51f1e7bd73a60c4d299a2202163fc662e1b03479\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-06\",\n        \"dark_eyed_junco\",\n        243303909,\n        141959574,\n        \"andywilson\",\n        71321,\n        \"ee5308b6f93fb40a6d795a7d8ca6f2ef55b4844513f4268ef34827e1c5e4c4a6\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-07\",\n        \"dark_eyed_junco\",\n        274980085,\n        159160633,\n        \"andy71\",\n        51209,\n        \"c54b45ca7fdc635bdb31eb89166c9fce84f0b0f8f0c17331fd5e42b582633c9b\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-08\",\n        \"dark_eyed_junco\",\n        213798376,\n        126031618,\n        \"nathanael15\",\n        57646,\n        \"d6b9e7d2dcc63cdd88b47cce328149aa9e8eb486ee2a5fc0f08b90601f2c7d9b\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-09\",\n        \"dark_eyed_junco\",\n        63482066,\n        39977347,\n        \"chrisleearm\",\n        41870,\n        \"90573e814dbde965c70a934d9702110c40670aa22cc9f80ff8849db877d1fd72\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-10\",\n        \"dark_eyed_junco\",\n        458710472,\n        255914048,\n        \"joy4birds\",\n        90973,\n        \"fd25ecb1bf86896b84e9e4e8329c742011011e6883631f47dbe3744ba7463153\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-11\",\n        \"dark_eyed_junco\",\n        20784016,\n        14046286,\n        \"gambolingquail\",\n        93957,\n        \"a454d6f987153317c9c57c05019b08ab0ebebc46d3cf72356b014231b69a1e9d\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-12\",\n        \"dark_eyed_junco\",\n        332842159,\n        189933284,\n        \"jan-konilu\",\n        110145,\n        \"d18706f536164a72a4ec9bacf47437edd89d6b547f86eefdf10ed0166cebf0dd\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-13\",\n        \"dark_eyed_junco\",\n        513004508,\n        270404136,\n        \"thevertebratepokedex\",\n        141252,\n        \"854f75e7527932d0409e14756725388de9913c0b450536da717e54d683de169f\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-14\",\n        \"dark_eyed_junco\",\n        593499256,\n        327583635,\n        \"orionid\",\n        108583,\n        \"7eab4cea6d46faa878732d82c5c2ece63ecc3e66781ea08d0bd950eee581d16d\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-15\",\n        \"dark_eyed_junco\",\n        256948665,\n        149140989,\n        \"igor322\",\n        86925,\n        \"9339b8cf4aa347b4eb176ef1209dfaa4fcb11e277889cc17de399d6af2c8633a\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-16\",\n        \"dark_eyed_junco\",\n        12533281,\n        9255403,\n        \"braincellsgone\",\n        40857,\n        \"be34c51655f59612858d888b05a4f927da65e1aede850c9a5f1be68fa00bfcbe\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-17\",\n        \"dark_eyed_junco\",\n        12580989,\n        9282523,\n        \"artemis224\",\n        216253,\n        \"15198c123c01fa0f8c03edc086408c8d95ffa553083c380e8ae236dbb7056093\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-18\",\n        \"dark_eyed_junco\",\n        63590391,\n        40041059,\n        \"bobbyblackmore\",\n        82853,\n        \"87746be0bca30fc40ddba739c1a35ed1fdd41882ebfe3561518b69d285f5ae5d\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-19\",\n        \"dark_eyed_junco\",\n        106718005,\n        66230973,\n        \"vicki936\",\n        41475,\n        \"e6903e9a69e987c45edd468ace0bf71adf8252063ff5d39c2130409d8fea68be\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-20\",\n        \"dark_eyed_junco\",\n        611049594,\n        336277421,\n        \"skylar_schell\",\n        15425,\n        \"7e876479febb3d44b1e494a25f778efe4e8bfd323279031cb08f9f6fe95f742e\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-21\",\n        \"dark_eyed_junco\",\n        591147962,\n        326403963,\n        \"toknowtheland\",\n        106373,\n        \"07770f3313dc969802355fa8b3e62a86edb115a64771bd38938e1fab33220c39\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-22\",\n        \"dark_eyed_junco\",\n        459696953,\n        256398190,\n        \"w_mark_c\",\n        262126,\n        \"5025e757c026e02160ff3f0e82f1f4434b888a6c9689e7ac76274c1fcec64f0c\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-23\",\n        \"dark_eyed_junco\",\n        484171775,\n        269292238,\n        \"aschuman\",\n        59459,\n        \"89dd933d12cf21c0e73eb01e96a9279e2e57b36676a72a9c860281d0d7703282\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-24\",\n        \"dark_eyed_junco\",\n        469746672,\n        261469406,\n        \"shannon_j\",\n        74167,\n        \"ec4c7ce15aa1c4eadd3bc5d46d433baf9560d6a0e4593d8d57aab8fa9bec9e18\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-25\",\n        \"dark_eyed_junco\",\n        357382685,\n        202362003,\n        \"dougbrown\",\n        56324,\n        \"70275d8e07192c99e121b67d206de6823b194f043e65b9766f1f9ce772953c78\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-26\",\n        \"dark_eyed_junco\",\n        7660371,\n        6119391,\n        \"jeffreyleeisanaturalist\",\n        108394,\n        \"0f667e9af8e8d3585807da58a2f315c5dbcb7eece5f6862006fb9da68a4bef42\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-27\",\n        \"dark_eyed_junco\",\n        437957476,\n        245430473,\n        \"eug302\",\n        106520,\n        \"394abfbcf192ecf5a76cb2f19dfa2a62070fa3874b084d1468219d6ee5db6cbf\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-28\",\n        \"dark_eyed_junco\",\n        339439778,\n        193239701,\n        \"rawcomposition\",\n        32258,\n        \"73fad57ed975df01c529e77eaca9eab9b00741ea2cfc91358856aefca620a10a\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-29\",\n        \"dark_eyed_junco\",\n        634100291,\n        347744522,\n        \"zorthesosen\",\n        169264,\n        \"932237001fe68e3d6d19c496fd448b8a82f254316211ac6f9d35708a1395ed81\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-30\",\n        \"dark_eyed_junco\",\n        6198359,\n        5055484,\n        \"glmory\",\n        108168,\n        \"d31767f42eaaf7f3527133fffb9a0271a9dbbe66fc036d5c694b605563d09965\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-31\",\n        \"dark_eyed_junco\",\n        6485959,\n        5243663,\n        \"fake_id\",\n        148254,\n        \"ce4dab6c5b37b7ebac07f69d4040d4686fea9fd4b753f91bdf15a3a440820712\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-32\",\n        \"dark_eyed_junco\",\n        246318772,\n        143610209,\n        \"wildreturn\",\n        62090,\n        \"839e57443659659522ed65dfd054a8fc0a6bfe400518d9f9a7519383d33a2c0d\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-33\",\n        \"dark_eyed_junco\",\n        253199424,\n        147174876,\n        \"sean579\",\n        121660,\n        \"6efc17eb4c0b184d64a605a76d3734d9bff5d26e8c8609d2f0b0dd4ccc8b3faa\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-34\",\n        \"dark_eyed_junco\",\n        244315013,\n        142500114,\n        \"enspring\",\n        63514,\n        \"1d5012c2b4504b4347fef36d8764eebd1bf07649a14cf874e6d0b3eba9fb3c4f\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-35\",\n        \"dark_eyed_junco\",\n        256191686,\n        148740024,\n        \"nartb\",\n        194532,\n        \"e88b1449a989fc62c5da1c592e4987fa4db71eadf1ef29c0c5c1159b8f77a298\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-36\",\n        \"dark_eyed_junco\",\n        241013731,\n        140770209,\n        \"nana10\",\n        196479,\n        \"fc773ca1b06e0e3673aac5c2213c66981ede11bf3ad4a0cdef2102c4f1d99843\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-37\",\n        \"dark_eyed_junco\",\n        240712071,\n        140608440,\n        \"saintaardvark\",\n        129995,\n        \"94c4dc30ca31804d6645a6be802cf87ccaffe974f0d3cb554b4fb3e9696ec6a5\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-38\",\n        \"dark_eyed_junco\",\n        242909841,\n        141750179,\n        \"calinsdad\",\n        61773,\n        \"c49c5d22969552de0fc04ff95eefb8a2ffc76df528477b93c2e2c38cdebe8f6c\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-39\",\n        \"dark_eyed_junco\",\n        109385584,\n        67667584,\n        \"ninetoes\",\n        64185,\n        \"64e9ee29cf017c292e1711b4c6482488a74c8f652e88e8b7c1df883dc91d75e2\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-40\",\n        \"dark_eyed_junco\",\n        102027986,\n        63561089,\n        \"michaelnaumoff\",\n        114346,\n        \"eccaa3c60fe575e28fc60f8f6896a99065bff4aee0a4fdc057d7220e7870ca4d\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-41\",\n        \"dark_eyed_junco\",\n        105652410,\n        65622040,\n        \"don54\",\n        159856,\n        \"5c4282e5e85af7a886eceeadc90977bcd11bf26658176ba5ef8182d2f4a2bfe1\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-42\",\n        \"dark_eyed_junco\",\n        267851501,\n        154986181,\n        \"shanebustapbj\",\n        203657,\n        \"217002a366779db0ccc3fddffa71e79472b1fcaa5d444049971dac4c31c910f5\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-43\",\n        \"dark_eyed_junco\",\n        268597047,\n        155389350,\n        \"j-dehoog\",\n        54916,\n        \"c0890844944f3d603f8220b0a75213051766c6a45b2ba1fed51381134832f69f\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-44\",\n        \"dark_eyed_junco\",\n        515881506,\n        286951393,\n        \"jubileej\",\n        108221,\n        \"9cbecae915f1c96dcc31d5fd77cb5902181152cc3b918693191cc11fce18822c\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-45\",\n        \"dark_eyed_junco\",\n        640462423,\n        351001488,\n        \"russnamitz\",\n        44508,\n        \"63276eb90201f890b977a54b95bf1d65592eef98089dbb73f2e27e66aaf49a65\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-46\",\n        \"dark_eyed_junco\",\n        649025513,\n        355899297,\n        \"rocksand2134\",\n        233215,\n        \"2f2545fa5e7901c0383578dea84131eb4be551b8f2bd55d9f308ebc838144f39\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-47\",\n        \"dark_eyed_junco\",\n        625101518,\n        343192310,\n        \"jtdavis05\",\n        106387,\n        \"4642076ebc5e9bcde85dbba65eb16a49d3fabe6c82cc707b4863ad4b7a4531ae\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-48\",\n        \"dark_eyed_junco\",\n        509902416,\n        283764048,\n        \"loarie\",\n        271588,\n        \"0c1e3897caca8ff3ff4f3eff7548e30ca50dfe2c2c972106303f9e21c259dbda\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-49\",\n        \"dark_eyed_junco\",\n        627787858,\n        344552039,\n        \"margohj\",\n        54717,\n        \"a41e557077525b064fa16a7c0b140d608d323663519cfbf424f4454d79951244\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-50\",\n        \"dark_eyed_junco\",\n        630444682,\n        345899876,\n        \"robinlanark\",\n        102578,\n        \"aa4ffd5dbd553d60ba687d57b21b52ce21572d3c711471abbc5e959bd1126da2\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-51\",\n        \"dark_eyed_junco\",\n        635784743,\n        348597723,\n        \"lina47741\",\n        317439,\n        \"392d1114b7fc1bb3b041d9895ceb049432a96951764e8949d5939315b96bcc54\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-52\",\n        \"dark_eyed_junco\",\n        518407567,\n        288278997,\n        \"zzphantom\",\n        231733,\n        \"de2a5be329b583911babe0758c9ce8c0982191f743b8082eb826623256709f71\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-53\",\n        \"dark_eyed_junco\",\n        620711072,\n        341026776,\n        \"ethologist\",\n        186675,\n        \"13f47961b2a1654ec40f6d1a2005afbf34c75060987fbb662001e66e10a023ed\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-54\",\n        \"dark_eyed_junco\",\n        638143588,\n        349800424,\n        \"tom_lazar\",\n        157629,\n        \"0be0b84bbc956c616ce35e9e704e8c9e01f0d24db7eeb157fcaa3b3952bd9b23\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-55\",\n        \"dark_eyed_junco\",\n        668612910,\n        366183670,\n        \"gendereuphorbia\",\n        76372,\n        \"4bfa9f780e9fd0327afec60a8b42e3378b8a9917a0461fd77a6df73b1a94e6bb\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-56\",\n        \"dark_eyed_junco\",\n        116078128,\n        71308227,\n        \"msieges\",\n        47042,\n        \"ecd4ade5abf058ac887fabe2706eb021ada3b9ab083d10728c9b1dc26de56bbc\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-57\",\n        \"dark_eyed_junco\",\n        117580446,\n        72145931,\n        \"hewwowhy\",\n        67000,\n        \"1a8ca3be627c192459440c2eb07bd85b348f0e876a7de32bfc6fe0fb2959a618\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-58\",\n        \"dark_eyed_junco\",\n        117044036,\n        71848977,\n        \"radrat\",\n        81035,\n        \"7362dd0a27d11b18d5a79abcce9a80304dc80dc0ea1fa84265a1cf0638687641\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-59\",\n        \"dark_eyed_junco\",\n        11110817,\n        8359397,\n        \"giselle9\",\n        71741,\n        \"b2e74f27ddca8d1e6dd4807251b07867cbfec8b6bf51d93ef7f142109f705520\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-00\",\n        \"house_finch\",\n        697940852,\n        381438133,\n        \"ben142\",\n        196735,\n        \"a89f8e0263fdabb404b462acaa592f5dd2ac88ee4615da444470de4a1fae82d5\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-01\",\n        \"house_finch\",\n        176982307,\n        105476125,\n        \"vicki936\",\n        22211,\n        \"c377fb361df0324c7a856d9344968886ece3b94bd67188c9325b8d2d284d3a2f\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-02\",\n        \"house_finch\",\n        117990649,\n        72375345,\n        \"kristen163\",\n        75945,\n        \"eeafad0dd2e91ecfe45c9d1f27dd0392a01bd81099549c60fd2e36a4b4342a9f\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-03\",\n        \"house_finch\",\n        389479656,\n        220010434,\n        \"aster-asti\",\n        82128,\n        \"a8848197b4e7890e07538d492480c4275b75d04e10c1ae95aee91aaafe3319c5\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-04\",\n        \"house_finch\",\n        98576538,\n        61594129,\n        \"enspring\",\n        46784,\n        \"8b355426d8fe6327f202c6a9458cee1b95de445bfbf7e48a4b5a2c7d0eb78a8c\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-05\",\n        \"house_finch\",\n        72470599,\n        45698380,\n        \"henrya\",\n        61724,\n        \"1b96d37a7078e1b725b80af4b10848da58b0d0c17a70c8ac01e326c0a749ee6b\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-06\",\n        \"house_finch\",\n        80751781,\n        50842166,\n        \"leahmfulton\",\n        44269,\n        \"6d6202de26f042d83ee6c5af550cd74e6ab10eb796eed2f78c83ac9e2368e4b7\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-07\",\n        \"house_finch\",\n        214612538,\n        126483167,\n        \"hamiltonturner\",\n        124116,\n        \"6b7687640c4641b974865da04cf9eaf1f86b774ebc19678f2fc39e55c8648930\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-08\",\n        \"house_finch\",\n        630196420,\n        345777550,\n        \"truthseqr\",\n        149455,\n        \"abb84d1e327dd82c07cbea3dd5583c07453e69b2cc220397b101e597da81bd6c\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-09\",\n        \"house_finch\",\n        213077180,\n        125637342,\n        \"jnicat\",\n        25823,\n        \"e1e3baff8d0bd72339e3e49089f7f4f2c1dd383ff005e49a964a1bacc4f8ebb8\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-10\",\n        \"house_finch\",\n        500744872,\n        278868417,\n        \"pbaff\",\n        149626,\n        \"2684cfb1fc40d7766610a5920ead0ad0c27c338ccb4fb9ed18baddda150568b0\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-11\",\n        \"house_finch\",\n        268678834,\n        155431721,\n        \"stevestevens\",\n        120325,\n        \"5d1e8c097c214d14ef7b895bf95769ae9bc25fa799a98bbfa6f12f2f84e6af79\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-12\",\n        \"house_finch\",\n        358373136,\n        202884575,\n        \"kcthetc1\",\n        52329,\n        \"b9929163e4fdadef26c753437ac7af3ad550aa047b15cba131c05d4d335799fe\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-13\",\n        \"house_finch\",\n        104227663,\n        64793290,\n        \"verdantpulsar\",\n        101003,\n        \"90fe37c477bad9ed30ab119e1a7445ffc2720e548a22bb65d66b2ff7b82337d5\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-14\",\n        \"house_finch\",\n        196156834,\n        116218899,\n        \"kgarrett\",\n        56801,\n        \"ce03d1d70a89b5e6e0088f4307f8077573c3571b91997725ca2e6c69be80e2d0\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-15\",\n        \"house_finch\",\n        454332148,\n        253709711,\n        \"rlaortiz\",\n        149985,\n        \"cf007ef8ac57bc0eb085c9eecf9fc95eb69a29df706998de237df24f9491c61d\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-16\",\n        \"house_finch\",\n        665287910,\n        295246528,\n        \"dinomariobob\",\n        74972,\n        \"5f6121c1f8dbfaccbb61c279a579c22600744eb4118c2eeef09e69c86f6e1a49\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-17\",\n        \"house_finch\",\n        509969159,\n        283798927,\n        \"damienxw\",\n        104277,\n        \"56d3656be473c362f1ccd09d15e62cbcfe9137d83bef7a3312d72444921c7805\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-18\",\n        \"house_finch\",\n        168402062,\n        100871757,\n        \"k-simpkins\",\n        86922,\n        \"209a884cf7618d0b85679ae3a72f237a6d03a5a3096f6ab1b2e5503c38c50d9d\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-19\",\n        \"house_finch\",\n        661042217,\n        362189851,\n        \"dougbrown\",\n        56426,\n        \"ad704994fda99779aa340ca1c637b1371f0513df6f36973263bb9ee88cf6713b\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-20\",\n        \"house_finch\",\n        701862058,\n        383472210,\n        \"dblanco\",\n        25222,\n        \"8562ed9c4e889b34f83b40c55dd93b821a6b5829a3b1c681374a53c3ca9bf01f\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-21\",\n        \"house_finch\",\n        709783562,\n        387605980,\n        \"fraskar\",\n        99555,\n        \"60a1926358c4ec04aeec9414381177600580c8237899c852e03472e53cf87f31\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-22\",\n        \"house_finch\",\n        703924789,\n        384540425,\n        \"meshhy\",\n        110802,\n        \"2df38d1aaab2c268c7a47f05e55bef1e83690eeeb91f34874c5b479a5ff0a2cb\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-23\",\n        \"house_finch\",\n        705516095,\n        385380385,\n        \"m_aniket\",\n        77797,\n        \"ba009764b66ed89efc9b0aaacd9565830c65039967961488d774ce3c9add5d64\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-24\",\n        \"house_finch\",\n        577527110,\n        319485540,\n        \"glmory\",\n        94407,\n        \"6d4cd80f8ef77f41575befae0d8534e9bc397685dceb7f629b5cbc24447a3613\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-25\",\n        \"house_finch\",\n        570686433,\n        315887836,\n        \"emilyheaton\",\n        229263,\n        \"27e0bd21c1b322aa5d74c5504dbf27bfb5a0fdeea9718b6ffcee7926f2cb2733\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-26\",\n        \"house_finch\",\n        266259557,\n        154114419,\n        \"andywilson\",\n        47792,\n        \"b0dc2de7134cc5af50b1482681b450d32b1f58765990d5d553c4838ae7936db0\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-27\",\n        \"house_finch\",\n        269729570,\n        156011712,\n        \"kbkash\",\n        59048,\n        \"3af407624c46ba4218d24a198ed22b812d4967380bd0e7f67e960f7fbb2defe7\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-28\",\n        \"house_finch\",\n        270277531,\n        156183453,\n        \"aparrot1\",\n        83424,\n        \"93c92a1bb7cf89e7106b0c3df65ce030c82bccade181d661952e940ddcc6282a\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-29\",\n        \"house_finch\",\n        658061500,\n        360629741,\n        \"mike_cove\",\n        107594,\n        \"deaaefaf712059b6c8a929a83e1ffd0295e12f0f291bc9869011703eecfcf372\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-30\",\n        \"house_finch\",\n        545518752,\n        302693775,\n        \"cvharris\",\n        208917,\n        \"fc5ffdbd30bc76fa4371a99f59543c9137411ddac0a691e2e8122ae7796e038b\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-31\",\n        \"house_finch\",\n        662587006,\n        362998939,\n        \"curran\",\n        113662,\n        \"644931afff18c942930717ac3ca563b63c18a0c78ab12ff88557aeede18ead63\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-32\",\n        \"house_finch\",\n        665549166,\n        364566546,\n        \"sealgyu\",\n        130087,\n        \"e6fb3e42979ce2709433e8141c4c665a70c4942166153578c7f0173328ffb2ad\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-33\",\n        \"house_finch\",\n        566433127,\n        313653086,\n        \"fake_id\",\n        150986,\n        \"cb8a50f48f65795850f57c4d8f84e49debc3deed59b6a60d5bee8ef95299650f\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-34\",\n        \"house_finch\",\n        404592864,\n        228025269,\n        \"logan_artz\",\n        78971,\n        \"d0cf631b138d199951d96538903eeb1b56e3b8da3f28c7b042d8868060f27a44\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-35\",\n        \"house_finch\",\n        423118612,\n        237726292,\n        \"conhawn\",\n        153347,\n        \"826ddd6fa8fad5bd1450c0c7beb77fa3261f41a0afa61d7681b7096bc4225c40\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-36\",\n        \"house_finch\",\n        290080883,\n        167451466,\n        \"vijaybarve\",\n        89129,\n        \"f0b32b4af4bdf0bc7eaf298b37a9be6dbcb175d9f72553efc239b6afd2746a69\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-37\",\n        \"house_finch\",\n        296347840,\n        170825854,\n        \"susanaber\",\n        96563,\n        \"47832989fc212f19631e55b63c23155817209c40c66af49e4e03e42863243770\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-38\",\n        \"house_finch\",\n        414288752,\n        233136904,\n        \"wafflemaster135\",\n        29548,\n        \"9f2039a678a69e8f8f383d43f80e3f4e6fcc81d5b99285a0e5b5e54834cf9125\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-39\",\n        \"house_finch\",\n        284751592,\n        164572867,\n        \"efalquet\",\n        80806,\n        \"b50e0a0a4a0bae79b5a8cd034645252044601a29b081f0842443848e802e82d2\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-40\",\n        \"house_finch\",\n        205914234,\n        121669780,\n        \"darylnolan\",\n        44763,\n        \"c04e8d7c002ad0038dcb62843e776ffb8fc86f425c2399442dd1653bbaa4b904\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-41\",\n        \"house_finch\",\n        96688995,\n        60467105,\n        \"erikschiff\",\n        145431,\n        \"dd056b1c0d89dcb9fa06b1c38936142e5c825eb2aeed0d7d2265dc7e147be792\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-42\",\n        \"house_finch\",\n        732358790,\n        399184296,\n        \"katrinamccollough\",\n        41331,\n        \"76c899fd9b9aa84ad41888c71a36ab99431fe7201dd417f2c19a66f77388420d\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-43\",\n        \"house_finch\",\n        83824778,\n        52720178,\n        \"beesbirdsbugs\",\n        48816,\n        \"72731eb63fa9c5058f9911d11be929af9a58cfea805159d54010e9095948c39a\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-44\",\n        \"house_finch\",\n        349400403,\n        198261102,\n        \"paulgraham\",\n        105965,\n        \"b0d1f556ee4926e6a6f87201f6b73955dd9c530788e0ce57d5e09ced74ede001\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-45\",\n        \"house_finch\",\n        484652323,\n        269544283,\n        \"robinlanark\",\n        29863,\n        \"a1d6e8ffbcef10a83a2d3674c7228af89f8d39297f44cd569d1e031da02bc516\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-46\",\n        \"house_finch\",\n        341357046,\n        194221751,\n        \"haconra\",\n        80643,\n        \"8e11937a5c6d5a47da47f562d52d4089d58ad3f2adefadc287b30c562c7ece85\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-47\",\n        \"house_finch\",\n        622140195,\n        341714166,\n        \"rachel141\",\n        133938,\n        \"aa776ef9fd3022b6b3db824a762bdde82eeb95518a8764763bd8199683bf1d04\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-48\",\n        \"house_finch\",\n        515309527,\n        286632225,\n        \"solunasilver\",\n        29507,\n        \"72a6b495ed99de80504c6b1a86ac1fbfc9e5736bc6f43274bd6093cc91f9ea05\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-49\",\n        \"house_finch\",\n        514877896,\n        286416757,\n        \"jubileej\",\n        153667,\n        \"26cd596525a244d2ba82ec2d575834c15671080814400cf7862f614c0db68d07\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-50\",\n        \"house_finch\",\n        125605349,\n        76831730,\n        \"sarahangulo\",\n        60174,\n        \"236eb345f1cf1ab4c82fb3b2cc48b948e614f8ddfd958c87a62ce5b44a04f080\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-51\",\n        \"house_finch\",\n        12881677,\n        9467101,\n        \"artemis224\",\n        173956,\n        \"91b5f9f66e8b84cd96169894024ebc3519de6f737d3a32ee3b70100c22b039b9\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-52\",\n        \"house_finch\",\n        124405788,\n        76034586,\n        \"radrat\",\n        36008,\n        \"bb9b3fe0b37a1ab6342b1e573f3faf3a287669b70fa9f00b5bfce83de4a376c6\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-53\",\n        \"house_finch\",\n        17468099,\n        12163814,\n        \"andy71\",\n        199644,\n        \"6bc02aaf56c1549dd78d6711f9121d48c33fbd76a8fb623895369f05604ad30d\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-54\",\n        \"house_finch\",\n        134699271,\n        82117365,\n        \"seanwashington1\",\n        35334,\n        \"d6e0fe4a8cc690fd6369deff8ea67f4810e1f1297a6f1b9f125aae905742db2a\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-55\",\n        \"house_finch\",\n        134287410,\n        81854554,\n        \"omcelroy\",\n        90394,\n        \"6b4a5ea4c8aa1722332ccdf420af0ca0906cfdb87348e9f6c0904d5a92d8c203\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-56\",\n        \"house_finch\",\n        17763955,\n        12337763,\n        \"jennifer510\",\n        167368,\n        \"35355c603049fdf21fc3bf1fed827b50740cfdba134542707026bb4e1f32149b\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-57\",\n        \"house_finch\",\n        18946521,\n        13021113,\n        \"deejay\",\n        60576,\n        \"2420363f6b02433f4b03f3323148ed5e4ac9d37c58a27e859f8e48354db866c8\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-58\",\n        \"house_finch\",\n        14590841,\n        10492098,\n        \"silvercat\",\n        120515,\n        \"23934e0376c70dd6d5ce5e9a24c4176291c877432d25257f07e27694495c879a\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-59\",\n        \"house_finch\",\n        124702263,\n        76228853,\n        \"janeyair\",\n        162431,\n        \"5c86d7cbd28e09d44814735f1e0a21e8266bd4e6a4396b047d092d2f5f3f2111\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-00\",\n        \"american_goldfinch\",\n        84579952,\n        53187208,\n        \"glennberry\",\n        59673,\n        \"72d36079e592e0a83c2f774f9073bfd4cc81253452c925d1673217ddd4b52a36\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-01\",\n        \"american_goldfinch\",\n        12533322,\n        9255418,\n        \"braincellsgone\",\n        55661,\n        \"6344e0125e74791f43ac6e07e5e1b9fbfce6d19bc62b6bb5d83b3caff9f7bcbc\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-02\",\n        \"american_goldfinch\",\n        131102823,\n        80016788,\n        \"radrat\",\n        98990,\n        \"232a944f7e3351d4916a12ef2f6d598e7b007caaa95a9b064a14c1ba3af2a6ff\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-03\",\n        \"american_goldfinch\",\n        175048222,\n        104466897,\n        \"eug302\",\n        44231,\n        \"11c723482cc75fcc3a723ac1c0818a68e2fcf74c4ca684cf60195bf0d33f274e\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-04\",\n        \"american_goldfinch\",\n        68849595,\n        43390778,\n        \"mefisher\",\n        154503,\n        \"66f07bc59bb3fdedd65a4537ebabd0cafd457826b8bf4bb633181f584a3edfd1\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-05\",\n        \"american_goldfinch\",\n        431916465,\n        242278180,\n        \"k-simpkins\",\n        45413,\n        \"d76e7adf33e3a84ebec24dde5438965e62ebec8595755453973846340e4f460d\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-06\",\n        \"american_goldfinch\",\n        230801384,\n        135330401,\n        \"enspring\",\n        42886,\n        \"78aebfb9b28c3e16dd9618a0e1ae06df67bfa4b8550c0879427d96915c475fed\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-07\",\n        \"american_goldfinch\",\n        377136648,\n        213398931,\n        \"nathan1177\",\n        66516,\n        \"7ad75838fdf2020a8e426e97507c7dd4355da93e6eece241128c28adbe302438\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-08\",\n        \"american_goldfinch\",\n        294667312,\n        169935316,\n        \"dande\",\n        163470,\n        \"39e7892e81eeef6af4887e61bc0998e17797688eb6394fcc8c9438391e875ee0\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-09\",\n        \"american_goldfinch\",\n        660472044,\n        361884286,\n        \"ben142\",\n        270599,\n        \"733d64cd50c61334682f0862f5c7859ded34cae5776ee0bc6594fee400dc4876\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-10\",\n        \"american_goldfinch\",\n        143215717,\n        86889530,\n        \"memoosborne\",\n        129438,\n        \"ef4a9a771cef9ba3c2c047eb106a6aa220236dd6aaa6aade5f4ef3a37c8abcba\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-11\",\n        \"american_goldfinch\",\n        403873164,\n        227647158,\n        \"drew_baxter\",\n        106009,\n        \"74e34c776f1b9a5d375a7dfae0e309cd42dbd133b82a1ecad4a49111ac7eed49\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-12\",\n        \"american_goldfinch\",\n        481153019,\n        267722510,\n        \"vicki936\",\n        255397,\n        \"3326ddbee3270241b681cb636466e742491f110e9841f453f5158864aadaea36\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-13\",\n        \"american_goldfinch\",\n        213311122,\n        125763980,\n        \"hickl\",\n        24740,\n        \"0c295b3761bced4215519ff24a4f734773b54be98de456a4444d0819456af7dd\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-14\",\n        \"american_goldfinch\",\n        417352824,\n        234736320,\n        \"joy4birds\",\n        81109,\n        \"357014c108519543471b94f39591667d1a67d87fd1fdc4702a3a449235d6bddd\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-15\",\n        \"american_goldfinch\",\n        72130720,\n        45486482,\n        \"dctphoto\",\n        178111,\n        \"17b2f3599e20161acc17fd63bf61e9f40488b9c4d5bfaefc8cae54de42997509\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-16\",\n        \"american_goldfinch\",\n        45148898,\n        28952026,\n        \"megachile\",\n        49358,\n        \"6bbc6ca0ac074e486c20ce4c3fad5dc863cb2e908efc2b1ef97494403a351252\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-17\",\n        \"american_goldfinch\",\n        133251099,\n        81250513,\n        \"raffib128\",\n        128110,\n        \"863c76587fe1de80a84b97c2e72f38ae1cf4fa972789e006d9b0b544d696c6ef\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-18\",\n        \"american_goldfinch\",\n        155797129,\n        93957328,\n        \"nathanael15\",\n        74086,\n        \"7059ae3bdeeb48fe949e5b70b788bd28c96aecd0fe49cb7be22db9a8697bf5a5\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-19\",\n        \"american_goldfinch\",\n        11400438,\n        8535447,\n        \"akneidel\",\n        26423,\n        \"4af74c1d04ddc7bbb7bb0e9eb2977a1daf48dd9b9477d71d69a7c4cb5d4f785b\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-20\",\n        \"american_goldfinch\",\n        55990791,\n        35505213,\n        \"conhawn\",\n        84915,\n        \"69f768c39a2180440bdcbfc6addc5d426341e8080d4cf9ba8241d57564b3e6fb\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-21\",\n        \"american_goldfinch\",\n        145464774,\n        88186208,\n        \"wildreturn\",\n        68204,\n        \"565d2a3b6d404e9ea0c24737ebcc5a3a82057b1452b4790df5e9552e8c50e592\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-22\",\n        \"american_goldfinch\",\n        637247336,\n        289067166,\n        \"dinomariobob\",\n        142312,\n        \"01c63d305fce8edcc3a494f0543154c6bd6aa52368e02be84cf6c23e95b94cdd\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-23\",\n        \"american_goldfinch\",\n        460988055,\n        257029007,\n        \"eric112\",\n        60314,\n        \"0db1b5f0e32d1faf861a437a20794fa18e80ea8966313933c145451c9319e2a9\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-24\",\n        \"american_goldfinch\",\n        59072170,\n        37280587,\n        \"bradenjudson\",\n        23468,\n        \"827d77bf9ca9cc456e867434a07b68cdceb4a20e09eb9e6b67757c80cd31ff42\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-25\",\n        \"american_goldfinch\",\n        66515260,\n        41915252,\n        \"reuvenm\",\n        58833,\n        \"90f0b687d9626fdf5d0111b794cf960cf18f3de4c8eab601fcb23c41b45c4df7\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-26\",\n        \"american_goldfinch\",\n        109875643,\n        67928198,\n        \"artemis224\",\n        217909,\n        \"99e1d7d30eb19d47c7974e9ddd7efe4d06329c952a41e15d0a4b2095b747df61\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-27\",\n        \"american_goldfinch\",\n        219478221,\n        129171341,\n        \"cgmayers\",\n        96993,\n        \"93bdee27bb526b60e0edfa518a923b3c10c41ba57221d6a23f4de15bf4a91600\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-28\",\n        \"american_goldfinch\",\n        99106353,\n        61899131,\n        \"thyg\",\n        50544,\n        \"0272bf89b379c2689247f1dbd6f4b36d7f197898baad91d7a8ebdf1c87f88fbd\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-29\",\n        \"american_goldfinch\",\n        223110598,\n        131133187,\n        \"marissa3\",\n        89214,\n        \"6a2bb601ac3a70e80e6ccb33d4a24af43128d1e7afe3e92d42e803234bdf532e\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-30\",\n        \"american_goldfinch\",\n        104649681,\n        65037800,\n        \"kemper\",\n        113821,\n        \"f4430daf9fca2cf6deb6b988148a5c2e5ae382af8c5db7e5cd37b26e8fbc9461\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-31\",\n        \"american_goldfinch\",\n        98669936,\n        61655828,\n        \"cloaca_enthusiast\",\n        31964,\n        \"3ddcb2e2e6863ef08a40fae7640e05b36ccec9fab9892ac494f095c4588b746f\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-32\",\n        \"american_goldfinch\",\n        347798463,\n        197465311,\n        \"aster-asti\",\n        116039,\n        \"e1236dd9850a412ba68386ffd342d180f582db187b83c1d33cb1e09dd51d0043\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-33\",\n        \"american_goldfinch\",\n        461261879,\n        257163676,\n        \"erikschiff\",\n        40881,\n        \"eb0b1ea08fcee160edd008ffc386b69b7b50bb89096c44873b978072c581d826\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-34\",\n        \"american_goldfinch\",\n        343726970,\n        195390446,\n        \"robinlanark\",\n        53508,\n        \"8b1f4e496755d1a3c4f567c6a48673998b8480ce7d38e7c52bbd94d7ee9901ed\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-35\",\n        \"american_goldfinch\",\n        352259924,\n        199722919,\n        \"carterdorscht\",\n        122243,\n        \"11edfdb53f892b1123df734cb7300e002171baeef21671a31411ff30d156bdc6\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-36\",\n        \"american_goldfinch\",\n        148829913,\n        90084462,\n        \"ian-wolfe\",\n        172110,\n        \"4f6a493313fc596665eb99f8e303ab0ee980b4932a9957993cd00b03b5360fa4\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-37\",\n        \"american_goldfinch\",\n        276388190,\n        159958623,\n        \"dianeclark6280\",\n        39786,\n        \"284ca7c9e814fede2c5a15e427a46d7dd82fa6984c4c862c380b0dddff2f86db\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-38\",\n        \"american_goldfinch\",\n        409778763,\n        230765968,\n        \"squidtk\",\n        210988,\n        \"b92ccce145f7a320f25edf92633e0c56aa92e8c83890b4a439ce4b3d669cb1a1\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-39\",\n        \"american_goldfinch\",\n        382773537,\n        216432059,\n        \"rocksand2134\",\n        195020,\n        \"ad33d43d9e5650dd441bbd9adb9cc9854a6a1bcc7088a8802a7fb4ccc1ee8b51\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-40\",\n        \"american_goldfinch\",\n        290287368,\n        167563678,\n        \"sean579\",\n        105190,\n        \"8703f245d4b98476f67775ca0c19b7abd7c5edf756c50b2ff502b3d058c08014\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-41\",\n        \"american_goldfinch\",\n        423112648,\n        237722733,\n        \"andrew2285\",\n        55719,\n        \"10709f5df3f4c509afd260ba51c0db5cee89c7c9076c74e9911e9613303d7754\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-42\",\n        \"american_goldfinch\",\n        422361237,\n        237335568,\n        \"wildaboutwildlife\",\n        34855,\n        \"55b0962389b589037e03378472e65ffa0e7ed0fb7e227a7441106a009f093924\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-43\",\n        \"american_goldfinch\",\n        291016386,\n        167956837,\n        \"samallonthesciencemon\",\n        128110,\n        \"489934bec3eb5682067cca11cb210c171198e616bb4920753b48bb0e1480bc32\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-44\",\n        \"american_goldfinch\",\n        419250127,\n        235728207,\n        \"paulgraham\",\n        39211,\n        \"be1f9ae51a6c4ee92a543ecfe1654698581d7647a3a0b46859a7cf054d8b0fd9\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-45\",\n        \"american_goldfinch\",\n        416004544,\n        234039610,\n        \"randv\",\n        105453,\n        \"942a87723af8aedc86459479437bc474817f19429806f36d8e1236538448e24f\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-46\",\n        \"american_goldfinch\",\n        416408092,\n        234251817,\n        \"kmayner\",\n        143385,\n        \"55259e21e667b88864e2b88ba8dacc8ac5be0d2606699be35c46d2b17fe67fb2\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-47\",\n        \"american_goldfinch\",\n        116188790,\n        71368917,\n        \"leahmfulton\",\n        48440,\n        \"ae01ef8ba2fb714253543a5e773e4e047e95448f4ff4d1e51bc889a20b14e1bc\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-48\",\n        \"american_goldfinch\",\n        118530525,\n        72675993,\n        \"seanwashington1\",\n        38505,\n        \"d688edde61a079907cdf5011a32914432f8f7f35c12d355e308aae29b1988738\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-49\",\n        \"american_goldfinch\",\n        121016576,\n        74071097,\n        \"geenance\",\n        110180,\n        \"fb4d1fef09311949043b3c9edc0d3a85d4c20386346093bb6ea39e1b95f756b9\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-50\",\n        \"american_goldfinch\",\n        3279608,\n        2872951,\n        \"joed\",\n        115964,\n        \"c2f1463bfef4c9b157f7e3ebd33fa37570592a4cdfa276f2b7c673f543cbc3ca\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-51\",\n        \"american_goldfinch\",\n        118140282,\n        72458899,\n        \"melaniegaddy\",\n        243713,\n        \"a3d70061032ac8c665f3d17a9118c8e592d53a3669182b23b502e3573a80d8f8\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-52\",\n        \"american_goldfinch\",\n        7669539,\n        6125334,\n        \"cuihenggang\",\n        61287,\n        \"36d9570131caf33f3b5cfa6ab99a26c417e7cd8e38d0c1351d1a2938bb6e9bf9\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-53\",\n        \"american_goldfinch\",\n        9258865,\n        7182023,\n        \"fake_id\",\n        151987,\n        \"131d2a52097c9ab18fad318b62e8213c1bd33acddc72065ec0e404c76220c5ee\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-54\",\n        \"american_goldfinch\",\n        12222432,\n        9054511,\n        \"myacadianforest\",\n        107758,\n        \"69b1b5274d74bb28601afc2758e57a88f8d63743b5410c70284a9f4877c721fb\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-55\",\n        \"american_goldfinch\",\n        128301260,\n        78400716,\n        \"natepow\",\n        67416,\n        \"0c988080a47484802806ef14e0d1fdb9205680eacecedbe24f8bbac2aa2873f9\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-56\",\n        \"american_goldfinch\",\n        129842959,\n        79296314,\n        \"nartb\",\n        117002,\n        \"f23b2e68f4c10e2679f0a9fe2f34a894bd2deea9c90c2143462e0aa5a83a3daa\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-57\",\n        \"american_goldfinch\",\n        78410307,\n        49351195,\n        \"chrisleearm\",\n        108066,\n        \"8a4b273f1a1bd548c48abcf57b86ed35028cbfb949720180a43f24976a26ac8e\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-58\",\n        \"american_goldfinch\",\n        72300928,\n        45598176,\n        \"henrya\",\n        68593,\n        \"a0c2869c330aaea08fba3bfeae99392c5588980b1ec21770c220c744125c3b42\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-59\",\n        \"american_goldfinch\",\n        194724999,\n        115367030,\n        \"vlatassa\",\n        159105,\n        \"c456a23238f8f81f93df7919b92dd3fa0123fa16a875c22ec1f9d0ae6ef17324\",\n        \"jpeg\",\n    ),\n)\nSAMPLE_SEED = 42\nSAMPLE_SPLIT = {\"train\": 36, \"validation\": 8, \"test\": 16}  # photographs per species; 6 species -> 216 / 48 / 96 pairs\nTIERS = (\"easy\", \"hard\")\nTIER_PARAMS: dict[str, dict[str, Any]] = {\n    \"easy\": {\"jitter\": 0.06, \"rotation\": 10.0, \"scale\": (0.9, 1.1), \"brightness\": (0.85, 1.15), \"contrast\": (0.85, 1.15), \"gamma\": (1.0, 1.0), \"blur\": 0.0, \"noise\": 4.0},\n    \"hard\": {\"jitter\": 0.18, \"rotation\": 150.0, \"scale\": (0.6, 1.4), \"brightness\": (0.6, 1.4), \"contrast\": (0.6, 1.4), \"gamma\": (0.7, 1.4), \"blur\": 1.2, \"noise\": 10.0},\n}\nMIN_RECORDS = 4\nMAX_RECORDS = 5_000\n_ID_RE = re.compile(r\"^[A-Za-z0-9_.:-]{1,64}$\")\n\n\ndef _sha256_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef photo_url(photo_id: int, ext: str = \"jpg\") -> str:\n    \"\"\"The served object for a pinned photo; `ext` is its recorded original extension (jpg, jpeg or png,\n    either case — the bucket key is case-sensitive).\"\"\"\n    if ext.lower() not in (\"jpg\", \"jpeg\", \"png\"):\n        raise ValueError(f\"unsupported photo extension {ext!r}\")\n    return f\"{CORPUS_BASE_URL}{photo_id}/medium.{ext}\"\n\n\ndef observation_url(observation_id: int) -> str:\n    return f\"https://www.inaturalist.org/observations/{observation_id}\"\n\n\ndef fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:\n    \"\"\"Return every pinned photo (bytes keyed by record id) from the cache or the open-data bucket.\"\"\"\n    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR\n    cache.mkdir(parents=True, exist_ok=True)\n    out = {}\n    for rid, _label, photo_id, _obs, _user, size, digest, ext in SAMPLE_RECORDS:\n        local = cache / f\"{photo_id}.jpg\"\n        data = local.read_bytes() if local.is_file() else b\"\"\n        if len(data) != size or _sha256_bytes(data) != digest:\n            url = photo_url(photo_id, ext)\n            if fetcher is not None:\n                data = fetcher(url)\n            else:\n                request = urllib.request.Request(\n                    url, headers={\"User-Agent\": \"dimer-lightglue-tutorial/1.0\"}\n                )\n                with urllib.request.urlopen(request, timeout=120) as response:  # noqa: S310 (pinned https URL)\n                    data = response.read()\n            if len(data) != size or _sha256_bytes(data) != digest:\n                raise ValueError(\n                    f\"{rid} ({photo_id}/medium.{ext}): fetched {len(data)} bytes with sha256 \"\n                    f\"{_sha256_bytes(data)[:16]}…, pinned {size} / {digest[:16]}…\"\n                )\n            local.write_bytes(data)\n        out[rid] = data\n    return out\n\n\ndef read_corpus(files: Mapping[str, bytes]) -> list[dict[str, Any]]:\n    \"\"\"Decode the verified photo bytes into `{id, image, species}` records with their provenance.\"\"\"\n    out = []\n    for rid, label, photo_id, obs_id, user, _size, _digest, _ext in SAMPLE_RECORDS:\n        if rid not in files:\n            raise ValueError(f\"corpus is missing {rid}\")\n        image = Image.open(io.BytesIO(files[rid]))\n        image.load()\n        out.append(\n            {\n                \"id\": rid,\n                \"image\": image.convert(\"RGB\"),\n                \"species\": label,\n                \"scientific_name\": SPECIES[label][0],\n                \"common_name\": SPECIES[label][1],\n                \"inat_photo_id\": photo_id,\n                \"inat_observation_url\": observation_url(obs_id),\n                \"observer\": user,\n            }\n        )\n    return out\n\n\n# --------------------------------------------------------------------------------------------------\n# pair synthesis: a seeded homography with an exact reference\n# --------------------------------------------------------------------------------------------------\n\n\ndef working_size(size: tuple[int, int], long_side: int = WORKING_LONG_SIDE) -> tuple[int, int]:\n    \"\"\"The size a photograph is brought to: long side `long_side`, both sides multiples of DIVISIBLE_BY.\"\"\"\n    width, height = size\n    scale = long_side / max(width, height)\n    w = max(DIVISIBLE_BY, int(width * scale) // DIVISIBLE_BY * DIVISIBLE_BY)\n    h = max(DIVISIBLE_BY, int(height * scale) // DIVISIBLE_BY * DIVISIBLE_BY)\n    return w, h\n\n\ndef prepare_image(image: Image.Image, long_side: int = WORKING_LONG_SIDE) -> Image.Image:\n    \"\"\"Resize to the working size (bicubic, aspect kept up to the multiple-of-8 crop) as RGB.\"\"\"\n    w, h = working_size(image.size, long_side)\n    return image.convert(\"RGB\").resize((w, h), Image.BICUBIC)\n\n\ndef sample_homography(size: tuple[int, int], rng: random.Random, params: Mapping[str, Any]) -> np.ndarray:\n    \"\"\"A random homography built from a rotation + scale about the centre followed by corner jitter.\"\"\"\n    width, height = size\n    cx, cy = (width - 1) / 2.0, (height - 1) / 2.0\n    angle = math.radians(rng.uniform(-params[\"rotation\"], params[\"rotation\"]))\n    scale = rng.uniform(*params[\"scale\"])\n    cos_a, sin_a = math.cos(angle) * scale, math.sin(angle) * scale\n    similarity = np.array(\n        [[cos_a, -sin_a, cx - cos_a * cx + sin_a * cy], [sin_a, cos_a, cy - sin_a * cx - cos_a * cy], [0.0, 0.0, 1.0]]\n    )\n    corners = np.array([[0.0, 0.0], [width - 1.0, 0.0], [width - 1.0, height - 1.0], [0.0, height - 1.0]])\n    jitter = params[\"jitter\"] * min(width, height)\n    moved = corners + np.array([[rng.uniform(-jitter, jitter), rng.uniform(-jitter, jitter)] for _ in range(4)])\n    from .metrics import dlt_homography\n\n    perspective = dlt_homography(corners, moved)\n    if perspective is None:  # degenerate draw (practically impossible); fall back to the similarity alone\n        return similarity\n    return perspective @ similarity\n\n\ndef _perspective_coefficients(homography: np.ndarray) -> list[float]:\n    \"\"\"PIL's PERSPECTIVE transform takes the inverse mapping (output pixel -> input pixel), 8 coefficients.\"\"\"\n    inverse = np.linalg.inv(homography)\n    inverse = inverse / inverse[2, 2]\n    return [float(v) for v in inverse.ravel()[:8]]\n\n\ndef warp_image(image: Image.Image, homography: np.ndarray) -> Image.Image:\n    \"\"\"image1 = image0 warped by `homography` (image0 coords -> image1 coords), same canvas, black outside.\"\"\"\n    return image.transform(image.size, Image.PERSPECTIVE, _perspective_coefficients(homography), Image.BICUBIC)\n\n\ndef photometric(image: Image.Image, rng: random.Random, params: Mapping[str, Any]) -> Image.Image:\n    \"\"\"Seeded brightness / contrast / gamma / blur / Gaussian-noise changes (never geometric).\"\"\"\n    out = ImageEnhance.Brightness(image).enhance(rng.uniform(*params[\"brightness\"]))\n    out = ImageEnhance.Contrast(out).enhance(rng.uniform(*params[\"contrast\"]))\n    gamma = rng.uniform(*params[\"gamma\"])\n    if params[\"blur\"] > 0:\n        out = out.filter(ImageFilter.GaussianBlur(rng.uniform(0.0, params[\"blur\"])))\n    array = np.asarray(out, dtype=np.float64) / 255.0\n    if gamma != 1.0:\n        array = np.power(np.clip(array, 0.0, 1.0), gamma)\n    if params[\"noise\"] > 0:\n        noise_rng = np.random.default_rng(rng.getrandbits(32))\n        array = array + noise_rng.normal(0.0, params[\"noise\"] / 255.0, array.shape)\n    return Image.fromarray((np.clip(array, 0.0, 1.0) * 255.0).round().astype(np.uint8))\n\n\ndef make_pair(image: Image.Image, *, seed: int, tier: str = \"hard\", record_id: str = \"pair\") -> dict[str, Any]:\n    \"\"\"One `{id, image0, image1, homography, tier}` record from a photograph and a seed.\"\"\"\n    if tier not in TIER_PARAMS:\n        raise ValueError(f\"tier must be one of {TIERS}\")\n    params = TIER_PARAMS[tier]\n    rng = random.Random(seed)\n    image0 = prepare_image(image)\n    homography = sample_homography(image0.size, rng, params)\n    image1 = photometric(warp_image(image0, homography), rng, params)\n    return {\n        \"id\": record_id,\n        \"image0\": image0,\n        \"image1\": image1,\n        \"homography\": homography.tolist(),\n        \"tier\": tier,\n        \"seed\": seed,\n    }\n\n\ndef make_pairs(records: Sequence[Mapping[str, Any]], *, seed: int = SAMPLE_SEED, tier: str | None = None) -> list[dict[str, Any]]:\n    \"\"\"One pair per image record (`{id, image, ...}`); tiers alternate easy / hard unless `tier` is fixed.\"\"\"\n    out = []\n    for index, record in enumerate(records):\n        chosen = tier or TIERS[index % len(TIERS)]\n        pair = make_pair(record[\"image\"], seed=seed * 100_003 + index, tier=chosen, record_id=str(record[\"id\"]))\n        for key in (\"species\", \"observer\", \"inat_photo_id\", \"inat_observation_url\", \"source_id\"):\n            if key in record:\n                pair[key] = record[key]\n        out.append(pair)\n    return out\n\n\ndef build_sample_dataset(\n    records: Sequence[Mapping[str, Any]],\n    *,\n    seed: int = SAMPLE_SEED,\n    sizes: Mapping[str, int] | None = None,\n) -> dict[str, list[dict[str, Any]]]:\n    \"\"\"Seeded stratified draw of photographs per species into train / validation / test, then one pair per\n    photograph (tiers alternating within each split). No photograph lands in two splits.\"\"\"\n    sizes = dict(sizes or SAMPLE_SPLIT)\n    rng = random.Random(seed)\n    by_label: dict[str, list[dict[str, Any]]] = {}\n    for record in records:\n        by_label.setdefault(str(record.get(\"species\", \"image\")), []).append(dict(record))\n    photos: dict[str, list[dict[str, Any]]] = {name: [] for name in sizes}\n    for label in sorted(by_label):\n        pool = by_label[label]\n        rng.shuffle(pool)\n        needed = sum(sizes.values())\n        if len(pool) < needed:\n            raise ValueError(f\"{label}: only {len(pool)} records available, need {needed}\")\n        cursor = 0\n        for name, per_class in sizes.items():\n            photos[name].extend(pool[cursor : cursor + per_class])\n            cursor += per_class\n    out: dict[str, list[dict[str, Any]]] = {}\n    for offset, name in enumerate(photos):\n        rng.shuffle(photos[name])\n        relabelled = [{**r, \"id\": f\"{name}-{i:03d}\", \"source_id\": r[\"id\"]} for i, r in enumerate(photos[name])]\n        out[name] = make_pairs(relabelled, seed=seed + offset)\n    return out\n\n\ndef fetch_sample_dataset(\n    *,\n    cache_dir: str | Path | None = None,\n    fetcher: Any = None,\n    seed: int = SAMPLE_SEED,\n    sizes: Mapping[str, int] | None = None,\n) -> dict[str, list[dict[str, Any]]]:\n    \"\"\"The tutorial splits from the pinned corpus.\"\"\"\n    return build_sample_dataset(\n        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes\n    )\n\n\n# --------------------------------------------------------------------------------------------------\n# validation\n# --------------------------------------------------------------------------------------------------\n\n\ndef _open(image: Any, label_name: str) -> Image.Image:\n    if isinstance(image, str | Path):\n        path = Path(image)\n        if not path.is_file():\n            raise ValueError(f\"{label_name}: image file not found: {path}\")\n        image = Image.open(path)\n        image.load()\n    if not isinstance(image, Image.Image):\n        raise ValueError(f\"{label_name}: image must be a PIL.Image.Image or a file path\")\n    width, height = image.size\n    if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:\n        raise ValueError(f\"{label_name}: image side outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: {image.size}\")\n    return image.convert(\"RGB\")\n\n\ndef check_homography(value: Any, label_name: str = \"homography\") -> np.ndarray:\n    array = np.asarray(value, dtype=np.float64)\n    if array.shape != (3, 3) or not np.all(np.isfinite(array)):\n        raise ValueError(f\"{label_name}: must be a finite 3x3 matrix\")\n    if abs(array[2, 2]) < 1e-12:\n        raise ValueError(f\"{label_name}: H[2, 2] must be non-zero\")\n    array = array / array[2, 2]\n    if abs(np.linalg.det(array)) < 1e-9:\n        raise ValueError(f\"{label_name}: matrix is singular\")\n    return array\n\n\ndef _check_record(record: Any, index: int) -> dict[str, Any]:\n    label_name = f\"records[{index}]\"\n    if not isinstance(record, Mapping):\n        raise ValueError(f\"{label_name} must be a mapping with id/image0/image1/homography\")\n    for key in (\"id\", \"image0\", \"image1\", \"homography\"):\n        if key not in record:\n            raise ValueError(f\"{label_name} is missing {key!r}\")\n    rid = record[\"id\"]\n    if not isinstance(rid, str) or not _ID_RE.match(rid):\n        raise ValueError(f\"{label_name}: id must match {_ID_RE.pattern}\")\n    image0 = _open(record[\"image0\"], label_name + \".image0\")\n    image1 = _open(record[\"image1\"], label_name + \".image1\")\n    for which, image in ((\"image0\", image0), (\"image1\", image1)):\n        if min(image.size) < MIN_SIDE or max(image.size) > MAX_SIDE:\n            raise ValueError(\n                f\"{label_name}.{which}: sides must lie in [{MIN_SIDE}, {MAX_SIDE}] px after preparation; got {image.size}\"\n            )\n    homography = check_homography(record[\"homography\"], label_name + \".homography\")\n    tier = record.get(\"tier\", \"unspecified\")\n    if not isinstance(tier, str) or not tier:\n        raise ValueError(f\"{label_name}: tier must be a non-empty string when given\")\n    item = {\"id\": rid, \"image0\": image0, \"image1\": image1, \"homography\": homography.tolist(), \"tier\": tier}\n    for key in (\"source_id\", \"seed\", \"species\", \"observer\", \"inat_photo_id\", \"inat_observation_url\"):\n        if key in record:\n            item[key] = record[key]\n    return item\n\n\ndef validate_dataset(\n    records: Sequence[Mapping[str, Any]],\n    *,\n    min_records: int = MIN_RECORDS,\n    max_records: int = MAX_RECORDS,\n) -> dict[str, Any]:\n    \"\"\"Structural validation of a pair dataset; raises ValueError before any model import. Nothing checks\n    that `image1` really is `image0` under `homography` — a wrong reference is scored without complaint.\"\"\"\n    if (\n        isinstance(records, Mapping)\n        or not isinstance(records, Sequence)\n        or isinstance(records, (str, bytes))\n    ):\n        raise ValueError(\"records must be a list of {id, image0, image1, homography} mappings\")\n    if not min_records <= len(records) <= max_records:\n        raise ValueError(f\"{len(records)} records; {min_records}..{max_records} are required\")\n    checked = []\n    ids: set[str] = set()\n    tiers: dict[str, int] = {}\n    for index, record in enumerate(records):\n        item = _check_record(record, index)\n        if item[\"id\"] in ids:\n            raise ValueError(f\"duplicate id {item['id']!r}\")\n        ids.add(item[\"id\"])\n        tiers[item[\"tier\"]] = tiers.get(item[\"tier\"], 0) + 1\n        checked.append(item)\n    sides = [max(r[\"image0\"].size) for r in checked]\n    return {\n        \"records\": checked,\n        \"n_records\": len(checked),\n        \"tiers\": dict(sorted(tiers.items())),\n        \"image_side\": {\"min\": min(sides), \"max\": max(sides)},\n        \"digest\": dataset_digest(checked),\n        \"model_id\": MODEL_ID,\n    }\n\n\ndef image_digest(image: Image.Image) -> str:\n    \"\"\"SHA-256 of the decoded RGB pixels (size + bytes), so a re-encoded copy of the same image matches.\"\"\"\n    rgb = image.convert(\"RGB\")\n    return _sha256_bytes(f\"{rgb.size[0]}x{rgb.size[1]}:\".encode() + rgb.tobytes())\n\n\ndef dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:\n    payload = [[r[\"id\"], image_digest(r[\"image0\"]), image_digest(r[\"image1\"]), r[\"homography\"]] for r in records]\n    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(\",\", \":\")).encode(\"utf-8\"))\n\n\ndef check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:\n    \"\"\"Assert no source image (by decoded-pixel digest of image0) appears in two splits (leakage check).\"\"\"\n    seen: dict[str, str] = {}\n    for name, records in splits.items():\n        for record in records:\n            key = image_digest(record[\"image0\"])\n            if key in seen and seen[key] != name:\n                raise ValueError(f\"image {record['id']!r} appears in both {seen[key]} and {name}\")\n            seen[key] = name\n    return {name: len(records) for name, records in splits.items()}\n\n\ndef observer_overlap(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, int]:\n    \"\"\"How many observers contributed photos to more than one split (an observation, not an assertion).\"\"\"\n    seen: dict[str, set[str]] = {}\n    for name, records in splits.items():\n        for record in records:\n            if record.get(\"observer\"):\n                seen.setdefault(str(record[\"observer\"]), set()).add(name)\n    return {\n        \"observers\": len(seen),\n        \"in_more_than_one_split\": sum(1 for s in seen.values() if len(s) > 1),\n    }\n\n\ndef split_dataset(\n    records: Sequence[Mapping[str, Any]],\n    *,\n    val_fraction: float = 0.15,\n    test_fraction: float = 0.2,\n    seed: int = 0,\n) -> dict[str, list[dict[str, Any]]]:\n    \"\"\"Seeded shuffle of BYOD image records (`{id, image}`) into train / validation / test after de-duplicating\n    images by decoded pixels, then one pair per image.\"\"\"\n    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):\n        raise ValueError(\"fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1\")\n    seen: set[str] = set()\n    unique = []\n    for record in records:\n        image = _open(record[\"image\"], str(record.get(\"id\")))\n        key = image_digest(image)\n        if key not in seen:\n            seen.add(key)\n            unique.append({**record, \"image\": image})\n    rng = random.Random(seed)\n    rng.shuffle(unique)\n    n_test = max(1, round(len(unique) * test_fraction))\n    n_val = round(len(unique) * val_fraction)\n    parts = {\"test\": unique[:n_test], \"validation\": unique[n_test : n_test + n_val], \"train\": unique[n_test + n_val :]}\n    if len(parts[\"train\"]) < MIN_RECORDS:\n        raise ValueError(f\"split leaves {len(parts['train'])} training images; at least {MIN_RECORDS} are required\")\n    out = {}\n    for offset, (name, part) in enumerate(parts.items()):\n        relabelled = [{**r, \"id\": f\"{name}-{i:03d}\", \"source_id\": r[\"id\"]} for i, r in enumerate(part)]\n        out[name] = make_pairs(relabelled, seed=seed + offset)\n    return out\n\n\ndef load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:\n    \"\"\"Read `{id, image}` records from a directory or a zip of image files (optionally listed in `images.csv`\n    with columns `id`, `file`); images are decoded, never extracted to disk.\"\"\"\n    source = Path(path)\n    if source.is_dir():\n        names = sorted(p.name for p in source.iterdir() if p.suffix.lower() in (\".jpg\", \".jpeg\", \".png\"))\n        loader = lambda name: Image.open(source / name)  # noqa: E731\n    elif source.is_file() and source.suffix.lower() == \".zip\":\n        archive = zipfile.ZipFile(source)\n        members = {Path(n).name: n for n in archive.namelist() if Path(n).suffix.lower() in (\".jpg\", \".jpeg\", \".png\")}\n        names = sorted(members)\n        loader = lambda name: Image.open(io.BytesIO(archive.read(members[name])))  # noqa: E731\n    else:\n        raise ValueError(\"BYOD datasets must be a directory or a .zip holding JPEG / PNG image files\")\n    if not names:\n        raise ValueError(\"BYOD dataset holds no JPEG / PNG image files\")\n    out = []\n    for name in names:\n        image = loader(name)\n        image.load()\n        out.append({\"id\": re.sub(r\"[^A-Za-z0-9_.:-]\", \"_\", Path(name).stem)[:64], \"image\": image.convert(\"RGB\")})\n    return out\n\n\ndef write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:\n    \"\"\"Write the pair table of a split (id, source, tier, seed, the homography, provenance).\"\"\"\n    out = Path(path)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    with open(out, \"w\", encoding=\"utf-8\", newline=\"\") as handle:\n        writer = csv.DictWriter(handle, fieldnames=[\"id\", \"source_id\", \"tier\", \"seed\", \"homography\", \"observer\", \"inat_observation_url\"])\n        writer.writeheader()\n        for record in records:\n            writer.writerow(\n                {\n                    \"id\": record[\"id\"],\n                    \"source_id\": record.get(\"source_id\", \"\"),\n                    \"tier\": record.get(\"tier\", \"\"),\n                    \"seed\": record.get(\"seed\", \"\"),\n                    \"homography\": json.dumps(record[\"homography\"]),\n                    \"observer\": record.get(\"observer\", \"\"),\n                    \"inat_observation_url\": record.get(\"inat_observation_url\", \"\"),\n                }\n            )\n    return out\n","xoftr_pipeline/__init__.py":"\"\"\"DIMER-oriented XoFTR image-matching pipeline: one pinned checkpoint, a vendored network, an\ninference contract with an exact-reference evaluation, a bounded coarse-matcher adaptation and a\nsafetensors adapter.\"\"\"\n\nfrom .config import (\n    ALT_MODEL_FILENAME,\n    ALT_MODEL_SHA256,\n    COARSE_THRESHOLD,\n    DEFAULT_MODEL_KEY,\n    DIVISIBLE_BY,\n    FINE_THRESHOLD,\n    MAX_SIDE,\n    MIN_SIDE,\n    MODEL_FILENAME,\n    MODEL_ID,\n    MODEL_ID_FORMER,\n    MODEL_LICENSE,\n    MODEL_REVISION,\n    MODEL_SHA256,\n    MODEL_SIZE_BYTES,\n    PARAMETER_COUNT,\n    STATE_TENSORS,\n    UNSAFE_WEIGHT_EXTENSIONS,\n)\nfrom .metrics import (\n    METRIC_DEFINITIONS,\n    corner_error,\n    dlt_homography,\n    identity_baseline,\n    matching_metrics,\n    pair_metrics,\n    patch_neighbour_baseline,\n    ransac_homography,\n    reprojection_errors,\n    warp_points,\n)\nfrom .model import (\n    DEFAULT_WEIGHTS_DIR,\n    MANIFEST_NAME,\n    build_model,\n    load_components,\n    stage_missing_files,\n    verify_checkpoint,\n    verify_snapshot,\n)\nfrom .modeling import UPSTREAM_COMMIT, UPSTREAM_REPOSITORY, XoFTR, default_config\nfrom .pipeline import (\n    ARTIFACT_FORMAT,\n    COARSE_LAYERS,\n    DEFAULT_TRAINABLE_COARSE_LAYERS,\n    INPUT_SCHEMA,\n    MAX_EVAL_RECORDS,\n    MIN_SCORED_RECORDS,\n    XoFTRPipeline,\n    evaluation_report,\n    load_pipeline,\n    validate_inputs,\n)\nfrom .provenance import build_provenance, write_provenance\nfrom .samples import (\n    CORPUS_BASE_URL,\n    CORPUS_BYTES,\n    CORPUS_LICENSE,\n    CORPUS_NAME,\n    CORPUS_RELEASE,\n    MAX_IMAGE_SIDE,\n    MAX_RECORDS,\n    MIN_RECORDS,\n    SAMPLE_RECORDS,\n    SAMPLE_SEED,\n    SAMPLE_SPLIT,\n    SPECIES,\n    TIER_PARAMS,\n    TIERS,\n    WORKING_LONG_SIDE,\n    build_sample_dataset,\n    check_homography,\n    check_split_disjoint,\n    dataset_digest,\n    fetch_corpus,\n    fetch_sample_dataset,\n    image_digest,\n    load_byod_dataset,\n    make_pair,\n    make_pairs,\n    observer_overlap,\n    prepare_image,\n    read_corpus,\n    sample_homography,\n    split_dataset,\n    validate_dataset,\n    warp_image,\n    working_size,\n    write_dataset_csv,\n)\n\n__all__ = [\n    \"ALT_MODEL_FILENAME\",\n    \"ALT_MODEL_SHA256\",\n    \"ARTIFACT_FORMAT\",\n    \"COARSE_LAYERS\",\n    \"COARSE_THRESHOLD\",\n    \"CORPUS_BASE_URL\",\n    \"CORPUS_BYTES\",\n    \"CORPUS_LICENSE\",\n    \"CORPUS_NAME\",\n    \"CORPUS_RELEASE\",\n    \"DEFAULT_MODEL_KEY\",\n    \"DEFAULT_TRAINABLE_COARSE_LAYERS\",\n    \"DEFAULT_WEIGHTS_DIR\",\n    \"DIVISIBLE_BY\",\n    \"FINE_THRESHOLD\",\n    \"INPUT_SCHEMA\",\n    \"MANIFEST_NAME\",\n    \"MAX_EVAL_RECORDS\",\n    \"MAX_IMAGE_SIDE\",\n    \"MAX_RECORDS\",\n    \"MAX_SIDE\",\n    \"METRIC_DEFINITIONS\",\n    \"MIN_RECORDS\",\n    \"MIN_SCORED_RECORDS\",\n    \"MIN_SIDE\",\n    \"MODEL_FILENAME\",\n    \"MODEL_ID\",\n    \"MODEL_ID_FORMER\",\n    \"MODEL_LICENSE\",\n    \"MODEL_REVISION\",\n    \"MODEL_SHA256\",\n    \"MODEL_SIZE_BYTES\",\n    \"PARAMETER_COUNT\",\n    \"SAMPLE_RECORDS\",\n    \"SAMPLE_SEED\",\n    \"SAMPLE_SPLIT\",\n    \"SPECIES\",\n    \"STATE_TENSORS\",\n    \"TIER_PARAMS\",\n    \"TIERS\",\n    \"UNSAFE_WEIGHT_EXTENSIONS\",\n    \"UPSTREAM_COMMIT\",\n    \"UPSTREAM_REPOSITORY\",\n    \"WORKING_LONG_SIDE\",\n    \"XoFTR\",\n    \"XoFTRPipeline\",\n    \"build_model\",\n    \"build_provenance\",\n    \"build_sample_dataset\",\n    \"check_homography\",\n    \"check_split_disjoint\",\n    \"corner_error\",\n    \"dataset_digest\",\n    \"default_config\",\n    \"dlt_homography\",\n    \"evaluation_report\",\n    \"fetch_corpus\",\n    \"fetch_sample_dataset\",\n    \"identity_baseline\",\n    \"image_digest\",\n    \"load_byod_dataset\",\n    \"load_components\",\n    \"load_pipeline\",\n    \"make_pair\",\n    \"make_pairs\",\n    \"matching_metrics\",\n    \"observer_overlap\",\n    \"pair_metrics\",\n    \"patch_neighbour_baseline\",\n    \"prepare_image\",\n    \"ransac_homography\",\n    \"read_corpus\",\n    \"reprojection_errors\",\n    \"sample_homography\",\n    \"split_dataset\",\n    \"stage_missing_files\",\n    \"validate_dataset\",\n    \"validate_inputs\",\n    \"verify_checkpoint\",\n    \"verify_snapshot\",\n    \"warp_image\",\n    \"warp_points\",\n    \"working_size\",\n    \"write_dataset_csv\",\n    \"write_provenance\",\n]\n","xoftr_pipeline/config.py":"from __future__ import annotations\n\n# The XoFTR checkpoints as hosted by the vismatch (formerly image-matching-models) project on the\n# Hub:\n# plain safetensors state dicts of the upstream network with a `matcher.` prefix on every key.\nMODEL_ID = \"vismatch/xoftr\"\nMODEL_ID_FORMER = \"image-matching-models/xoftr\"  # the same files under the project's former name\nMODEL_REVISION = \"d8ee7d89be3c9e5c157db3886db1c0f0e038b321\"\nMODEL_LICENSE = \"Apache-2.0\"\n\n# The served weight file: the 640-px training variant (the image-matching-models default).\nMODEL_FILENAME = \"xoftr_640.safetensors\"\nMODEL_SHA256 = \"4d5ed62e8b41f862ecc5c660e31f1c450402966623d6a28e85acf7fbd794cc69\"\nMODEL_SIZE_BYTES = 44_419_304\n# The 840-px sibling hosted beside it (recorded, not staged by default).\nALT_MODEL_FILENAME = \"xoftr_840.safetensors\"\nALT_MODEL_SHA256 = \"3385e8d5121116805d99f700aaceddbbe9760a8dd22585e55404172e1ea0d488\"\nALT_MODEL_SIZE_BYTES = 44_419_304\nSTATE_TENSORS = 247  # 231 float32 parameters + 16 int64 buffers (relative-position index tables)\nPARAMETER_COUNT = 11_091_722\n\nDEFAULT_MODEL_KEY = \"xoftr\"\nUNSAFE_WEIGHT_EXTENSIONS = (\n    \".bin\",\n    \".pt\",\n    \".pth\",\n    \".ckpt\",\n    \".pkl\",\n    \".pickle\",\n    \".h5\",\n    \".msgpack\",\n)\nALLOWED_CHECKPOINT_FILES = (MODEL_FILENAME,)\n\n# Inference contract.\nCOARSE_THRESHOLD = 0.3  # upstream MATCH_COARSE.THR\nFINE_THRESHOLD = 0.1  # upstream FINE.THR\nDIVISIBLE_BY = 8  # image sides are cropped down to a multiple of this (the 1/8 coarse grid)\nMIN_SIDE = 64\nMAX_SIDE = 1024\n","xoftr_pipeline/metrics.py":"\"\"\"Homography-supervised matching metrics and two non-neural baselines, in numpy.\n\nA record pairs an image with a warped copy of itself under a known 3 × 3 homography `H` (image0 → image1),\nso every match has an exact reference: the reprojection error of `(x0, y0)` mapped by `H` against `(x1, y1)`.\nFor a set of records the pipeline reports:\n\n- **precision at 3 px** (the fraction of returned matches with reprojection error under 3 px — the\n  matching-precision reading), also at 1 px and 5 px;\n- **matches per pair** and **inliers per pair** at 3 px (how much a downstream solver has to work with);\n- **median reprojection error** of the inliers (sub-pixel accuracy);\n- **homography accuracy at 3 px / 5 px**: the fraction of pairs whose homography, estimated from the\n  matches by a normalised DLT inside a plain RANSAC loop, moves the four image corners by less than the\n  threshold on average against the reference `H` (the usual HPatches-style reading; pairs with fewer than\n  four inliers count as failures).\n\nTwo baselines a matcher must beat: the **identity guess** (every grid point maps to itself — correct only\nwhere the warp is small) and a **patch nearest neighbour** (for each grid point of image0 the best\nnormalised-cross-correlation 15 × 15 patch of image1 within a search window — a classifier-free matcher\nthat knows the images through raw intensities). Both return the same match structure the model does and\nare scored by the same code.\n\"\"\"\n# ruff: noqa: E501  -- fleet metrics module written at the 110-column fleet width; this repo lints at 100\n\nfrom __future__ import annotations\n\nimport math\nfrom collections.abc import Mapping, Sequence\nfrom typing import Any\n\nimport numpy as np\nfrom PIL import Image\n\nMETRIC_DEFINITIONS = {\n    \"precision_3px\": \"fraction of returned matches whose reprojection error under the reference homography is below 3 px, averaged over pairs (a pair with no matches scores 0); in 0..1\",\n    \"precision_1px\": \"the same at 1 px\",\n    \"precision_5px\": \"the same at 5 px\",\n    \"matches_per_pair\": \"mean number of returned matches per pair\",\n    \"inliers_per_pair\": \"mean number of returned matches under 3 px per pair\",\n    \"median_error_px\": \"median reprojection error of the inliers under 3 px, pooled over pairs; px\",\n    \"homography_acc_3px\": \"fraction of pairs whose RANSAC-DLT homography from the matches moves the four corners by less than 3 px on average against the reference; in 0..1\",\n    \"homography_acc_5px\": \"the same at 5 px\",\n}\nTHRESHOLDS = (1.0, 3.0, 5.0)\nINLIER_PX = 3.0\n\n\ndef warp_points(points: np.ndarray, homography: np.ndarray) -> np.ndarray:\n    \"\"\"Apply a 3 × 3 homography to (N, 2) pixel coordinates.\"\"\"\n    pts = np.asarray(points, dtype=np.float64)\n    if pts.ndim != 2 or pts.shape[1] != 2:\n        raise ValueError(\"points must be an (N, 2) array\")\n    hom = np.concatenate([pts, np.ones((len(pts), 1))], axis=1) @ np.asarray(homography, dtype=np.float64).T\n    with np.errstate(divide=\"ignore\", invalid=\"ignore\"):  # points at infinity under a degenerate candidate\n        return hom[:, :2] / hom[:, 2:3]\n\n\ndef reprojection_errors(kpts0: np.ndarray, kpts1: np.ndarray, homography: np.ndarray) -> np.ndarray:\n    \"\"\"Per-match distance between `H · kpts0` and `kpts1`, in px.\"\"\"\n    kpts0 = np.asarray(kpts0, dtype=np.float64).reshape(-1, 2)\n    kpts1 = np.asarray(kpts1, dtype=np.float64).reshape(-1, 2)\n    if len(kpts0) != len(kpts1):\n        raise ValueError(\"kpts0 and kpts1 must have the same length\")\n    if len(kpts0) == 0:\n        return np.zeros((0,), dtype=np.float64)\n    errors = np.linalg.norm(warp_points(kpts0, homography) - kpts1, axis=1)\n    return np.where(np.isfinite(errors), errors, np.inf)\n\n\ndef _normalise(points: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n    mean = points.mean(axis=0)\n    scale = math.sqrt(2.0) / max(float(np.sqrt(((points - mean) ** 2).sum(axis=1)).mean()), 1e-9)\n    transform = np.array([[scale, 0.0, -scale * mean[0]], [0.0, scale, -scale * mean[1]], [0.0, 0.0, 1.0]])\n    hom = np.concatenate([points, np.ones((len(points), 1))], axis=1) @ transform.T\n    return hom[:, :2], transform\n\n\ndef dlt_homography(kpts0: np.ndarray, kpts1: np.ndarray) -> np.ndarray | None:\n    \"\"\"Normalised direct linear transform from at least four correspondences; None when degenerate.\"\"\"\n    kpts0 = np.asarray(kpts0, dtype=np.float64)\n    kpts1 = np.asarray(kpts1, dtype=np.float64)\n    if len(kpts0) < 4:\n        return None\n    p0, t0 = _normalise(kpts0)\n    p1, t1 = _normalise(kpts1)\n    rows = []\n    for (x, y), (u, v) in zip(p0, p1, strict=True):\n        rows.append([-x, -y, -1.0, 0.0, 0.0, 0.0, u * x, u * y, u])\n        rows.append([0.0, 0.0, 0.0, -x, -y, -1.0, v * x, v * y, v])\n    a = np.asarray(rows)\n    try:\n        _u, sigma, vt = np.linalg.svd(a)\n    except np.linalg.LinAlgError:\n        return None\n    if sigma[-2] < 1e-12:  # rank-deficient: collinear points\n        return None\n    h_norm = vt[-1].reshape(3, 3)\n    homography = np.linalg.inv(t1) @ h_norm @ t0\n    if abs(homography[2, 2]) < 1e-12:\n        return None\n    return homography / homography[2, 2]\n\n\ndef ransac_homography(\n    kpts0: np.ndarray,\n    kpts1: np.ndarray,\n    *,\n    threshold: float = INLIER_PX,\n    iterations: int = 500,\n    seed: int = 0,\n) -> tuple[np.ndarray | None, np.ndarray]:\n    \"\"\"A plain RANSAC over four-point DLT samples, refit on the consensus set. Returns (H or None, inlier mask).\"\"\"\n    kpts0 = np.asarray(kpts0, dtype=np.float64).reshape(-1, 2)\n    kpts1 = np.asarray(kpts1, dtype=np.float64).reshape(-1, 2)\n    n = len(kpts0)\n    if n < 4:\n        return None, np.zeros((n,), dtype=bool)\n    rng = np.random.default_rng(seed)\n    best_mask = np.zeros((n,), dtype=bool)\n    for _ in range(iterations):\n        sample = rng.choice(n, size=4, replace=False)\n        candidate = dlt_homography(kpts0[sample], kpts1[sample])\n        if candidate is None:\n            continue\n        mask = reprojection_errors(kpts0, kpts1, candidate) < threshold\n        if mask.sum() > best_mask.sum():\n            best_mask = mask\n            if best_mask.sum() == n:\n                break\n    if best_mask.sum() < 4:\n        return None, best_mask\n    refit = dlt_homography(kpts0[best_mask], kpts1[best_mask])\n    if refit is None:\n        return None, best_mask\n    return refit, reprojection_errors(kpts0, kpts1, refit) < threshold\n\n\ndef corner_error(estimated: np.ndarray, reference: np.ndarray, size: tuple[int, int]) -> float:\n    \"\"\"Mean displacement of the four image corners between two homographies, in px.\"\"\"\n    width, height = size\n    corners = np.array([[0.0, 0.0], [width - 1.0, 0.0], [width - 1.0, height - 1.0], [0.0, height - 1.0]])\n    return float(np.linalg.norm(warp_points(corners, estimated) - warp_points(corners, reference), axis=1).mean())\n\n\ndef pair_metrics(match: Mapping[str, Any], homography: np.ndarray, size: tuple[int, int]) -> dict[str, Any]:\n    \"\"\"Per-pair scores for one match result `{kpts0, kpts1}` against the reference homography.\"\"\"\n    kpts0 = np.asarray(match[\"kpts0\"], dtype=np.float64).reshape(-1, 2)\n    kpts1 = np.asarray(match[\"kpts1\"], dtype=np.float64).reshape(-1, 2)\n    errors = reprojection_errors(kpts0, kpts1, homography)\n    out: dict[str, Any] = {\"n_matches\": int(len(errors))}\n    for t in THRESHOLDS:\n        out[f\"precision_{int(t)}px\"] = float((errors < t).mean()) if len(errors) else 0.0\n    inliers = errors[errors < INLIER_PX]\n    out[\"n_inliers\"] = int(len(inliers))\n    out[\"inlier_errors\"] = inliers.tolist()\n    estimated, _mask = ransac_homography(kpts0, kpts1)\n    out[\"corner_error_px\"] = corner_error(estimated, homography, size) if estimated is not None else math.inf\n    return out\n\n\ndef matching_metrics(per_pair: Sequence[Mapping[str, Any]]) -> dict[str, Any]:\n    \"\"\"Aggregate `pair_metrics` rows over a set of pairs.\"\"\"\n    if not per_pair:\n        raise ValueError(\"at least one pair is required\")\n    pooled = [e for row in per_pair for e in row[\"inlier_errors\"]]\n    out: dict[str, Any] = {\n        \"n\": len(per_pair),\n        \"matches_per_pair\": float(np.mean([row[\"n_matches\"] for row in per_pair])),\n        \"inliers_per_pair\": float(np.mean([row[\"n_inliers\"] for row in per_pair])),\n        \"median_error_px\": float(np.median(pooled)) if pooled else math.inf,\n        \"homography_acc_3px\": float(np.mean([row[\"corner_error_px\"] < 3.0 for row in per_pair])),\n        \"homography_acc_5px\": float(np.mean([row[\"corner_error_px\"] < 5.0 for row in per_pair])),\n        \"definitions\": METRIC_DEFINITIONS,\n    }\n    for t in THRESHOLDS:\n        key = f\"precision_{int(t)}px\"\n        out[key] = float(np.mean([row[key] for row in per_pair]))\n    return out\n\n\n# --------------------------------------------------------------------------------------------------\n# non-neural baselines\n# --------------------------------------------------------------------------------------------------\n\n\ndef grid_points(size: tuple[int, int], step: int = 32, margin: int = 16) -> np.ndarray:\n    width, height = size\n    xs = np.arange(margin, width - margin, step, dtype=np.float64)\n    ys = np.arange(margin, height - margin, step, dtype=np.float64)\n    gx, gy = np.meshgrid(xs, ys)\n    return np.stack([gx.ravel(), gy.ravel()], axis=1)\n\n\ndef identity_baseline(image0: Image.Image, image1: Image.Image, *, step: int = 32) -> dict[str, Any]:\n    \"\"\"Every grid point of image0 is matched to the same coordinates in image1 (no motion assumed).\"\"\"\n    pts = grid_points(image0.size, step=step)\n    return {\"kpts0\": pts, \"kpts1\": pts.copy(), \"confidence\": np.ones(len(pts)), \"baseline\": \"identity guess\"}\n\n\ndef _gray(image: Image.Image) -> np.ndarray:\n    return np.asarray(image.convert(\"L\"), dtype=np.float64)\n\n\ndef _ncc(patch: np.ndarray, window: np.ndarray) -> np.ndarray:\n    \"\"\"Normalised cross-correlation of a (p, p) patch over every (p, p) position of a (h, w) window.\"\"\"\n    p = patch.shape[0]\n    h, w = window.shape\n    if h < p or w < p:\n        return np.zeros((0, 0))\n    strides = np.lib.stride_tricks.sliding_window_view(window, (p, p))  # (h-p+1, w-p+1, p, p)\n    tiles = strides.reshape(strides.shape[0], strides.shape[1], -1)\n    tiles = tiles - tiles.mean(axis=2, keepdims=True)\n    flat = (patch - patch.mean()).ravel()\n    denom = np.sqrt((tiles**2).sum(axis=2) * (flat**2).sum()) + 1e-9\n    return (tiles @ flat) / denom\n\n\ndef patch_neighbour_baseline(\n    image0: Image.Image,\n    image1: Image.Image,\n    *,\n    step: int = 32,\n    patch: int = 15,\n    search: int = 48,\n) -> dict[str, Any]:\n    \"\"\"For each grid point of image0, the position in image1 (within ±`search` px) whose `patch` × `patch`\n    neighbourhood has the highest normalised cross-correlation with the point's own patch.\"\"\"\n    g0, g1 = _gray(image0), _gray(image1)\n    half = patch // 2\n    pts0 = grid_points(image0.size, step=step, margin=max(16, half + 1))\n    kpts0, kpts1, conf = [], [], []\n    for x, y in pts0:\n        xi, yi = int(round(x)), int(round(y))\n        tile = g0[yi - half : yi + half + 1, xi - half : xi + half + 1]\n        y0, y1 = max(0, yi - search - half), min(g1.shape[0], yi + search + half + 1)\n        x0, x1 = max(0, xi - search - half), min(g1.shape[1], xi + search + half + 1)\n        scores = _ncc(tile, g1[y0:y1, x0:x1])\n        if scores.size == 0:\n            continue\n        best = np.unravel_index(int(np.argmax(scores)), scores.shape)\n        kpts0.append([x, y])\n        kpts1.append([x0 + best[1] + half, y0 + best[0] + half])\n        conf.append(float(scores[best]))\n    return {\n        \"kpts0\": np.asarray(kpts0, dtype=np.float64).reshape(-1, 2),\n        \"kpts1\": np.asarray(kpts1, dtype=np.float64).reshape(-1, 2),\n        \"confidence\": np.asarray(conf, dtype=np.float64),\n        \"baseline\": f\"patch nearest neighbour ({patch}x{patch} NCC, ±{search} px search)\",\n    }\n","xoftr_pipeline/model.py":"from __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom collections.abc import Callable\nfrom pathlib import Path\nfrom typing import Any\n\nimport torch\nfrom huggingface_hub import snapshot_download\n\nfrom .config import (\n    ALLOWED_CHECKPOINT_FILES,\n    COARSE_THRESHOLD,\n    DEFAULT_MODEL_KEY,\n    FINE_THRESHOLD,\n    MODEL_FILENAME,\n    MODEL_ID,\n    MODEL_REVISION,\n    MODEL_SHA256,\n    MODEL_SIZE_BYTES,\n    PARAMETER_COUNT,\n    STATE_TENSORS,\n    UNSAFE_WEIGHT_EXTENSIONS,\n)\n\nMANIFEST_NAME = \"dimer-base-manifest.json\"\n#: Fleet snapshot scheme (DIMER NOTEBOOK_SPEC 1.1 MOD13): the pinned files live in a repository-\n#: local snapshot directory named by the model key and described by the committed manifest; a\n#: standalone notebook carries that manifest inline and stages/verifies a working-directory copy.\nDEFAULT_WEIGHTS_DIR = Path(__file__).resolve().parents[2] / \"weights\" / DEFAULT_MODEL_KEY\n\n\ndef _sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open(\"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef verify_checkpoint(\n    snapshot_path: str | Path,\n    *,\n    require_configs: bool = False,\n    return_manifest_verified: bool = False,\n) -> Path | tuple[Path, bool]:\n    root = Path(snapshot_path)\n    if not root.is_dir():\n        raise RuntimeError(f\"Checkpoint directory does not exist: {root}\")\n\n    weight_path = root / MODEL_FILENAME\n    if not weight_path.is_file():\n        raise RuntimeError(f\"Pinned checkpoint is missing {MODEL_FILENAME}\")\n\n    unsafe = sorted(\n        p.name\n        for p in root.iterdir()\n        if p.is_file() and p.suffix.lower() in UNSAFE_WEIGHT_EXTENSIONS\n    )\n    if unsafe:\n        raise RuntimeError(f\"Refusing unsafe weight files: {unsafe}\")\n\n    manifest_path = root / MANIFEST_NAME\n    manifest_verified = False\n\n    if manifest_path.is_file():\n        try:\n            manifest = json.loads(manifest_path.read_text(encoding=\"utf-8\"))\n        except Exception as exc:\n            raise RuntimeError(f\"Corrupt manifest {MANIFEST_NAME}: {exc}\") from exc\n\n        files = manifest.get(\"files\") or []\n        if not files:\n            raise RuntimeError(f\"Manifest {MANIFEST_NAME} contains no files\")\n\n        for entry in files:\n            rel_path = entry.get(\"path\")\n            if not rel_path:\n                continue\n            target = root / rel_path\n            if not target.is_file():\n                raise RuntimeError(f\"Manifest file missing: {rel_path}\")\n            exp_bytes = entry.get(\"bytes\")\n            if exp_bytes is not None and target.stat().st_size != exp_bytes:\n                raise RuntimeError(\n                    f\"Size mismatch for {rel_path}: {target.stat().st_size} != {exp_bytes}\"\n                )\n            exp_sha = entry.get(\"sha256\")\n            if exp_sha is not None and _sha256(target) != exp_sha:\n                raise RuntimeError(f\"SHA-256 mismatch for {rel_path}\")\n\n        manifest_verified = True\n\n    size = weight_path.stat().st_size\n    if size != MODEL_SIZE_BYTES:\n        raise RuntimeError(f\"Unexpected {MODEL_FILENAME} size: {size}; expected {MODEL_SIZE_BYTES}\")\n\n    digest = _sha256(weight_path)\n    if digest != MODEL_SHA256:\n        raise RuntimeError(\n            f\"Unexpected {MODEL_FILENAME} SHA-256: {digest}; expected {MODEL_SHA256}\"\n        )\n\n    # The network's configuration is carried in code (modeling.default_config); with\n    # require_configs there is nothing else to require.\n\n    if return_manifest_verified:\n        return root, manifest_verified\n    return root\n\n\ndef _read_manifest(root: Path) -> dict[str, Any]:\n    \"\"\"Load and identity-check ``<root>/dimer-base-manifest.json``.\"\"\"\n    manifest_path = root / MANIFEST_NAME\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f\"snapshot manifest not found: {manifest_path}\")\n    try:\n        manifest = json.loads(manifest_path.read_text(encoding=\"utf-8\"))\n    except ValueError as exc:\n        raise RuntimeError(f\"Corrupt manifest {MANIFEST_NAME}: {exc}\") from exc\n    if manifest.get(\"modelId\") != MODEL_ID or manifest.get(\"revision\") != MODEL_REVISION:\n        raise ValueError(\n            f\"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, \"\n            f\"package pins {MODEL_ID}@{MODEL_REVISION}; refusing\"\n        )\n    if not manifest.get(\"files\"):\n        raise RuntimeError(f\"Manifest {MANIFEST_NAME} contains no files\")\n    return manifest\n\n\ndef verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:\n    \"\"\"Manifest-driven verification of a fleet snapshot directory; raise on the first mismatch.\n\n    The identity in the manifest must be the pinned one; every manifest entry is then size- and\n    SHA-256-checked by :func:`verify_checkpoint` (the existing verifier, which also asserts the\n    weight file's pinned digest and byte count and refuses unsafe formats). Returns\n    ``{\"path\": ..., **manifest}``.\n    \"\"\"\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    manifest = _read_manifest(root)\n    _, manifest_verified = verify_checkpoint(\n        root, require_configs=True, return_manifest_verified=True\n    )\n    if not manifest_verified:\n        raise RuntimeError(f\"manifest at {root} was not verified\")  # pragma: no cover\n    return {\"path\": str(root), **manifest}\n\n\ndef _hub_download(relative_path: str, root: Path) -> None:\n    \"\"\"Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory.\"\"\"\n    from huggingface_hub import hf_hub_download\n\n    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))\n\n\ndef stage_missing_files(\n    path: str | Path | None = None,\n    *,\n    allow_download: bool = False,\n    downloader: Callable[[str, Path], None] | None = None,\n) -> list[str]:\n    \"\"\"Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest and\n    the small files but git-ignores the weights). Returns the relative paths fetched;\n    :func:`verify_snapshot` still runs after.\"\"\"\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    manifest = _read_manifest(root)\n    missing = [entry[\"path\"] for entry in manifest[\"files\"] if not (root / entry[\"path\"]).is_file()]\n    if not missing:\n        return []\n    if not allow_download:\n        raise FileNotFoundError(\n            f\"snapshot at {root} is missing {missing}; \"\n            f\"pass allow_download=True to fetch them at {MODEL_REVISION}\"\n        )\n    fetch = downloader or _hub_download\n    for relative_path in missing:\n        fetch(relative_path, root)\n    return missing\n\n\ndef resolve_weights_path(\n    weights_path: str | Path | None = None,\n    cache_dir: str | Path | None = None,\n) -> tuple[Path, str]:\n    \"\"\"Resolve weights path with precedence:\n\n    1. Explicit argument `weights_path` -> 'explicit_path'\n    2. Environment variable `XOFTR_WEIGHTS_DIR` -> 'env_var'\n    3. Source checkout convention `weights/xoftr` -> 'repo_offline'\n       (only if pyproject.toml exists at repo root and weights/ contains xoftr_640.safetensors)\n    4. Hugging Face Hub snapshot download -> 'hf_hub'\n    \"\"\"\n    if weights_path is not None:\n        return Path(weights_path), \"explicit_path\"\n\n    env_dir = os.environ.get(\"XOFTR_WEIGHTS_DIR\")\n    if env_dir:\n        return Path(env_dir), \"env_var\"\n\n    repo_root = Path(__file__).resolve().parents[2]\n    if (repo_root / \"pyproject.toml\").is_file():\n        repo_weights = repo_root / \"weights\" / DEFAULT_MODEL_KEY\n        if (repo_weights / MODEL_FILENAME).is_file():\n            return repo_weights, \"repo_offline\"\n\n    hub_path = Path(\n        snapshot_download(\n            repo_id=MODEL_ID,\n            revision=MODEL_REVISION,\n            allow_patterns=list(ALLOWED_CHECKPOINT_FILES),\n            cache_dir=str(cache_dir) if cache_dir is not None else None,\n        )\n    )\n    return hub_path, \"hf_hub\"\n\n\n_resolve_weights_path = resolve_weights_path\n\n\ndef _resolve_device(device: str | torch.device | None) -> torch.device:\n    if device is not None:\n        return torch.device(device)\n    return torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n\n\ndef build_model(\n    *, coarse_threshold: float = COARSE_THRESHOLD, fine_threshold: float = FINE_THRESHOLD\n) -> Any:\n    \"\"\"The vendored XoFTR network at random initialisation, in inference configuration.\"\"\"\n    from .modeling import XoFTR, default_config\n\n    return XoFTR(default_config(coarse_thr=coarse_threshold, fine_thr=fine_threshold))\n\n\ndef load_components(\n    *,\n    device: str | torch.device | None = None,\n    cache_dir: str | Path | None = None,\n    weights_path: str | Path | None = None,\n    return_metadata: bool = False,\n    coarse_threshold: float = COARSE_THRESHOLD,\n    fine_threshold: float = FINE_THRESHOLD,\n) -> tuple[Any, torch.device, Path] | tuple[Any, torch.device, Path, dict[str, Any]]:\n    \"\"\"Acquire, verify, and load the one supported XoFTR checkpoint into the vendored network\n    (strict state-dict load from safetensors; no pickle, no Hub code).\"\"\"\n    from safetensors.torch import load_file\n\n    candidate_path, source = resolve_weights_path(\n        weights_path=weights_path,\n        cache_dir=cache_dir,\n    )\n\n    verified, manifest_verified = verify_checkpoint(\n        candidate_path,\n        require_configs=True,\n        return_manifest_verified=True,\n    )\n    target_device = _resolve_device(device)\n\n    weight_file = verified / MODEL_FILENAME\n    state = load_file(str(weight_file))\n    if len(state) != STATE_TENSORS:\n        raise RuntimeError(f\"{MODEL_FILENAME}: {len(state)} tensors, expected {STATE_TENSORS}\")\n    model = build_model(coarse_threshold=coarse_threshold, fine_threshold=fine_threshold)\n    # the vendored class strips the checkpoint's `matcher.` prefix\n    model.load_state_dict(state, strict=True)\n    n_params = sum(p.numel() for p in model.parameters())\n    if n_params != PARAMETER_COUNT:\n        raise RuntimeError(\n            f\"vendored network has {n_params} parameters, expected {PARAMETER_COUNT}\"\n        )\n    model = model.eval().to(target_device)\n\n    metadata: dict[str, Any] = {\n        \"checkpoint_path\": verified,\n        \"checkpoint_source\": source,\n        \"manifest_verified\": manifest_verified,\n        \"weight_sha256\": _sha256(weight_file),\n        \"weight_size_bytes\": weight_file.stat().st_size,\n        \"device\": str(target_device),\n        \"state_tensors\": len(state),\n        \"parameters\": n_params,\n    }\n\n    if return_metadata:\n        return model, target_device, verified, metadata\n    return model, target_device, verified\n","xoftr_pipeline/modeling.py":"\"\"\"XoFTR (Tuzcuoğlu, Köksal, Sofu, Kalkan and Alatan, CVPRW 2024) inference network, vendored from\nhttps://github.com/OnderT/XoFTR at commit e0fbea431b30be9742effbf5577c90aa8eb938f9 (Apache-2.0):\n``src/xoftr/backbone/resnet.py``, ``src/xoftr/utils/position_encoding.py`` and ``src/xoftr/xoftr_module/*``\nconcatenated in dependency order with the package-relative imports removed, plus the inference configuration\nfrom ``src/config/default.py`` (``get_cfg_defaults(inference=True)`` lowered to a plain dict, as\nimage-matching-models builds it). No training utilities, datasets, Lightning or kornia code is carried; the\nsix ``einops.rearrange`` patterns upstream uses are provided by a local ``rearrange`` shim (plain ``reshape`` /\n``permute``), so torch is the only dependency.\n\nThe state-dict key layout of the Hub checkpoints (``vismatch/xoftr``, formerly ``image-matching-models/xoftr``)\nmatches this module exactly: ``backbone.*``, ``pos_encoding.*``, ``loftr_coarse.*``, ``coarse_matching.*``,\n``fine_process.*``, ``fine_matching.*``.\n\"\"\"\n# ruff: noqa: E501, N801, N802, N803, N806, E741, B905, F841, E712  -- vendored code kept as upstream wrote it, for auditability\n\nfrom __future__ import annotations\n\nimport copy\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.nn import Dropout, Module\n\n\ndef rearrange(tensor: torch.Tensor, pattern: str, **axes: int) -> torch.Tensor:\n    \"\"\"The six ``einops.rearrange`` patterns the upstream modules use, as plain ``view`` / ``permute`` so the\n    vendored code reads exactly as upstream without an einops dependency. Any other pattern is refused.\"\"\"\n    if pattern == \"b (h0c w0c) (h1c w1c) -> b h0c w0c h1c w1c\":\n        return tensor.reshape(tensor.shape[0], axes[\"h0c\"], axes[\"w0c\"], axes[\"h1c\"], axes[\"w1c\"])\n    if pattern == \"b h0c w0c h1c w1c -> b (h0c w0c) (h1c w1c)\":\n        return tensor.reshape(tensor.shape[0], axes[\"h0c\"] * axes[\"w0c\"], axes[\"h1c\"] * axes[\"w1c\"])\n    if pattern == \"n (h w) c -> n c h w\":\n        n, _, c = tensor.shape\n        return tensor.reshape(n, axes[\"h\"], axes[\"w\"], c).permute(0, 3, 1, 2)\n    if pattern == \"n c h w -> n (h w) 1 c\":\n        return tensor.flatten(2).permute(0, 2, 1).unsqueeze(2)\n    if pattern == \"n (c ww) l -> n l ww c\":\n        n, cww, l = tensor.shape\n        return tensor.reshape(n, cww // axes[\"ww\"], axes[\"ww\"], l).permute(0, 3, 2, 1)\n    if pattern == \"n c h w -> n (h w) c\":\n        return tensor.flatten(2).permute(0, 2, 1)\n    raise ValueError(f\"unsupported rearrange pattern: {pattern!r}\")\n\nUPSTREAM_REPOSITORY = \"https://github.com/OnderT/XoFTR\"\nUPSTREAM_COMMIT = \"e0fbea431b30be9742effbf5577c90aa8eb938f9\"\nRESOLUTION = (8, 2)\n\n\ndef default_config(*, coarse_thr: float = 0.3, fine_thr: float = 0.1, denser: bool = False) -> dict:\n    \"\"\"``get_cfg_defaults(inference=True)`` -> ``lower_config`` -> ``[\"xoftr\"]`` from upstream ``src/config/default.py``,\n    with the three knobs image-matching-models exposes.\"\"\"\n    return {\n        \"resolution\": RESOLUTION,\n        \"fine_window_size\": 5,\n        \"medium_window_size\": 3,\n        \"resnet\": {\"initial_dim\": 128, \"block_dims\": [128, 196, 256]},\n        \"coarse\": {\n            \"inference\": True,\n            \"d_model\": 256,\n            \"d_ffn\": 256,\n            \"nhead\": 8,\n            \"layer_names\": [\"self\", \"cross\"] * 4,\n            \"attention\": \"linear\",\n        },\n        \"match_coarse\": {\n            \"inference\": True,\n            \"d_model\": 256,\n            \"thr\": coarse_thr,\n            \"border_rm\": 2,\n            \"match_type\": \"dual_softmax\",\n            \"dsmax_temperature\": 0.1,\n            \"train_coarse_percent\": 0.2,\n            \"train_pad_num_gt_min\": 200,\n        },\n        \"fine\": {\n            \"denser\": denser,\n            \"inference\": True,\n            \"dsmax_temperature\": 0.1,\n            \"thr\": fine_thr,\n            \"mlp_hidden_dim_coef\": 2,\n            \"nhead_fine_level\": 8,\n            \"nhead_medium_level\": 7,\n        },\n        \"loss\": {\n            \"focal_alpha\": 0.25,\n            \"focal_gamma\": 2.0,\n            \"pos_weight\": 1.0,\n            \"neg_weight\": 1.0,\n            \"coarse_weight\": 0.5,\n            \"fine_weight\": 0.3,\n            \"sub_weight\": 1 * 10**4,\n        },\n    }\n\n\n\n# ----------------------------------------------------------------------------------------------------\n# upstream src/xoftr/backbone/resnet.py\n# ----------------------------------------------------------------------------------------------------\n\ndef conv1x1(in_planes, out_planes, stride=1):\n    \"\"\"1x1 convolution without padding\"\"\"\n    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, padding=0, bias=False)\n\n\ndef conv3x3(in_planes, out_planes, stride=1):\n    \"\"\"3x3 convolution with padding\"\"\"\n    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)\n\n\nclass BasicBlock(nn.Module):\n    def __init__(self, in_planes, planes, stride=1):\n        super().__init__()\n        self.conv1 = conv3x3(in_planes, planes, stride)\n        self.conv2 = conv3x3(planes, planes)\n        self.bn1 = nn.BatchNorm2d(planes)\n        self.bn2 = nn.BatchNorm2d(planes)\n        self.relu = nn.ReLU(inplace=True)\n\n        if stride == 1:\n            self.downsample = None\n        else:\n            self.downsample = nn.Sequential(\n                conv1x1(in_planes, planes, stride=stride),\n                nn.BatchNorm2d(planes)\n            )\n\n    def forward(self, x):\n        y = x\n        y = self.relu(self.bn1(self.conv1(y)))\n        y = self.bn2(self.conv2(y))\n\n        if self.downsample is not None:\n            x = self.downsample(x)\n\n        return self.relu(x+y)\n\nclass ResNet_8_2(nn.Module):\n    \"\"\"\n    ResNet, output resolution are 1/8 and 1/2.\n    Each block has 2 layers.\n    \"\"\"\n\n    def __init__(self, config):\n        super().__init__()\n        # Config\n        block = BasicBlock\n        initial_dim = config['initial_dim']\n        block_dims = config['block_dims']\n\n        # Class Variable\n        self.in_planes = initial_dim\n\n        # Networks\n        self.conv1 = nn.Conv2d(1, initial_dim, kernel_size=7, stride=2, padding=3, bias=False)\n        self.bn1 = nn.BatchNorm2d(initial_dim)\n        self.relu = nn.ReLU(inplace=True)\n\n        self.layer1 = self._make_layer(block, block_dims[0], stride=1)  # 1/2\n        self.layer2 = self._make_layer(block, block_dims[1], stride=2)  # 1/4\n        self.layer3 = self._make_layer(block, block_dims[2], stride=2)  # 1/8\n\n        self.layer3_outconv = conv1x1(block_dims[2], block_dims[2])\n\n\n        for m in self.modules():\n            if isinstance(m, nn.Conv2d):\n                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')\n            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):\n                nn.init.constant_(m.weight, 1)\n                nn.init.constant_(m.bias, 0)\n\n    def _make_layer(self, block, dim, stride=1):\n        layer1 = block(self.in_planes, dim, stride=stride)\n        layer2 = block(dim, dim, stride=1)\n        layers = (layer1, layer2)\n\n        self.in_planes = dim\n        return nn.Sequential(*layers)\n\n    def forward(self, x):\n        # ResNet Backbone\n        x0 = self.relu(self.bn1(self.conv1(x)))\n        x1 = self.layer1(x0)  # 1/2\n        x2 = self.layer2(x1)  # 1/4\n        x3 = self.layer3(x2)  # 1/8\n\n        x3_out = self.layer3_outconv(x3)\n\n        return x3_out, x2, x1\n\n\n# ----------------------------------------------------------------------------------------------------\n# upstream src/xoftr/utils/position_encoding.py\n# ----------------------------------------------------------------------------------------------------\n\nclass PositionEncodingSine(nn.Module):\n    \"\"\"\n    This is a sinusoidal position encoding that generalized to 2-dimensional images\n    \"\"\"\n\n    def __init__(self, d_model, max_shape=(256, 256)):\n        \"\"\"\n        Args:\n            max_shape (tuple): for 1/8 featmap, the max length of 256 corresponds to 2048 pixels\n        \"\"\"\n        super().__init__()\n\n        pe = torch.zeros((d_model, *max_shape))\n        y_position = torch.ones(max_shape).cumsum(0).float().unsqueeze(0)\n        x_position = torch.ones(max_shape).cumsum(1).float().unsqueeze(0)\n        div_term = torch.exp(torch.arange(0, d_model//2, 2).float() * (-math.log(10000.0) / (d_model//2)))\n\n        div_term = div_term[:, None, None]  # [C//4, 1, 1]\n        pe[0::4, :, :] = torch.sin(x_position * div_term)\n        pe[1::4, :, :] = torch.cos(x_position * div_term)\n        pe[2::4, :, :] = torch.sin(y_position * div_term)\n        pe[3::4, :, :] = torch.cos(y_position * div_term)\n\n        self.register_buffer('pe', pe.unsqueeze(0), persistent=False)  # [1, C, H, W]\n\n    def forward(self, x):\n        \"\"\"\n        Args:\n            x: [N, C, H, W]\n        \"\"\"\n        return x + self.pe[:, :, :x.size(2), :x.size(3)]\n\n\n# ----------------------------------------------------------------------------------------------------\n# upstream src/xoftr/xoftr_module/linear_attention.py\n# ----------------------------------------------------------------------------------------------------\n\n\"\"\"\nLinear Transformer proposed in \"Transformers are RNNs: Fast Autoregressive Transformers with Linear Attention\"\nModified from: https://github.com/idiap/fast-transformers/blob/master/fast_transformers/attention/linear_attention.py\n\"\"\"\n\n\n\ndef elu_feature_map(x):\n    return torch.nn.functional.elu(x) + 1\n\n\nclass LinearAttention(Module):\n    def __init__(self, eps=1e-6):\n        super().__init__()\n        self.feature_map = elu_feature_map\n        self.eps = eps\n\n    def forward(self, queries, keys, values, q_mask=None, kv_mask=None):\n        \"\"\" Multi-Head linear attention proposed in \"Transformers are RNNs\"\n        Args:\n            queries: [N, L, H, D]\n            keys: [N, S, H, D]\n            values: [N, S, H, D]\n            q_mask: [N, L]\n            kv_mask: [N, S]\n        Returns:\n            queried_values: (N, L, H, D)\n        \"\"\"\n        Q = self.feature_map(queries)\n        K = self.feature_map(keys)\n\n        # set padded position to zero\n        if q_mask is not None:\n            Q = Q * q_mask[:, :, None, None]\n        if kv_mask is not None:\n            K = K * kv_mask[:, :, None, None]\n            values = values * kv_mask[:, :, None, None]\n\n        v_length = values.size(1)\n        values = values / v_length  # prevent fp16 overflow\n        KV = torch.einsum(\"nshd,nshv->nhdv\", K, values)  # (S,D)' @ S,V\n        Z = 1 / (torch.einsum(\"nlhd,nhd->nlh\", Q, K.sum(dim=1)) + self.eps)\n        queried_values = torch.einsum(\"nlhd,nhdv,nlh->nlhv\", Q, KV, Z) * v_length\n\n        return queried_values.contiguous()\n\n\nclass FullAttention(Module):\n    def __init__(self, use_dropout=False, attention_dropout=0.1):\n        super().__init__()\n        self.use_dropout = use_dropout\n        self.dropout = Dropout(attention_dropout)\n\n    def forward(self, queries, keys, values, q_mask=None, kv_mask=None):\n        \"\"\" Multi-head scaled dot-product attention, a.k.a full attention.\n        Args:\n            queries: [N, L, H, D]\n            keys: [N, S, H, D]\n            values: [N, S, H, D]\n            q_mask: [N, L]\n            kv_mask: [N, S]\n        Returns:\n            queried_values: (N, L, H, D)\n        \"\"\"\n\n        # Compute the unnormalized attention and apply the masks\n        QK = torch.einsum(\"nlhd,nshd->nlsh\", queries, keys)\n        if kv_mask is not None:\n            QK.masked_fill_(~(q_mask[:, :, None, None] * kv_mask[:, None, :, None]), float('-inf'))\n\n        # Compute the attention and the weighted average\n        softmax_temp = 1. / queries.size(3)**.5  # sqrt(D)\n        A = torch.softmax(softmax_temp * QK, dim=2)\n        if self.use_dropout:\n            A = self.dropout(A)\n\n        queried_values = torch.einsum(\"nlsh,nshd->nlhd\", A, values)\n\n        return queried_values.contiguous()\n\n\n# ----------------------------------------------------------------------------------------------------\n# upstream src/xoftr/xoftr_module/transformer.py\n# ----------------------------------------------------------------------------------------------------\n\nclass LoFTREncoderLayer(nn.Module):\n    def __init__(self,\n                 d_model,\n                 nhead,\n                 attention='linear'):\n        super().__init__()\n\n        self.dim = d_model // nhead\n        self.nhead = nhead\n\n        # multi-head attention\n        self.q_proj = nn.Linear(d_model, d_model, bias=False)\n        self.k_proj = nn.Linear(d_model, d_model, bias=False)\n        self.v_proj = nn.Linear(d_model, d_model, bias=False)\n        self.attention = LinearAttention() if attention == 'linear' else FullAttention()\n        self.merge = nn.Linear(d_model, d_model, bias=False)\n\n        # feed-forward network\n        self.mlp = nn.Sequential(\n            nn.Linear(d_model*2, d_model*2, bias=False),\n            nn.ReLU(True),\n            nn.Linear(d_model*2, d_model, bias=False),\n        )\n\n        # norm and dropout\n        self.norm1 = nn.LayerNorm(d_model)\n        self.norm2 = nn.LayerNorm(d_model)\n\n    def forward(self, x, source, x_mask=None, source_mask=None):\n        \"\"\"\n        Args:\n            x (torch.Tensor): [N, L, C]\n            source (torch.Tensor): [N, S, C]\n            x_mask (torch.Tensor): [N, L] (optional)\n            source_mask (torch.Tensor): [N, S] (optional)\n        \"\"\"\n        bs = x.size(0)\n        query, key, value = x, source, source\n\n        # multi-head attention\n        query = self.q_proj(query).view(bs, -1, self.nhead, self.dim)  # [N, L, (H, D)]\n        key = self.k_proj(key).view(bs, -1, self.nhead, self.dim)  # [N, S, (H, D)]\n        value = self.v_proj(value).view(bs, -1, self.nhead, self.dim)\n        message = self.attention(query, key, value, q_mask=x_mask, kv_mask=source_mask)  # [N, L, (H, D)]\n        message = self.merge(message.view(bs, -1, self.nhead*self.dim))  # [N, L, C]\n        message = self.norm1(message)\n\n        # feed-forward network\n        message = self.mlp(torch.cat([x, message], dim=2))\n        message = self.norm2(message)\n\n        return x + message\n\n\nclass LocalFeatureTransformer(nn.Module):\n    \"\"\"A Local Feature Transformer (LoFTR) module.\"\"\"\n\n    def __init__(self, config):\n        super().__init__()\n\n        self.config = config\n        self.d_model = config['d_model']\n        self.nhead = config['nhead']\n        self.layer_names = config['layer_names']\n        encoder_layer = LoFTREncoderLayer(config['d_model'], config['nhead'], config['attention'])\n        self.layers = nn.ModuleList([copy.deepcopy(encoder_layer) for _ in range(len(self.layer_names))])\n        self._reset_parameters()\n\n    def _reset_parameters(self):\n        for p in self.parameters():\n            if p.dim() > 1:\n                nn.init.xavier_uniform_(p)\n\n    def forward(self, feat0, feat1, mask0=None, mask1=None):\n        \"\"\"\n        Args:\n            feat0 (torch.Tensor): [N, L, C]\n            feat1 (torch.Tensor): [N, S, C]\n            mask0 (torch.Tensor): [N, L] (optional)\n            mask1 (torch.Tensor): [N, S] (optional)\n        \"\"\"\n\n        assert self.d_model == feat0.size(2), \"the feature number of src and transformer must be equal\"\n\n        for layer, name in zip(self.layers, self.layer_names):\n            if name == 'self':\n                feat0 = layer(feat0, feat0, mask0, mask0)\n                feat1 = layer(feat1, feat1, mask1, mask1)\n            elif name == 'cross':\n                feat0 = layer(feat0, feat1, mask0, mask1)\n                feat1 = layer(feat1, feat0, mask1, mask0)\n            else:\n                raise KeyError\n\n        return feat0, feat1\n\n\n# ----------------------------------------------------------------------------------------------------\n# upstream src/xoftr/xoftr_module/coarse_matching.py\n# ----------------------------------------------------------------------------------------------------\n\nINF = 1e9\n\ndef mask_border(m, b: int, v):\n    \"\"\" Mask borders with value\n    Args:\n        m (torch.Tensor): [N, H0, W0, H1, W1]\n        b (int)\n        v (m.dtype)\n    \"\"\"\n    if b <= 0:\n        return\n\n    m[:, :b] = v\n    m[:, :, :b] = v\n    m[:, :, :, :b] = v\n    m[:, :, :, :, :b] = v\n    m[:, -b:] = v\n    m[:, :, -b:] = v\n    m[:, :, :, -b:] = v\n    m[:, :, :, :, -b:] = v\n\n\ndef mask_border_with_padding(m, bd, v, p_m0, p_m1):\n    if bd <= 0:\n        return\n\n    m[:, :bd] = v\n    m[:, :, :bd] = v\n    m[:, :, :, :bd] = v\n    m[:, :, :, :, :bd] = v\n\n    h0s, w0s = p_m0.sum(1).max(-1)[0].int(), p_m0.sum(-1).max(-1)[0].int()\n    h1s, w1s = p_m1.sum(1).max(-1)[0].int(), p_m1.sum(-1).max(-1)[0].int()\n    for b_idx, (h0, w0, h1, w1) in enumerate(zip(h0s, w0s, h1s, w1s)):\n        m[b_idx, h0 - bd:] = v\n        m[b_idx, :, w0 - bd:] = v\n        m[b_idx, :, :, h1 - bd:] = v\n        m[b_idx, :, :, :, w1 - bd:] = v\n\n\ndef compute_max_candidates(p_m0, p_m1):\n    \"\"\"Compute the max candidates of all pairs within a batch\n\n    Args:\n        p_m0, p_m1 (torch.Tensor): padded masks\n    \"\"\"\n    h0s, w0s = p_m0.sum(1).max(-1)[0], p_m0.sum(-1).max(-1)[0]\n    h1s, w1s = p_m1.sum(1).max(-1)[0], p_m1.sum(-1).max(-1)[0]\n    max_cand = torch.sum(\n        torch.min(torch.stack([h0s * w0s, h1s * w1s], -1), -1)[0])\n    return max_cand\n\n\nclass CoarseMatching(nn.Module):\n    def __init__(self, config):\n        super().__init__()\n        self.config = config\n        # general config\n        d_model = config['d_model']\n        self.thr = config['thr']\n        self.inference = config['inference']\n        self.border_rm = config['border_rm']\n        # -- # for trainig fine-level XoFTR\n        self.train_coarse_percent = config['train_coarse_percent']\n        self.train_pad_num_gt_min = config['train_pad_num_gt_min']\n        self.final_proj = nn.Linear(d_model, d_model, bias=True)\n\n        self.temperature = config['dsmax_temperature']\n\n    def forward(self, feat_c0, feat_c1, data, mask_c0=None, mask_c1=None):\n        \"\"\"\n        Args:\n            feat0 (torch.Tensor): [N, L, C]\n            feat1 (torch.Tensor): [N, S, C]\n            data (dict)\n            mask_c0 (torch.Tensor): [N, L] (optional)\n            mask_c1 (torch.Tensor): [N, S] (optional)\n        Update:\n            data (dict): {\n                'b_ids' (torch.Tensor): [M'],\n                'i_ids' (torch.Tensor): [M'],\n                'j_ids' (torch.Tensor): [M'],\n                'gt_mask' (torch.Tensor): [M'],\n                'mkpts0_c' (torch.Tensor): [M, 2],\n                'mkpts1_c' (torch.Tensor): [M, 2],\n                'mconf' (torch.Tensor): [M]}\n            NOTE: M' != M during training.\n        \"\"\"\n\n        feat_c0 = self.final_proj(feat_c0)\n        feat_c1 = self.final_proj(feat_c1)\n\n        # normalize\n        feat_c0, feat_c1 = map(lambda feat: feat / feat.shape[-1]**.5,\n                               [feat_c0, feat_c1])\n\n        sim_matrix = torch.einsum(\"nlc,nsc->nls\", feat_c0,\n                                    feat_c1) / self.temperature\n        if mask_c0 is not None:\n            sim_matrix.masked_fill_(\n                ~(mask_c0[..., None] * mask_c1[:, None]).bool(),\n                -INF)\n        if self.inference:\n            # predict coarse matches from conf_matrix\n            data.update(**self.get_coarse_match_inference(sim_matrix, data))\n        else:\n            conf_matrix_0_to_1 = F.softmax(sim_matrix, 2)\n            conf_matrix_1_to_0 = F.softmax(sim_matrix, 1)\n            data.update({'conf_matrix_0_to_1': conf_matrix_0_to_1,\n                        'conf_matrix_1_to_0': conf_matrix_1_to_0\n                        })\n            # predict coarse matches from conf_matrix\n            data.update(**self.get_coarse_match_training(conf_matrix_0_to_1, conf_matrix_1_to_0, data))\n\n    @torch.no_grad()\n    def get_coarse_match_training(self, conf_matrix_0_to_1, conf_matrix_1_to_0, data):\n        \"\"\"\n        Args:\n            conf_matrix_0_to_1 (torch.Tensor): [N, L, S]\n            conf_matrix_1_to_0 (torch.Tensor): [N, L, S]\n            data (dict): with keys ['hw0_i', 'hw1_i', 'hw0_c', 'hw1_c']\n        Returns:\n            coarse_matches (dict): {\n                'b_ids' (torch.Tensor): [M'],\n                'i_ids' (torch.Tensor): [M'],\n                'j_ids' (torch.Tensor): [M'],\n                'gt_mask' (torch.Tensor): [M'],\n                'm_bids' (torch.Tensor): [M],\n                'mkpts0_c' (torch.Tensor): [M, 2],\n                'mkpts1_c' (torch.Tensor): [M, 2],\n                'mconf' (torch.Tensor): [M]}\n        \"\"\"\n        axes_lengths = {\n            'h0c': data['hw0_c'][0],\n            'w0c': data['hw0_c'][1],\n            'h1c': data['hw1_c'][0],\n            'w1c': data['hw1_c'][1]\n        }\n        _device = conf_matrix_0_to_1.device\n\n        # confidence thresholding\n        # {(nearest neighbour for 0 to 1) U (nearest neighbour for 1 to 0)}\n        mask = torch.logical_or((conf_matrix_0_to_1 > self.thr) * (conf_matrix_0_to_1 == conf_matrix_0_to_1.max(dim=2, keepdim=True)[0]),\n                               (conf_matrix_1_to_0 > self.thr) * (conf_matrix_1_to_0 == conf_matrix_1_to_0.max(dim=1, keepdim=True)[0]))\n\n        mask = rearrange(mask, 'b (h0c w0c) (h1c w1c) -> b h0c w0c h1c w1c',\n                         **axes_lengths)\n        if 'mask0' not in data:\n            mask_border(mask, self.border_rm, False)\n        else:\n            mask_border_with_padding(mask, self.border_rm, False,\n                                     data['mask0'], data['mask1'])\n        mask = rearrange(mask, 'b h0c w0c h1c w1c -> b (h0c w0c) (h1c w1c)',\n                         **axes_lengths)\n\n        # find all valid coarse matches\n        b_ids, i_ids, j_ids = mask.nonzero(as_tuple=True)\n\n        mconf = torch.maximum(conf_matrix_0_to_1[b_ids, i_ids, j_ids], conf_matrix_1_to_0[b_ids, i_ids, j_ids])\n\n        # random sampling of training samples for fine-level XoFTR\n        # (optional) pad samples with gt coarse-level matches\n        if self.training:\n            # NOTE:\n            # the sampling is performed across all pairs in a batch without manually balancing\n            # samples for fine-level increases w.r.t. batch_size\n            if 'mask0' not in data:\n                num_candidates_max = mask.size(0) * max(\n                    mask.size(1), mask.size(2))\n            else:\n                num_candidates_max = compute_max_candidates(\n                    data['mask0'], data['mask1'])\n            num_matches_train = int(num_candidates_max *\n                                    self.train_coarse_percent)\n            num_matches_pred = len(b_ids)\n            assert self.train_pad_num_gt_min < num_matches_train, \"min-num-gt-pad should be less than num-train-matches\"\n\n            # pred_indices is to select from prediction\n            if num_matches_pred <= num_matches_train - self.train_pad_num_gt_min:\n                pred_indices = torch.arange(num_matches_pred, device=_device)\n            else:\n                pred_indices = torch.randint(\n                    num_matches_pred,\n                    (num_matches_train - self.train_pad_num_gt_min, ),\n                    device=_device)\n\n            # gt_pad_indices is to select from gt padding. e.g. max(3787-4800, 200)\n            gt_pad_indices = torch.randint(\n                    len(data['spv_b_ids']),\n                    (max(num_matches_train - num_matches_pred,\n                        self.train_pad_num_gt_min), ),\n                    device=_device)\n            mconf_gt = torch.zeros(len(data['spv_b_ids']), device=_device)  # set conf of gt paddings to all zero\n\n            b_ids, i_ids, j_ids, mconf = map(\n                lambda x, y: torch.cat([x[pred_indices], y[gt_pad_indices]],\n                                       dim=0),\n                *zip([b_ids, data['spv_b_ids']], [i_ids, data['spv_i_ids']],\n                     [j_ids, data['spv_j_ids']], [mconf, mconf_gt]))\n\n        # these matches are selected patches that feed into fine-level network\n        coarse_matches = {'b_ids': b_ids, 'i_ids': i_ids, 'j_ids': j_ids}\n\n        # update with matches in original image resolution\n        scale = data['hw0_i'][0] / data['hw0_c'][0]\n        scale0 = scale * data['scale0'][b_ids] if 'scale0' in data else scale\n        scale1 = scale * data['scale1'][b_ids] if 'scale1' in data else scale\n        mkpts0_c = torch.stack(\n            [i_ids % data['hw0_c'][1], torch.div(i_ids, data['hw0_c'][1], rounding_mode='trunc')],\n            dim=1) * scale0\n        mkpts1_c = torch.stack(\n            [j_ids % data['hw1_c'][1], torch.div(j_ids, data['hw1_c'][1], rounding_mode='trunc')],\n            dim=1) * scale1\n\n        # these matches is the current prediction (for visualization)\n        coarse_matches.update({\n            'gt_mask': mconf == 0,\n            'm_bids': b_ids[mconf != 0],  # mconf == 0 => gt matches\n            'mkpts0_c': mkpts0_c[mconf != 0],\n            'mkpts1_c': mkpts1_c[mconf != 0],\n            'mconf': mconf[mconf != 0]\n        })\n\n        return coarse_matches\n\n    @torch.no_grad()\n    def get_coarse_match_inference(self, sim_matrix, data):\n        \"\"\"\n        Args:\n            sim_matrix (torch.Tensor): [N, L, S]\n            data (dict): with keys ['hw0_i', 'hw1_i', 'hw0_c', 'hw1_c']\n        Returns:\n            coarse_matches (dict): {\n                'b_ids' (torch.Tensor): [M'],\n                'i_ids' (torch.Tensor): [M'],\n                'j_ids' (torch.Tensor): [M'],\n                'gt_mask' (torch.Tensor): [M'],\n                'm_bids' (torch.Tensor): [M],\n                'mkpts0_c' (torch.Tensor): [M, 2],\n                'mkpts1_c' (torch.Tensor): [M, 2],\n                'mconf' (torch.Tensor): [M]}\n        \"\"\"\n        axes_lengths = {\n            'h0c': data['hw0_c'][0],\n            'w0c': data['hw0_c'][1],\n            'h1c': data['hw1_c'][0],\n            'w1c': data['hw1_c'][1]\n        }\n\n        # softmax for 0 to 1\n        conf_matrix_ = F.softmax(sim_matrix, 2)\n\n        # confidence thresholding and nearest neighbour for 0 to 1\n        mask = (conf_matrix_ > self.thr) * (conf_matrix_ == conf_matrix_.max(dim=2, keepdim=True)[0])\n\n        # unlike training, reuse the same conf martix to decrease the vram consumption\n        # softmax for 0 to 1\n        conf_matrix_ = F.softmax(sim_matrix, 1)\n\n        # update mask {(nearest neighbour for 0 to 1) U (nearest neighbour for 1 to 0)}\n        mask = torch.logical_or(mask,\n                                 (conf_matrix_ > self.thr) * (conf_matrix_ == conf_matrix_.max(dim=1, keepdim=True)[0]))\n\n        mask = rearrange(mask, 'b (h0c w0c) (h1c w1c) -> b h0c w0c h1c w1c',\n                    **axes_lengths)\n        if 'mask0' not in data:\n            mask_border(mask, self.border_rm, False)\n        else:\n            mask_border_with_padding(mask, self.border_rm, False,\n                                     data['mask0'], data['mask1'])\n        mask = rearrange(mask, 'b h0c w0c h1c w1c -> b (h0c w0c) (h1c w1c)',\n                         **axes_lengths)\n\n        # find all valid coarse matches\n        b_ids, i_ids, j_ids = mask.nonzero(as_tuple=True)\n\n        # mconf = torch.maximum(conf_matrix_0_to_1[b_ids, i_ids, j_ids], conf_matrix_1_to_0[b_ids, i_ids, j_ids])\n\n        # these matches are selected patches that feed into fine-level network\n        coarse_matches = {'b_ids': b_ids, 'i_ids': i_ids, 'j_ids': j_ids}\n\n        # update with matches in original image resolution\n        scale = data['hw0_i'][0] / data['hw0_c'][0]\n        scale0 = scale * data['scale0'][b_ids] if 'scale0' in data else scale\n        scale1 = scale * data['scale1'][b_ids] if 'scale1' in data else scale\n        mkpts0_c = torch.stack(\n            [i_ids % data['hw0_c'][1], torch.div(i_ids, data['hw0_c'][1], rounding_mode='trunc')],\n            dim=1) * scale0\n        mkpts1_c = torch.stack(\n            [j_ids % data['hw1_c'][1], torch.div(j_ids, data['hw1_c'][1], rounding_mode='trunc')],\n            dim=1) * scale1\n\n        # these matches are the current coarse level predictions\n        coarse_matches.update({\n            'm_bids': b_ids,  # mconf == 0 => gt matches\n            'mkpts0_c': mkpts0_c,\n            'mkpts1_c': mkpts1_c,\n        })\n\n        return coarse_matches\n\n\n# ----------------------------------------------------------------------------------------------------\n# upstream src/xoftr/xoftr_module/fine_process.py\n# ----------------------------------------------------------------------------------------------------\n\nclass Mlp(nn.Module):\n    \"\"\"Multi-Layer Perceptron (MLP)\"\"\"\n\n    def __init__(self,\n                 in_dim,\n                 hidden_dim=None,\n                 out_dim=None,\n                 act_layer=nn.GELU):\n        \"\"\"\n        Args:\n            in_dim: input features dimension\n            hidden_dim: hidden features dimension\n            out_dim: output features dimension\n            act_layer: activation function\n        \"\"\"\n        super().__init__()\n        out_dim = out_dim or in_dim\n        hidden_dim = hidden_dim or in_dim\n        self.fc1 = nn.Linear(in_dim, hidden_dim)\n        self.act = act_layer()\n        self.fc2 = nn.Linear(hidden_dim, out_dim)\n        self.out_dim = out_dim\n\n    def forward(self, x):\n        x_size = x.size()\n        x = x.view(-1, x_size[-1])\n        x = self.fc1(x)\n        x = self.act(x)\n        x = self.fc2(x)\n        x = x.view(*x_size[:-1], self.out_dim)\n        return x\n\n\nclass VanillaAttention(nn.Module):\n    def __init__(self,\n                 dim,\n                 num_heads=8,\n                 proj_bias=False):\n        super().__init__()\n        \"\"\"\n        Args:\n            dim: feature dimension\n            num_heads: number of attention head\n            proj_bias: bool use query, key, value bias\n        \"\"\"\n        self.num_heads = num_heads\n        self.head_dim = dim // num_heads\n        self.softmax_temp = self.head_dim ** -0.5\n        self.kv_proj = nn.Linear(dim, dim * 2, bias=proj_bias)\n        self.q_proj = nn.Linear(dim, dim, bias=proj_bias)\n        self.merge = nn.Linear(dim, dim)\n\n    def forward(self, x_q, x_kv=None):\n        \"\"\"\n        Args:\n            x_q (torch.Tensor): [N, L, C]\n            x_kv (torch.Tensor): [N, S, C]\n        \"\"\"\n        if x_kv is None:\n            x_kv = x_q\n        bs, _, dim = x_q.shape\n        bs, _, dim = x_kv.shape\n        # [N, S, 2, H, D] => [2, N, H, S, D]\n        kv = self.kv_proj(x_kv).reshape(bs, -1, 2, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)\n        # [N, L, H, D] => [N, H, L, D]\n        q = self.q_proj(x_q).reshape(bs, -1, self.num_heads, self.head_dim).permute(0, 2, 1, 3)\n        k, v = kv[0].transpose(-2, -1).contiguous(), kv[1].contiguous() # [N, H, D, S], [N, H, S, D]\n        attn = (q @ k) * self.softmax_temp # [N, H, L, S]\n        attn = attn.softmax(dim=-1)\n        x_q = (attn @ v).transpose(1, 2).reshape(bs, -1, dim)\n        x_q = self.merge(x_q)\n        return x_q\n\n\nclass CrossBidirectionalAttention(nn.Module):\n    def __init__(self, dim, num_heads, proj_bias = False):\n        super().__init__()\n        \"\"\"\n        Args:\n            dim: feature dimension\n            num_heads: number of attention head\n            proj_bias: bool use query, key, value bias\n        \"\"\"\n\n        self.num_heads = num_heads\n        self.head_dim = dim // num_heads\n        self.softmax_temp = self.head_dim ** -0.5\n        self.qk_proj = nn.Linear(dim, dim, bias=proj_bias)\n        self.v_proj = nn.Linear(dim, dim, bias=proj_bias)\n        self.merge = nn.Linear(dim, dim, bias=proj_bias)\n        self.temperature = nn.Parameter(torch.tensor([0.0]), requires_grad=True)\n        # print(self.temperature)\n\n    def map_(self, func, x0, x1):\n        return func(x0), func(x1)\n\n    def forward(self, x0, x1):\n        \"\"\"\n        Args:\n            x0 (torch.Tensor): [N, L, C]\n            x1 (torch.Tensor): [N, S, C]\n        \"\"\"\n        bs = x0.size(0)\n\n        qk0, qk1 = self.map_(self.qk_proj, x0, x1)\n        v0, v1 = self.map_(self.v_proj, x0, x1)\n        qk0, qk1, v0, v1 = map(\n            lambda t: t.reshape(bs, -1, self.num_heads, self.head_dim).permute(0, 2, 1, 3).contiguous(),\n            (qk0, qk1, v0, v1))\n\n        qk0, qk1 = qk0 * self.softmax_temp**0.5, qk1 * self.softmax_temp**0.5\n        sim = qk0 @ qk1.transpose(-2,-1).contiguous()\n        attn01 = F.softmax(sim, dim=-1)\n        attn10 = F.softmax(sim.transpose(-2, -1).contiguous(), dim=-1)\n        x0 = attn01 @ v1\n        x1 = attn10 @ v0\n        x0, x1 = self.map_(lambda t: t.transpose(1, 2).flatten(start_dim=-2),\n                        x0, x1)\n        x0, x1 = self.map_(self.merge, x0, x1)\n\n        return x0, x1\n\n\nclass SwinPosEmbMLP(nn.Module):\n    def __init__(self,\n                 dim):\n        super().__init__()\n        self.pos_embed = None\n        self.pos_mlp = nn.Sequential(nn.Linear(2, 512, bias=True),\n                                        nn.ReLU(),\n                                        nn.Linear(512, dim, bias=False))\n\n    def forward(self, x):\n        seq_length = x.shape[1]\n        if self.pos_embed is None or self.training:\n            seq_length = int(seq_length**0.5)\n            coords = torch.arange(0, seq_length, device=x.device, dtype = x.dtype)\n            grid = torch.stack(torch.meshgrid([coords, coords])).contiguous().unsqueeze(0)\n            grid -= seq_length // 2\n            grid /= (seq_length // 2)\n            self.pos_embed = self.pos_mlp(grid.flatten(2).transpose(1,2))\n        x = x + self.pos_embed\n        return x\n\n\nclass WindowSelfAttention(nn.Module):\n    def __init__(self, dim, num_heads, mlp_hidden_coef, use_pre_pos_embed=False):\n        super().__init__()\n        self.mlp = Mlp(in_dim=dim*2, hidden_dim=dim*mlp_hidden_coef, out_dim=dim, act_layer=nn.GELU)\n        self.gamma = nn.Parameter(torch.ones(dim))\n        self.norm1 = nn.LayerNorm(dim)\n        self.norm2 = nn.LayerNorm(dim)\n        self.attn = VanillaAttention(dim, num_heads=num_heads)\n        self.pos_embed = SwinPosEmbMLP(dim)\n        self.pos_embed_pre = SwinPosEmbMLP(dim) if use_pre_pos_embed else nn.Identity()\n\n    def forward(self, x, x_pre):\n        ww = x.shape[1]\n        ww_pre = x_pre.shape[1]\n        x = self.pos_embed(x)\n        x_pre = self.pos_embed_pre(x_pre)\n        x = torch.cat((x, x_pre), dim=1)\n        x = x + self.gamma*self.norm1(self.mlp(torch.cat([x, self.attn(self.norm2(x))], dim=-1)))\n        x, x_pre = x.split([ww, ww_pre], dim=1)\n        return x, x_pre\n\n\nclass WindowCrossAttention(nn.Module):\n    def __init__(self, dim, num_heads, mlp_hidden_coef):\n        super().__init__()\n        self.norm1 = nn.LayerNorm(dim)\n        self.norm2 = nn.LayerNorm(dim)\n        self.mlp = Mlp(in_dim=dim*2, hidden_dim=dim*mlp_hidden_coef, out_dim=dim, act_layer=nn.GELU)\n        self.cross_attn = CrossBidirectionalAttention(dim, num_heads=num_heads, proj_bias=False)\n        self.gamma = nn.Parameter(torch.ones(dim))\n\n    def forward(self, x0, x1):\n        m_x0, m_x1 = self.cross_attn(self.norm1(x0), self.norm1(x1))\n        x0 = x0 + self.gamma*self.norm2(self.mlp(torch.cat([x0, m_x0], dim=-1)))\n        x1 = x1 + self.gamma*self.norm2(self.mlp(torch.cat([x1, m_x1], dim=-1)))\n        return x0, x1\n\n\nclass FineProcess(nn.Module):\n    def __init__(self, config):\n        super().__init__()\n        # Config\n        block_dims = config['resnet']['block_dims']\n        self.block_dims = block_dims\n        self.W_f = config['fine_window_size']\n        self.W_m = config['medium_window_size']\n        nhead_f = config[\"fine\"]['nhead_fine_level']\n        nhead_m = config[\"fine\"]['nhead_medium_level']\n        mlp_hidden_coef = config[\"fine\"]['mlp_hidden_dim_coef']\n\n        # Networks\n        self.conv_merge = nn.Sequential(nn.Conv2d(block_dims[2]*2, block_dims[1], kernel_size=1, stride=1, padding=0, bias=False),\n                                        nn.Conv2d(block_dims[1], block_dims[1], kernel_size=3, stride=1, padding=1, groups=block_dims[1], bias=False),\n                                        nn.BatchNorm2d(block_dims[1])\n                                        )\n        self.out_conv_m = nn.Conv2d(block_dims[1], block_dims[1], kernel_size=1, stride=1, padding=0, bias=False)\n        self.out_conv_f = nn.Conv2d(block_dims[0], block_dims[0], kernel_size=1, stride=1, padding=0, bias=False)\n        self.self_attn_m = WindowSelfAttention(block_dims[1], num_heads=nhead_m,\n                                                mlp_hidden_coef=mlp_hidden_coef, use_pre_pos_embed=False)\n        self.cross_attn_m = WindowCrossAttention(block_dims[1], num_heads=nhead_m,\n                                                  mlp_hidden_coef=mlp_hidden_coef)\n        self.self_attn_f = WindowSelfAttention(block_dims[0], num_heads=nhead_f,\n                                                mlp_hidden_coef=mlp_hidden_coef, use_pre_pos_embed=True)\n        self.cross_attn_f = WindowCrossAttention(block_dims[0], num_heads=nhead_f,\n                                                  mlp_hidden_coef=mlp_hidden_coef)\n        self.down_proj_m_f = nn.Linear(block_dims[1], block_dims[0], bias=False)\n\n        for m in self.modules():\n            if isinstance(m, nn.Conv2d):\n                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')\n            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):\n                nn.init.constant_(m.weight, 1)\n                nn.init.constant_(m.bias, 0)\n\n    def pre_process(self, feat_f0, feat_f1, feat_m0, feat_m1, feat_c0, feat_c1, feat_c0_pre, feat_c1_pre, data):\n        W_f = self.W_f\n        W_m = self.W_m\n        data.update({'W_f': W_f,\n                'W_m': W_m})\n\n        # merge coarse features before and after loftr layer, and down proj channel dimesions\n        feat_c0 = rearrange(feat_c0, 'n (h w) c -> n c h w', h =data[\"hw0_c\"][0], w =data[\"hw0_c\"][1])\n        feat_c1 = rearrange(feat_c1, 'n (h w) c -> n c h w', h =data[\"hw1_c\"][0], w =data[\"hw1_c\"][1])\n        feat_c0 = self.conv_merge(torch.cat([feat_c0, feat_c0_pre], dim=1))\n        feat_c1 = self.conv_merge(torch.cat([feat_c1, feat_c1_pre], dim=1))\n        feat_c0 = rearrange(feat_c0, 'n c h w -> n (h w) 1 c')\n        feat_c1 = rearrange(feat_c1, 'n c h w -> n (h w) 1 c')\n\n        stride_f = data['hw0_f'][0] // data['hw0_c'][0]\n        stride_m = data['hw0_m'][0] // data['hw0_c'][0]\n\n        if feat_m0.shape[2] == feat_m1.shape[2] and feat_m0.shape[3] == feat_m1.shape[3]:\n            feat_m = self.out_conv_m(torch.cat([feat_m0, feat_m1], dim=0))\n            feat_m0, feat_m1 = torch.chunk(feat_m, 2, dim=0)\n            feat_f = self.out_conv_f(torch.cat([feat_f0, feat_f1], dim=0))\n            feat_f0, feat_f1 = torch.chunk(feat_f, 2, dim=0)\n        else:\n            feat_m0 = self.out_conv_m(feat_m0)\n            feat_m1 = self.out_conv_m(feat_m1)\n            feat_f0 = self.out_conv_f(feat_f0)\n            feat_f1 = self.out_conv_f(feat_f1)\n\n        # 1. unfold (crop windows) all local windows\n        feat_m0_unfold = F.unfold(feat_m0, kernel_size=(W_m, W_m), stride=stride_m, padding=W_m//2)\n        feat_m0_unfold = rearrange(feat_m0_unfold, 'n (c ww) l -> n l ww c', ww=W_m**2)\n        feat_m1_unfold = F.unfold(feat_m1, kernel_size=(W_m, W_m), stride=stride_m, padding=W_m//2)\n        feat_m1_unfold = rearrange(feat_m1_unfold, 'n (c ww) l -> n l ww c', ww=W_m**2)\n\n        feat_f0_unfold = F.unfold(feat_f0, kernel_size=(W_f, W_f), stride=stride_f, padding=W_f//2)\n        feat_f0_unfold = rearrange(feat_f0_unfold, 'n (c ww) l -> n l ww c', ww=W_f**2)\n        feat_f1_unfold = F.unfold(feat_f1, kernel_size=(W_f, W_f), stride=stride_f, padding=W_f//2)\n        feat_f1_unfold = rearrange(feat_f1_unfold, 'n (c ww) l -> n l ww c', ww=W_f**2)\n\n        # 2. select only the predicted matches\n        feat_c0 = feat_c0[data['b_ids'], data['i_ids']] # [n, ww, cm]\n        feat_c1 = feat_c1[data['b_ids'], data['j_ids']]\n\n        feat_m0_unfold = feat_m0_unfold[data['b_ids'], data['i_ids']]  # [n, ww, cm]\n        feat_m1_unfold = feat_m1_unfold[data['b_ids'], data['j_ids']]\n\n        feat_f0_unfold = feat_f0_unfold[data['b_ids'], data['i_ids']]  # [n, ww, cf]\n        feat_f1_unfold = feat_f1_unfold[data['b_ids'], data['j_ids']]\n\n        return feat_c0, feat_c1, feat_m0_unfold, feat_m1_unfold, feat_f0_unfold, feat_f1_unfold\n\n    def forward(self, feat_f0, feat_f1, feat_m0, feat_m1, feat_c0, feat_c1, feat_c0_pre, feat_c1_pre, data):\n        \"\"\"\n        Args:\n            feat_f0 (torch.Tensor): [N, C, H, W]\n            feat_f1 (torch.Tensor): [N, C, H, W]\n            feat_m0 (torch.Tensor): [N, C, H, W]\n            feat_m1 (torch.Tensor): [N, C, H, W]\n            feat_c0 (torch.Tensor): [N, L, C]\n            feat_c1 (torch.Tensor): [N, S, C]\n            feat_c0_pre (torch.Tensor): [N, C, H, W]\n            feat_c1_pre (torch.Tensor): [N, C, H, W]\n            data (dict): with keys ['hw0_c', 'hw1_c', 'hw0_m', 'hw1_m', 'hw0_f', 'hw1_f', 'b_ids', 'j_ids']\n        \"\"\"\n\n        # upstream note: \"Check for this case\" (kept as written; a to-do marker in the vendored source)\n        if data['b_ids'].shape[0] == 0:\n            feat0 = torch.empty(0, self.W_f**2, self.block_dims[0], device=feat_f0.device)\n            feat1 = torch.empty(0, self.W_f**2, self.block_dims[0], device=feat_f0.device)\n            return feat0, feat1\n\n        feat_c0, feat_c1, feat_m0_unfold, feat_m1_unfold, \\\n            feat_f0_unfold, feat_f1_unfold = self.pre_process(feat_f0, feat_f1, feat_m0, feat_m1,\n                                                               feat_c0, feat_c1, feat_c0_pre, feat_c1_pre, data)\n\n        # self attention (c + m)\n        feat_m_unfold, _ = self.self_attn_m(torch.cat([feat_m0_unfold, feat_m1_unfold], dim=0),\n                                                         torch.cat([feat_c0, feat_c1], dim=0))\n        feat_m0_unfold, feat_m1_unfold = torch.chunk(feat_m_unfold, 2, dim=0)\n\n        # cross attention (m0 <-> m1)\n        feat_m0_unfold, feat_m1_unfold = self.cross_attn_m(feat_m0_unfold, feat_m1_unfold)\n\n        # down proj m\n        feat_m_unfold = self.down_proj_m_f(torch.cat([feat_m0_unfold, feat_m1_unfold], dim=0))\n        feat_m0_unfold, feat_m1_unfold = torch.chunk(feat_m_unfold, 2, dim=0)\n\n        # self attention (m + f)\n        feat_f_unfold, _ = self.self_attn_f(torch.cat([feat_f0_unfold, feat_f1_unfold], dim=0),\n                                                         torch.cat([feat_m0_unfold, feat_m1_unfold], dim=0))\n        feat_f0_unfold, feat_f1_unfold = torch.chunk(feat_f_unfold, 2, dim=0)\n\n        # cross attention (f0 <-> f1)\n        feat_f0_unfold, feat_f1_unfold = self.cross_attn_f(feat_f0_unfold, feat_f1_unfold)\n\n        return feat_f0_unfold, feat_f1_unfold\n\n\n# ----------------------------------------------------------------------------------------------------\n# upstream src/xoftr/xoftr_module/fine_matching.py\n# ----------------------------------------------------------------------------------------------------\n\nclass FineSubMatching(nn.Module):\n    \"\"\"Fine-level and Sub-pixel matching\"\"\"\n\n    def __init__(self, config):\n        super().__init__()\n        self.temperature = config['fine']['dsmax_temperature']\n        self.W_f = config['fine_window_size']\n        self.denser = config['fine']['denser']\n        self.inference = config['fine']['inference']\n        dim_f = config['resnet']['block_dims'][0]\n        self.fine_thr = config['fine']['thr']\n        self.fine_proj = nn.Linear(dim_f, dim_f, bias=False)\n        self.subpixel_mlp = nn.Sequential(nn.Linear(2*dim_f, 2*dim_f, bias=False),\n                                           nn.ReLU(),\n                                           nn.Linear(2*dim_f, 4, bias=False))\n\n    def forward(self, feat_f0_unfold, feat_f1_unfold, data):\n        \"\"\"\n        Args:\n            feat_f0_unfold (torch.Tensor): [M, WW, C]\n            feat_f1_unfold (torch.Tensor): [M, WW, C]\n            data (dict)\n        Update:\n            data (dict):{\n                'expec_f' (torch.Tensor): [M, 3],\n                'mkpts0_f' (torch.Tensor): [M, 2],\n                'mkpts1_f' (torch.Tensor): [M, 2]}\n        \"\"\"\n\n        feat_f0 = self.fine_proj(feat_f0_unfold)\n        feat_f1 = self.fine_proj(feat_f1_unfold)\n\n        M, WW, C = feat_f0.shape\n        W_f = self.W_f\n\n        # corner case: if no coarse matches found\n        if M == 0:\n            assert self.training == False, \"M is always >0, when training, see coarse_matching.py\"\n            # logger.warning('No matches found in coarse-level.')\n            data.update({\n                'mkpts0_f': data['mkpts0_c'],\n                'mkpts1_f': data['mkpts1_c'],\n                'mconf_f': torch.zeros(0, device=feat_f0_unfold.device),\n                # 'mkpts0_f_train': data['mkpts0_c'],\n                # 'mkpts1_f_train': data['mkpts1_c'],\n                # 'conf_matrix_fine': torch.zeros(1, W_f*W_f, W_f*W_f, device=feat_f0.device)\n            })\n            return\n\n        # normalize\n        feat_f0, feat_f1 = map(lambda feat: feat / feat.shape[-1]**.5,\n                               [feat_f0, feat_f1])\n        sim_matrix = torch.einsum(\"nlc,nsc->nls\", feat_f0,\n                                      feat_f1) / self.temperature\n\n        conf_matrix_fine = F.softmax(sim_matrix, 1) * F.softmax(sim_matrix, 2)\n        data.update({'conf_matrix_fine': conf_matrix_fine})\n\n        # predict fine-level and sub-pixel matches from conf_matrix\n        data.update(**self.get_fine_sub_match(conf_matrix_fine, feat_f0_unfold, feat_f1_unfold, data))\n\n    def get_fine_sub_match(self, conf_matrix_fine, feat_f0_unfold, feat_f1_unfold, data):\n        \"\"\"\n        Args:\n            conf_matrix_fine (torch.Tensor): [M, WW, WW]\n            feat_f0_unfold (torch.Tensor): [M, WW, C]\n            feat_f1_unfold (torch.Tensor): [M, WW, C]\n            data (dict)\n        Update:\n            data (dict):{\n                'm_bids' (torch.Tensor): [M]\n                'expec_f' (torch.Tensor): [M, 3],\n                'mkpts0_f' (torch.Tensor): [M, 2],\n                'mkpts1_f' (torch.Tensor): [M, 2]}\n        \"\"\"\n\n        with torch.no_grad():\n            W_f = self.W_f\n\n            # 1. confidence thresholding\n            mask = conf_matrix_fine > self.fine_thr\n\n            if mask.sum() == 0:\n                mask[0,0,0] = 1\n                conf_matrix_fine[0,0,0] = 1\n\n            if not self.denser:\n                # match only the highest confidence\n                mask = mask \\\n                    * (conf_matrix_fine == conf_matrix_fine.amax(dim=[1,2], keepdim=True))\n            else:\n                # 2. mutual nearest, match all features in fine window\n                mask = mask \\\n                    * (conf_matrix_fine == conf_matrix_fine.max(dim=2, keepdim=True)[0]) \\\n                    * (conf_matrix_fine == conf_matrix_fine.max(dim=1, keepdim=True)[0])\n\n            # 3. find all valid fine matches\n            # this only works when at most one `True` in each row\n            mask_v, all_j_ids = mask.max(dim=2)\n            b_ids, i_ids = torch.where(mask_v)\n            j_ids = all_j_ids[b_ids, i_ids]\n            mconf = conf_matrix_fine[b_ids, i_ids, j_ids]\n\n            # 4. update with matches in original image resolution\n\n            # indices from coarse matches\n            b_ids_c, i_ids_c, j_ids_c = data['b_ids'], data['i_ids'], data['j_ids']\n\n            # scale (coarse level / fine-level)\n            scale_f_c = data['hw0_f'][0] // data['hw0_c'][0]\n\n            # coarse level matches scaled to fine-level (1/2)\n            mkpts0_c_scaled_to_f = torch.stack(\n            [i_ids_c % data['hw0_c'][1], torch.div(i_ids_c, data['hw0_c'][1], rounding_mode='trunc')],\n            dim=1) * scale_f_c\n\n            mkpts1_c_scaled_to_f = torch.stack(\n                [j_ids_c % data['hw1_c'][1], torch.div(j_ids_c, data['hw1_c'][1], rounding_mode='trunc')],\n                dim=1) * scale_f_c\n\n            # updated b_ids after second thresholding\n            updated_b_ids = b_ids_c[b_ids]\n\n            # scales (image res / fine level)\n            scale = data['hw0_i'][0] / data['hw0_f'][0]\n            scale0 = scale * data['scale0'][updated_b_ids] if 'scale0' in data else scale\n            scale1 = scale * data['scale1'][updated_b_ids] if 'scale1' in data else scale\n\n            # fine-level discrete matches on window coordiantes\n            mkpts0_f_window = torch.stack(\n            [i_ids % W_f, torch.div(i_ids, W_f, rounding_mode='trunc')],\n            dim=1)\n\n            mkpts1_f_window = torch.stack(\n            [j_ids % W_f, torch.div(j_ids, W_f, rounding_mode='trunc')],\n            dim=1)\n\n        # sub-pixel refinement\n        sub_ref = self.subpixel_mlp(torch.cat([feat_f0_unfold[b_ids, i_ids],\n                                                     feat_f1_unfold[b_ids, j_ids]], dim=-1))\n        sub_ref0, sub_ref1 = torch.chunk(sub_ref, 2, dim=-1)\n        sub_ref0 = torch.tanh(sub_ref0) * 0.5\n        sub_ref1 = torch.tanh(sub_ref1) * 0.5\n\n        # final sub-pixel matches by (coarse-level + fine-level windowed + sub-pixel refinement)\n        mkpts0_f_train = (mkpts0_f_window + mkpts0_c_scaled_to_f[b_ids] - (W_f//2) + sub_ref0) * scale0\n        mkpts1_f_train = (mkpts1_f_window + mkpts1_c_scaled_to_f[b_ids] - (W_f//2) + sub_ref1) * scale1\n        mkpts0_f = mkpts0_f_train.clone().detach()\n        mkpts1_f = mkpts1_f_train.clone().detach()\n\n        # These matches is the current prediction (for visualization)\n        sub_pixel_matches = {\n            'm_bids': b_ids_c[b_ids[mconf != 0]],  # mconf == 0 => gt matches\n            'mkpts0_f': mkpts0_f[mconf != 0],\n            'mkpts1_f': mkpts1_f[mconf != 0],\n            'mconf_f': mconf[mconf != 0]\n        }\n\n        # These matches are used for training\n        if not self.inference:\n            sub_pixel_matches.update({\n                'mkpts0_f_train': mkpts0_f_train[mconf != 0],\n                'mkpts1_f_train': mkpts1_f_train[mconf != 0],\n            })\n\n        return sub_pixel_matches\n\n\n# ----------------------------------------------------------------------------------------------------\n# upstream src/xoftr/xoftr.py\n# ----------------------------------------------------------------------------------------------------\n\nclass XoFTR(nn.Module):\n    def __init__(self, config):\n        super().__init__()\n        # Misc\n        self.config = config\n\n        # Modules\n        self.backbone = ResNet_8_2(config['resnet'])\n        self.pos_encoding = PositionEncodingSine(config['coarse']['d_model'])\n        self.loftr_coarse = LocalFeatureTransformer(config['coarse'])\n        self.coarse_matching = CoarseMatching(config['match_coarse'])\n        self.fine_process = FineProcess(config)\n        self.fine_matching= FineSubMatching(config)\n\n\n    def forward(self, data):\n        \"\"\"\n        Update:\n            data (dict): {\n                'image0': (torch.Tensor): (N, 1, H, W)\n                'image1': (torch.Tensor): (N, 1, H, W)\n                'mask0'(optional) : (torch.Tensor): (N, H, W) '0' indicates a padded position\n                'mask1'(optional) : (torch.Tensor): (N, H, W)\n            }\n        \"\"\"\n        # 1. Local Feature CNN\n        data.update({\n            'bs': data['image0'].size(0),\n            'hw0_i': data['image0'].shape[2:], 'hw1_i': data['image1'].shape[2:]\n        })\n\n        eps = 1e-6\n\n        image0_mean = data['image0'].mean(dim=[2,3], keepdim=True)\n        image0_std = data['image0'].std(dim=[2,3], keepdim=True)\n        image0 = (data['image0'] - image0_mean) / (image0_std + eps)\n\n        image1_mean = data['image1'].mean(dim=[2,3], keepdim=True)\n        image1_std = data['image1'].std(dim=[2,3], keepdim=True)\n        image1 = (data['image1'] - image1_mean) / (image1_std + eps)\n\n        if data['hw0_i'] == data['hw1_i']:  # faster & better BN convergence\n            feats_c, feats_m, feats_f = self.backbone(torch.cat([image0, image1], dim=0))\n            (feat_c0, feat_c1) = feats_c.split(data['bs'])\n            (feat_m0, feat_m1) = feats_m.split(data['bs'])\n            (feat_f0, feat_f1) = feats_f.split(data['bs'])\n        else:  # handle different input shapes\n            feat_c0, feat_m0, feat_f0 = self.backbone(image0)\n            feat_c1, feat_m1, feat_f1 = self.backbone(image1)\n\n        data.update({\n            'hw0_c': feat_c0.shape[2:], 'hw1_c': feat_c1.shape[2:],\n            'hw0_m': feat_m0.shape[2:], 'hw1_m': feat_m1.shape[2:],\n            'hw0_f': feat_f0.shape[2:], 'hw1_f': feat_f1.shape[2:]\n        })\n\n        # save coarse features for fine matching\n        feat_c0_pre, feat_c1_pre = feat_c0.clone(), feat_c1.clone()\n\n        # 2. coarse-level loftr module\n        # add featmap with positional encoding, then flatten it to sequence [N, HW, C]\n        feat_c0 = rearrange(self.pos_encoding(feat_c0), 'n c h w -> n (h w) c')\n        feat_c1 = rearrange(self.pos_encoding(feat_c1), 'n c h w -> n (h w) c')\n\n        mask_c0 = mask_c1 = None  # mask is useful in training\n        if 'mask0' in data:\n            mask_c0, mask_c1 = data['mask0'].flatten(-2), data['mask1'].flatten(-2)\n        feat_c0, feat_c1 = self.loftr_coarse(feat_c0, feat_c1, mask_c0, mask_c1)\n\n        # 3. match coarse-level\n        self.coarse_matching(feat_c0, feat_c1, data, mask_c0=mask_c0, mask_c1=mask_c1)\n\n        # 4. fine-level matching module\n        feat_f0_unfold, feat_f1_unfold = self.fine_process(feat_f0, feat_f1,\n                                                           feat_m0, feat_m1,\n                                                           feat_c0, feat_c1,\n                                                           feat_c0_pre, feat_c1_pre,\n                                                           data)\n\n        # 5. match fine-level and sub-pixel refinement\n        self.fine_matching(feat_f0_unfold, feat_f1_unfold, data)\n\n    def load_state_dict(self, state_dict, *args, **kwargs):\n        for k in list(state_dict.keys()):\n            if k.startswith('matcher.'):\n                state_dict[k.replace('matcher.', '', 1)] = state_dict.pop(k)\n        return super().load_state_dict(state_dict, *args, **kwargs)\n","xoftr_pipeline/pipeline.py":"\"\"\"XoFTR matching pipeline: the inference contract (`match`), homography-supervised evaluation, a bounded\nadaptation of the coarse transformer, and a digest-manifested safetensors adapter.\"\"\"\n# ruff: noqa: E501  -- docstrings and record literals kept on single lines at the fleet width\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport time\nfrom collections.abc import Callable, Mapping, Sequence\nfrom io import BytesIO\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport torch\nfrom PIL import Image\n\nfrom .config import (\n    COARSE_THRESHOLD,\n    DEFAULT_MODEL_KEY,\n    DIVISIBLE_BY,\n    FINE_THRESHOLD,\n    MAX_SIDE,\n    MIN_SIDE,\n    MODEL_FILENAME,\n    MODEL_ID,\n    MODEL_REVISION,\n    MODEL_SHA256,\n    PARAMETER_COUNT,\n)\nfrom .metrics import identity_baseline, matching_metrics, pair_metrics, patch_neighbour_baseline\nfrom .model import load_components, stage_missing_files, verify_snapshot\n\nImageInput = str | Path | bytes | Image.Image\n\n# --------------------------------------------------------------------------\n# Adaptation contract (E2E): bounded fine-tuning of the coarse transformer's last layers with the\n# upstream coarse focal loss, supervised by the pairs' exact homographies.\n# --------------------------------------------------------------------------\nCOARSE_LAYERS = 8  # loftr_coarse: ['self', 'cross'] * 4\nDEFAULT_TRAINABLE_COARSE_LAYERS = 2  # the last self + cross pair + the coarse projection (1,378,560 params)\nMAX_EVAL_RECORDS = 5_000\nMIN_SCORED_RECORDS = 30  # below this a scored set is labelled a small sample\nARTIFACT_FORMAT = \"org.valcorza.xoftr.adapter.v1\"\nARTIFACT_FORMAT_VERSION = \"1.0\"\nARTIFACT_WEIGHTS_NAME = \"adapter.safetensors\"\nARTIFACT_MANIFEST_NAME = \"manifest.json\"\nFOCAL_ALPHA = 0.25  # upstream LOSS.FOCAL_ALPHA / FOCAL_GAMMA / POS_WEIGHT\nFOCAL_GAMMA = 2.0\nPOS_WEIGHT = 1.0\n\nINPUT_SCHEMA: dict[str, Any] = {\n    \"images\": (\n        \"PIL.Image.Image, raw bytes, or a local path decodable by Pillow; any mode, converted to \"\n        \"grey-scale; remote URLs are refused\"\n    ),\n    \"image_size\": (\n        f\"sides in [{MIN_SIDE}, {MAX_SIDE}] px; each image is cropped down to a multiple of {DIVISIBLE_BY} \"\n        \"(the 1/8 coarse grid) before matching and keypoints are reported in the cropped frame\"\n    ),\n    \"thresholds\": {\"coarse\": COARSE_THRESHOLD, \"fine\": FINE_THRESHOLD},\n    \"output\": \"kpts0 / kpts1 (M, 2) float32 pixel coordinates (x, y) and confidence (M,) in (0, 1]\",\n    \"validation\": (\n        \"size and decodability only. Nothing checks that the two images show the same scene: any two \"\n        \"images are matched, and a pair with no overlap still returns whatever passes the thresholds\"\n    ),\n}\n\n\ndef _sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with open(path, \"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1 << 20), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _coerce_image(value: ImageInput) -> Image.Image:\n    if isinstance(value, Image.Image):\n        return value.convert(\"RGB\")\n    if isinstance(value, bytes):\n        image = Image.open(BytesIO(value))\n        image.load()\n        return image.convert(\"RGB\")\n    if isinstance(value, str | Path):\n        text = str(value)\n        if text.lower().startswith((\"http://\", \"https://\")):\n            raise ValueError(\"remote image URLs are not accepted; pass a local path, bytes or a PIL image\")\n        path = Path(text)\n        if not path.is_file():\n            raise ValueError(f\"image file not found: {path}\")\n        image = Image.open(path)\n        image.load()\n        return image.convert(\"RGB\")\n    raise ValueError(\"image must be a local path, bytes or a PIL.Image.Image\")\n\n\ndef _check_size(image: Image.Image, what: str) -> None:\n    if min(image.size) < MIN_SIDE or max(image.size) > MAX_SIDE:\n        raise ValueError(f\"{what}: sides must lie in [{MIN_SIDE}, {MAX_SIDE}] px; got {image.size}\")\n\n\ndef _to_tensor(image: Image.Image) -> torch.Tensor:\n    \"\"\"Grey-scale float tensor (1, 1, H, W) in [0, 1], sides cropped to multiples of DIVISIBLE_BY.\"\"\"\n    width, height = image.size\n    w, h = width // DIVISIBLE_BY * DIVISIBLE_BY, height // DIVISIBLE_BY * DIVISIBLE_BY\n    grey = image.convert(\"L\").crop((0, 0, w, h))\n    array = np.asarray(grey, dtype=np.float32) / 255.0\n    return torch.from_numpy(array)[None, None]\n\n\ndef validate_inputs(\n    image0: ImageInput, image1: ImageInput, *, names: Sequence[str] | None = None\n) -> dict[str, Any]:\n    \"\"\"Validation stage: exactly the checks `match` applies, reported as an input manifest before the model runs.\"\"\"\n    findings: list[dict[str, Any]] = []\n    observations = []\n    labels = list(names) if names else [\"image0\", \"image1\"]\n    for label, value in zip(labels, (image0, image1), strict=True):\n        image = _coerce_image(value)\n        _check_size(image, label)\n        width, height = image.size\n        observations.append(\n            {\n                \"name\": label,\n                \"size\": [width, height],\n                \"cropped_to\": [width // DIVISIBLE_BY * DIVISIBLE_BY, height // DIVISIBLE_BY * DIVISIBLE_BY],\n                \"mode\": \"grey-scale after conversion\",\n            }\n        )\n    return {\"schema\": INPUT_SCHEMA, \"images\": observations, \"findings\": findings, \"verdict\": \"accepted\"}\n\n\nclass XoFTRPipeline:\n    def __init__(\n        self,\n        model: Any,\n        *,\n        device: str | torch.device = \"cpu\",\n        checkpoint_path: Path | str | None = None,\n        checkpoint_source: str | None = None,\n        manifest_verified: bool = False,\n        weight_sha256: str | None = None,\n        weight_size_bytes: int | None = None,\n    ) -> None:\n        self.model = model\n        self.device = torch.device(device)\n        self.checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else None\n        self.checkpoint_source = checkpoint_source\n        self.manifest_verified = manifest_verified\n        self.weight_sha256 = weight_sha256\n        self.weight_size_bytes = weight_size_bytes\n        self.adapter: dict[str, Any] | None = None\n        if hasattr(self.model, \"parameters\"):\n            for param in self.model.parameters():\n                param.requires_grad_(False)\n\n    @classmethod\n    def from_pretrained(\n        cls,\n        *,\n        device: str | torch.device | None = None,\n        cache_dir: str | Path | None = None,\n        weights_path: str | Path | None = None,\n        weights_dir: str | Path | None = None,\n        allow_download: bool = False,\n        coarse_threshold: float = COARSE_THRESHOLD,\n        fine_threshold: float = FINE_THRESHOLD,\n    ) -> XoFTRPipeline:\n        \"\"\"Load the one supported checkpoint into the vendored network.\n\n        ``weights_dir`` names a fleet snapshot directory holding ``dimer-base-manifest.json``: absent manifest\n        entries are staged with :func:`stage_missing_files` (only when ``allow_download=True``), the directory\n        is verified against the manifest and the pinned digest by :func:`verify_snapshot`, and\n        :func:`load_components` loads it as an explicit path (nothing goes through ``snapshot_download``).\n        \"\"\"\n        if weights_dir is not None:\n            if weights_path is not None:\n                raise ValueError(\"pass either weights_dir or weights_path, not both\")\n            stage_missing_files(weights_dir, allow_download=allow_download)\n            verify_snapshot(weights_dir)\n            weights_path = weights_dir\n        model, target_device, _, metadata = load_components(\n            device=device,\n            cache_dir=cache_dir,\n            weights_path=weights_path,\n            return_metadata=True,\n            coarse_threshold=coarse_threshold,\n            fine_threshold=fine_threshold,\n        )\n        return cls(\n            model,\n            device=target_device,\n            checkpoint_path=metadata.get(\"checkpoint_path\"),\n            checkpoint_source=metadata.get(\"checkpoint_source\"),\n            manifest_verified=metadata.get(\"manifest_verified\", False),\n            weight_sha256=metadata.get(\"weight_sha256\"),\n            weight_size_bytes=metadata.get(\"weight_size_bytes\"),\n        )\n\n    # ------------------------------------------------------------------ inference contract\n\n    def match(self, image0: ImageInput, image1: ImageInput) -> dict[str, Any]:\n        \"\"\"Dense-to-sparse matches between two images: `{kpts0, kpts1, confidence, size0, size1}` with\n        keypoints as (M, 2) float32 (x, y) pixel coordinates in each image's cropped frame.\"\"\"\n        img0, img1 = _coerce_image(image0), _coerce_image(image1)\n        _check_size(img0, \"image0\")\n        _check_size(img1, \"image1\")\n        t0, t1 = _to_tensor(img0).to(self.device), _to_tensor(img1).to(self.device)\n        batch = {\"image0\": t0, \"image1\": t1}\n        with torch.inference_mode():\n            self.model(batch)\n        kpts0 = batch[\"mkpts0_f\"].detach().cpu().numpy().astype(np.float32)\n        kpts1 = batch[\"mkpts1_f\"].detach().cpu().numpy().astype(np.float32)\n        conf = batch[\"mconf_f\"].detach().cpu().numpy().astype(np.float32)\n        return {\n            \"kpts0\": kpts0.reshape(-1, 2),\n            \"kpts1\": kpts1.reshape(-1, 2),\n            \"confidence\": conf.reshape(-1),\n            \"size0\": [int(t0.shape[-1]), int(t0.shape[-2])],\n            \"size1\": [int(t1.shape[-1]), int(t1.shape[-2])],\n            \"coarse_matches\": int(batch[\"mkpts0_c\"].shape[0]),\n        }\n\n    # ------------------------------------------------------------------ evaluation\n\n    def evaluate(\n        self,\n        records: Sequence[Mapping[str, Any]],\n        *,\n        matcher: Callable[[Image.Image, Image.Image], Mapping[str, Any]] | None = None,\n    ) -> dict[str, Any]:\n        \"\"\"Homography-supervised scoring of a validated pair dataset with `metrics.matching_metrics`; `matcher`\n        substitutes a baseline for the model (same record structure, same scoring).\"\"\"\n        from .samples import validate_dataset\n\n        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)[\"records\"]\n        started = time.perf_counter()\n        rows = []\n        for record in checked:\n            result = (\n                matcher(record[\"image0\"], record[\"image1\"])\n                if matcher is not None\n                else self.match(record[\"image0\"], record[\"image1\"])\n            )\n            size = tuple(result.get(\"size0\", record[\"image0\"].size))\n            row = pair_metrics(result, np.asarray(record[\"homography\"]), (int(size[0]), int(size[1])))\n            row.update({\"id\": record[\"id\"], \"tier\": record[\"tier\"]})\n            rows.append(row)\n        out = matching_metrics(rows)\n        tiers = sorted({r[\"tier\"] for r in rows})\n        out[\"by_tier\"] = {\n            tier: {\n                k: v\n                for k, v in matching_metrics([r for r in rows if r[\"tier\"] == tier]).items()\n                if k != \"definitions\"\n            }\n            for tier in tiers\n        }\n        out.update(\n            {\n                \"per_pair\": [{k: v for k, v in r.items() if k != \"inlier_errors\"} for r in rows],\n                \"verdict\": \"measured\" if len(checked) >= MIN_SCORED_RECORDS else \"measured-small-sample\",\n                \"adapted\": self.adapter is not None,\n                \"matcher\": \"model\" if matcher is None else \"baseline\",\n                \"seconds\": round(time.perf_counter() - started, 3),\n                \"model_id\": MODEL_ID,\n                \"model_revision\": MODEL_REVISION,\n            }\n        )\n        return out\n\n    def evaluate_baselines(self, records: Sequence[Mapping[str, Any]]) -> dict[str, dict[str, Any]]:\n        \"\"\"The two non-neural references scored exactly as the model is.\"\"\"\n        out = {}\n        for name, fn in ((\"identity\", identity_baseline), (\"patch_neighbour\", patch_neighbour_baseline)):\n            result = self.evaluate(records, matcher=fn)\n            result[\"baseline\"] = name\n            out[name] = result\n        return out\n\n    # ------------------------------------------------------------------ adaptation\n\n    def _coarse_forward(self, t0: torch.Tensor, t1: torch.Tensor) -> tuple[torch.Tensor, tuple[int, int], tuple[int, int]]:\n        \"\"\"The upstream forward up to the coarse similarity matrix (backbone → positional encoding → coarse\n        transformer → coarse projection), with gradients; returns sim / temperature and the coarse grids.\"\"\"\n        model = self.model\n        eps = 1e-6\n        image0 = (t0 - t0.mean(dim=[2, 3], keepdim=True)) / (t0.std(dim=[2, 3], keepdim=True) + eps)\n        image1 = (t1 - t1.mean(dim=[2, 3], keepdim=True)) / (t1.std(dim=[2, 3], keepdim=True) + eps)\n        feat_c0, _m0, _f0 = model.backbone(image0)\n        feat_c1, _m1, _f1 = model.backbone(image1)\n        hw0 = (int(feat_c0.shape[2]), int(feat_c0.shape[3]))\n        hw1 = (int(feat_c1.shape[2]), int(feat_c1.shape[3]))\n        feat_c0 = model.pos_encoding(feat_c0).flatten(2).permute(0, 2, 1)\n        feat_c1 = model.pos_encoding(feat_c1).flatten(2).permute(0, 2, 1)\n        feat_c0, feat_c1 = model.loftr_coarse(feat_c0, feat_c1, None, None)\n        feat_c0 = model.coarse_matching.final_proj(feat_c0)\n        feat_c1 = model.coarse_matching.final_proj(feat_c1)\n        feat_c0, feat_c1 = feat_c0 / feat_c0.shape[-1] ** 0.5, feat_c1 / feat_c1.shape[-1] ** 0.5\n        sim = torch.einsum(\"nlc,nsc->nls\", feat_c0, feat_c1) / model.coarse_matching.temperature\n        return sim, hw0, hw1\n\n    @staticmethod\n    def coarse_ground_truth(\n        homography: np.ndarray, hw0: tuple[int, int], hw1: tuple[int, int], scale: int = DIVISIBLE_BY\n    ) -> torch.Tensor:\n        \"\"\"The upstream coarse supervision for a homography: every coarse cell of image0 (its top-left pixel)\n        warped into image1 and rounded to the nearest cell, and the reverse, both marked positive (the union\n        rule of `spvs_coarse`); out-of-bounds cells are dropped. Returns (1, h0·w0, h1·w1) with 0 / 1 entries.\"\"\"\n        from .metrics import warp_points\n\n        h0, w0 = hw0\n        h1, w1 = hw1\n        gt = torch.zeros(1, h0 * w0, h1 * w1)\n        ys, xs = np.meshgrid(np.arange(h0), np.arange(w0), indexing=\"ij\")\n        pts0 = np.stack([xs.ravel(), ys.ravel()], axis=1) * float(scale)\n        warped = np.rint(warp_points(pts0, homography) / scale).astype(np.int64)\n        ok = (warped[:, 0] >= 0) & (warped[:, 0] < w1) & (warped[:, 1] >= 0) & (warped[:, 1] < h1)\n        i_ids = np.nonzero(ok)[0]\n        j_ids = warped[ok, 0] + warped[ok, 1] * w1\n        gt[0, i_ids, j_ids] = 1.0\n        ys1, xs1 = np.meshgrid(np.arange(h1), np.arange(w1), indexing=\"ij\")\n        pts1 = np.stack([xs1.ravel(), ys1.ravel()], axis=1) * float(scale)\n        back = np.rint(warp_points(pts1, np.linalg.inv(homography)) / scale).astype(np.int64)\n        ok1 = (back[:, 0] >= 0) & (back[:, 0] < w0) & (back[:, 1] >= 0) & (back[:, 1] < h0)\n        j1 = np.nonzero(ok1)[0]\n        i1 = back[ok1, 0] + back[ok1, 1] * w0\n        gt[0, i1, j1] = 1.0\n        gt[0, 0, 0] = 0.0\n        return gt\n\n    @staticmethod\n    def coarse_loss(sim: torch.Tensor, gt: torch.Tensor) -> torch.Tensor:\n        \"\"\"Upstream `compute_coarse_loss`: the focal term on the positive cells of both softmax directions.\"\"\"\n        conf01 = torch.softmax(sim, 2).clamp(1e-6, 1 - 1e-6)\n        conf10 = torch.softmax(sim, 1).clamp(1e-6, 1 - 1e-6)\n        pos = gt > 0\n        if not bool(pos.any()):\n            return sim.sum() * 0.0\n        loss = -FOCAL_ALPHA * (1 - conf01[pos]) ** FOCAL_GAMMA * conf01[pos].log()\n        loss = loss - FOCAL_ALPHA * (1 - conf10[pos]) ** FOCAL_GAMMA * conf10[pos].log()\n        return POS_WEIGHT * loss.mean()\n\n    def _trainable_names(self, trainable_coarse_layers: int) -> list[str]:\n        if (\n            isinstance(trainable_coarse_layers, bool)\n            or not isinstance(trainable_coarse_layers, int)\n            or not 1 <= trainable_coarse_layers <= COARSE_LAYERS\n        ):\n            raise ValueError(f\"trainable_coarse_layers must be an int in 1..{COARSE_LAYERS}\")\n        first = COARSE_LAYERS - trainable_coarse_layers\n        prefixes = tuple(f\"loftr_coarse.layers.{k}.\" for k in range(first, COARSE_LAYERS)) + (\n            \"coarse_matching.final_proj.\",\n        )\n        return [name for name, _p in self.model.named_parameters() if name.startswith(prefixes)]\n\n    def adapt(\n        self,\n        train: Sequence[Mapping[str, Any]],\n        val: Sequence[Mapping[str, Any]] | None = None,\n        *,\n        epochs: int = 3,\n        lr: float = 5e-5,\n        batch_size: int = 4,\n        trainable_coarse_layers: int = DEFAULT_TRAINABLE_COARSE_LAYERS,\n        seed: int = 0,\n        progress: Callable[[dict[str, Any]], None] | None = None,\n    ) -> dict[str, Any]:\n        \"\"\"Bounded fine-tuning of the coarse matcher on validated pairs.\n\n        Only the last `trainable_coarse_layers` layers of the coarse transformer (`loftr_coarse`; a self and a\n        cross layer by default) and the coarse projection (`coarse_matching.final_proj`) train — 1,378,560 of\n        11,091,722 parameters; the backbone, the positional encoding and the whole fine level stay frozen. Each\n        pair runs the upstream forward to the coarse similarity matrix and is scored with the upstream coarse\n        focal loss against the homography's coarse ground truth; `batch_size` pairs are accumulated per AdamW\n        step (pairs have different sizes, so they are not stacked), gradients are clipped at 1.0, the order is\n        seeded, no scheduler. Epoch 0 records the frozen model's validation metrics; the epoch with the highest\n        validation precision at 3 px is kept (ties broken by homography accuracy at 3 px). Transactional: any\n        failure restores the base tensors.\"\"\"\n        from .samples import validate_dataset\n\n        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:\n            raise ValueError(\"epochs must be an int in 1..20\")\n        if not (0.0 < lr <= 1e-3):\n            raise ValueError(\"lr must be in (0, 1e-3]\")\n        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 32:\n            raise ValueError(\"batch_size must be an int in 1..32\")\n        names = self._trainable_names(trainable_coarse_layers)\n        train_checked = validate_dataset(train)[\"records\"]\n        val_checked = validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)[\"records\"] if val else []\n        model = self.model\n        torch.manual_seed(seed)\n        started = time.perf_counter()\n        wanted = set(names)\n        for name, param in model.named_parameters():\n            param.requires_grad_(name in wanted)\n        params = [p for p in model.parameters() if p.requires_grad]\n        n_trainable = sum(p.numel() for p in params)\n        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)\n        tensors = [(_to_tensor(r[\"image0\"]), _to_tensor(r[\"image1\"]), np.asarray(r[\"homography\"])) for r in train_checked]\n\n        def score_val() -> dict[str, Any] | None:\n            if not val_checked:\n                return None\n            model.eval()\n            result = self.evaluate(val_checked)\n            return {k: result[k] for k in (\"precision_3px\", \"homography_acc_3px\", \"inliers_per_pair\", \"matches_per_pair\", \"n\")}\n\n        def key(entry: dict[str, Any]) -> tuple[float, float]:\n            return (entry[\"val\"][\"precision_3px\"], entry[\"val\"][\"homography_acc_3px\"]) if entry[\"val\"] else (-math.inf, -math.inf)\n\n        history: list[dict[str, Any]] = []\n        entry: dict[str, Any] = {\"epoch\": 0, \"train_loss\": None, \"val\": score_val(), \"note\": \"frozen model\"}\n        history.append(entry)\n        if progress:\n            progress(entry)\n        best_key = key(entry)\n        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}\n        initial_state = {k: v.clone() for k, v in best_state.items()}\n        best_epoch = 0\n        generator = torch.Generator().manual_seed(seed)\n        try:\n            for epoch in range(1, epochs + 1):\n                model.train()\n                model.backbone.eval()  # frozen BatchNorm statistics\n                order = torch.randperm(len(tensors), generator=generator).tolist()\n                losses = []\n                for start in range(0, len(order), batch_size):\n                    optimiser.zero_grad(set_to_none=True)\n                    chunk = order[start : start + batch_size]\n                    total = 0.0\n                    for i in chunk:\n                        t0, t1, homography = tensors[i]\n                        sim, hw0, hw1 = self._coarse_forward(t0.to(self.device), t1.to(self.device))\n                        gt = self.coarse_ground_truth(homography, hw0, hw1).to(self.device)\n                        loss = self.coarse_loss(sim, gt) / len(chunk)\n                        loss.backward()\n                        total += float(loss.detach())\n                    torch.nn.utils.clip_grad_norm_(params, 1.0)\n                    optimiser.step()\n                    losses.append(total)\n                model.eval()\n                entry = {\"epoch\": epoch, \"train_loss\": sum(losses) / len(losses), \"val\": score_val()}\n                history.append(entry)\n                if progress:\n                    progress(entry)\n                if not entry[\"val\"] or key(entry) > best_key:\n                    best_key = key(entry)\n                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}\n                    best_epoch = epoch\n        except BaseException:\n            restore = dict(model.state_dict())\n            restore.update(initial_state)\n            model.load_state_dict(restore, strict=True)\n            model.eval()\n            for param in model.parameters():\n                param.requires_grad_(False)\n            self.adapter = None\n            raise\n        merged = dict(model.state_dict())\n        merged.update(best_state)\n        model.load_state_dict(merged, strict=True)\n        model.eval()\n        for param in model.parameters():\n            param.requires_grad_(False)\n        self.adapter = {\n            \"trainable_coarse_layers\": trainable_coarse_layers,\n            \"trainable_names\": names,\n            \"n_trainable\": n_trainable,\n            \"n_total\": sum(p.numel() for p in model.parameters()),\n            \"epochs\": epochs,\n            \"best_epoch\": best_epoch,\n            \"selection\": \"highest validation precision at 3 px (ties: homography accuracy at 3 px)\"\n            if val_checked\n            else \"final epoch (no validation split)\",\n            \"lr\": lr,\n            \"batch_size\": batch_size,\n            \"n_train\": len(train_checked),\n            \"n_val\": len(val_checked),\n            \"seed\": seed,\n            \"history\": history,\n            \"seconds\": round(time.perf_counter() - started, 2),\n        }\n        return dict(self.adapter)\n\n    # ------------------------------------------------------------------ artifacts\n\n    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:\n        \"\"\"Write the adapted coarse-matcher tensors as safetensors plus a base manifest.\"\"\"\n        if self.adapter is None:\n            raise ValueError(\"nothing to save: call adapt() first\")\n        from safetensors.torch import save_file\n\n        out = Path(output_dir)\n        out.mkdir(parents=True, exist_ok=True)\n        names = set(self.adapter[\"trainable_names\"])\n        tensors = {k: v.detach().cpu().contiguous() for k, v in self.model.state_dict().items() if k in names}\n        weights_path = out / ARTIFACT_WEIGHTS_NAME\n        save_file(tensors, str(weights_path), metadata={\"format\": \"pt\"})\n        manifest = {\n            \"format\": ARTIFACT_FORMAT,\n            \"format_version\": ARTIFACT_FORMAT_VERSION,\n            \"base_model\": {\n                \"id\": MODEL_ID,\n                \"revision\": MODEL_REVISION,\n                \"key\": DEFAULT_MODEL_KEY,\n                \"weight_file\": MODEL_FILENAME,\n                \"weight_sha256\": MODEL_SHA256,\n            },\n            \"adapter\": {k: v for k, v in self.adapter.items() if k not in (\"history\", \"trainable_names\")},\n            \"history\": self.adapter[\"history\"],\n            \"tensors\": sorted(tensors),\n            \"files\": [\n                {\n                    \"path\": ARTIFACT_WEIGHTS_NAME,\n                    \"bytes\": weights_path.stat().st_size,\n                    \"sha256\": _sha256_file(weights_path),\n                }\n            ],\n            \"metadata\": dict(metadata or {}),\n        }\n        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding=\"utf-8\")\n        return out\n\n    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:\n        \"\"\"Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite\n        exactly the tensors it carries.\"\"\"\n        root = Path(artifact_dir)\n        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding=\"utf-8\"))\n        if manifest.get(\"format\") != ARTIFACT_FORMAT:\n            raise ValueError(f\"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}\")\n        if manifest.get(\"format_version\") != ARTIFACT_FORMAT_VERSION:\n            raise ValueError(f\"artifact format_version {manifest.get('format_version')!r} != {ARTIFACT_FORMAT_VERSION!r}\")\n        base = manifest.get(\"base_model\") or {}\n        if (base.get(\"id\"), base.get(\"revision\"), base.get(\"weight_sha256\")) != (MODEL_ID, MODEL_REVISION, MODEL_SHA256):\n            raise ValueError(\"artifact was adapted from a different base model, revision or weight file\")\n        if base.get(\"weight_file\", MODEL_FILENAME) != MODEL_FILENAME:\n            raise ValueError(\"artifact was adapted from a different base weight file\")\n        files = manifest.get(\"files\")\n        if not isinstance(files, list) or len(files) != 1 or files[0].get(\"path\") != ARTIFACT_WEIGHTS_NAME:\n            raise ValueError(f\"artifact manifest must list exactly {ARTIFACT_WEIGHTS_NAME!r}\")\n        weights_path = (root / ARTIFACT_WEIGHTS_NAME).resolve()\n        if weights_path.parent != root.resolve():\n            raise ValueError(\"artifact weight path must resolve inside the artifact directory\")\n        layers = (manifest.get(\"adapter\") or {}).get(\"trainable_coarse_layers\")\n        expected = sorted(self._trainable_names(layers))\n        if sorted(manifest.get(\"tensors\") or []) != expected:\n            raise ValueError(\"artifact tensor list does not match its recorded configuration\")\n        entry = files[0]\n        if not weights_path.is_file():\n            raise FileNotFoundError(f\"artifact weights missing: {weights_path}\")\n        if _sha256_file(weights_path) != entry[\"sha256\"] or weights_path.stat().st_size != entry[\"bytes\"]:\n            raise ValueError(f\"{entry['path']}: digest or size mismatch; refusing to load\")\n        from safetensors.torch import load_file\n\n        tensors = load_file(str(weights_path))\n        if sorted(tensors) != expected:\n            raise ValueError(\"artifact tensor names differ from its manifest\")\n        state = self.model.state_dict()\n        for key, value in tensors.items():\n            if key not in state or not key.startswith((\"loftr_coarse.\", \"coarse_matching.\")):\n                raise ValueError(f\"artifact tensor {key} is not an adaptable coarse-matcher tensor of the base\")\n            if tuple(value.shape) != tuple(state[key].shape):\n                raise ValueError(f\"artifact tensor {key} has shape {tuple(value.shape)}, base has {tuple(state[key].shape)}\")\n        merged = dict(state)\n        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})\n        self.model.load_state_dict(merged, strict=True)\n        self.model.eval()\n        self.adapter = {**manifest[\"adapter\"], \"trainable_names\": manifest[\"tensors\"], \"history\": manifest.get(\"history\", [])}\n        return manifest\n\n    @classmethod\n    def from_artifact(\n        cls,\n        artifact_dir: str | Path,\n        *,\n        device: str | torch.device | None = None,\n        weights_dir: str | Path | None = None,\n        allow_download: bool = False,\n    ) -> XoFTRPipeline:\n        \"\"\"A fresh pipeline from the pinned base with an adapter overlaid.\"\"\"\n        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)\n        pipe.load_artifact(artifact_dir)\n        return pipe\n\n\ndef load_pipeline(**kwargs: Any) -> XoFTRPipeline:\n    return XoFTRPipeline.from_pretrained(**kwargs)\n\n\ndef evaluation_report(result: Mapping[str, Any], reference: Mapping[str, Any] | None = None, *, sample_kind: str = \"synthetic\") -> dict[str, Any]:\n    \"\"\"Evaluation stage for the drawn-shape sanity pair: a machine-readable report even when nothing is\n    measurable. With a `reference` homography and image size the report carries the pair metrics with the\n    verdict `sample-sanity`; without it the verdict is `not-measurable`.\"\"\"\n    base: dict[str, Any] = {\n        \"task\": \"detector-free image matching (coarse-to-fine, sub-pixel refined)\",\n        \"score_semantics\": (\n            \"match confidences are dual-softmax / fine-level scores in (0, 1], not calibrated probabilities \"\n            \"that a match is correct; the decision rule is the upstream coarse / fine thresholds; no \"\n            \"geometric verification ships\"\n        ),\n        \"sample_kind\": sample_kind,\n        \"n_matches\": int(len(np.asarray(result.get(\"kpts0\", [])).reshape(-1, 2))),\n    }\n    if not reference:\n        return {**base, \"metrics\": [], \"verdict\": \"not-measurable\"}\n    row = pair_metrics(result, np.asarray(reference[\"homography\"]), tuple(reference[\"size\"]))\n    metrics = [\n        {\"id\": \"precision_3px\", \"value\": row[\"precision_3px\"], \"definition\": \"fraction of matches under 3 px reprojection error\"},\n        {\"id\": \"n_inliers\", \"value\": row[\"n_inliers\"], \"definition\": \"matches under 3 px\"},\n        {\"id\": \"corner_error_px\", \"value\": row[\"corner_error_px\"], \"definition\": \"mean corner displacement of the RANSAC-DLT homography vs the reference\"},\n    ]\n    return {**base, \"metrics\": metrics, \"verdict\": \"sample-sanity\"}\n\n\n__all__ = [\n    \"ARTIFACT_FORMAT\",\n    \"COARSE_LAYERS\",\n    \"DEFAULT_TRAINABLE_COARSE_LAYERS\",\n    \"INPUT_SCHEMA\",\n    \"MAX_EVAL_RECORDS\",\n    \"MIN_SCORED_RECORDS\",\n    \"PARAMETER_COUNT\",\n    \"XoFTRPipeline\",\n    \"evaluation_report\",\n    \"load_pipeline\",\n    \"validate_inputs\",\n]\n","xoftr_pipeline/provenance.py":"from __future__ import annotations\n\nimport json\nimport platform\nimport sys\nfrom importlib.metadata import PackageNotFoundError, version\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .config import (\n    COARSE_THRESHOLD,\n    DIVISIBLE_BY,\n    FINE_THRESHOLD,\n    MAX_SIDE,\n    MIN_SIDE,\n    MODEL_FILENAME,\n    MODEL_ID,\n    MODEL_REVISION,\n    MODEL_SHA256,\n    MODEL_SIZE_BYTES,\n)\nfrom .modeling import UPSTREAM_COMMIT, UPSTREAM_REPOSITORY\n\n_RUNTIME_PACKAGES = (\n    \"huggingface-hub\",\n    \"numpy\",\n    \"pillow\",\n    \"safetensors\",\n    \"torch\",\n)\n\n\ndef _package_version(name: str) -> str | None:\n    try:\n        return version(name)\n    except PackageNotFoundError:\n        return None\n\n\ndef build_provenance(\n    *,\n    pipeline: Any | None = None,\n    checkpoint_path: str | Path | None = None,\n    include_runtime: bool = True,\n) -> dict[str, Any]:\n    checkpoint_source = None\n    manifest_verified = False\n    weight_sha256 = MODEL_SHA256\n    weight_size = MODEL_SIZE_BYTES\n    device_str = None\n    resolved_checkpoint_path = None\n    adapter = None\n\n    if pipeline is not None:\n        if getattr(pipeline, \"checkpoint_path\", None) is not None:\n            resolved_checkpoint_path = str(pipeline.checkpoint_path)\n        checkpoint_source = getattr(pipeline, \"checkpoint_source\", None)\n        manifest_verified = bool(getattr(pipeline, \"manifest_verified\", False))\n        if getattr(pipeline, \"weight_sha256\", None):\n            weight_sha256 = pipeline.weight_sha256\n        if getattr(pipeline, \"weight_size_bytes\", None):\n            weight_size = pipeline.weight_size_bytes\n        if getattr(pipeline, \"device\", None) is not None:\n            device_str = str(pipeline.device)\n        if getattr(pipeline, \"adapter\", None):\n            skip = (\"history\", \"trainable_names\")\n            adapter = {k: v for k, v in pipeline.adapter.items() if k not in skip}\n    if checkpoint_path is not None:\n        resolved_checkpoint_path = str(checkpoint_path)\n\n    provenance: dict[str, Any] = {\n        \"schema_version\": 1,\n        \"model\": {\n            \"id\": MODEL_ID,\n            \"revision\": MODEL_REVISION,\n            \"weight_file\": MODEL_FILENAME,\n            \"weight_sha256\": weight_sha256,\n            \"weight_size_bytes\": weight_size,\n            \"checkpoint_source\": checkpoint_source,\n            \"checkpoint_path\": resolved_checkpoint_path,\n            \"manifest_verified\": manifest_verified,\n            \"vendored_code\": {\"repository\": UPSTREAM_REPOSITORY, \"commit\": UPSTREAM_COMMIT},\n        },\n        \"preprocessing\": {\n            \"grey_scale\": True,\n            \"crop_to_multiple_of\": DIVISIBLE_BY,\n            \"side_range_px\": [MIN_SIDE, MAX_SIDE],\n            \"per_image_standardisation\": \"inside the network (mean / std over the image)\",\n        },\n        \"inference\": {\n            \"coarse_threshold\": COARSE_THRESHOLD,\n            \"fine_threshold\": FINE_THRESHOLD,\n            \"match_confidence_semantics\": \"dual_softmax_and_fine_scores_not_calibrated_probability\",\n            \"geometric_verification\": (\n                \"none in the pipeline; the evaluation's RANSAC-DLT is a metric, not a filter\"\n            ),\n            \"device\": device_str,\n        },\n        \"adapter\": adapter,\n    }\n    if include_runtime:\n        provenance[\"runtime\"] = {\n            \"python\": platform.python_version(),\n            \"implementation\": platform.python_implementation(),\n            \"platform\": sys.platform,\n            \"packages\": {name: _package_version(name) for name in _RUNTIME_PACKAGES},\n        }\n    return provenance\n\n\ndef write_provenance(\n    path: str | Path,\n    *,\n    pipeline: Any | None = None,\n    checkpoint_path: str | Path | None = None,\n    include_runtime: bool = True,\n) -> Path:\n    record = build_provenance(\n        pipeline=pipeline, checkpoint_path=checkpoint_path, include_runtime=include_runtime\n    )\n    out = Path(path)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    out.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding=\"utf-8\")\n    return out\n","xoftr_pipeline/samples.py":"\"\"\"Image-pair dataset contract for adapting the matcher: the pinned iNaturalist photograph corpus, seeded\nhomography pairs with exact references, validation, splitting, BYOD loaders and CSV export.\n\nThe images are **real**: 360 CC0-licensed, research-grade iNaturalist photographs of six North American bird\nspecies (the fleet's SigLIP sample; 60 per species, one per observer), pinned here by photo id, byte size and\nSHA-256 of the served `medium` JPEG and fetched from the iNaturalist open-data bucket at run time, refused on any\nbyte-size or SHA-256 mismatch; the repository redistributes none of them, and every record keeps its\nobservation page and observer login. Each photograph becomes one **pair**: the photograph (long side scaled to\n640 px, sides cropped to multiples of 8) and a copy warped by a seeded homography with seeded photometric\nchanges, so the reference `H` (image0 → image1) is exact and every returned match has a reprojection error.\nTwo difficulty tiers alternate per photograph: `easy` (corner jitter up to 6 % of the side, rotation ±10°,\nscale 0.9–1.1, mild brightness / contrast / noise) and `hard` (jitter up to 18 %, rotation ±35°, scale 0.6–1.4,\nstrong brightness / contrast / gamma / blur / noise).\n\nA record is ``{id, image0, image1, homography}`` (PIL images or paths, and a 3 × 3 list); `SPECIES` maps the\nphotograph's species key to its names for provenance.\n\"\"\"\n# ruff: noqa: E501  -- fleet dataset module written at the 110-column fleet width; this repo lints at 100\n\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport random\nimport re\nimport urllib.request\nimport zipfile\nfrom collections.abc import Mapping, Sequence\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nfrom PIL import Image, ImageEnhance, ImageFilter\n\nfrom .config import DIVISIBLE_BY, MAX_SIDE, MIN_SIDE, MODEL_ID\n\nMAX_IMAGE_SIDE = 4096  # pixels; larger uploads are rejected before any decode-to-tensor work\nWORKING_LONG_SIDE = 640  # the pairs are built at the checkpoint's training resolution\n\nCORPUS_NAME = \"iNaturalist CC0 bird photographs (six species)\"\nCORPUS_RELEASE = (\n    \"iNaturalist open-data bucket, research-grade CC0 photos selected 2026-09-19 (60 per species)\"\n)\nCORPUS_BASE_URL = \"https://inaturalist-open-data.s3.amazonaws.com/photos/\"\nCORPUS_LICENSE = (\n    \"CC0 1.0 (each photo's own license_code on iNaturalist; observers credited in the records)\"\n)\nCORPUS_BYTES = 39_223_447\nDEFAULT_CACHE_DIR = Path(\"weights\") / \"inat-birds\"\nSPECIES: dict[str, tuple[str, str]] = {\n    \"song_sparrow\": (\"Melospiza melodia\", \"Song Sparrow\"),\n    \"chipping_sparrow\": (\"Spizella passerina\", \"Chipping Sparrow\"),\n    \"white_throated_sparrow\": (\"Zonotrichia albicollis\", \"White-throated Sparrow\"),\n    \"dark_eyed_junco\": (\"Junco hyemalis\", \"Dark-eyed Junco\"),\n    \"house_finch\": (\"Haemorhous mexicanus\", \"House Finch\"),\n    \"american_goldfinch\": (\"Spinus tristis\", \"American Goldfinch\"),\n}\n# (id, species, iNat photo id, iNat observation id, observer login, bytes, sha256 of the served\n#  <photo id>/medium.<ext>, ext) — the bucket serves each photo under its original extension\n#  (jpg or jpeg); the digest pins the served bytes\nSAMPLE_RECORDS: tuple[tuple[str, str, int, int, str, int, str, str], ...] = (\n    (\n        \"song_sparrow-00\",\n        \"song_sparrow\",\n        129376982,\n        79016324,\n        \"andywilson\",\n        43427,\n        \"7a9d9304a82f202e992655ec5f65477cd3d7c1dce03aa89a214c2daa38f9d61d\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-01\",\n        \"song_sparrow\",\n        480991086,\n        267636534,\n        \"lyneisfilm\",\n        162073,\n        \"11f77ff277dd2703c1000f2c787136ff0c3ca7ffad7017056892fe789d65efec\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-02\",\n        \"song_sparrow\",\n        546060381,\n        302980489,\n        \"swpollinators\",\n        27899,\n        \"4e70b9519c6e5f7384a4495b91b45465f2b1599f86491d8f9f9b12635a4046f6\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-03\",\n        \"song_sparrow\",\n        308625896,\n        177450028,\n        \"radrat\",\n        70961,\n        \"1211da4fdb24ae85ef0c6c3e2d03542c430457856aec661fe8f5f2de0027eee5\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-04\",\n        \"song_sparrow\",\n        494793016,\n        275349085,\n        \"k-simpkins\",\n        58410,\n        \"255538cf450197257e86ed3d41dc69fb78e594434e9cb338c6314288c6cff26e\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-05\",\n        \"song_sparrow\",\n        674054489,\n        369029444,\n        \"ben142\",\n        220573,\n        \"1ae24622888d9d449ffd6b5c65cac1b9b14870fd8e12dbed8aa0acc2f5030123\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-06\",\n        \"song_sparrow\",\n        339623726,\n        193339933,\n        \"rawcomposition\",\n        25012,\n        \"d2cde085277a71886a2bf211a1eec26941a73752375708726a8af29aa4995a8b\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-07\",\n        \"song_sparrow\",\n        181658744,\n        107953669,\n        \"gcart043\",\n        98482,\n        \"a662a6abb24f42b256fb6e2d6f02c3e128a1053ef534f454461aa5cadcc03fe4\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-08\",\n        \"song_sparrow\",\n        222768957,\n        130949329,\n        \"davidfbird\",\n        110773,\n        \"0feee62753f409d0aa365e9aa017ddef436ad70a2673dc847feb3b5386af27ff\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-09\",\n        \"song_sparrow\",\n        148994242,\n        90171417,\n        \"glennberry\",\n        102702,\n        \"64333977d24957d723a1d26886004f222d810557617685579f72a6b553e508fa\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-10\",\n        \"song_sparrow\",\n        637315932,\n        349374463,\n        \"sooji\",\n        136572,\n        \"a5cebbc0cc2325d3805c4ac854e103f7ba1c22e3a34fb204f30d58681e865033\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-11\",\n        \"song_sparrow\",\n        471146686,\n        262252507,\n        \"jeanpaulboerekamps\",\n        98601,\n        \"4301f06b52b8dcf1e137567c32412e384edb71cb90e2e35459a0405b2e56529b\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-12\",\n        \"song_sparrow\",\n        640811426,\n        351179648,\n        \"erikschiff\",\n        107695,\n        \"27aecce184a485ee888c0e8101cb4a34899b8a185b62dc064f3dc2ec9902182b\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-13\",\n        \"song_sparrow\",\n        123859010,\n        75689904,\n        \"w_mark_c\",\n        190819,\n        \"6b6057a1c50b83ffeb4d9e34367b3e6a9b355236dbcae9820b509e1484f8fe33\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-14\",\n        \"song_sparrow\",\n        108520869,\n        67204020,\n        \"dugald\",\n        52496,\n        \"e483364889fb95c540db84b5edf5a2400febf7d2eb62a1e49303b13ad329d1b8\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-15\",\n        \"song_sparrow\",\n        63537800,\n        40010230,\n        \"nathanael15\",\n        51441,\n        \"5ae55f868e779a4e8ee34f6ad077b40c41f20aa699d967f373ebd659a89137a8\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-16\",\n        \"song_sparrow\",\n        435251292,\n        244042351,\n        \"carterdorscht\",\n        166662,\n        \"d0cdd9ddcf7202a91ac2c47910a0c230639bce50283e8511fb8311151763233a\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-17\",\n        \"song_sparrow\",\n        120710033,\n        73898230,\n        \"tys_rbg\",\n        126820,\n        \"515b3b32b64e88d401990bfd1d8e2c1281443b5d1e763a09e86d3d3664e1df41\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-18\",\n        \"song_sparrow\",\n        349361283,\n        198243065,\n        \"sean579\",\n        42820,\n        \"5eb0031a8d6066650c66b265fb1724413273e095e4e531054d2817eb75910074\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-19\",\n        \"song_sparrow\",\n        393408927,\n        222067370,\n        \"irenemacaulay_\",\n        101681,\n        \"22e326807de963352b4307790bd40c5506adb0fab1f9844968edaf4bc5665e1c\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-20\",\n        \"song_sparrow\",\n        614258628,\n        337847585,\n        \"joy4birds\",\n        70552,\n        \"aa767aa5a74a76cfd985aba5282e589f670a9ce0a8fcdc6a096c0357992dec67\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-21\",\n        \"song_sparrow\",\n        608802621,\n        335154467,\n        \"jamesadney\",\n        112564,\n        \"6e94bce1b5135f48b0b19b64e21a0c76cff4f9b9a28fcbc2b7f5c8ddda6ef513\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-22\",\n        \"song_sparrow\",\n        634100352,\n        347744524,\n        \"zorthesosen\",\n        214044,\n        \"6a70dba26dfb3e98a244a9ec9badb812ddaa8b3612e7a36f66aba057bb757ecd\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-23\",\n        \"song_sparrow\",\n        50065474,\n        31954532,\n        \"truthseqr\",\n        82521,\n        \"4186fbf344e92038358d4338102aa440098bff5f558ab1d197f72fefbcec0b15\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-24\",\n        \"song_sparrow\",\n        8656044,\n        6803564,\n        \"glmory\",\n        297964,\n        \"e8cab773436ccfaad4699e5112ef4edb516236d413d6dbf034eea7bf9c88c8f5\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-25\",\n        \"song_sparrow\",\n        131051149,\n        79988590,\n        \"funvill\",\n        72572,\n        \"0a694b5bb03aee6eb5a3a6132e855d47172b21b84cc4cc7c64367d1a31de7894\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-26\",\n        \"song_sparrow\",\n        12923156,\n        9491600,\n        \"gambolingquail\",\n        69559,\n        \"6a92bff4fc76820c6f21e15c5f254385dd976a32264911bfc08417a1c2bf053d\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-27\",\n        \"song_sparrow\",\n        12077500,\n        8959545,\n        \"reuvenm\",\n        74010,\n        \"3e3c5e94c839f45610ef3ef7ffaf3575f4bfe365fc94454c30bfdb718ca3c593\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-28\",\n        \"song_sparrow\",\n        132730299,\n        80955309,\n        \"steph123456\",\n        154309,\n        \"5f8854dd231a302c643b22521b49ed3007203d74b8b8413a23abc79ffe0b0270\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-29\",\n        \"song_sparrow\",\n        124840110,\n        76316566,\n        \"terrimewbornagain\",\n        81046,\n        \"2020092b0b67397a78cc2b0df267fb671b3ba18e03fb0e35e664e0784061e3c6\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-30\",\n        \"song_sparrow\",\n        130188079,\n        79482300,\n        \"fake_id\",\n        180462,\n        \"1d9ad7b45536b85d46fa0fd9a29d4a8ae828bc223836796a3eb9df1fe8fe4fc4\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-31\",\n        \"song_sparrow\",\n        136405967,\n        83075831,\n        \"tom_lazar\",\n        178554,\n        \"f86c3c5899d20ac8fb73afb76d45666c610245636d7285a6d517b0081b9d6cc9\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-32\",\n        \"song_sparrow\",\n        118885385,\n        72867352,\n        \"ellyne\",\n        105156,\n        \"aab53fb1e7c12adc697711215778c81983f51af83e20bfffe841b80c1d144673\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-33\",\n        \"song_sparrow\",\n        676467457,\n        370293984,\n        \"sholsenbeck\",\n        59104,\n        \"b7c230ca942cedc1c53f42902e1f70bc2d09497e57d2e9f31dfe72a5f6c84d16\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-34\",\n        \"song_sparrow\",\n        688316087,\n        376461614,\n        \"doublecritch\",\n        109416,\n        \"7f11c69df733f99d6e8663e91d807327f8cc57f404049fdbe95e2fd1ce1066e1\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-35\",\n        \"song_sparrow\",\n        691785499,\n        378254663,\n        \"karuquebec\",\n        144522,\n        \"9862b079bcf496aa8d9ce9e716e65b03a3d9a70a5c2077a223630cf9dcc6d5cc\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-36\",\n        \"song_sparrow\",\n        691561091,\n        378140427,\n        \"vaughnshirey\",\n        87827,\n        \"a2c197f91c0543ec7bd3a9aac4867a34417b2eb2444bf3062060f9f1e3580045\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-37\",\n        \"song_sparrow\",\n        582182748,\n        321880490,\n        \"dougbrown\",\n        44795,\n        \"465ce1214c40efab6afdddfbfd34fc5076e5640d130b0442e8e8ac2731ec8a1f\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-38\",\n        \"song_sparrow\",\n        704199014,\n        384692464,\n        \"enspring\",\n        85269,\n        \"1aa25fd1ed376b9245127432925832cf9253e5d156f4f7d35298eff9a8002043\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-39\",\n        \"song_sparrow\",\n        583754170,\n        322666295,\n        \"rociherrera\",\n        97743,\n        \"5e093fc6d59b13bbd7ee0889613cef115ef7bb14432634d0af753a8124951737\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-40\",\n        \"song_sparrow\",\n        590270305,\n        325961498,\n        \"damienxw\",\n        133375,\n        \"a12a152c94ef9c27e5f6dc557e74bf57f6c9ca3015a5e28ee9c30a27ed0e7e19\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-41\",\n        \"song_sparrow\",\n        423534602,\n        237940479,\n        \"don54\",\n        25229,\n        \"69b0f0dfb05e8219700e3878c9b7e4a2d43d0a2df3905414630913397a565dea\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-42\",\n        \"song_sparrow\",\n        418537114,\n        235351206,\n        \"memoosborne\",\n        95540,\n        \"3432cf5a95939b898989c5a962c66b2d71b92b17195d5e695020a776361af3fa\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-43\",\n        \"song_sparrow\",\n        280724703,\n        162358184,\n        \"shirleymorrison\",\n        43015,\n        \"f64af9b3fc4559f251ba0f1e95bf8889bfa629123586532adeb2b81e4c2871b2\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-44\",\n        \"song_sparrow\",\n        641009256,\n        351278414,\n        \"robinlanark\",\n        49640,\n        \"e4af33fd4a504a876b6e7a111a5024cf4197fc888c90f2039df63e9be13d5f06\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-45\",\n        \"song_sparrow\",\n        531314531,\n        295151228,\n        \"steveplumb\",\n        128710,\n        \"8806861909c0833b29f43b10e9428e104deaccb99d41b42039b76f7da0ae4600\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-46\",\n        \"song_sparrow\",\n        114487462,\n        70425556,\n        \"leahmfulton\",\n        50091,\n        \"d0b3c9408b72514f8485b19f61116b94be002be99368db96e126d625a17bd343\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-47\",\n        \"song_sparrow\",\n        118804811,\n        72823547,\n        \"heibudas\",\n        191643,\n        \"8dbe5a56d5fea83b6c25257aa0f9e552ea971f1e0311f583691d24fc9668556f\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-48\",\n        \"song_sparrow\",\n        118498513,\n        72671334,\n        \"seanwashington1\",\n        80800,\n        \"940762ff238d94815922d06051ca00fe4a025279b1da08921a44733084adcdb6\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-49\",\n        \"song_sparrow\",\n        119696805,\n        73327300,\n        \"rambryum\",\n        68983,\n        \"65f7007d6b90654b77b7f4ec0557ff2d0f310c572368cb3492dd69f9d082e90d\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-50\",\n        \"song_sparrow\",\n        4251416,\n        3670620,\n        \"swells\",\n        49271,\n        \"8c53fa5c02e603227117293720d269c59a13555abd537a83640210805fbd844d\",\n        \"JPG\",\n    ),\n    (\n        \"song_sparrow-51\",\n        \"song_sparrow\",\n        303954368,\n        174929167,\n        \"dianeclark6280\",\n        157478,\n        \"14446920b7c55472de3b172f417fbbeac312090add7356a3c947cfb16911dc95\",\n        \"png\",\n    ),\n    (\n        \"song_sparrow-52\",\n        \"song_sparrow\",\n        392544542,\n        221653008,\n        \"emckenziewhat\",\n        138683,\n        \"7fd2f3ad3c835e541a7c720372d662a8f953f88f4d8223864a03e4b0f70a42d3\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-53\",\n        \"song_sparrow\",\n        270614122,\n        156495271,\n        \"radkins21\",\n        92783,\n        \"7bf8ceb5944de93cbf01aebe4d464b9acb848569c07fbe6e732b701d309f0b25\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-54\",\n        \"song_sparrow\",\n        139634990,\n        84920252,\n        \"raffib128\",\n        56293,\n        \"865cf92a61d89cb3420c9d3cbf585a0724df2e314918b05d705804aee356f1a5\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-55\",\n        \"song_sparrow\",\n        25251350,\n        16733911,\n        \"kemper\",\n        54034,\n        \"99f19fa1a37d23712538e43f6a82339f1fd37defe283de3baacc17719f05857e\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-56\",\n        \"song_sparrow\",\n        139129879,\n        84639025,\n        \"wewantashrubbery\",\n        104736,\n        \"aabbffe279d90ffdf737175cc720bea92d634483b830c90acecf920901419643\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-57\",\n        \"song_sparrow\",\n        128242284,\n        78367372,\n        \"stevestevens\",\n        112226,\n        \"918dfc8094d0a697beb1dcd04499cfd7b0c4b12f9dac4d915d1566ab5690b29b\",\n        \"jpeg\",\n    ),\n    (\n        \"song_sparrow-58\",\n        \"song_sparrow\",\n        478733235,\n        266446909,\n        \"saintaardvark\",\n        298253,\n        \"d3c8d3a58b44d28c6af4748e4e4195a9a2af7dc6fe91541429dc44aa3a659d21\",\n        \"jpg\",\n    ),\n    (\n        \"song_sparrow-59\",\n        \"song_sparrow\",\n        623006598,\n        342147975,\n        \"ryman56\",\n        60213,\n        \"73e18df350a69ce00158f03c7ec4fda8672539e0a7292b75f1dfa22dbec7ee30\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-00\",\n        \"chipping_sparrow\",\n        198992636,\n        117809422,\n        \"k-simpkins\",\n        62563,\n        \"cf9f3b0c1863808e21af596b2e609b047ddbc28cb2ed076625e2546425ff0adf\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-01\",\n        \"chipping_sparrow\",\n        248210057,\n        144599194,\n        \"w_mark_c\",\n        193389,\n        \"497d0a0fef81c326bcc87b5d1eb97fe987d559b8f2b71bb60fe422ae34dce150\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-02\",\n        \"chipping_sparrow\",\n        156350853,\n        94266719,\n        \"ellyne\",\n        142332,\n        \"6a60ebac34476a372b4790a87d823432cdda8f72590930b5d18ae166cf4c7ba2\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-03\",\n        \"chipping_sparrow\",\n        16128796,\n        11327134,\n        \"reuvenm\",\n        70532,\n        \"5ad36c9cdd6c92e225a1b8ab3c04d2f65bc4e971d0243958f8ac090b0996115c\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-04\",\n        \"chipping_sparrow\",\n        391300648,\n        220982684,\n        \"carterdorscht\",\n        208567,\n        \"c974a676c3c227c2844ff822d429c784bc87bb41f40d5074a0ebadd4c6785b21\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-05\",\n        \"chipping_sparrow\",\n        339456083,\n        193252042,\n        \"rawcomposition\",\n        46329,\n        \"41c9260ca9107430e3a8090ba01cebf3e3c25f2dad15bea4f77c8e824d9fc496\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-06\",\n        \"chipping_sparrow\",\n        40457652,\n        26076708,\n        \"andywilson\",\n        156894,\n        \"b70edd35e00fe672a39d5f0441ea4e9bb9fac19b0f2415ae13e22160bc6091a6\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-07\",\n        \"chipping_sparrow\",\n        84151914,\n        52921135,\n        \"davidfbird\",\n        145714,\n        \"ea65ce5ed881ded9a8157b5756fa907948f41e0eefb6735b48a1c43993e977f9\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-08\",\n        \"chipping_sparrow\",\n        523674610,\n        291074747,\n        \"rwp84\",\n        47983,\n        \"41eac0a5fb578b089f7524c4d2e6cb38c5508aa81815d6cfc46948c081daa800\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-09\",\n        \"chipping_sparrow\",\n        292695018,\n        168861389,\n        \"tim_kirsten\",\n        64812,\n        \"cbb2a03f5dbd2209d56f1cca8b49f342dfbaf44e51fe1014ead75b2018c6678d\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-10\",\n        \"chipping_sparrow\",\n        92501489,\n        57964052,\n        \"tniernberger\",\n        78380,\n        \"44ddb9923026a97ee77ed362bc944c3d2cbbe4fbd4636000880e81613634e6c5\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-11\",\n        \"chipping_sparrow\",\n        220257388,\n        129611350,\n        \"gcart043\",\n        129377,\n        \"1bdce32e7ac58321dd6beacc084afaf45973469215d66b61a0f7c612da3db361\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-12\",\n        \"chipping_sparrow\",\n        478480946,\n        266314208,\n        \"russnamitz\",\n        61359,\n        \"8200cc2ec779d47be1b5afa261d934910a251a39f5620d3dc705106fb2bf5e0b\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-13\",\n        \"chipping_sparrow\",\n        538480223,\n        298914072,\n        \"hiltonward\",\n        201271,\n        \"2a6479556f14a20a8c9a69ff0c1026fe4deb376944234d1d4429c558042c4346\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-14\",\n        \"chipping_sparrow\",\n        58192408,\n        36778771,\n        \"bradenjudson\",\n        22352,\n        \"7c630d7a5b24d94e677e563a8ecb36f57fb29babe6ac1568f11537f10a5edf75\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-15\",\n        \"chipping_sparrow\",\n        80410971,\n        50636049,\n        \"radrat\",\n        55646,\n        \"0c71ca735502e8c94db81302ecd006428206eb16d75472eb0c706e62e2656667\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-16\",\n        \"chipping_sparrow\",\n        300376697,\n        172991803,\n        \"matthias55\",\n        81986,\n        \"7a9a5cbfb6a7d0581c98c75b5fdeb53a049d60f352eda43a39b1f9c2a3347f3b\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-17\",\n        \"chipping_sparrow\",\n        370917657,\n        209626382,\n        \"craigmartin\",\n        101736,\n        \"feb94d319fe5b12c01e63a80dc8e44c45e15a0bfb6534ac5d2c0641205dfe1d5\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-18\",\n        \"chipping_sparrow\",\n        264119989,\n        152960401,\n        \"laurelthrone\",\n        194107,\n        \"13e7336628f9ad784757dca82557c79eda22fdce561e0b52a10a503e72979b8e\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-19\",\n        \"chipping_sparrow\",\n        220323755,\n        129631264,\n        \"enspring\",\n        85687,\n        \"886b7e6faa39943f0c9754e4aef637e2a2f5ad57fe8e8c7ec08360ea95af45b1\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-20\",\n        \"chipping_sparrow\",\n        210319996,\n        124105091,\n        \"bunnymom20\",\n        188328,\n        \"2f6b6f7ac5ed9d91a361c8fed102484f4fca599ec39c9a2bfe9af1df53b28899\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-21\",\n        \"chipping_sparrow\",\n        117013164,\n        71832078,\n        \"terrimewbornagain\",\n        49527,\n        \"4456adb52cdafdefdaeb6dd8cec5955f93c1b8f40abdbb1182b2f605a0e1d486\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-22\",\n        \"chipping_sparrow\",\n        660951395,\n        362143224,\n        \"mike_cove\",\n        148079,\n        \"41d85eaa47b3a41a97ada54728a84c0bfbfed3bcff92de3b6ba88d4c9aface48\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-23\",\n        \"chipping_sparrow\",\n        691160217,\n        377927837,\n        \"ben142\",\n        235922,\n        \"f8137e51e90f0a1964c64fac1a775beaa9ae96c90a1d9aad4f1c870f143e34fd\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-24\",\n        \"chipping_sparrow\",\n        707633397,\n        386482138,\n        \"leannestacy\",\n        238800,\n        \"3fef93babe26eed119ab3b2363902007c46a58a02f91beb0d3bc650821f0dc28\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-25\",\n        \"chipping_sparrow\",\n        696064427,\n        380468188,\n        \"rdnwoods\",\n        309547,\n        \"b56f8f043112021367cbb02861021ac2ef6d2831359315f49d1445bf92477a07\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-26\",\n        \"chipping_sparrow\",\n        702557361,\n        383835541,\n        \"seanwashington1\",\n        159095,\n        \"2940c235b38f84101504b6f50e59c359d0f13e3dc384cadbd43976d1e5e9de91\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-27\",\n        \"chipping_sparrow\",\n        660337945,\n        361814902,\n        \"kristinpiston\",\n        158651,\n        \"a2272e4168edf260001ace410774fbe874ea6b1f6632b8502efc1bc02ae0f866\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-28\",\n        \"chipping_sparrow\",\n        546647394,\n        303298450,\n        \"kzoebel\",\n        86457,\n        \"c37a4aff4263de570ca82f94de92f82d9f9711922e5880547af836104f1883df\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-29\",\n        \"chipping_sparrow\",\n        713495256,\n        389515472,\n        \"lina47741\",\n        287986,\n        \"52e8a37dca8d985a57fdd0d90205e1d813388d961dbf4280d5cbccca9c6af5d3\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-30\",\n        \"chipping_sparrow\",\n        269951645,\n        156136769,\n        \"conhawn\",\n        106540,\n        \"11b48d8bf97084d9ff57aa69517b61539f03b72525022930192330fcb05a160c\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-31\",\n        \"chipping_sparrow\",\n        535714979,\n        297462154,\n        \"dissectedfrog\",\n        200498,\n        \"cd2f2e1be8c76240a1eeff9535e9a60b8f4fb9d771c7e487d65cde48b7e24cf6\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-32\",\n        \"chipping_sparrow\",\n        557301829,\n        308874987,\n        \"mar_y_sierra_silvestre\",\n        115174,\n        \"ce58e87584df623a86643705b08750228903705ca9b02d93b0206401a24eee9e\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-33\",\n        \"chipping_sparrow\",\n        652401709,\n        357680572,\n        \"dianeclark6280\",\n        46782,\n        \"b77c0eea6899d49f8ab97fd1ad6fc772b6055e5195628fa9f6f989e2d51cedd1\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-34\",\n        \"chipping_sparrow\",\n        657945260,\n        360569392,\n        \"jasonleduc\",\n        36363,\n        \"e3471104742fa68b9eeb149255b0136c53306df23d14f2b2633cea80c9e46eea\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-35\",\n        \"chipping_sparrow\",\n        434553038,\n        243681621,\n        \"stariplativky\",\n        57489,\n        \"e253bd9d70dea9d0438b795f6bebcbc2b26c48b16a90729f0d3155695fa812fd\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-36\",\n        \"chipping_sparrow\",\n        207498409,\n        122551474,\n        \"askalotl\",\n        98446,\n        \"d84e28017be8142c22758e59f14b8c52990e79584a68e6f929123aab4997ac68\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-37\",\n        \"chipping_sparrow\",\n        179570776,\n        106840941,\n        \"artemis224\",\n        138087,\n        \"74cad843089d2aada7a124f9e704ae1c1419b7cfcade7e2ae29e2c4eda0a6d36\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-38\",\n        \"chipping_sparrow\",\n        81287617,\n        51172999,\n        \"thyg\",\n        55659,\n        \"e1f075addb4bd1feea3ab256e37945a464d2e8c77768b7be131d0655382268d7\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-39\",\n        \"chipping_sparrow\",\n        343116928,\n        195090930,\n        \"umamimomma\",\n        52983,\n        \"a8948a0189d19b3d7b8df65271f4844b14bd5118f510d0b9ef458c942ac41ce2\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-40\",\n        \"chipping_sparrow\",\n        480169680,\n        267207764,\n        \"cvharris\",\n        144339,\n        \"d1ab8643c48f2ea823f9b61843016795def21b290c35ce3d6045e4933b5dfa0a\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-41\",\n        \"chipping_sparrow\",\n        352953833,\n        200088501,\n        \"aster-asti\",\n        105786,\n        \"82c35e04e4e22fe35c9b364bce7737fa5226cb5c469c761adfc61ef0956da9db\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-42\",\n        \"chipping_sparrow\",\n        622149696,\n        341721858,\n        \"wilderbombyx\",\n        293652,\n        \"227c39a25941a81e9551b5ccef39f95d320e7daa02c96de247123ba62f7b9824\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-43\",\n        \"chipping_sparrow\",\n        503555630,\n        280377982,\n        \"jtdavis05\",\n        42352,\n        \"24ccb474659505cd785502295f385ec369a36c6248296aff85261ed3475a4372\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-44\",\n        \"chipping_sparrow\",\n        509135437,\n        283351340,\n        \"martyndrabik\",\n        125948,\n        \"1604bf4f5512207c59637a84c78842285c40f76a4cf5335bf6d39afe896ed03c\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-45\",\n        \"chipping_sparrow\",\n        8076820,\n        6402608,\n        \"msieges\",\n        166588,\n        \"6c379f94141a0bcdeb8c9743ea140ded50f8c7864baecd6cef0ee1c724b9fd80\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-46\",\n        \"chipping_sparrow\",\n        131917390,\n        80486493,\n        \"tom_lazar\",\n        137355,\n        \"10ab0a9c5af2ca2203a18e4e3df799dcf129494d7b637a3c46ffdcf0f3b3856c\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-47\",\n        \"chipping_sparrow\",\n        141348323,\n        85865218,\n        \"j-dehoog\",\n        71473,\n        \"d3fae9ca7f515c43c216d749f085231d19ddfd101bd37cc4ffae25fc946a34fb\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-48\",\n        \"chipping_sparrow\",\n        28879424,\n        18850970,\n        \"andy71\",\n        194983,\n        \"ea5af97ddef4f743596c176653c86e457f6bddde65d86f5ab16ddff939574683\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-49\",\n        \"chipping_sparrow\",\n        148831027,\n        90084486,\n        \"ian-wolfe\",\n        181942,\n        \"cfab51f0c0598120f5312b354ae249bafb15124dd74723a8c29cdd7319206e49\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-50\",\n        \"chipping_sparrow\",\n        641370647,\n        351455126,\n        \"hrachski\",\n        130262,\n        \"9bc25019bf6a71d73daac63b810d18039d360887504014525598cffce2035908\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-51\",\n        \"chipping_sparrow\",\n        648268506,\n        355462611,\n        \"potatocthulhu\",\n        193138,\n        \"f526135f7b71e0f9f8d80c17d8e6c1a3cd89ba068333b9104e66d02ced7a7eaa\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-52\",\n        \"chipping_sparrow\",\n        645412478,\n        353758550,\n        \"ctlqh\",\n        154844,\n        \"e34228baa058ddb18c6aacf193c20574dab67d8c1f4d6940eec6e1acb03d993b\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-53\",\n        \"chipping_sparrow\",\n        121777317,\n        74504842,\n        \"hickl\",\n        48576,\n        \"a7f0980bf777cacb9f61855d157b94d895f6dd629485b2a038ce20519c53856c\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-54\",\n        \"chipping_sparrow\",\n        147992761,\n        89606238,\n        \"benkeen\",\n        461732,\n        \"8b6e8f03650c270624ebf82f85dd0b37db3e693157d4d3d7a1787c855846da9d\",\n        \"png\",\n    ),\n    (\n        \"chipping_sparrow-55\",\n        \"chipping_sparrow\",\n        27976676,\n        18310312,\n        \"schoenitz\",\n        66424,\n        \"64a0186d15abaf1537a34bd329b67625f7580b7aa544dded6bc626dd389b72fe\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-56\",\n        \"chipping_sparrow\",\n        190706448,\n        112884695,\n        \"jbeusmans\",\n        109296,\n        \"1777945eeb6f916ec6f9c0dc10c81014e8dc0ba22d87268e7c7dc1173193b1e6\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-57\",\n        \"chipping_sparrow\",\n        184174664,\n        109291439,\n        \"chrismcv\",\n        91566,\n        \"c5d9a1dbaf1fad839680c417c7be7d4a680396e38e7c2aba0ee78fd213f7d32c\",\n        \"jpeg\",\n    ),\n    (\n        \"chipping_sparrow-58\",\n        \"chipping_sparrow\",\n        72765715,\n        45879059,\n        \"henryfrye\",\n        146855,\n        \"f0b957698f5a600e3a6c45471848013f9c74d1602895e40d513e2116a9b7320a\",\n        \"jpg\",\n    ),\n    (\n        \"chipping_sparrow-59\",\n        \"chipping_sparrow\",\n        74100878,\n        46721618,\n        \"aredmer\",\n        132045,\n        \"ce6f4d4324d684cfc0f40d6fe9c4a2b03a7024080e7448083ae7e3fc0c06ac6e\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-00\",\n        \"white_throated_sparrow\",\n        339621218,\n        193338380,\n        \"rawcomposition\",\n        31152,\n        \"d1c08bfaca721bf0873437455b4cc010c6860d08b4777136c775007b0b9d07b6\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-01\",\n        \"white_throated_sparrow\",\n        166821399,\n        99992799,\n        \"dziakj1\",\n        125954,\n        \"c595a41fbc8948b0d918b59117340dc320e2ba80d29e92bb9dacaaed5b404852\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-02\",\n        \"white_throated_sparrow\",\n        469820434,\n        261505977,\n        \"joy4birds\",\n        111767,\n        \"7ce091492c73c68395667bb45578dfff11457511b0958d1d0d320d1df3e55ceb\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-03\",\n        \"white_throated_sparrow\",\n        260413022,\n        150969515,\n        \"andywilson\",\n        45958,\n        \"fb7c533c92239775da6a5b353ff398f6bdd8f65c13445fc1dd72986af5a46466\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-04\",\n        \"white_throated_sparrow\",\n        99351488,\n        62040646,\n        \"bradenjudson\",\n        23399,\n        \"e103968e2a6c9efb6f0bcb548a6457aef950a4a720838a624b6796b90411538e\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-05\",\n        \"white_throated_sparrow\",\n        628148203,\n        344731686,\n        \"lavenderdame\",\n        106872,\n        \"36379abf3af51d865ba6e6804ba3dd48fc9efd12ecc0b7d97e03fc04a16178a8\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-06\",\n        \"white_throated_sparrow\",\n        104660609,\n        65043951,\n        \"allan7\",\n        42443,\n        \"10791891945e83a0c908c14fea07a257a0f25f51c0e708bee35184213833a635\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-07\",\n        \"white_throated_sparrow\",\n        250718938,\n        145903421,\n        \"stevestevens\",\n        138735,\n        \"bd52e6d2f48c247f72db0fa393ec4fa13af8510c95f0ca9134cbef71caddb6d4\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-08\",\n        \"white_throated_sparrow\",\n        15105971,\n        10793852,\n        \"schylerbrown\",\n        31467,\n        \"6496e7e6d3abf13b1a538f769e3cc402280cf1e9a2a1ebdf81c68dcfd7ed01e2\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-09\",\n        \"white_throated_sparrow\",\n        171460784,\n        102554447,\n        \"w_mark_c\",\n        251287,\n        \"3828c41af9209b408fe0d8ec6541edf35d13fc730b74fd27a82980d4f3771137\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-10\",\n        \"white_throated_sparrow\",\n        244260317,\n        142471526,\n        \"deejay\",\n        75694,\n        \"d67efb1ec61f6700b8a6c6552e2da9cd981e68ae81af16d6f1e0ef17fcaff84f\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-11\",\n        \"white_throated_sparrow\",\n        194732675,\n        115373159,\n        \"wildreturn\",\n        128989,\n        \"7ed4bde480b35734576bb4c5f9d1453e77c1f095380432b43b1b682050255831\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-12\",\n        \"white_throated_sparrow\",\n        341990462,\n        194541980,\n        \"ethologist\",\n        149260,\n        \"3e2710fac082cc347e6cd114d42d39f25ec47b82c25936922232eba916808cdb\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-13\",\n        \"white_throated_sparrow\",\n        267625208,\n        154862375,\n        \"laurelthrone\",\n        148616,\n        \"0cac1bc7053c891ce5ae1c33e3f94b69f1b19b958bd08c686db2ccaa7ba0fbd1\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-14\",\n        \"white_throated_sparrow\",\n        330539586,\n        188793537,\n        \"efalquet\",\n        119214,\n        \"c3db4583124472b28dbfb4c6fd9b3829e451d508a6b0a095eccc28f17e7982b4\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-15\",\n        \"white_throated_sparrow\",\n        193091403,\n        114346779,\n        \"ian-wolfe\",\n        73751,\n        \"e28974c8782fccb00f5ea5420140e01248ead859f151563a95a26bf67acba479\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-16\",\n        \"white_throated_sparrow\",\n        691050773,\n        377873006,\n        \"dinomariobob\",\n        167744,\n        \"cc5e25afa5041ce2d6ea4e0d726793843f3a867f30b8d9ccf55892d6617da887\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-17\",\n        \"white_throated_sparrow\",\n        440986574,\n        246671481,\n        \"suzannehale\",\n        141857,\n        \"4a354c189225de2e7b4e94a6df9cbd3a671dac0c8a7d2d4c75e933f93d8b83bf\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-18\",\n        \"white_throated_sparrow\",\n        113217820,\n        69733707,\n        \"kemper\",\n        97802,\n        \"4de7da3ac53a086f2c343555be5a2b62a683715db9f80636a76673cc380fa7f6\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-19\",\n        \"white_throated_sparrow\",\n        575307217,\n        318327478,\n        \"k-simpkins\",\n        90416,\n        \"af2b4e72a098bd90b920c8a49632e1bbff18b73954d8ac541ca81ed4baf0530b\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-20\",\n        \"white_throated_sparrow\",\n        340420076,\n        193765555,\n        \"don54\",\n        62714,\n        \"7f6eea434fc700166af1d343951d15f3a9c83eff06cfb0518c3c4291019a5556\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-21\",\n        \"white_throated_sparrow\",\n        562344160,\n        311510652,\n        \"rrfc\",\n        45842,\n        \"f7c6394649765db6e3313a03ae013291db1ae20d89d3b8f0bdf6641757f31ed0\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-22\",\n        \"white_throated_sparrow\",\n        74598266,\n        47039878,\n        \"ianrwhyte\",\n        141502,\n        \"b0633999572a4499a80310955ab928a86f4fe08774e669b4c4949cd26254428b\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-23\",\n        \"white_throated_sparrow\",\n        588001234,\n        324825124,\n        \"portablecity\",\n        370187,\n        \"11e3a97f43de5d61266d15028fe9023eef3900cfea2d027bd94ad847ecba9607\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-24\",\n        \"white_throated_sparrow\",\n        694103481,\n        379467387,\n        \"memoosborne\",\n        120656,\n        \"82684f82f62f6e036e206eec315fff8e1ba36d308b667eba06750b75d1fada3a\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-25\",\n        \"white_throated_sparrow\",\n        655749108,\n        359408087,\n        \"russnamitz\",\n        58449,\n        \"2a7d2b4226f1941d0950adc04bfd8fe62345f1138c521d0f5f0adf8bc5838da1\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-26\",\n        \"white_throated_sparrow\",\n        599110899,\n        330395670,\n        \"jd_flores\",\n        121343,\n        \"5eeef4f5533a4b0a0222dcf7be000a2b34f3331f3be2b939703155ba8c8ba3ae\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-27\",\n        \"white_throated_sparrow\",\n        653888173,\n        358443775,\n        \"toknowtheland\",\n        144197,\n        \"20a4933b00967fe4488296b2b6e89c12ecbcca0e40da591bd95927cca46ae2f7\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-28\",\n        \"white_throated_sparrow\",\n        295037357,\n        170132018,\n        \"sturuss\",\n        117077,\n        \"d309f75ad5f4008c906ba7b3d6138aa017f919a0a17a1d5b431fa7ab47d5f2d3\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-29\",\n        \"white_throated_sparrow\",\n        405042885,\n        228270930,\n        \"carterdorscht\",\n        122157,\n        \"f1fb32bcfb78f66f50ccd20f2418832afe72c7a5847bdeb8e4dc19546455cfaf\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-30\",\n        \"white_throated_sparrow\",\n        84971412,\n        53425343,\n        \"davidfbird\",\n        168442,\n        \"95c5363704d17d6076f3c7a4c9f5e41d235edf7d5dc9d6022116774f2fd544e6\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-31\",\n        \"white_throated_sparrow\",\n        476335868,\n        265210970,\n        \"aster-asti\",\n        91852,\n        \"cd300cbc823e8ebc0902b7bd80c99eb323e5900bbabf63303b0d9399624c3e54\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-32\",\n        \"white_throated_sparrow\",\n        481170792,\n        267732071,\n        \"kcthetc1\",\n        46021,\n        \"2194a78339aae99549df56d2d3d7fc91e0bf98a371e7d0da7958b4312fa3b487\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-33\",\n        \"white_throated_sparrow\",\n        623602892,\n        342440773,\n        \"imperialwoodpecker26\",\n        112678,\n        \"66653f47ce6a54d38cc1f54b0a85e965f5c5ee7cce65b02c78c1c185cb586c85\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-34\",\n        \"white_throated_sparrow\",\n        622732944,\n        342010560,\n        \"tom_lazar\",\n        34257,\n        \"584070c4b5231db49087d1507aabec8f51adc4e09dcdda8b93b3a78159d47146\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-35\",\n        \"white_throated_sparrow\",\n        505347090,\n        281327144,\n        \"erikschiff\",\n        85652,\n        \"91d50bda6afc3cd5596bf4d3176999bf473c31fb6680b64445ec1cbeaabea847\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-36\",\n        \"white_throated_sparrow\",\n        7072170,\n        5706496,\n        \"jmutter\",\n        171582,\n        \"de90ea2739e8c65ca89f0e511abaf44219501e18377eb49d9e5c7f10d3a6d5b2\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-37\",\n        \"white_throated_sparrow\",\n        16303076,\n        11431253,\n        \"robw\",\n        61348,\n        \"e7d81cfe027c7b0914c79a0431f0ce1431db1ebe78368a2fe5f31e2aab59c3c7\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-38\",\n        \"white_throated_sparrow\",\n        7149657,\n        5761350,\n        \"bradleysaul\",\n        108304,\n        \"adf755b8393fb6b4db99e788d4112c2c72197c33ecffeec2ca7b3ba5dd9d650e\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-39\",\n        \"white_throated_sparrow\",\n        27513220,\n        17998912,\n        \"mefisher\",\n        49587,\n        \"f1716d49782f70455ec97ecdda634505b6b2afc375105b1d779353e704dda07b\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-40\",\n        \"white_throated_sparrow\",\n        642121459,\n        351843522,\n        \"jeffcherry\",\n        53348,\n        \"c9e82d7e47ca2e51c856f9bd4d8f2965d8f69948f081c08c9fa6ebf44fcde711\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-41\",\n        \"white_throated_sparrow\",\n        642355950,\n        351962427,\n        \"robinlanark\",\n        54504,\n        \"c0da16d5146afac102fa274fffea2e0b203282078ed692f3b7ad4a1340b0d4cd\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-42\",\n        \"white_throated_sparrow\",\n        638199267,\n        349826931,\n        \"wildaboutwildlife\",\n        32981,\n        \"845f7d38d6f35d2e6e9dca61234c9566778fec1e274500c97e47977fcecd232f\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-43\",\n        \"white_throated_sparrow\",\n        119839965,\n        73407944,\n        \"terrimewbornagain\",\n        61951,\n        \"dbfbaaa2f72caa812c026122f41512caef00f8f8fdd21c2307fcdf9d79921757\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-44\",\n        \"white_throated_sparrow\",\n        6236370,\n        5078910,\n        \"ruggedbynature\",\n        126403,\n        \"3d6030593bf57bc5734fd6c0706c1e350bb117a32370a16b194ef4488668f123\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-45\",\n        \"white_throated_sparrow\",\n        32221828,\n        20882923,\n        \"christinan\",\n        132485,\n        \"b92ce41128bb1c5b53a0c487b46ba80eea83b9a443ee9c72f100005e5d80e411\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-46\",\n        \"white_throated_sparrow\",\n        36598789,\n        23643589,\n        \"jessm-c\",\n        45861,\n        \"eda75171590418596e372c6cba058364acb64f1279d250dff6c5df61fa17868b\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-47\",\n        \"white_throated_sparrow\",\n        163198257,\n        98065830,\n        \"ellyne\",\n        109261,\n        \"864fb42a82561684c1ac22ae71bd023ddeb15c6052b7732ebd2570a98b552a1a\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-48\",\n        \"white_throated_sparrow\",\n        162150240,\n        97493574,\n        \"c_burns802\",\n        188464,\n        \"0fb2fb2e04bca82c436315b77a974adc665d2d1fb4e419dd61afa068ab412254\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-49\",\n        \"white_throated_sparrow\",\n        161756778,\n        97282492,\n        \"raffib128\",\n        67237,\n        \"77384a65eb120bdb345c3015cb8c22fcec9c89df49b0e0c00954e3476834da9f\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-50\",\n        \"white_throated_sparrow\",\n        165146429,\n        99107203,\n        \"philippthompson\",\n        84518,\n        \"8582544ca8b3cb1c73174fc42a9d66971fdae0c90cd4739083eb4ef4983eb6e7\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-51\",\n        \"white_throated_sparrow\",\n        177451999,\n        105722646,\n        \"natepow\",\n        44983,\n        \"c81a68c46264d1a5fcae3042e7ab24653c3615665cef6293c31622d5f342d840\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-52\",\n        \"white_throated_sparrow\",\n        67630852,\n        42591255,\n        \"schoenitz\",\n        138616,\n        \"34594c49a4f601b17e49dfc4d4098db7b73ff5a69d3830b773378e4d695ff181\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-53\",\n        \"white_throated_sparrow\",\n        99106006,\n        61898962,\n        \"glennberry\",\n        122576,\n        \"fd8cd1c0b0c1fdd7636b8588c86c6d4f3cddc233e9bd4bccade5e2ed275d0e28\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-54\",\n        \"white_throated_sparrow\",\n        102162606,\n        63634844,\n        \"eug302\",\n        73329,\n        \"692114df14fb8a75fdf38428a196467a4a394e5ded95605fbb48cf64903ccb1b\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-55\",\n        \"white_throated_sparrow\",\n        370403033,\n        209307218,\n        \"kuykenwil\",\n        124312,\n        \"d0700068b06ad0f79eeac331a92e5d633bd76be264c3d8801a9a6eb301193969\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-56\",\n        \"white_throated_sparrow\",\n        500790892,\n        278893388,\n        \"quillipede\",\n        186273,\n        \"0db918123c7f529ca76c04129886eb9aeb947bd3e90cdfd968c4605931c16f85\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-57\",\n        \"white_throated_sparrow\",\n        364738073,\n        205369541,\n        \"sandra1142\",\n        144424,\n        \"4cd383751871426904f4d8ff7526a312ba33a6c894faf7121c6ec0ff6ce0d7b1\",\n        \"jpeg\",\n    ),\n    (\n        \"white_throated_sparrow-58\",\n        \"white_throated_sparrow\",\n        374502828,\n        211884837,\n        \"halliefromcali\",\n        62476,\n        \"0073652fddd7378dc936f280c18d7e44c4a7160aadc2f051e7b5e2896b1a476c\",\n        \"jpg\",\n    ),\n    (\n        \"white_throated_sparrow-59\",\n        \"white_throated_sparrow\",\n        109241547,\n        67591707,\n        \"blkillin\",\n        130154,\n        \"e5b8c1caa0f7cc7926d46eac42df582ccb0882b4a0733fc0740b8b4597a4e014\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-00\",\n        \"dark_eyed_junco\",\n        172110799,\n        102901486,\n        \"schylerbrown\",\n        182973,\n        \"185209c7a1111fc626a068e136ec3cfcdff3174d15f1c60af209d7b32fe7bef9\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-01\",\n        \"dark_eyed_junco\",\n        46691943,\n        29901256,\n        \"haida_gwaii\",\n        46823,\n        \"a92dca21e6e58c375fc313f0d1da2c86c11408beb0df8fd96fc60ae035ae80d5\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-02\",\n        \"dark_eyed_junco\",\n        707222551,\n        386266764,\n        \"ben142\",\n        289608,\n        \"7bf320edf4d34a4848f3e9d175cfafa66cc5bab6a1a8f97c41c670263de3ff89\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-03\",\n        \"dark_eyed_junco\",\n        346777340,\n        196961623,\n        \"zacharyfoster\",\n        46712,\n        \"ea8f9f0eebb4d2f86193c705344a8fab09bf634bc2217a0874ee57c4f0f5b4ab\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-04\",\n        \"dark_eyed_junco\",\n        8793471,\n        6892999,\n        \"truthseqr\",\n        45302,\n        \"f8241eab39797c4e097a1432b13658466287d61ed515448457907c17e60cf5e2\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-05\",\n        \"dark_eyed_junco\",\n        192557376,\n        114006980,\n        \"k-simpkins\",\n        172398,\n        \"206f012add8a5d0fa434e07c51f1e7bd73a60c4d299a2202163fc662e1b03479\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-06\",\n        \"dark_eyed_junco\",\n        243303909,\n        141959574,\n        \"andywilson\",\n        71321,\n        \"ee5308b6f93fb40a6d795a7d8ca6f2ef55b4844513f4268ef34827e1c5e4c4a6\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-07\",\n        \"dark_eyed_junco\",\n        274980085,\n        159160633,\n        \"andy71\",\n        51209,\n        \"c54b45ca7fdc635bdb31eb89166c9fce84f0b0f8f0c17331fd5e42b582633c9b\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-08\",\n        \"dark_eyed_junco\",\n        213798376,\n        126031618,\n        \"nathanael15\",\n        57646,\n        \"d6b9e7d2dcc63cdd88b47cce328149aa9e8eb486ee2a5fc0f08b90601f2c7d9b\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-09\",\n        \"dark_eyed_junco\",\n        63482066,\n        39977347,\n        \"chrisleearm\",\n        41870,\n        \"90573e814dbde965c70a934d9702110c40670aa22cc9f80ff8849db877d1fd72\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-10\",\n        \"dark_eyed_junco\",\n        458710472,\n        255914048,\n        \"joy4birds\",\n        90973,\n        \"fd25ecb1bf86896b84e9e4e8329c742011011e6883631f47dbe3744ba7463153\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-11\",\n        \"dark_eyed_junco\",\n        20784016,\n        14046286,\n        \"gambolingquail\",\n        93957,\n        \"a454d6f987153317c9c57c05019b08ab0ebebc46d3cf72356b014231b69a1e9d\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-12\",\n        \"dark_eyed_junco\",\n        332842159,\n        189933284,\n        \"jan-konilu\",\n        110145,\n        \"d18706f536164a72a4ec9bacf47437edd89d6b547f86eefdf10ed0166cebf0dd\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-13\",\n        \"dark_eyed_junco\",\n        513004508,\n        270404136,\n        \"thevertebratepokedex\",\n        141252,\n        \"854f75e7527932d0409e14756725388de9913c0b450536da717e54d683de169f\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-14\",\n        \"dark_eyed_junco\",\n        593499256,\n        327583635,\n        \"orionid\",\n        108583,\n        \"7eab4cea6d46faa878732d82c5c2ece63ecc3e66781ea08d0bd950eee581d16d\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-15\",\n        \"dark_eyed_junco\",\n        256948665,\n        149140989,\n        \"igor322\",\n        86925,\n        \"9339b8cf4aa347b4eb176ef1209dfaa4fcb11e277889cc17de399d6af2c8633a\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-16\",\n        \"dark_eyed_junco\",\n        12533281,\n        9255403,\n        \"braincellsgone\",\n        40857,\n        \"be34c51655f59612858d888b05a4f927da65e1aede850c9a5f1be68fa00bfcbe\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-17\",\n        \"dark_eyed_junco\",\n        12580989,\n        9282523,\n        \"artemis224\",\n        216253,\n        \"15198c123c01fa0f8c03edc086408c8d95ffa553083c380e8ae236dbb7056093\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-18\",\n        \"dark_eyed_junco\",\n        63590391,\n        40041059,\n        \"bobbyblackmore\",\n        82853,\n        \"87746be0bca30fc40ddba739c1a35ed1fdd41882ebfe3561518b69d285f5ae5d\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-19\",\n        \"dark_eyed_junco\",\n        106718005,\n        66230973,\n        \"vicki936\",\n        41475,\n        \"e6903e9a69e987c45edd468ace0bf71adf8252063ff5d39c2130409d8fea68be\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-20\",\n        \"dark_eyed_junco\",\n        611049594,\n        336277421,\n        \"skylar_schell\",\n        15425,\n        \"7e876479febb3d44b1e494a25f778efe4e8bfd323279031cb08f9f6fe95f742e\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-21\",\n        \"dark_eyed_junco\",\n        591147962,\n        326403963,\n        \"toknowtheland\",\n        106373,\n        \"07770f3313dc969802355fa8b3e62a86edb115a64771bd38938e1fab33220c39\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-22\",\n        \"dark_eyed_junco\",\n        459696953,\n        256398190,\n        \"w_mark_c\",\n        262126,\n        \"5025e757c026e02160ff3f0e82f1f4434b888a6c9689e7ac76274c1fcec64f0c\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-23\",\n        \"dark_eyed_junco\",\n        484171775,\n        269292238,\n        \"aschuman\",\n        59459,\n        \"89dd933d12cf21c0e73eb01e96a9279e2e57b36676a72a9c860281d0d7703282\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-24\",\n        \"dark_eyed_junco\",\n        469746672,\n        261469406,\n        \"shannon_j\",\n        74167,\n        \"ec4c7ce15aa1c4eadd3bc5d46d433baf9560d6a0e4593d8d57aab8fa9bec9e18\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-25\",\n        \"dark_eyed_junco\",\n        357382685,\n        202362003,\n        \"dougbrown\",\n        56324,\n        \"70275d8e07192c99e121b67d206de6823b194f043e65b9766f1f9ce772953c78\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-26\",\n        \"dark_eyed_junco\",\n        7660371,\n        6119391,\n        \"jeffreyleeisanaturalist\",\n        108394,\n        \"0f667e9af8e8d3585807da58a2f315c5dbcb7eece5f6862006fb9da68a4bef42\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-27\",\n        \"dark_eyed_junco\",\n        437957476,\n        245430473,\n        \"eug302\",\n        106520,\n        \"394abfbcf192ecf5a76cb2f19dfa2a62070fa3874b084d1468219d6ee5db6cbf\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-28\",\n        \"dark_eyed_junco\",\n        339439778,\n        193239701,\n        \"rawcomposition\",\n        32258,\n        \"73fad57ed975df01c529e77eaca9eab9b00741ea2cfc91358856aefca620a10a\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-29\",\n        \"dark_eyed_junco\",\n        634100291,\n        347744522,\n        \"zorthesosen\",\n        169264,\n        \"932237001fe68e3d6d19c496fd448b8a82f254316211ac6f9d35708a1395ed81\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-30\",\n        \"dark_eyed_junco\",\n        6198359,\n        5055484,\n        \"glmory\",\n        108168,\n        \"d31767f42eaaf7f3527133fffb9a0271a9dbbe66fc036d5c694b605563d09965\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-31\",\n        \"dark_eyed_junco\",\n        6485959,\n        5243663,\n        \"fake_id\",\n        148254,\n        \"ce4dab6c5b37b7ebac07f69d4040d4686fea9fd4b753f91bdf15a3a440820712\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-32\",\n        \"dark_eyed_junco\",\n        246318772,\n        143610209,\n        \"wildreturn\",\n        62090,\n        \"839e57443659659522ed65dfd054a8fc0a6bfe400518d9f9a7519383d33a2c0d\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-33\",\n        \"dark_eyed_junco\",\n        253199424,\n        147174876,\n        \"sean579\",\n        121660,\n        \"6efc17eb4c0b184d64a605a76d3734d9bff5d26e8c8609d2f0b0dd4ccc8b3faa\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-34\",\n        \"dark_eyed_junco\",\n        244315013,\n        142500114,\n        \"enspring\",\n        63514,\n        \"1d5012c2b4504b4347fef36d8764eebd1bf07649a14cf874e6d0b3eba9fb3c4f\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-35\",\n        \"dark_eyed_junco\",\n        256191686,\n        148740024,\n        \"nartb\",\n        194532,\n        \"e88b1449a989fc62c5da1c592e4987fa4db71eadf1ef29c0c5c1159b8f77a298\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-36\",\n        \"dark_eyed_junco\",\n        241013731,\n        140770209,\n        \"nana10\",\n        196479,\n        \"fc773ca1b06e0e3673aac5c2213c66981ede11bf3ad4a0cdef2102c4f1d99843\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-37\",\n        \"dark_eyed_junco\",\n        240712071,\n        140608440,\n        \"saintaardvark\",\n        129995,\n        \"94c4dc30ca31804d6645a6be802cf87ccaffe974f0d3cb554b4fb3e9696ec6a5\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-38\",\n        \"dark_eyed_junco\",\n        242909841,\n        141750179,\n        \"calinsdad\",\n        61773,\n        \"c49c5d22969552de0fc04ff95eefb8a2ffc76df528477b93c2e2c38cdebe8f6c\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-39\",\n        \"dark_eyed_junco\",\n        109385584,\n        67667584,\n        \"ninetoes\",\n        64185,\n        \"64e9ee29cf017c292e1711b4c6482488a74c8f652e88e8b7c1df883dc91d75e2\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-40\",\n        \"dark_eyed_junco\",\n        102027986,\n        63561089,\n        \"michaelnaumoff\",\n        114346,\n        \"eccaa3c60fe575e28fc60f8f6896a99065bff4aee0a4fdc057d7220e7870ca4d\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-41\",\n        \"dark_eyed_junco\",\n        105652410,\n        65622040,\n        \"don54\",\n        159856,\n        \"5c4282e5e85af7a886eceeadc90977bcd11bf26658176ba5ef8182d2f4a2bfe1\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-42\",\n        \"dark_eyed_junco\",\n        267851501,\n        154986181,\n        \"shanebustapbj\",\n        203657,\n        \"217002a366779db0ccc3fddffa71e79472b1fcaa5d444049971dac4c31c910f5\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-43\",\n        \"dark_eyed_junco\",\n        268597047,\n        155389350,\n        \"j-dehoog\",\n        54916,\n        \"c0890844944f3d603f8220b0a75213051766c6a45b2ba1fed51381134832f69f\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-44\",\n        \"dark_eyed_junco\",\n        515881506,\n        286951393,\n        \"jubileej\",\n        108221,\n        \"9cbecae915f1c96dcc31d5fd77cb5902181152cc3b918693191cc11fce18822c\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-45\",\n        \"dark_eyed_junco\",\n        640462423,\n        351001488,\n        \"russnamitz\",\n        44508,\n        \"63276eb90201f890b977a54b95bf1d65592eef98089dbb73f2e27e66aaf49a65\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-46\",\n        \"dark_eyed_junco\",\n        649025513,\n        355899297,\n        \"rocksand2134\",\n        233215,\n        \"2f2545fa5e7901c0383578dea84131eb4be551b8f2bd55d9f308ebc838144f39\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-47\",\n        \"dark_eyed_junco\",\n        625101518,\n        343192310,\n        \"jtdavis05\",\n        106387,\n        \"4642076ebc5e9bcde85dbba65eb16a49d3fabe6c82cc707b4863ad4b7a4531ae\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-48\",\n        \"dark_eyed_junco\",\n        509902416,\n        283764048,\n        \"loarie\",\n        271588,\n        \"0c1e3897caca8ff3ff4f3eff7548e30ca50dfe2c2c972106303f9e21c259dbda\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-49\",\n        \"dark_eyed_junco\",\n        627787858,\n        344552039,\n        \"margohj\",\n        54717,\n        \"a41e557077525b064fa16a7c0b140d608d323663519cfbf424f4454d79951244\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-50\",\n        \"dark_eyed_junco\",\n        630444682,\n        345899876,\n        \"robinlanark\",\n        102578,\n        \"aa4ffd5dbd553d60ba687d57b21b52ce21572d3c711471abbc5e959bd1126da2\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-51\",\n        \"dark_eyed_junco\",\n        635784743,\n        348597723,\n        \"lina47741\",\n        317439,\n        \"392d1114b7fc1bb3b041d9895ceb049432a96951764e8949d5939315b96bcc54\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-52\",\n        \"dark_eyed_junco\",\n        518407567,\n        288278997,\n        \"zzphantom\",\n        231733,\n        \"de2a5be329b583911babe0758c9ce8c0982191f743b8082eb826623256709f71\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-53\",\n        \"dark_eyed_junco\",\n        620711072,\n        341026776,\n        \"ethologist\",\n        186675,\n        \"13f47961b2a1654ec40f6d1a2005afbf34c75060987fbb662001e66e10a023ed\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-54\",\n        \"dark_eyed_junco\",\n        638143588,\n        349800424,\n        \"tom_lazar\",\n        157629,\n        \"0be0b84bbc956c616ce35e9e704e8c9e01f0d24db7eeb157fcaa3b3952bd9b23\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-55\",\n        \"dark_eyed_junco\",\n        668612910,\n        366183670,\n        \"gendereuphorbia\",\n        76372,\n        \"4bfa9f780e9fd0327afec60a8b42e3378b8a9917a0461fd77a6df73b1a94e6bb\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-56\",\n        \"dark_eyed_junco\",\n        116078128,\n        71308227,\n        \"msieges\",\n        47042,\n        \"ecd4ade5abf058ac887fabe2706eb021ada3b9ab083d10728c9b1dc26de56bbc\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-57\",\n        \"dark_eyed_junco\",\n        117580446,\n        72145931,\n        \"hewwowhy\",\n        67000,\n        \"1a8ca3be627c192459440c2eb07bd85b348f0e876a7de32bfc6fe0fb2959a618\",\n        \"jpg\",\n    ),\n    (\n        \"dark_eyed_junco-58\",\n        \"dark_eyed_junco\",\n        117044036,\n        71848977,\n        \"radrat\",\n        81035,\n        \"7362dd0a27d11b18d5a79abcce9a80304dc80dc0ea1fa84265a1cf0638687641\",\n        \"jpeg\",\n    ),\n    (\n        \"dark_eyed_junco-59\",\n        \"dark_eyed_junco\",\n        11110817,\n        8359397,\n        \"giselle9\",\n        71741,\n        \"b2e74f27ddca8d1e6dd4807251b07867cbfec8b6bf51d93ef7f142109f705520\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-00\",\n        \"house_finch\",\n        697940852,\n        381438133,\n        \"ben142\",\n        196735,\n        \"a89f8e0263fdabb404b462acaa592f5dd2ac88ee4615da444470de4a1fae82d5\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-01\",\n        \"house_finch\",\n        176982307,\n        105476125,\n        \"vicki936\",\n        22211,\n        \"c377fb361df0324c7a856d9344968886ece3b94bd67188c9325b8d2d284d3a2f\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-02\",\n        \"house_finch\",\n        117990649,\n        72375345,\n        \"kristen163\",\n        75945,\n        \"eeafad0dd2e91ecfe45c9d1f27dd0392a01bd81099549c60fd2e36a4b4342a9f\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-03\",\n        \"house_finch\",\n        389479656,\n        220010434,\n        \"aster-asti\",\n        82128,\n        \"a8848197b4e7890e07538d492480c4275b75d04e10c1ae95aee91aaafe3319c5\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-04\",\n        \"house_finch\",\n        98576538,\n        61594129,\n        \"enspring\",\n        46784,\n        \"8b355426d8fe6327f202c6a9458cee1b95de445bfbf7e48a4b5a2c7d0eb78a8c\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-05\",\n        \"house_finch\",\n        72470599,\n        45698380,\n        \"henrya\",\n        61724,\n        \"1b96d37a7078e1b725b80af4b10848da58b0d0c17a70c8ac01e326c0a749ee6b\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-06\",\n        \"house_finch\",\n        80751781,\n        50842166,\n        \"leahmfulton\",\n        44269,\n        \"6d6202de26f042d83ee6c5af550cd74e6ab10eb796eed2f78c83ac9e2368e4b7\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-07\",\n        \"house_finch\",\n        214612538,\n        126483167,\n        \"hamiltonturner\",\n        124116,\n        \"6b7687640c4641b974865da04cf9eaf1f86b774ebc19678f2fc39e55c8648930\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-08\",\n        \"house_finch\",\n        630196420,\n        345777550,\n        \"truthseqr\",\n        149455,\n        \"abb84d1e327dd82c07cbea3dd5583c07453e69b2cc220397b101e597da81bd6c\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-09\",\n        \"house_finch\",\n        213077180,\n        125637342,\n        \"jnicat\",\n        25823,\n        \"e1e3baff8d0bd72339e3e49089f7f4f2c1dd383ff005e49a964a1bacc4f8ebb8\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-10\",\n        \"house_finch\",\n        500744872,\n        278868417,\n        \"pbaff\",\n        149626,\n        \"2684cfb1fc40d7766610a5920ead0ad0c27c338ccb4fb9ed18baddda150568b0\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-11\",\n        \"house_finch\",\n        268678834,\n        155431721,\n        \"stevestevens\",\n        120325,\n        \"5d1e8c097c214d14ef7b895bf95769ae9bc25fa799a98bbfa6f12f2f84e6af79\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-12\",\n        \"house_finch\",\n        358373136,\n        202884575,\n        \"kcthetc1\",\n        52329,\n        \"b9929163e4fdadef26c753437ac7af3ad550aa047b15cba131c05d4d335799fe\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-13\",\n        \"house_finch\",\n        104227663,\n        64793290,\n        \"verdantpulsar\",\n        101003,\n        \"90fe37c477bad9ed30ab119e1a7445ffc2720e548a22bb65d66b2ff7b82337d5\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-14\",\n        \"house_finch\",\n        196156834,\n        116218899,\n        \"kgarrett\",\n        56801,\n        \"ce03d1d70a89b5e6e0088f4307f8077573c3571b91997725ca2e6c69be80e2d0\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-15\",\n        \"house_finch\",\n        454332148,\n        253709711,\n        \"rlaortiz\",\n        149985,\n        \"cf007ef8ac57bc0eb085c9eecf9fc95eb69a29df706998de237df24f9491c61d\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-16\",\n        \"house_finch\",\n        665287910,\n        295246528,\n        \"dinomariobob\",\n        74972,\n        \"5f6121c1f8dbfaccbb61c279a579c22600744eb4118c2eeef09e69c86f6e1a49\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-17\",\n        \"house_finch\",\n        509969159,\n        283798927,\n        \"damienxw\",\n        104277,\n        \"56d3656be473c362f1ccd09d15e62cbcfe9137d83bef7a3312d72444921c7805\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-18\",\n        \"house_finch\",\n        168402062,\n        100871757,\n        \"k-simpkins\",\n        86922,\n        \"209a884cf7618d0b85679ae3a72f237a6d03a5a3096f6ab1b2e5503c38c50d9d\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-19\",\n        \"house_finch\",\n        661042217,\n        362189851,\n        \"dougbrown\",\n        56426,\n        \"ad704994fda99779aa340ca1c637b1371f0513df6f36973263bb9ee88cf6713b\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-20\",\n        \"house_finch\",\n        701862058,\n        383472210,\n        \"dblanco\",\n        25222,\n        \"8562ed9c4e889b34f83b40c55dd93b821a6b5829a3b1c681374a53c3ca9bf01f\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-21\",\n        \"house_finch\",\n        709783562,\n        387605980,\n        \"fraskar\",\n        99555,\n        \"60a1926358c4ec04aeec9414381177600580c8237899c852e03472e53cf87f31\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-22\",\n        \"house_finch\",\n        703924789,\n        384540425,\n        \"meshhy\",\n        110802,\n        \"2df38d1aaab2c268c7a47f05e55bef1e83690eeeb91f34874c5b479a5ff0a2cb\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-23\",\n        \"house_finch\",\n        705516095,\n        385380385,\n        \"m_aniket\",\n        77797,\n        \"ba009764b66ed89efc9b0aaacd9565830c65039967961488d774ce3c9add5d64\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-24\",\n        \"house_finch\",\n        577527110,\n        319485540,\n        \"glmory\",\n        94407,\n        \"6d4cd80f8ef77f41575befae0d8534e9bc397685dceb7f629b5cbc24447a3613\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-25\",\n        \"house_finch\",\n        570686433,\n        315887836,\n        \"emilyheaton\",\n        229263,\n        \"27e0bd21c1b322aa5d74c5504dbf27bfb5a0fdeea9718b6ffcee7926f2cb2733\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-26\",\n        \"house_finch\",\n        266259557,\n        154114419,\n        \"andywilson\",\n        47792,\n        \"b0dc2de7134cc5af50b1482681b450d32b1f58765990d5d553c4838ae7936db0\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-27\",\n        \"house_finch\",\n        269729570,\n        156011712,\n        \"kbkash\",\n        59048,\n        \"3af407624c46ba4218d24a198ed22b812d4967380bd0e7f67e960f7fbb2defe7\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-28\",\n        \"house_finch\",\n        270277531,\n        156183453,\n        \"aparrot1\",\n        83424,\n        \"93c92a1bb7cf89e7106b0c3df65ce030c82bccade181d661952e940ddcc6282a\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-29\",\n        \"house_finch\",\n        658061500,\n        360629741,\n        \"mike_cove\",\n        107594,\n        \"deaaefaf712059b6c8a929a83e1ffd0295e12f0f291bc9869011703eecfcf372\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-30\",\n        \"house_finch\",\n        545518752,\n        302693775,\n        \"cvharris\",\n        208917,\n        \"fc5ffdbd30bc76fa4371a99f59543c9137411ddac0a691e2e8122ae7796e038b\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-31\",\n        \"house_finch\",\n        662587006,\n        362998939,\n        \"curran\",\n        113662,\n        \"644931afff18c942930717ac3ca563b63c18a0c78ab12ff88557aeede18ead63\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-32\",\n        \"house_finch\",\n        665549166,\n        364566546,\n        \"sealgyu\",\n        130087,\n        \"e6fb3e42979ce2709433e8141c4c665a70c4942166153578c7f0173328ffb2ad\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-33\",\n        \"house_finch\",\n        566433127,\n        313653086,\n        \"fake_id\",\n        150986,\n        \"cb8a50f48f65795850f57c4d8f84e49debc3deed59b6a60d5bee8ef95299650f\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-34\",\n        \"house_finch\",\n        404592864,\n        228025269,\n        \"logan_artz\",\n        78971,\n        \"d0cf631b138d199951d96538903eeb1b56e3b8da3f28c7b042d8868060f27a44\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-35\",\n        \"house_finch\",\n        423118612,\n        237726292,\n        \"conhawn\",\n        153347,\n        \"826ddd6fa8fad5bd1450c0c7beb77fa3261f41a0afa61d7681b7096bc4225c40\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-36\",\n        \"house_finch\",\n        290080883,\n        167451466,\n        \"vijaybarve\",\n        89129,\n        \"f0b32b4af4bdf0bc7eaf298b37a9be6dbcb175d9f72553efc239b6afd2746a69\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-37\",\n        \"house_finch\",\n        296347840,\n        170825854,\n        \"susanaber\",\n        96563,\n        \"47832989fc212f19631e55b63c23155817209c40c66af49e4e03e42863243770\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-38\",\n        \"house_finch\",\n        414288752,\n        233136904,\n        \"wafflemaster135\",\n        29548,\n        \"9f2039a678a69e8f8f383d43f80e3f4e6fcc81d5b99285a0e5b5e54834cf9125\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-39\",\n        \"house_finch\",\n        284751592,\n        164572867,\n        \"efalquet\",\n        80806,\n        \"b50e0a0a4a0bae79b5a8cd034645252044601a29b081f0842443848e802e82d2\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-40\",\n        \"house_finch\",\n        205914234,\n        121669780,\n        \"darylnolan\",\n        44763,\n        \"c04e8d7c002ad0038dcb62843e776ffb8fc86f425c2399442dd1653bbaa4b904\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-41\",\n        \"house_finch\",\n        96688995,\n        60467105,\n        \"erikschiff\",\n        145431,\n        \"dd056b1c0d89dcb9fa06b1c38936142e5c825eb2aeed0d7d2265dc7e147be792\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-42\",\n        \"house_finch\",\n        732358790,\n        399184296,\n        \"katrinamccollough\",\n        41331,\n        \"76c899fd9b9aa84ad41888c71a36ab99431fe7201dd417f2c19a66f77388420d\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-43\",\n        \"house_finch\",\n        83824778,\n        52720178,\n        \"beesbirdsbugs\",\n        48816,\n        \"72731eb63fa9c5058f9911d11be929af9a58cfea805159d54010e9095948c39a\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-44\",\n        \"house_finch\",\n        349400403,\n        198261102,\n        \"paulgraham\",\n        105965,\n        \"b0d1f556ee4926e6a6f87201f6b73955dd9c530788e0ce57d5e09ced74ede001\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-45\",\n        \"house_finch\",\n        484652323,\n        269544283,\n        \"robinlanark\",\n        29863,\n        \"a1d6e8ffbcef10a83a2d3674c7228af89f8d39297f44cd569d1e031da02bc516\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-46\",\n        \"house_finch\",\n        341357046,\n        194221751,\n        \"haconra\",\n        80643,\n        \"8e11937a5c6d5a47da47f562d52d4089d58ad3f2adefadc287b30c562c7ece85\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-47\",\n        \"house_finch\",\n        622140195,\n        341714166,\n        \"rachel141\",\n        133938,\n        \"aa776ef9fd3022b6b3db824a762bdde82eeb95518a8764763bd8199683bf1d04\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-48\",\n        \"house_finch\",\n        515309527,\n        286632225,\n        \"solunasilver\",\n        29507,\n        \"72a6b495ed99de80504c6b1a86ac1fbfc9e5736bc6f43274bd6093cc91f9ea05\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-49\",\n        \"house_finch\",\n        514877896,\n        286416757,\n        \"jubileej\",\n        153667,\n        \"26cd596525a244d2ba82ec2d575834c15671080814400cf7862f614c0db68d07\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-50\",\n        \"house_finch\",\n        125605349,\n        76831730,\n        \"sarahangulo\",\n        60174,\n        \"236eb345f1cf1ab4c82fb3b2cc48b948e614f8ddfd958c87a62ce5b44a04f080\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-51\",\n        \"house_finch\",\n        12881677,\n        9467101,\n        \"artemis224\",\n        173956,\n        \"91b5f9f66e8b84cd96169894024ebc3519de6f737d3a32ee3b70100c22b039b9\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-52\",\n        \"house_finch\",\n        124405788,\n        76034586,\n        \"radrat\",\n        36008,\n        \"bb9b3fe0b37a1ab6342b1e573f3faf3a287669b70fa9f00b5bfce83de4a376c6\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-53\",\n        \"house_finch\",\n        17468099,\n        12163814,\n        \"andy71\",\n        199644,\n        \"6bc02aaf56c1549dd78d6711f9121d48c33fbd76a8fb623895369f05604ad30d\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-54\",\n        \"house_finch\",\n        134699271,\n        82117365,\n        \"seanwashington1\",\n        35334,\n        \"d6e0fe4a8cc690fd6369deff8ea67f4810e1f1297a6f1b9f125aae905742db2a\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-55\",\n        \"house_finch\",\n        134287410,\n        81854554,\n        \"omcelroy\",\n        90394,\n        \"6b4a5ea4c8aa1722332ccdf420af0ca0906cfdb87348e9f6c0904d5a92d8c203\",\n        \"jpg\",\n    ),\n    (\n        \"house_finch-56\",\n        \"house_finch\",\n        17763955,\n        12337763,\n        \"jennifer510\",\n        167368,\n        \"35355c603049fdf21fc3bf1fed827b50740cfdba134542707026bb4e1f32149b\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-57\",\n        \"house_finch\",\n        18946521,\n        13021113,\n        \"deejay\",\n        60576,\n        \"2420363f6b02433f4b03f3323148ed5e4ac9d37c58a27e859f8e48354db866c8\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-58\",\n        \"house_finch\",\n        14590841,\n        10492098,\n        \"silvercat\",\n        120515,\n        \"23934e0376c70dd6d5ce5e9a24c4176291c877432d25257f07e27694495c879a\",\n        \"jpeg\",\n    ),\n    (\n        \"house_finch-59\",\n        \"house_finch\",\n        124702263,\n        76228853,\n        \"janeyair\",\n        162431,\n        \"5c86d7cbd28e09d44814735f1e0a21e8266bd4e6a4396b047d092d2f5f3f2111\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-00\",\n        \"american_goldfinch\",\n        84579952,\n        53187208,\n        \"glennberry\",\n        59673,\n        \"72d36079e592e0a83c2f774f9073bfd4cc81253452c925d1673217ddd4b52a36\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-01\",\n        \"american_goldfinch\",\n        12533322,\n        9255418,\n        \"braincellsgone\",\n        55661,\n        \"6344e0125e74791f43ac6e07e5e1b9fbfce6d19bc62b6bb5d83b3caff9f7bcbc\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-02\",\n        \"american_goldfinch\",\n        131102823,\n        80016788,\n        \"radrat\",\n        98990,\n        \"232a944f7e3351d4916a12ef2f6d598e7b007caaa95a9b064a14c1ba3af2a6ff\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-03\",\n        \"american_goldfinch\",\n        175048222,\n        104466897,\n        \"eug302\",\n        44231,\n        \"11c723482cc75fcc3a723ac1c0818a68e2fcf74c4ca684cf60195bf0d33f274e\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-04\",\n        \"american_goldfinch\",\n        68849595,\n        43390778,\n        \"mefisher\",\n        154503,\n        \"66f07bc59bb3fdedd65a4537ebabd0cafd457826b8bf4bb633181f584a3edfd1\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-05\",\n        \"american_goldfinch\",\n        431916465,\n        242278180,\n        \"k-simpkins\",\n        45413,\n        \"d76e7adf33e3a84ebec24dde5438965e62ebec8595755453973846340e4f460d\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-06\",\n        \"american_goldfinch\",\n        230801384,\n        135330401,\n        \"enspring\",\n        42886,\n        \"78aebfb9b28c3e16dd9618a0e1ae06df67bfa4b8550c0879427d96915c475fed\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-07\",\n        \"american_goldfinch\",\n        377136648,\n        213398931,\n        \"nathan1177\",\n        66516,\n        \"7ad75838fdf2020a8e426e97507c7dd4355da93e6eece241128c28adbe302438\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-08\",\n        \"american_goldfinch\",\n        294667312,\n        169935316,\n        \"dande\",\n        163470,\n        \"39e7892e81eeef6af4887e61bc0998e17797688eb6394fcc8c9438391e875ee0\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-09\",\n        \"american_goldfinch\",\n        660472044,\n        361884286,\n        \"ben142\",\n        270599,\n        \"733d64cd50c61334682f0862f5c7859ded34cae5776ee0bc6594fee400dc4876\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-10\",\n        \"american_goldfinch\",\n        143215717,\n        86889530,\n        \"memoosborne\",\n        129438,\n        \"ef4a9a771cef9ba3c2c047eb106a6aa220236dd6aaa6aade5f4ef3a37c8abcba\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-11\",\n        \"american_goldfinch\",\n        403873164,\n        227647158,\n        \"drew_baxter\",\n        106009,\n        \"74e34c776f1b9a5d375a7dfae0e309cd42dbd133b82a1ecad4a49111ac7eed49\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-12\",\n        \"american_goldfinch\",\n        481153019,\n        267722510,\n        \"vicki936\",\n        255397,\n        \"3326ddbee3270241b681cb636466e742491f110e9841f453f5158864aadaea36\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-13\",\n        \"american_goldfinch\",\n        213311122,\n        125763980,\n        \"hickl\",\n        24740,\n        \"0c295b3761bced4215519ff24a4f734773b54be98de456a4444d0819456af7dd\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-14\",\n        \"american_goldfinch\",\n        417352824,\n        234736320,\n        \"joy4birds\",\n        81109,\n        \"357014c108519543471b94f39591667d1a67d87fd1fdc4702a3a449235d6bddd\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-15\",\n        \"american_goldfinch\",\n        72130720,\n        45486482,\n        \"dctphoto\",\n        178111,\n        \"17b2f3599e20161acc17fd63bf61e9f40488b9c4d5bfaefc8cae54de42997509\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-16\",\n        \"american_goldfinch\",\n        45148898,\n        28952026,\n        \"megachile\",\n        49358,\n        \"6bbc6ca0ac074e486c20ce4c3fad5dc863cb2e908efc2b1ef97494403a351252\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-17\",\n        \"american_goldfinch\",\n        133251099,\n        81250513,\n        \"raffib128\",\n        128110,\n        \"863c76587fe1de80a84b97c2e72f38ae1cf4fa972789e006d9b0b544d696c6ef\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-18\",\n        \"american_goldfinch\",\n        155797129,\n        93957328,\n        \"nathanael15\",\n        74086,\n        \"7059ae3bdeeb48fe949e5b70b788bd28c96aecd0fe49cb7be22db9a8697bf5a5\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-19\",\n        \"american_goldfinch\",\n        11400438,\n        8535447,\n        \"akneidel\",\n        26423,\n        \"4af74c1d04ddc7bbb7bb0e9eb2977a1daf48dd9b9477d71d69a7c4cb5d4f785b\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-20\",\n        \"american_goldfinch\",\n        55990791,\n        35505213,\n        \"conhawn\",\n        84915,\n        \"69f768c39a2180440bdcbfc6addc5d426341e8080d4cf9ba8241d57564b3e6fb\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-21\",\n        \"american_goldfinch\",\n        145464774,\n        88186208,\n        \"wildreturn\",\n        68204,\n        \"565d2a3b6d404e9ea0c24737ebcc5a3a82057b1452b4790df5e9552e8c50e592\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-22\",\n        \"american_goldfinch\",\n        637247336,\n        289067166,\n        \"dinomariobob\",\n        142312,\n        \"01c63d305fce8edcc3a494f0543154c6bd6aa52368e02be84cf6c23e95b94cdd\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-23\",\n        \"american_goldfinch\",\n        460988055,\n        257029007,\n        \"eric112\",\n        60314,\n        \"0db1b5f0e32d1faf861a437a20794fa18e80ea8966313933c145451c9319e2a9\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-24\",\n        \"american_goldfinch\",\n        59072170,\n        37280587,\n        \"bradenjudson\",\n        23468,\n        \"827d77bf9ca9cc456e867434a07b68cdceb4a20e09eb9e6b67757c80cd31ff42\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-25\",\n        \"american_goldfinch\",\n        66515260,\n        41915252,\n        \"reuvenm\",\n        58833,\n        \"90f0b687d9626fdf5d0111b794cf960cf18f3de4c8eab601fcb23c41b45c4df7\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-26\",\n        \"american_goldfinch\",\n        109875643,\n        67928198,\n        \"artemis224\",\n        217909,\n        \"99e1d7d30eb19d47c7974e9ddd7efe4d06329c952a41e15d0a4b2095b747df61\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-27\",\n        \"american_goldfinch\",\n        219478221,\n        129171341,\n        \"cgmayers\",\n        96993,\n        \"93bdee27bb526b60e0edfa518a923b3c10c41ba57221d6a23f4de15bf4a91600\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-28\",\n        \"american_goldfinch\",\n        99106353,\n        61899131,\n        \"thyg\",\n        50544,\n        \"0272bf89b379c2689247f1dbd6f4b36d7f197898baad91d7a8ebdf1c87f88fbd\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-29\",\n        \"american_goldfinch\",\n        223110598,\n        131133187,\n        \"marissa3\",\n        89214,\n        \"6a2bb601ac3a70e80e6ccb33d4a24af43128d1e7afe3e92d42e803234bdf532e\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-30\",\n        \"american_goldfinch\",\n        104649681,\n        65037800,\n        \"kemper\",\n        113821,\n        \"f4430daf9fca2cf6deb6b988148a5c2e5ae382af8c5db7e5cd37b26e8fbc9461\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-31\",\n        \"american_goldfinch\",\n        98669936,\n        61655828,\n        \"cloaca_enthusiast\",\n        31964,\n        \"3ddcb2e2e6863ef08a40fae7640e05b36ccec9fab9892ac494f095c4588b746f\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-32\",\n        \"american_goldfinch\",\n        347798463,\n        197465311,\n        \"aster-asti\",\n        116039,\n        \"e1236dd9850a412ba68386ffd342d180f582db187b83c1d33cb1e09dd51d0043\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-33\",\n        \"american_goldfinch\",\n        461261879,\n        257163676,\n        \"erikschiff\",\n        40881,\n        \"eb0b1ea08fcee160edd008ffc386b69b7b50bb89096c44873b978072c581d826\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-34\",\n        \"american_goldfinch\",\n        343726970,\n        195390446,\n        \"robinlanark\",\n        53508,\n        \"8b1f4e496755d1a3c4f567c6a48673998b8480ce7d38e7c52bbd94d7ee9901ed\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-35\",\n        \"american_goldfinch\",\n        352259924,\n        199722919,\n        \"carterdorscht\",\n        122243,\n        \"11edfdb53f892b1123df734cb7300e002171baeef21671a31411ff30d156bdc6\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-36\",\n        \"american_goldfinch\",\n        148829913,\n        90084462,\n        \"ian-wolfe\",\n        172110,\n        \"4f6a493313fc596665eb99f8e303ab0ee980b4932a9957993cd00b03b5360fa4\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-37\",\n        \"american_goldfinch\",\n        276388190,\n        159958623,\n        \"dianeclark6280\",\n        39786,\n        \"284ca7c9e814fede2c5a15e427a46d7dd82fa6984c4c862c380b0dddff2f86db\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-38\",\n        \"american_goldfinch\",\n        409778763,\n        230765968,\n        \"squidtk\",\n        210988,\n        \"b92ccce145f7a320f25edf92633e0c56aa92e8c83890b4a439ce4b3d669cb1a1\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-39\",\n        \"american_goldfinch\",\n        382773537,\n        216432059,\n        \"rocksand2134\",\n        195020,\n        \"ad33d43d9e5650dd441bbd9adb9cc9854a6a1bcc7088a8802a7fb4ccc1ee8b51\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-40\",\n        \"american_goldfinch\",\n        290287368,\n        167563678,\n        \"sean579\",\n        105190,\n        \"8703f245d4b98476f67775ca0c19b7abd7c5edf756c50b2ff502b3d058c08014\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-41\",\n        \"american_goldfinch\",\n        423112648,\n        237722733,\n        \"andrew2285\",\n        55719,\n        \"10709f5df3f4c509afd260ba51c0db5cee89c7c9076c74e9911e9613303d7754\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-42\",\n        \"american_goldfinch\",\n        422361237,\n        237335568,\n        \"wildaboutwildlife\",\n        34855,\n        \"55b0962389b589037e03378472e65ffa0e7ed0fb7e227a7441106a009f093924\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-43\",\n        \"american_goldfinch\",\n        291016386,\n        167956837,\n        \"samallonthesciencemon\",\n        128110,\n        \"489934bec3eb5682067cca11cb210c171198e616bb4920753b48bb0e1480bc32\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-44\",\n        \"american_goldfinch\",\n        419250127,\n        235728207,\n        \"paulgraham\",\n        39211,\n        \"be1f9ae51a6c4ee92a543ecfe1654698581d7647a3a0b46859a7cf054d8b0fd9\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-45\",\n        \"american_goldfinch\",\n        416004544,\n        234039610,\n        \"randv\",\n        105453,\n        \"942a87723af8aedc86459479437bc474817f19429806f36d8e1236538448e24f\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-46\",\n        \"american_goldfinch\",\n        416408092,\n        234251817,\n        \"kmayner\",\n        143385,\n        \"55259e21e667b88864e2b88ba8dacc8ac5be0d2606699be35c46d2b17fe67fb2\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-47\",\n        \"american_goldfinch\",\n        116188790,\n        71368917,\n        \"leahmfulton\",\n        48440,\n        \"ae01ef8ba2fb714253543a5e773e4e047e95448f4ff4d1e51bc889a20b14e1bc\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-48\",\n        \"american_goldfinch\",\n        118530525,\n        72675993,\n        \"seanwashington1\",\n        38505,\n        \"d688edde61a079907cdf5011a32914432f8f7f35c12d355e308aae29b1988738\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-49\",\n        \"american_goldfinch\",\n        121016576,\n        74071097,\n        \"geenance\",\n        110180,\n        \"fb4d1fef09311949043b3c9edc0d3a85d4c20386346093bb6ea39e1b95f756b9\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-50\",\n        \"american_goldfinch\",\n        3279608,\n        2872951,\n        \"joed\",\n        115964,\n        \"c2f1463bfef4c9b157f7e3ebd33fa37570592a4cdfa276f2b7c673f543cbc3ca\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-51\",\n        \"american_goldfinch\",\n        118140282,\n        72458899,\n        \"melaniegaddy\",\n        243713,\n        \"a3d70061032ac8c665f3d17a9118c8e592d53a3669182b23b502e3573a80d8f8\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-52\",\n        \"american_goldfinch\",\n        7669539,\n        6125334,\n        \"cuihenggang\",\n        61287,\n        \"36d9570131caf33f3b5cfa6ab99a26c417e7cd8e38d0c1351d1a2938bb6e9bf9\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-53\",\n        \"american_goldfinch\",\n        9258865,\n        7182023,\n        \"fake_id\",\n        151987,\n        \"131d2a52097c9ab18fad318b62e8213c1bd33acddc72065ec0e404c76220c5ee\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-54\",\n        \"american_goldfinch\",\n        12222432,\n        9054511,\n        \"myacadianforest\",\n        107758,\n        \"69b1b5274d74bb28601afc2758e57a88f8d63743b5410c70284a9f4877c721fb\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-55\",\n        \"american_goldfinch\",\n        128301260,\n        78400716,\n        \"natepow\",\n        67416,\n        \"0c988080a47484802806ef14e0d1fdb9205680eacecedbe24f8bbac2aa2873f9\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-56\",\n        \"american_goldfinch\",\n        129842959,\n        79296314,\n        \"nartb\",\n        117002,\n        \"f23b2e68f4c10e2679f0a9fe2f34a894bd2deea9c90c2143462e0aa5a83a3daa\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-57\",\n        \"american_goldfinch\",\n        78410307,\n        49351195,\n        \"chrisleearm\",\n        108066,\n        \"8a4b273f1a1bd548c48abcf57b86ed35028cbfb949720180a43f24976a26ac8e\",\n        \"jpg\",\n    ),\n    (\n        \"american_goldfinch-58\",\n        \"american_goldfinch\",\n        72300928,\n        45598176,\n        \"henrya\",\n        68593,\n        \"a0c2869c330aaea08fba3bfeae99392c5588980b1ec21770c220c744125c3b42\",\n        \"jpeg\",\n    ),\n    (\n        \"american_goldfinch-59\",\n        \"american_goldfinch\",\n        194724999,\n        115367030,\n        \"vlatassa\",\n        159105,\n        \"c456a23238f8f81f93df7919b92dd3fa0123fa16a875c22ec1f9d0ae6ef17324\",\n        \"jpeg\",\n    ),\n)\nSAMPLE_SEED = 42\nSAMPLE_SPLIT = {\"train\": 36, \"validation\": 8, \"test\": 16}  # photographs per species; 6 species -> 216 / 48 / 96 pairs\nTIERS = (\"easy\", \"hard\")\nTIER_PARAMS: dict[str, dict[str, Any]] = {\n    \"easy\": {\"jitter\": 0.06, \"rotation\": 10.0, \"scale\": (0.9, 1.1), \"brightness\": (0.85, 1.15), \"contrast\": (0.85, 1.15), \"gamma\": (1.0, 1.0), \"blur\": 0.0, \"noise\": 4.0},\n    \"hard\": {\"jitter\": 0.18, \"rotation\": 35.0, \"scale\": (0.6, 1.4), \"brightness\": (0.6, 1.4), \"contrast\": (0.6, 1.4), \"gamma\": (0.7, 1.4), \"blur\": 1.2, \"noise\": 10.0},\n}\nMIN_RECORDS = 4\nMAX_RECORDS = 5_000\n_ID_RE = re.compile(r\"^[A-Za-z0-9_.:-]{1,64}$\")\n\n\ndef _sha256_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef photo_url(photo_id: int, ext: str = \"jpg\") -> str:\n    \"\"\"The served object for a pinned photo; `ext` is its recorded original extension (jpg, jpeg or png,\n    either case — the bucket key is case-sensitive).\"\"\"\n    if ext.lower() not in (\"jpg\", \"jpeg\", \"png\"):\n        raise ValueError(f\"unsupported photo extension {ext!r}\")\n    return f\"{CORPUS_BASE_URL}{photo_id}/medium.{ext}\"\n\n\ndef observation_url(observation_id: int) -> str:\n    return f\"https://www.inaturalist.org/observations/{observation_id}\"\n\n\ndef fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:\n    \"\"\"Return every pinned photo (bytes keyed by record id) from the cache or the open-data bucket.\"\"\"\n    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR\n    cache.mkdir(parents=True, exist_ok=True)\n    out = {}\n    for rid, _label, photo_id, _obs, _user, size, digest, ext in SAMPLE_RECORDS:\n        local = cache / f\"{photo_id}.jpg\"\n        data = local.read_bytes() if local.is_file() else b\"\"\n        if len(data) != size or _sha256_bytes(data) != digest:\n            url = photo_url(photo_id, ext)\n            if fetcher is not None:\n                data = fetcher(url)\n            else:\n                request = urllib.request.Request(\n                    url, headers={\"User-Agent\": \"dimer-xoftr-tutorial/1.0\"}\n                )\n                with urllib.request.urlopen(request, timeout=120) as response:  # noqa: S310 (pinned https URL)\n                    data = response.read()\n            if len(data) != size or _sha256_bytes(data) != digest:\n                raise ValueError(\n                    f\"{rid} ({photo_id}/medium.{ext}): fetched {len(data)} bytes with sha256 \"\n                    f\"{_sha256_bytes(data)[:16]}…, pinned {size} / {digest[:16]}…\"\n                )\n            local.write_bytes(data)\n        out[rid] = data\n    return out\n\n\ndef read_corpus(files: Mapping[str, bytes]) -> list[dict[str, Any]]:\n    \"\"\"Decode the verified photo bytes into `{id, image, species}` records with their provenance.\"\"\"\n    out = []\n    for rid, label, photo_id, obs_id, user, _size, _digest, _ext in SAMPLE_RECORDS:\n        if rid not in files:\n            raise ValueError(f\"corpus is missing {rid}\")\n        image = Image.open(io.BytesIO(files[rid]))\n        image.load()\n        out.append(\n            {\n                \"id\": rid,\n                \"image\": image.convert(\"RGB\"),\n                \"species\": label,\n                \"scientific_name\": SPECIES[label][0],\n                \"common_name\": SPECIES[label][1],\n                \"inat_photo_id\": photo_id,\n                \"inat_observation_url\": observation_url(obs_id),\n                \"observer\": user,\n            }\n        )\n    return out\n\n\n# --------------------------------------------------------------------------------------------------\n# pair synthesis: a seeded homography with an exact reference\n# --------------------------------------------------------------------------------------------------\n\n\ndef working_size(size: tuple[int, int], long_side: int = WORKING_LONG_SIDE) -> tuple[int, int]:\n    \"\"\"The size a photograph is brought to: long side `long_side`, both sides multiples of DIVISIBLE_BY.\"\"\"\n    width, height = size\n    scale = long_side / max(width, height)\n    w = max(DIVISIBLE_BY, int(width * scale) // DIVISIBLE_BY * DIVISIBLE_BY)\n    h = max(DIVISIBLE_BY, int(height * scale) // DIVISIBLE_BY * DIVISIBLE_BY)\n    return w, h\n\n\ndef prepare_image(image: Image.Image, long_side: int = WORKING_LONG_SIDE) -> Image.Image:\n    \"\"\"Resize to the working size (bicubic, aspect kept up to the multiple-of-8 crop) as RGB.\"\"\"\n    w, h = working_size(image.size, long_side)\n    return image.convert(\"RGB\").resize((w, h), Image.BICUBIC)\n\n\ndef sample_homography(size: tuple[int, int], rng: random.Random, params: Mapping[str, Any]) -> np.ndarray:\n    \"\"\"A random homography built from a rotation + scale about the centre followed by corner jitter.\"\"\"\n    width, height = size\n    cx, cy = (width - 1) / 2.0, (height - 1) / 2.0\n    angle = math.radians(rng.uniform(-params[\"rotation\"], params[\"rotation\"]))\n    scale = rng.uniform(*params[\"scale\"])\n    cos_a, sin_a = math.cos(angle) * scale, math.sin(angle) * scale\n    similarity = np.array(\n        [[cos_a, -sin_a, cx - cos_a * cx + sin_a * cy], [sin_a, cos_a, cy - sin_a * cx - cos_a * cy], [0.0, 0.0, 1.0]]\n    )\n    corners = np.array([[0.0, 0.0], [width - 1.0, 0.0], [width - 1.0, height - 1.0], [0.0, height - 1.0]])\n    jitter = params[\"jitter\"] * min(width, height)\n    moved = corners + np.array([[rng.uniform(-jitter, jitter), rng.uniform(-jitter, jitter)] for _ in range(4)])\n    from .metrics import dlt_homography\n\n    perspective = dlt_homography(corners, moved)\n    if perspective is None:  # degenerate draw (practically impossible); fall back to the similarity alone\n        return similarity\n    return perspective @ similarity\n\n\ndef _perspective_coefficients(homography: np.ndarray) -> list[float]:\n    \"\"\"PIL's PERSPECTIVE transform takes the inverse mapping (output pixel -> input pixel), 8 coefficients.\"\"\"\n    inverse = np.linalg.inv(homography)\n    inverse = inverse / inverse[2, 2]\n    return [float(v) for v in inverse.ravel()[:8]]\n\n\ndef warp_image(image: Image.Image, homography: np.ndarray) -> Image.Image:\n    \"\"\"image1 = image0 warped by `homography` (image0 coords -> image1 coords), same canvas, black outside.\"\"\"\n    return image.transform(image.size, Image.PERSPECTIVE, _perspective_coefficients(homography), Image.BICUBIC)\n\n\ndef photometric(image: Image.Image, rng: random.Random, params: Mapping[str, Any]) -> Image.Image:\n    \"\"\"Seeded brightness / contrast / gamma / blur / Gaussian-noise changes (never geometric).\"\"\"\n    out = ImageEnhance.Brightness(image).enhance(rng.uniform(*params[\"brightness\"]))\n    out = ImageEnhance.Contrast(out).enhance(rng.uniform(*params[\"contrast\"]))\n    gamma = rng.uniform(*params[\"gamma\"])\n    if params[\"blur\"] > 0:\n        out = out.filter(ImageFilter.GaussianBlur(rng.uniform(0.0, params[\"blur\"])))\n    array = np.asarray(out, dtype=np.float64) / 255.0\n    if gamma != 1.0:\n        array = np.power(np.clip(array, 0.0, 1.0), gamma)\n    if params[\"noise\"] > 0:\n        noise_rng = np.random.default_rng(rng.getrandbits(32))\n        array = array + noise_rng.normal(0.0, params[\"noise\"] / 255.0, array.shape)\n    return Image.fromarray((np.clip(array, 0.0, 1.0) * 255.0).round().astype(np.uint8))\n\n\ndef make_pair(image: Image.Image, *, seed: int, tier: str = \"hard\", record_id: str = \"pair\") -> dict[str, Any]:\n    \"\"\"One `{id, image0, image1, homography, tier}` record from a photograph and a seed.\"\"\"\n    if tier not in TIER_PARAMS:\n        raise ValueError(f\"tier must be one of {TIERS}\")\n    params = TIER_PARAMS[tier]\n    rng = random.Random(seed)\n    image0 = prepare_image(image)\n    homography = sample_homography(image0.size, rng, params)\n    image1 = photometric(warp_image(image0, homography), rng, params)\n    return {\n        \"id\": record_id,\n        \"image0\": image0,\n        \"image1\": image1,\n        \"homography\": homography.tolist(),\n        \"tier\": tier,\n        \"seed\": seed,\n    }\n\n\ndef make_pairs(records: Sequence[Mapping[str, Any]], *, seed: int = SAMPLE_SEED, tier: str | None = None) -> list[dict[str, Any]]:\n    \"\"\"One pair per image record (`{id, image, ...}`); tiers alternate easy / hard unless `tier` is fixed.\"\"\"\n    out = []\n    for index, record in enumerate(records):\n        chosen = tier or TIERS[index % len(TIERS)]\n        pair = make_pair(record[\"image\"], seed=seed * 100_003 + index, tier=chosen, record_id=str(record[\"id\"]))\n        for key in (\"species\", \"observer\", \"inat_photo_id\", \"inat_observation_url\", \"source_id\"):\n            if key in record:\n                pair[key] = record[key]\n        out.append(pair)\n    return out\n\n\ndef build_sample_dataset(\n    records: Sequence[Mapping[str, Any]],\n    *,\n    seed: int = SAMPLE_SEED,\n    sizes: Mapping[str, int] | None = None,\n) -> dict[str, list[dict[str, Any]]]:\n    \"\"\"Seeded stratified draw of photographs per species into train / validation / test, then one pair per\n    photograph (tiers alternating within each split). No photograph lands in two splits.\"\"\"\n    sizes = dict(sizes or SAMPLE_SPLIT)\n    rng = random.Random(seed)\n    by_label: dict[str, list[dict[str, Any]]] = {}\n    for record in records:\n        by_label.setdefault(str(record.get(\"species\", \"image\")), []).append(dict(record))\n    photos: dict[str, list[dict[str, Any]]] = {name: [] for name in sizes}\n    for label in sorted(by_label):\n        pool = by_label[label]\n        rng.shuffle(pool)\n        needed = sum(sizes.values())\n        if len(pool) < needed:\n            raise ValueError(f\"{label}: only {len(pool)} records available, need {needed}\")\n        cursor = 0\n        for name, per_class in sizes.items():\n            photos[name].extend(pool[cursor : cursor + per_class])\n            cursor += per_class\n    out: dict[str, list[dict[str, Any]]] = {}\n    for offset, name in enumerate(photos):\n        rng.shuffle(photos[name])\n        relabelled = [{**r, \"id\": f\"{name}-{i:03d}\", \"source_id\": r[\"id\"]} for i, r in enumerate(photos[name])]\n        out[name] = make_pairs(relabelled, seed=seed + offset)\n    return out\n\n\ndef fetch_sample_dataset(\n    *,\n    cache_dir: str | Path | None = None,\n    fetcher: Any = None,\n    seed: int = SAMPLE_SEED,\n    sizes: Mapping[str, int] | None = None,\n) -> dict[str, list[dict[str, Any]]]:\n    \"\"\"The tutorial splits from the pinned corpus.\"\"\"\n    return build_sample_dataset(\n        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes\n    )\n\n\n# --------------------------------------------------------------------------------------------------\n# validation\n# --------------------------------------------------------------------------------------------------\n\n\ndef _open(image: Any, label_name: str) -> Image.Image:\n    if isinstance(image, str | Path):\n        path = Path(image)\n        if not path.is_file():\n            raise ValueError(f\"{label_name}: image file not found: {path}\")\n        image = Image.open(path)\n        image.load()\n    if not isinstance(image, Image.Image):\n        raise ValueError(f\"{label_name}: image must be a PIL.Image.Image or a file path\")\n    width, height = image.size\n    if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:\n        raise ValueError(f\"{label_name}: image side outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: {image.size}\")\n    return image.convert(\"RGB\")\n\n\ndef check_homography(value: Any, label_name: str = \"homography\") -> np.ndarray:\n    array = np.asarray(value, dtype=np.float64)\n    if array.shape != (3, 3) or not np.all(np.isfinite(array)):\n        raise ValueError(f\"{label_name}: must be a finite 3x3 matrix\")\n    if abs(array[2, 2]) < 1e-12:\n        raise ValueError(f\"{label_name}: H[2, 2] must be non-zero\")\n    array = array / array[2, 2]\n    if abs(np.linalg.det(array)) < 1e-9:\n        raise ValueError(f\"{label_name}: matrix is singular\")\n    return array\n\n\ndef _check_record(record: Any, index: int) -> dict[str, Any]:\n    label_name = f\"records[{index}]\"\n    if not isinstance(record, Mapping):\n        raise ValueError(f\"{label_name} must be a mapping with id/image0/image1/homography\")\n    for key in (\"id\", \"image0\", \"image1\", \"homography\"):\n        if key not in record:\n            raise ValueError(f\"{label_name} is missing {key!r}\")\n    rid = record[\"id\"]\n    if not isinstance(rid, str) or not _ID_RE.match(rid):\n        raise ValueError(f\"{label_name}: id must match {_ID_RE.pattern}\")\n    image0 = _open(record[\"image0\"], label_name + \".image0\")\n    image1 = _open(record[\"image1\"], label_name + \".image1\")\n    for which, image in ((\"image0\", image0), (\"image1\", image1)):\n        if min(image.size) < MIN_SIDE or max(image.size) > MAX_SIDE:\n            raise ValueError(\n                f\"{label_name}.{which}: sides must lie in [{MIN_SIDE}, {MAX_SIDE}] px after preparation; got {image.size}\"\n            )\n    homography = check_homography(record[\"homography\"], label_name + \".homography\")\n    tier = record.get(\"tier\", \"unspecified\")\n    if not isinstance(tier, str) or not tier:\n        raise ValueError(f\"{label_name}: tier must be a non-empty string when given\")\n    item = {\"id\": rid, \"image0\": image0, \"image1\": image1, \"homography\": homography.tolist(), \"tier\": tier}\n    for key in (\"source_id\", \"seed\", \"species\", \"observer\", \"inat_photo_id\", \"inat_observation_url\"):\n        if key in record:\n            item[key] = record[key]\n    return item\n\n\ndef validate_dataset(\n    records: Sequence[Mapping[str, Any]],\n    *,\n    min_records: int = MIN_RECORDS,\n    max_records: int = MAX_RECORDS,\n) -> dict[str, Any]:\n    \"\"\"Structural validation of a pair dataset; raises ValueError before any model import. Nothing checks\n    that `image1` really is `image0` under `homography` — a wrong reference is scored without complaint.\"\"\"\n    if (\n        isinstance(records, Mapping)\n        or not isinstance(records, Sequence)\n        or isinstance(records, (str, bytes))\n    ):\n        raise ValueError(\"records must be a list of {id, image0, image1, homography} mappings\")\n    if not min_records <= len(records) <= max_records:\n        raise ValueError(f\"{len(records)} records; {min_records}..{max_records} are required\")\n    checked = []\n    ids: set[str] = set()\n    tiers: dict[str, int] = {}\n    for index, record in enumerate(records):\n        item = _check_record(record, index)\n        if item[\"id\"] in ids:\n            raise ValueError(f\"duplicate id {item['id']!r}\")\n        ids.add(item[\"id\"])\n        tiers[item[\"tier\"]] = tiers.get(item[\"tier\"], 0) + 1\n        checked.append(item)\n    sides = [max(r[\"image0\"].size) for r in checked]\n    return {\n        \"records\": checked,\n        \"n_records\": len(checked),\n        \"tiers\": dict(sorted(tiers.items())),\n        \"image_side\": {\"min\": min(sides), \"max\": max(sides)},\n        \"digest\": dataset_digest(checked),\n        \"model_id\": MODEL_ID,\n    }\n\n\ndef image_digest(image: Image.Image) -> str:\n    \"\"\"SHA-256 of the decoded RGB pixels (size + bytes), so a re-encoded copy of the same image matches.\"\"\"\n    rgb = image.convert(\"RGB\")\n    return _sha256_bytes(f\"{rgb.size[0]}x{rgb.size[1]}:\".encode() + rgb.tobytes())\n\n\ndef dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:\n    payload = [[r[\"id\"], image_digest(r[\"image0\"]), image_digest(r[\"image1\"]), r[\"homography\"]] for r in records]\n    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(\",\", \":\")).encode(\"utf-8\"))\n\n\ndef check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:\n    \"\"\"Assert no source image (by decoded-pixel digest of image0) appears in two splits (leakage check).\"\"\"\n    seen: dict[str, str] = {}\n    for name, records in splits.items():\n        for record in records:\n            key = image_digest(record[\"image0\"])\n            if key in seen and seen[key] != name:\n                raise ValueError(f\"image {record['id']!r} appears in both {seen[key]} and {name}\")\n            seen[key] = name\n    return {name: len(records) for name, records in splits.items()}\n\n\ndef observer_overlap(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, int]:\n    \"\"\"How many observers contributed photos to more than one split (an observation, not an assertion).\"\"\"\n    seen: dict[str, set[str]] = {}\n    for name, records in splits.items():\n        for record in records:\n            if record.get(\"observer\"):\n                seen.setdefault(str(record[\"observer\"]), set()).add(name)\n    return {\n        \"observers\": len(seen),\n        \"in_more_than_one_split\": sum(1 for s in seen.values() if len(s) > 1),\n    }\n\n\ndef split_dataset(\n    records: Sequence[Mapping[str, Any]],\n    *,\n    val_fraction: float = 0.15,\n    test_fraction: float = 0.2,\n    seed: int = 0,\n) -> dict[str, list[dict[str, Any]]]:\n    \"\"\"Seeded shuffle of BYOD image records (`{id, image}`) into train / validation / test after de-duplicating\n    images by decoded pixels, then one pair per image.\"\"\"\n    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):\n        raise ValueError(\"fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1\")\n    seen: set[str] = set()\n    unique = []\n    for record in records:\n        image = _open(record[\"image\"], str(record.get(\"id\")))\n        key = image_digest(image)\n        if key not in seen:\n            seen.add(key)\n            unique.append({**record, \"image\": image})\n    rng = random.Random(seed)\n    rng.shuffle(unique)\n    n_test = max(1, round(len(unique) * test_fraction))\n    n_val = round(len(unique) * val_fraction)\n    parts = {\"test\": unique[:n_test], \"validation\": unique[n_test : n_test + n_val], \"train\": unique[n_test + n_val :]}\n    if len(parts[\"train\"]) < MIN_RECORDS:\n        raise ValueError(f\"split leaves {len(parts['train'])} training images; at least {MIN_RECORDS} are required\")\n    out = {}\n    for offset, (name, part) in enumerate(parts.items()):\n        relabelled = [{**r, \"id\": f\"{name}-{i:03d}\", \"source_id\": r[\"id\"]} for i, r in enumerate(part)]\n        out[name] = make_pairs(relabelled, seed=seed + offset)\n    return out\n\n\ndef load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:\n    \"\"\"Read `{id, image}` records from a directory or a zip of image files (optionally listed in `images.csv`\n    with columns `id`, `file`); images are decoded, never extracted to disk.\"\"\"\n    source = Path(path)\n    if source.is_dir():\n        names = sorted(p.name for p in source.iterdir() if p.suffix.lower() in (\".jpg\", \".jpeg\", \".png\"))\n        loader = lambda name: Image.open(source / name)  # noqa: E731\n    elif source.is_file() and source.suffix.lower() == \".zip\":\n        archive = zipfile.ZipFile(source)\n        members = {Path(n).name: n for n in archive.namelist() if Path(n).suffix.lower() in (\".jpg\", \".jpeg\", \".png\")}\n        names = sorted(members)\n        loader = lambda name: Image.open(io.BytesIO(archive.read(members[name])))  # noqa: E731\n    else:\n        raise ValueError(\"BYOD datasets must be a directory or a .zip holding JPEG / PNG image files\")\n    if not names:\n        raise ValueError(\"BYOD dataset holds no JPEG / PNG image files\")\n    out = []\n    for name in names:\n        image = loader(name)\n        image.load()\n        out.append({\"id\": re.sub(r\"[^A-Za-z0-9_.:-]\", \"_\", Path(name).stem)[:64], \"image\": image.convert(\"RGB\")})\n    return out\n\n\ndef write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:\n    \"\"\"Write the pair table of a split (id, source, tier, seed, the homography, provenance).\"\"\"\n    out = Path(path)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    with open(out, \"w\", encoding=\"utf-8\", newline=\"\") as handle:\n        writer = csv.DictWriter(handle, fieldnames=[\"id\", \"source_id\", \"tier\", \"seed\", \"homography\", \"observer\", \"inat_observation_url\"])\n        writer.writeheader()\n        for record in records:\n            writer.writerow(\n                {\n                    \"id\": record[\"id\"],\n                    \"source_id\": record.get(\"source_id\", \"\"),\n                    \"tier\": record.get(\"tier\", \"\"),\n                    \"seed\": record.get(\"seed\", \"\"),\n                    \"homography\": json.dumps(record[\"homography\"]),\n                    \"observer\": record.get(\"observer\", \"\"),\n                    \"inat_observation_url\": record.get(\"inat_observation_url\", \"\"),\n                }\n            )\n    return out\n"}
for relative, source in EMBEDDED_SOURCES.items():
    path=EMBEDDED_ROOT/relative
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(source,encoding="utf-8")

LG_WEIGHTS=RUNTIME_ROOT/"weights"/"lightglue-aliked"
XO_WEIGHTS=RUNTIME_ROOT/"weights"/"xoftr"
LG_WEIGHTS.mkdir(parents=True,exist_ok=True)
XO_WEIGHTS.mkdir(parents=True,exist_ok=True)
(LG_WEIGHTS/"dimer-base-manifest.json").write_text("{\n  \"format\": \"dimer_release_snapshot\",\n  \"formatVersion\": 1,\n  \"modelKey\": \"lightglue-aliked\",\n  \"modelId\": \"cvg/LightGlue\",\n  \"revision\": \"v0.1_arxiv\",\n  \"revisionKind\": \"github-release-tag\",\n  \"upstreamCodeRevision\": \"eb42fee2d71449efb0aa5c10549752b5d75384d8\",\n  \"license\": \"Apache-2.0 (LightGlue) / BSD-3-Clause (ALIKED)\",\n  \"files\": [\n    {\n      \"path\": \"aliked_lightglue.pth\",\n      \"bytes\": 47632827,\n      \"sha256\": \"d975e965b105311a6143194852297dff4f02aea5cc2e10cecfed966ca0e22503\",\n      \"sourceUrl\": \"https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/aliked_lightglue.pth\",\n      \"pickleAuditSha256\": \"e7b998d087a5dcadd37713daf30b63cc571160c3180ebc138500ab662197e932\",\n      \"convertsTo\": {\n        \"path\": \"aliked_lightglue.safetensors\",\n        \"bytes\": 47564948,\n        \"sha256\": \"9c630a386c74c534428370ce46253e1d0968655db180f97074cb6ad797bd2bc6\"\n      }\n    },\n    {\n      \"path\": \"aliked-n16.pth\",\n      \"bytes\": 2738091,\n      \"sha256\": \"5be8704840ed662d9d8c561bf7279c222092674e7eb05fd0feab94899e9d82f2\",\n      \"sourceUrl\": \"https://raw.githubusercontent.com/Shiaoming/ALIKED/683d7c65197395c0b3f01ebe76e1084a27e73a65/models/aliked-n16.pth\",\n      \"sourceRepository\": \"Shiaoming/ALIKED\",\n      \"sourceCommit\": \"683d7c65197395c0b3f01ebe76e1084a27e73a65\",\n      \"pickleAuditSha256\": \"5b9f0ba08490293d6c17b9cef219991e1a6edda31609429679f8dca1af5a7b10\",\n      \"convertsTo\": {\n        \"path\": \"aliked-n16.safetensors\",\n        \"bytes\": 2719928,\n        \"sha256\": \"3c8ca40c0c985cd4d641e96e4b408b14d067b5b3521ac17b36590447d49d115a\"\n      }\n    }\n  ],\n  \"totalBytes\": 50370918\n}\n",encoding="utf-8")
(XO_WEIGHTS/"dimer-base-manifest.json").write_text("{\r\n  \"format\": \"dimer_hf_snapshot\",\r\n  \"formatVersion\": 1,\r\n  \"modelKey\": \"xoftr\",\r\n  \"modelId\": \"vismatch/xoftr\",\r\n  \"revision\": \"d8ee7d89be3c9e5c157db3886db1c0f0e038b321\",\r\n  \"files\": [\r\n    {\r\n      \"path\": \"README.md\",\r\n      \"bytes\": 166,\r\n      \"sha256\": \"97e4f408247d1f95334caa95b992bbab2081aff8ba069063c9c069359acd4bb4\"\r\n    },\r\n    {\r\n      \"path\": \"vismatch.yaml\",\r\n      \"bytes\": 114,\r\n      \"sha256\": \"64c2401cb5a86b410b2d669c41d4cbdbe97cd9d685a98dbf3619568bf6cbb8f8\"\r\n    },\r\n    {\r\n      \"path\": \"xoftr_640.safetensors\",\r\n      \"bytes\": 44419304,\r\n      \"sha256\": \"4d5ed62e8b41f862ecc5c660e31f1c450402966623d6a28e85acf7fbd794cc69\"\r\n    }\r\n  ],\r\n  \"totalBytes\": 44419584\r\n}\r\n",encoding="utf-8")

SOURCE_PROVENANCE={
    "lightglue":{"repository":"kurtvalcorza/lightglue-matching-pipeline","commit":"86996ba0671c8cdcf1c922d95d9f53ca156f3130"},
    "xoftr":{"repository":"kurtvalcorza/xoftr-image-matching-pipeline","commit":"bb795eb064bf12742beb2e842e2a1aa7dc4093d8"},
}
print({"embedded_files":len(EMBEDDED_SOURCES),"source_provenance":SOURCE_PROVENANCE})


## 2. Runtime and optional BYOD

Both carried packages are compatible with one isolated environment. The default sample path downloads the packages' shared, pinned 360-photo CC0 iNaturalist corpus, then evaluates only the deterministic **96-photo test split**.

### BYOD

For your own measured image pairs, set `USE_BYOD=True` and provide a local ZIP in `BYOD_ZIP_PATH`.

The archive must contain `pairs.csv` with:

`id,image0,image1,tier,h00,h01,h02,h10,h11,h12,h20,h21,h22`

The nine `h..` values define the ground-truth 3×3 homography from image0 coordinates to image1 coordinates. Image paths must be relative and point to files inside the ZIP.

A homography is required because this notebook's primary evaluation is geometric. If you only have two photographs without a reference transformation, use the model-specific inference notebooks instead of treating an unlabelled visual match as measured evidence.


In [ ]:
# @title 2.1 Install isolated dependencies
RUNTIME_PINS=[
    "numpy==2.5.3",
    "pillow==11.3.0",
    "safetensors==0.8.0",
    "torch==2.14.0",
    "torchvision==0.29.0",
    "huggingface-hub==0.36.2",
]
ENV_DIR=RUNTIME_ROOT/"matching_env"
PYTHON=ENV_DIR/("Scripts/python.exe" if os.name=="nt" else "bin/python")
marker=ENV_DIR/"dimer_pins.json"
pins_text=json.dumps(RUNTIME_PINS,sort_keys=True)
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=True).create(ENV_DIR)
if not marker.exists() or marker.read_text()!=pins_text:
    subprocess.run([str(PYTHON),"-m","pip","install","--disable-pip-version-check","-q",*RUNTIME_PINS],check=True)
    marker.write_text(pins_text)
print({"runtime_python":str(PYTHON),"pins":RUNTIME_PINS})


In [ ]:
# @title 2.2 Validate/extract BYOD archive when enabled
MAX_ARCHIVE_EXPANDED_BYTES=2*1024**3

def safe_extract_zip(path,destination):
    destination.mkdir(parents=True,exist_ok=True)
    seen=set()
    expanded=0
    with zipfile.ZipFile(path) as archive:
        for info in archive.infolist():
            member=PurePosixPath(info.filename)
            if member.is_absolute() or ".." in member.parts:
                raise ValueError(f"Unsafe archive path: {info.filename!r}")
            if info.filename in seen:
                raise ValueError(f"Duplicate archive member: {info.filename!r}")
            seen.add(info.filename)
            mode=(info.external_attr>>16)&0o170000
            if mode==stat.S_IFLNK:
                raise ValueError(f"Symlink entries are not allowed: {info.filename!r}")
            expanded+=info.file_size
            if expanded>MAX_ARCHIVE_EXPANDED_BYTES:
                raise ValueError("Archive expands beyond the 2 GiB workshop ceiling.")
        archive.extractall(destination)
    return destination

BYOD_ROOT=None
if USE_BYOD:
    if not BYOD_ZIP_PATH:
        raise ValueError("USE_BYOD=True requires BYOD_ZIP_PATH.")
    archive=Path(BYOD_ZIP_PATH)
    if not archive.is_file():
        raise FileNotFoundError(archive)
    BYOD_ROOT=safe_extract_zip(archive,RUNTIME_ROOT/"byod_pairs")
    candidates=list(BYOD_ROOT.rglob("pairs.csv"))
    if len(candidates)!=1:
        raise ValueError("BYOD archive must contain exactly one pairs.csv.")
    BYOD_ROOT=candidates[0].parent
    print({"byod_root":str(BYOD_ROOT),"pairs_csv":str(candidates[0])})
else:
    print("Using the pinned iNaturalist homography benchmark.")


## 3. Build the shared benchmark and run both frozen matchers

For the default sample, the runner reconstructs the same deterministic photograph split used by the live carriers and synthesizes **three different pair sets from the same 96 held-out photographs**.

The metrics are:

- `precision_1px`, `precision_3px`, `precision_5px`: fraction of returned correspondences whose homography reprojection error is below the threshold;
- `matches_per_pair`: average number of returned matches;
- `inliers_per_pair`: average number of matches within 3 px;
- `median_error_px`: pooled median inlier reprojection error;
- `homography_acc_3px` / `homography_acc_5px`: fraction of pairs whose recovered homography has mean corner error below the threshold.

Two non-neural references—identity and local patch-neighbour matching—are scored under the exact same metric implementation.


In [ ]:
# @title 3.1 Materialize and run the standalone comparative runner
RUNNER_SOURCE=r"""
import csv
import json
import os
import platform
import random
import sys
import time
from pathlib import Path, PurePosixPath

import numpy as np
import torch
from PIL import Image

cfg=json.loads(Path(sys.argv[1]).read_text())
out_path=Path(sys.argv[2])
sys.path.insert(0,cfg["embedded_root"])

from lightglue_pipeline.pipeline import LightGluePipeline
from lightglue_pipeline import samples as lg_samples
from xoftr_pipeline.pipeline import XoFTRPipeline
from xoftr_pipeline import samples as xo_samples
from xoftr_pipeline.metrics import identity_baseline, patch_neighbour_baseline

device="cuda" if torch.cuda.is_available() else "cpu"
if device!="cuda":
    raise RuntimeError("CUDA is required by the reference comparative path.")

def photo_test_split():
    files=xo_samples.fetch_corpus(cache_dir=Path(cfg["runtime_root"])/"inat_cache")
    records=xo_samples.read_corpus(files)
    rng=random.Random(xo_samples.SAMPLE_SEED)
    by_label={}
    for record in records:
        by_label.setdefault(str(record.get("species","image")),[]).append(dict(record))
    photos={name:[] for name in xo_samples.SAMPLE_SPLIT}
    for label in sorted(by_label):
        pool=by_label[label]
        rng.shuffle(pool)
        cursor=0
        for name,per_class in xo_samples.SAMPLE_SPLIT.items():
            photos[name].extend(pool[cursor:cursor+per_class])
            cursor+=per_class
    for name in photos:
        rng.shuffle(photos[name])
        photos[name]=[{**r,"id":f"{name}-{i:03d}","source_id":r["id"]} for i,r in enumerate(photos[name])]
    return photos["test"][:int(cfg["max_eval_pairs"])]

def load_byod():
    root=Path(cfg["byod_root"])
    csv_path=root/"pairs.csv"
    rows=list(csv.DictReader(csv_path.open(encoding="utf-8")))
    if not rows or len(rows)>int(cfg["max_eval_pairs"]):
        raise ValueError(f"pairs.csv must contain 1..{cfg['max_eval_pairs']} rows")
    required=["id","image0","image1","tier","h00","h01","h02","h10","h11","h12","h20","h21","h22"]
    if any(name not in rows[0] for name in required):
        raise ValueError(f"pairs.csv must contain {required}")
    out=[]
    resolved_root=root.resolve()
    for row in rows:
        images=[]
        for field in ("image0","image1"):
            rel=PurePosixPath(row[field])
            if rel.is_absolute() or ".." in rel.parts:
                raise ValueError(f"Unsafe pair path: {row[field]}")
            path=(root/Path(*rel.parts)).resolve()
            if resolved_root not in path.parents and path!=resolved_root:
                raise ValueError("Pair path escapes BYOD root")
            if not path.is_file():
                raise FileNotFoundError(path)
            with Image.open(path) as image:
                image.load()
                images.append(image.convert("RGB"))
        H=np.asarray([[float(row[f"h{i}{j}"]) for j in range(3)] for i in range(3)],dtype=float)
        out.append({"id":row["id"],"image0":images[0],"image1":images[1],"homography":H.tolist(),"tier":row["tier"] or "byod"})
    return {"byod":out}

if cfg["use_byod"]:
    suites=load_byod()
else:
    test_photos=photo_test_split()
    seed=xo_samples.SAMPLE_SEED+2
    suites={
        "easy":xo_samples.make_pairs(test_photos,seed=seed,tier="easy"),
        "standard_hard":xo_samples.make_pairs(test_photos,seed=seed,tier="hard"),
        "extreme_rotation":lg_samples.make_pairs(test_photos,seed=seed,tier="hard"),
    }

lg=LightGluePipeline.from_pretrained(device=device,weights_dir=cfg["lightglue_weights"],allow_download=True)
xo=XoFTRPipeline.from_pretrained(device=device,weights_dir=cfg["xoftr_weights"],allow_download=True)

def compact(report):
    keys=[
        "n","precision_1px","precision_3px","precision_5px","matches_per_pair","inliers_per_pair",
        "median_error_px","homography_acc_3px","homography_acc_5px","seconds","verdict"
    ]
    return {key:report[key] for key in keys if key in report}

results={"lightglue":{},"xoftr":{},"identity":{},"patch_neighbour":{}}
per_pair={"lightglue":{},"xoftr":{}}
for tier,pairs in suites.items():
    lg_report=lg.evaluate(pairs)
    xo_report=xo.evaluate(pairs)
    identity=xo.evaluate(pairs,matcher=identity_baseline)
    patch=xo.evaluate(pairs,matcher=patch_neighbour_baseline)
    results["lightglue"][tier]=compact(lg_report)
    results["xoftr"][tier]=compact(xo_report)
    results["identity"][tier]=compact(identity)
    results["patch_neighbour"][tier]=compact(patch)
    per_pair["lightglue"][tier]=lg_report["per_pair"]
    per_pair["xoftr"][tier]=xo_report["per_pair"]

out={
    "runtime":{"python":platform.python_version(),"torch":torch.__version__,"numpy":np.__version__,"device":device},
    "sources":cfg["source_provenance"],
    "models":{
        "lightglue":{"model_id":"cvg/LightGlue","revision":"v0.1_arxiv","extractor":"Shiaoming/ALIKED@683d7c65197395c0b3f01ebe76e1084a27e73a65"},
        "xoftr":{"model_id":"vismatch/xoftr","revision":"d8ee7d89be3c9e5c157db3886db1c0f0e038b321"},
    },
    "suites":{name:len(pairs) for name,pairs in suites.items()},
    "results":results,
    "per_pair":per_pair,
}
out_path.write_text(json.dumps(out,indent=2))
"""
runner_path=RUNTIME_ROOT/"image_matching_runner.py"
runner_path.write_text(RUNNER_SOURCE,encoding="utf-8")
runner_cfg={
    "embedded_root":str(EMBEDDED_ROOT),
    "runtime_root":str(RUNTIME_ROOT),
    "lightglue_weights":str(LG_WEIGHTS),
    "xoftr_weights":str(XO_WEIGHTS),
    "use_byod":USE_BYOD,
    "byod_root":str(BYOD_ROOT) if BYOD_ROOT else None,
    "max_eval_pairs":MAX_EVAL_PAIRS,
    "source_provenance":SOURCE_PROVENANCE,
}
config_path=RUNTIME_ROOT/"image_matching_runner_config.json"
config_path.write_text(json.dumps(runner_cfg,indent=2))
result_path=RUNTIME_ROOT/"image_matching_results.json"
started=time.perf_counter()
run=subprocess.run([str(PYTHON),str(runner_path),str(config_path),str(result_path)],text=True,capture_output=True)
WALL_SECONDS=time.perf_counter()-started
if run.returncode!=0:
    print(run.stdout)
    print(run.stderr)
    raise RuntimeError(f"Image-matching runner failed with exit code {run.returncode}")
RUN_RESULTS=json.loads(result_path.read_text())
print({"runtime":RUN_RESULTS["runtime"],"suites":RUN_RESULTS["suites"],"wall_seconds":round(WALL_SECONDS,2)})


## 4. Common metric table

Do not read this table as a universal ranking. The tiers are controlled perturbations of a single photograph corpus, and the ±150° condition was deliberately introduced because rotation is a known difficult regime for the LightGlue+ALIKED configuration.

The useful questions are:

- Does precision collapse before the number of matches collapses?
- Does a matcher return many correspondences but fail to recover the global homography?
- Does a model remain geometrically reliable as rotation increases?
- How far above the non-neural references is each learned matcher?


In [ ]:
# @title 4.1 Assemble the comparison table
rows=[]
for condition,tier_reports in RUN_RESULTS["results"].items():
    for tier,report in tier_reports.items():
        rows.append({"condition":condition,"tier":tier,**report})
metrics=pd.DataFrame(rows)
tier_order={"easy":0,"standard_hard":1,"extreme_rotation":2,"byod":0}
metrics["_tier_order"]=metrics["tier"].map(tier_order).fillna(99)
metrics=metrics.sort_values(["_tier_order","condition"]).drop(columns="_tier_order")
display(metrics)

learned=metrics[metrics["condition"].isin(["lightglue","xoftr"])].copy()


## 5. Visualize degradation across tiers

The line chart shows the selected geometric metric across increasing transformation severity. It is meant to expose **failure shape**, not just a single average.

For `precision_3px`, for example, 1.0 means every returned match is within 3 pixels of the location implied by the known homography; it does not mean the model found every possible correspondence.


In [ ]:
# @title 5.1 Metric-versus-tier plot
if not USE_BYOD:
    order=["easy","standard_hard","extreme_rotation"]
    fig=plt.figure(figsize=(9,5))
    for condition in ("lightglue","xoftr","identity","patch_neighbour"):
        block=metrics[metrics["condition"]==condition].set_index("tier")
        plt.plot(order,[block.loc[tier,PLOT_METRIC] for tier in order],marker="o",label=condition)
    plt.xlabel("transformation tier")
    plt.ylabel(PLOT_METRIC)
    plt.title(f"Image-matching degradation: {PLOT_METRIC}")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plot_path=OUTPUT_ROOT/f"image_matching_{PLOT_METRIC}_by_tier.png"
    fig.savefig(plot_path,dpi=140)
    plt.show()
else:
    print("Tier-degradation plot is skipped for arbitrary BYOD tier names.")


## 6. Pair-level failure analysis

Aggregate values can hide catastrophic pairs. The next table combines pair-level records from both learned matchers so you can look for images where:

- one system finds substantially more inliers;
- both systems return matches but neither recovers the reference homography; or
- extreme rotation causes a sharp precision drop.

The workshop intentionally does **not** tune thresholds after inspecting these held-out pairs.


In [ ]:
# @title 6.1 Inspect pair-level metrics
pair_rows=[]
for condition,tier_map in RUN_RESULTS["per_pair"].items():
    for tier,items in tier_map.items():
        for item in items:
            pair_rows.append({"condition":condition,"tier":tier,**item})
pair_metrics=pd.DataFrame(pair_rows)
display(
    pair_metrics.sort_values(["tier","precision_3px"],ascending=[True,True])
    .head(30)
)


## 7. Export evidence and provenance

The exported records preserve:

- common aggregate metrics;
- learned-matcher pair-level metrics;
- exact embedded source commits;
- exact model/checkpoint identities;
- runtime versions;
- sample-suite sizes; and
- the notebook's total wall time.

No adapted weight artifact exists because this is a genuine `TASK-INFERENCE` workshop.


In [ ]:
# @title 7.1 Machine-readable exports
metrics.to_csv(OUTPUT_ROOT/"multimodel_image_matching_metrics.csv",index=False)
pair_metrics.to_csv(OUTPUT_ROOT/"multimodel_image_matching_per_pair.csv",index=False)
provenance={
    "notebook":{
        "specification":"2.1",
        "profile":"TASK-INFERENCE",
        "mode":"WORKSHOP",
        "standalone":True,
        "host_repository":"kurtvalcorza/lightglue-matching-pipeline",
        "generator":"tools/build_multimodel_image_matching_workshop.py",
        "clean_runtime_evidence":"pending",
    },
    "source_provenance":SOURCE_PROVENANCE,
    "models":RUN_RESULTS["models"],
    "runtime":{**RUN_RESULTS["runtime"],"gpu":GPU_INFO,"wall_seconds":WALL_SECONDS},
    "suites":RUN_RESULTS["suites"],
    "byod":bool(USE_BYOD),
}
(OUTPUT_ROOT/"multimodel_image_matching_provenance.json").write_text(json.dumps(provenance,indent=2),encoding="utf-8")
print(sorted(path.name for path in OUTPUT_ROOT.iterdir()))


## 8. Interpretation and limits

This workshop establishes a **controlled geometric comparison** of the two pinned live DIMER matchers. It does not establish that one matcher is generally better.

Limitations include:

- all default pairs come from one 360-photo CC0 bird corpus;
- the benchmark transformations are synthetic homographies and photometric perturbations, not real camera motion or full 3D scene change;
- the easy / standard / extreme tiers differ mostly in perturbation severity and rotation range;
- LightGlue+ALIKED is sparse and XoFTR is detector-free/semi-dense, so raw match counts are not directly a quality score;
- the models use different learned representations and decision thresholds;
- runtime is specific to this session's GPU; and
- a deployment choice should be tested on the target domain, camera geometry, expected viewpoint changes, and latency budget.

### Suggested exercises

1. Find pairs where match count increases but homography accuracy decreases. Why can that happen?
2. Compare `precision_1px` and `precision_5px`: what does the gap say about localization quality?
3. Build a BYOD benchmark from controlled planar scenes with measured homographies.
4. Add real viewpoint/scale cases rather than only synthetic warps.
5. In the model-specific notebooks, fine-tune each matcher separately and compare adapted versus frozen behavior without conflating the two adaptation recipes.

### References

- DIMER Notebook Specification 2.1: `ml-worker/integrations/dimer/fleet-specs/NOTEBOOK_SPEC.md`
- DIMER model fleet: `ml-worker/integrations/dimer/fleet-inventory/MODEL_MATRIX.md`
- LightGlue: Lindenberger, Sarlin & Pollefeys, ICCV 2023
- ALIKED: Zhao et al.
- XoFTR: detector-free image matching


In [ ]:
# @title Run-all completion summary
summary={
    "notebook_spec":"2.1",
    "profile":"TASK-INFERENCE",
    "mode":"WORKSHOP",
    "models":["LightGlue + ALIKED","XoFTR"],
    "suites":RUN_RESULTS["suites"],
    "outputs":sorted(path.name for path in OUTPUT_ROOT.iterdir()),
    "clean_runtime_evidence":"pending until exact hosted-runtime execution is recorded",
}
display(pd.Series(summary,name="value").to_frame())
print("Run-all complete: source materialization, pinned model acquisition, common homography benchmark, geometric evaluation, failure analysis, and export succeeded.")
